# AI-Powered GIS Site Analysis for Park Design

**A modular, explainable, configuration-driven notebook for automated site analysis in landscape architecture.**

This notebook:
1. Accepts a site from you (place name, Google Maps URL, lat/lon, or an uploaded boundary file)
2. Retrieves GIS and environmental data from multiple sources (OpenStreetMap, Google Maps, ESRI/ArcGIS, Google Earth Engine, elevation and climate services), with graceful fallback when a source or API key is unavailable
3. Scores the site against a **configurable criteria tree** parsed from an analysis document, with every score fully explained (data used, formula, weight, assumptions, confidence)
4. Produces a dashboard (radar charts, KPI cards, bar charts, heatmaps, interactive maps) and exports PDF / Excel / CSV / HTML reports

### Notebook architecture

| # | Module | Purpose |
|---|--------|---------|
| 01 | User Input | Site selection, boundary resolution |
| 02 | Data Collection | Pull GIS/environmental data from all sources |
| 03 | GIS Preprocessing | Reproject, clip, align datasets to the site |
| 04 | Data Cleaning | Validate, fill gaps, flag missing data |
| 05 | Spatial Analysis | Compute raw metrics (areas, distances, coverage %, etc.) |
| 06 | Scoring Engine | Map metrics → criteria scores via the config |
| 07 | Dashboard | Radar/bar/heatmap/KPI/map visualizations |
| 08 | AI Recommendations | Strengths/weaknesses/opportunities/risks |
| 09 | Report Generation | PDF, Excel, CSV, HTML outputs |
| 10 | Export | Bundle and download all artifacts |

### Design principles
- **Configuration-driven**: nothing about a specific site or a specific park's requirements is hardcoded. The criteria tree, weights, formulas, and data source priorities all live in editable config cells/files.
- **Explainable**: every score is a structured object carrying its inputs, formula, weight, and confidence — never a bare number.
- **Graceful degradation**: every external API call is wrapped so a missing key, quota error, or network failure logs a warning and falls back to the next available source (or a "no data" state that lowers confidence) rather than crashing the notebook.
- **Extensible**: adding a new data source, criterion, or project type (urban planning, campus, residential, infrastructure) should mean adding a config entry or a small connector function, not restructuring the notebook.

---
**Before running:** Runtime → Change runtime type → make sure you have internet access enabled (default in Colab). API keys are configured in the cell below — the notebook works without any of them, using free/open fallbacks, but coverage and quality improve substantially with keys.


## Environment Setup

In [ ]:
# @title Install dependencies
# This installs the full breadth of libraries this notebook can use. Free/open-source
# packages install directly; ArcGIS Python API and Earth Engine are included but only
# activate if you provide credentials in the next cell — otherwise their code paths
# fall back automatically.

import sys
import subprocess

PACKAGES = [
    "geopandas",
    "rasterio",
    "shapely",
    "folium",
    "plotly",
    "pandas",
    "osmnx",
    "contextily",
    "requests",
    "matplotlib",
    "reportlab",         # PDF generation
    "openpyxl",          # Excel export
    "xlsxwriter",         # Excel export (richer formatting)
    "earthengine-api",    # Google Earth Engine (optional, needs auth)
    "geemap",              # Earth Engine visualization helper (optional)
    "arcgis",               # ArcGIS Python API (optional, needs auth/licensing)
    "elevation",              # SRTM elevation tile fetch (free, no key)
    "rasterstats",
    "pyproj",
    "geopy",
    "google-genai",       # Gemini API - used for the optional AI-assisted boundary vision analysis in Module 01
    "pillow",               # image handling, needed alongside google-genai for image inputs
]
# Deliberately NOT upgrading numpy here. Colab ships with a numpy build that every other
# pre-installed compiled package (rasterio, scipy, etc.) was built against; force-upgrading
# it via --upgrade can leave some already-loaded compiled extension out of sync with the new
# numpy ABI, producing obscure errors like "module 'numpy._core...' has no attribute
# '_blas_supports_fpe'" the moment an affected package is imported later. If a specific
# package genuinely needs a newer numpy, let pip's own dependency resolver pull it in as a
# side effect of installing that package, rather than upgrading it unconditionally up front.

def pip_install(packages):
    base_cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *packages]
    result = subprocess.run(base_cmd, check=False, capture_output=True, text=True)
    if result.returncode != 0 and "externally-managed-environment" in (result.stderr or ""):
        # Some non-Colab environments (e.g. Debian-managed Python) require this flag.
        # Colab itself does not need it and will simply succeed on the first attempt.
        subprocess.run(base_cmd + ["--break-system-packages"], check=False)
    elif result.returncode != 0:
        err_tail = result.stderr[-1500:] if result.stderr else "unknown error"
        print(f"Warning: some packages may have failed to install:\n{err_tail}")

print("Installing packages (this can take a few minutes on a fresh Colab runtime)...")
pip_install(PACKAGES)
print("Done. If any package failed, its dependent features will auto-disable later with a warning rather than crashing the notebook.")

# Sanity check: catch a numpy/compiled-package ABI mismatch immediately, with a clear fix,
# rather than letting it surface later as a cryptic AttributeError deep in some other cell.
try:
    import numpy as _np_check
    _np_check.array([1, 2, 3]).sum()
    from numpy.testing import assert_allclose as _assert_check  # touches the same code path that broke previously
except Exception as _np_err:
    print(f"\n⚠️  numpy sanity check failed ({_np_err.__class__.__name__}: {_np_err}).")
    print("This usually means numpy and a compiled package (rasterio/scipy/etc.) are out of sync in this "
          "session. Fix: Runtime -> Restart session, then Run All again from a clean kernel. If it persists, "
          "run `!pip install --force-reinstall numpy` in a fresh cell before Run All.")


Installing packages (this can take a few minutes on a fresh Colab runtime)...
Done. If any package failed, its dependent features will auto-disable later with a warning rather than crashing the notebook.


In [ ]:
# @title Core imports and global logging/warning setup
import os
import json
import math
import warnings
import logging
import datetime as dt
from dataclasses import dataclass, field, asdict
from typing import Optional, Any, Union

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore")

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger("park_gis")

# ---------------------------------------------------------------------------
# Hard timeout guard (defined first - _try_import below depends on it)
# ---------------------------------------------------------------------------
import signal as _signal
import contextlib as _contextlib


class HardTimeoutError(Exception):
    pass


@_contextlib.contextmanager
def hard_timeout(seconds):
    '''Guards any call against hanging indefinitely on a slow/blocked network - some
    libraries (osmnx, the elevation package, and even `import arcgis` itself) issue
    their own HTTP/subprocess requests or perform network probing with retry logic
    that can ignore or outlast a per-request timeout argument. This uses a POSIX
    SIGALRM so it works regardless of what the wrapped code does internally.
    Not available on Windows; falls back to no-op there since Colab runs on Linux.'''
    if not hasattr(_signal, "SIGALRM"):
        yield
        return

    def _handler(signum, frame):
        raise HardTimeoutError(f"Operation exceeded {seconds}s hard timeout")

    old_handler = _signal.signal(_signal.SIGALRM, _handler)
    _signal.alarm(seconds)
    try:
        yield
    finally:
        _signal.alarm(0)
        _signal.signal(_signal.SIGALRM, old_handler)


# ---------------------------------------------------------------------------
# Fallback-aware import helper
# ---------------------------------------------------------------------------
# Many optional libraries (Earth Engine, ArcGIS API) require auth or may not be
# installed successfully in every environment - some (arcgis in particular) can
# even hang on the import statement itself due to network probing at import time.
# We import them defensively, under a hard timeout, so a missing/broken/slow
# package degrades a *feature*, not the whole notebook.

_AVAILABLE = {}
IMPORT_TIMEOUT_S = 30           # default: plenty for pure-local imports
SLOW_IMPORT_TIMEOUT_S = 120     # for packages with large dependency trees that can be slow
                                  # to import on a cold runtime (osmnx pulls in networkx, etc.)
                                  # without making any network calls themselves - slow is fine,
                                  # this just guards against an actual hang.

def _try_import(name, pip_name=None, timeout_s=None):
    timeout_s = timeout_s or IMPORT_TIMEOUT_S
    try:
        with hard_timeout(timeout_s):
            module = __import__(name)
        _AVAILABLE[name] = True
        return module
    except HardTimeoutError:
        logger.warning(f"Optional dependency '{pip_name or name}' import exceeded {timeout_s}s "
                        f"- skipping. Features relying on it will be skipped or use a fallback source. "
                        f"If this is a large/slow-loading package rather than a genuine hang, try "
                        f"re-running this cell (module caching usually makes the second import instant).")
        _AVAILABLE[name] = False
        return None
    except Exception as e:
        logger.warning(f"Optional dependency '{pip_name or name}' unavailable ({e.__class__.__name__}). "
                        f"Features relying on it will be skipped or use a fallback source.")
        _AVAILABLE[name] = False
        return None


gpd = _try_import("geopandas")
rasterio = _try_import("rasterio")
shapely = _try_import("shapely")
folium = _try_import("folium")
plotly = _try_import("plotly")
osmnx = _try_import("osmnx", timeout_s=SLOW_IMPORT_TIMEOUT_S)
contextily = _try_import("contextily")
ee = _try_import("ee", pip_name="earthengine-api")
geemap = _try_import("geemap", timeout_s=SLOW_IMPORT_TIMEOUT_S)
arcgis = _try_import("arcgis")  # kept at the tight default - this is the one known to hang on network probing
genai = _try_import("google.genai", pip_name="google-genai")
PIL_Image = None
if _try_import("PIL", pip_name="pillow"):
    try:
        from PIL import Image as PIL_Image
    except Exception:
        PIL_Image = None

print("Dependency availability:")
for k, v in _AVAILABLE.items():
    print(f"  {'✓' if v else '✗'} {k}")

if not osmnx:
    print("\n⚠️  osmnx failed to import within the timeout. This is usually just a slow cold import on a "
          "fresh Colab runtime, not a real failure. Try Runtime → Restart session, then Run All again — "
          "pip's package cache will make the import much faster the second time.")

# ---------------------------------------------------------------------------
# Shared OSM/Overpass helper - defined here (not in Module 02) so any module,
# including Module 01's boundary resolver, can use it regardless of cell order.
# ---------------------------------------------------------------------------
OSM_FETCH_TIMEOUT_S = 90  # Overpass queries can legitimately take 30-60s+ for larger site
                            # radii or densely-tagged urban areas - this is a generous cap,
                            # not an expected typical duration.
if osmnx:
    try:
        osmnx.settings.timeout = OSM_FETCH_TIMEOUT_S
        osmnx.settings.log_console = False
    except Exception:
        pass


def _osmnx_features(polygon, tags):
    '''Wraps osmnx.features_from_polygon so a genuine "zero matching features" result
    (which osmnx raises as an exception rather than returning an empty GeoDataFrame)
    is distinguished from a real error. Returns an empty GeoDataFrame on no-match,
    re-raises everything else (including HardTimeoutError) for the caller to handle.'''
    try:
        with hard_timeout(OSM_FETCH_TIMEOUT_S):
            return osmnx.features_from_polygon(polygon, tags=tags)
    except HardTimeoutError:
        raise
    except Exception as e:
        if "No matching features" in str(e) or "InsufficientResponseError" in e.__class__.__name__:
            return gpd.GeoDataFrame() if gpd else pd.DataFrame()
        raise


Dependency availability:
  ✓ geopandas
  ✓ rasterio
  ✓ shapely
  ✓ folium
  ✓ plotly
  ✓ osmnx
  ✓ contextily
  ✓ ee
  ✓ geemap
  ✓ arcgis
  ✓ google.genai
  ✓ PIL


### API Keys & Credentials (optional but recommended)

Fill in whichever of these you have. **All are optional** — leave any as `None` / empty string and the notebook will fall back to open/free sources for that capability and lower the confidence score on any criterion that would have benefited from it.

| Key | Used for | Get it from |
|---|---|---|
| `GOOGLE_MAPS_API_KEY` | Basemaps, geocoding, Places (POIs), Static Maps imagery | https://console.cloud.google.com/google/maps-apis |
| `ARCGIS_USERNAME` / `ARCGIS_PASSWORD` (or `ARCGIS_API_KEY`) | ESRI basemaps, ESRI Living Atlas layers (land cover, demographics) | https://developers.arcgis.com |
| `GEE_PROJECT_ID` + Earth Engine auth (interactive, triggered on first use) | Sentinel-2/Landsat imagery, canopy, land cover, climate rasters | https://earthengine.google.com |
| `GEMINI_API_KEY` | Optional AI-assisted boundary vision analysis in Module 01 (free tier: Gemini 2.5 Flash) | https://aistudio.google.com/app/apikey |

**⚠️ Never paste an API key directly into a chat conversation, this notebook's code cells, or anywhere else it could be logged or shared.** Always enter keys through the Colab Secrets panel only. If a key is ever exposed outside of Secrets, treat it as compromised and revoke/regenerate it immediately from the provider's console.

**⚠️ `GEE_PROJECT_ID` is a *Cloud project ID* (e.g. `my-park-analysis-2026`), not an API key.** It comes from a Google Cloud project that has the Earth Engine API enabled and is registered at https://code.earthengine.google.com/register — it will never look like `AIzaSy...` (that pattern is an API key, and will fail Earth Engine init with a "Project not found" error). If you don't have a GEE project set up yet, leave this blank; Earth Engine layers will simply report `unavailable` and every other module still works.

**Recommendation:** In Colab, use the 🔑 *Secrets* panel (left sidebar) to store these instead of pasting them in plaintext, then load with `from google.colab import userdata; userdata.get("GOOGLE_MAPS_API_KEY")`. The cell below tries Secrets first and falls back to the plain variables if you're not on Colab or haven't set Secrets.


In [ ]:
# @title Configure API keys (edit values or use Colab Secrets)

def _get_secret_or_var(secret_name, fallback_value=None):
    '''Reads a Colab Secret (or falls back to a plain variable) and sanitizes the result.
    Secrets pasted from other sources can carry invisible embedded whitespace/newlines
    (e.g. 'ask-lucy\\r\\nask-lucy' from a double-paste or copy including a line break) -
    these silently corrupt any HTTP header or API call built from the raw value, often
    producing a confusing low-level error far from where the bad value was read. We
    take the first non-empty line and strip surrounding whitespace as a defensive default.'''
    val = None
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
    except Exception:
        pass
    if not val:
        val = fallback_value
    if val and isinstance(val, str):
        first_line = val.strip().splitlines()[0].strip() if val.strip() else ""
        if first_line != val.strip():
            logger_msg = (f"Secret '{secret_name}' contained extra whitespace/newlines "
                           f"(e.g. from a double-paste) - using the sanitized value "
                           f"'{first_line}' instead of the raw stored value. Consider "
                           f"re-entering this secret in the Colab Secrets panel to clean it up.")
            print(f"⚠️  {logger_msg}")
        val = first_line or None
    return val

# --- Edit these fallback values directly if you're not using Colab Secrets ---
GOOGLE_MAPS_API_KEY = _get_secret_or_var("GOOGLE_MAPS_API_KEY", fallback_value=None)
ARCGIS_USERNAME     = _get_secret_or_var("ARCGIS_USERNAME", fallback_value=None)
ARCGIS_PASSWORD     = _get_secret_or_var("ARCGIS_PASSWORD", fallback_value=None)
ARCGIS_API_KEY      = _get_secret_or_var("ARCGIS_API_KEY", fallback_value=None)
GEE_PROJECT_ID       = _get_secret_or_var("GEE_PROJECT_ID", fallback_value=None)
GEMINI_API_KEY        = _get_secret_or_var("GEMINI_API_KEY", fallback_value=None)

API_KEYS = {
    "google_maps": GOOGLE_MAPS_API_KEY,
    "arcgis_username": ARCGIS_USERNAME,
    "arcgis_password": ARCGIS_PASSWORD,
    "arcgis_api_key": ARCGIS_API_KEY,
    "gee_project_id": GEE_PROJECT_ID,
    "gemini_api_key": GEMINI_API_KEY,
}

def key_status(keys: dict) -> pd.DataFrame:
    rows = [{"credential": k, "configured": bool(v)} for k, v in keys.items()]
    return pd.DataFrame(rows)

print("Credential status (True = configured, False = will use free/open fallback):")
key_status(API_KEYS)


Credential status (True = configured, False = will use free/open fallback):


,credential,configured
0,google_maps,True
1,arcgis_username,True
2,arcgis_password,True
3,arcgis_api_key,True
4,gee_project_id,True
5,gemini_api_key,True


---
# Module 01 — User Input

Resolves whatever the user provides (place name, Google Maps URL, coordinates, or an uploaded boundary file)
into a single normalized `Site` object used by every downstream module:

- `site.centroid` — (lat, lon)
- `site.boundary` — a Shapely polygon (drawn/uploaded boundary, or an auto-generated bounding box if only a point was given)
- `site.bbox` — (minx, miny, maxx, maxy) in WGS84
- `site.crs` — coordinate reference system
- `site.source_method` — how the site was resolved, kept for the explainability trail


In [ ]:
# @title Site data structure

@dataclass
class Site:
    name: str
    centroid: tuple            # (lat, lon)
    boundary: Any               # shapely Polygon in WGS84
    bbox: tuple                  # (minx, miny, maxx, maxy) WGS84
    crs: str = "EPSG:4326"
    source_method: str = "unknown"
    raw_input: Optional[str] = None
    notes: list = field(default_factory=list)

    def summary(self) -> pd.DataFrame:
        return pd.DataFrame([{
            "name": self.name,
            "centroid_lat": round(self.centroid[0], 6),
            "centroid_lon": round(self.centroid[1], 6),
            "bbox": tuple(round(v, 6) for v in self.bbox),
            "crs": self.crs,
            "source_method": self.source_method,
        }])

    def area_m2(self) -> Optional[float]:
        '''Reprojects to a local UTM zone to get a metric area.'''
        if not gpd or self.boundary is None:
            return None
        try:
            gdf = gpd.GeoDataFrame({"geometry": [self.boundary]}, crs=self.crs)
            utm_crs = gdf.estimate_utm_crs()
            return float(gdf.to_crs(utm_crs).area.iloc[0])
        except Exception as e:
            logger.warning(f"Could not compute metric area: {e}")
            return None


In [ ]:
# @title Input resolvers (place name / Google Maps URL / lat-lon / uploaded boundary)
import re

DEFAULT_SITE_RADIUS_M = 500  # used to build a bounding box around a point when no boundary is given

def _bbox_from_point(lat, lon, radius_m=DEFAULT_SITE_RADIUS_M):
    '''Builds a square-ish bbox (in WGS84 degrees) around a point using a metric buffer.'''
    if gpd and shapely:
        from shapely.geometry import Point
        gdf = gpd.GeoDataFrame({"geometry": [Point(lon, lat)]}, crs="EPSG:4326")
        utm_crs = gdf.estimate_utm_crs()
        buffered = gdf.to_crs(utm_crs).buffer(radius_m).to_crs("EPSG:4326")
        poly = buffered.iloc[0]
        return poly, poly.bounds
    else:
        # crude fallback without geopandas: ~ meters to degrees at given latitude
        dlat = radius_m / 111_320
        dlon = radius_m / (111_320 * math.cos(math.radians(lat)) or 1e-6)
        bbox = (lon - dlon, lat - dlat, lon + dlon, lat + dlat)
        poly = None
        if shapely:
            from shapely.geometry import box
            poly = box(*bbox)
        return poly, bbox


def resolve_from_latlon(lat: float, lon: float, name: str = "Site", radius_m: int = DEFAULT_SITE_RADIUS_M) -> Site:
    poly, bbox = _bbox_from_point(lat, lon, radius_m)
    return Site(
        name=name,
        centroid=(lat, lon),
        boundary=poly,
        bbox=bbox,
        source_method="lat_lon",
        raw_input=f"{lat},{lon}",
        notes=[f"Boundary auto-generated as a {radius_m}m buffer around the given point (no boundary was supplied)."],
    )


def resolve_from_place_name(place_name: str, radius_m: int = DEFAULT_SITE_RADIUS_M) -> Site:
    '''Geocodes a place name. Tries Google Geocoding API first (if key configured), falls back to Nominatim (OSM, free).'''
    lat = lon = None
    method_used = None

    if API_KEYS.get("google_maps"):
        try:
            import requests
            resp = requests.get(
                "https://maps.googleapis.com/maps/api/geocode/json",
                params={"address": place_name, "key": API_KEYS["google_maps"]},
                timeout=10,
            )
            data = resp.json()
            if data.get("status") == "OK":
                loc = data["results"][0]["geometry"]["location"]
                lat, lon = loc["lat"], loc["lng"]
                method_used = "google_geocoding"
            else:
                logger.warning(f"Google Geocoding returned status={data.get('status')}; falling back to Nominatim.")
        except Exception as e:
            logger.warning(f"Google Geocoding failed ({e}); falling back to Nominatim.")

    if lat is None:
        try:
            from geopy.geocoders import Nominatim
            geolocator = Nominatim(user_agent="park_gis_site_analysis")
            location = geolocator.geocode(place_name, timeout=10)
            if location:
                lat, lon = location.latitude, location.longitude
                method_used = "nominatim_osm"
        except Exception as e:
            logger.error(f"Nominatim geocoding also failed: {e}")

    if lat is None:
        raise ValueError(
            f"Could not geocode '{place_name}' via Google or Nominatim. "
            "Try a more specific place name, or provide lat/lon directly."
        )

    site = resolve_from_latlon(lat, lon, name=place_name, radius_m=radius_m)
    site.source_method = method_used
    site.raw_input = place_name
    return site


def resolve_from_google_maps_url(url: str, radius_m: int = DEFAULT_SITE_RADIUS_M) -> Site:
    '''Extracts coordinates from common Google Maps URL formats:
    - .../@lat,lon,zoom
    - ?q=lat,lon  or ?q=place+name
    - /place/Name/@lat,lon,...
    '''
    lat = lon = None

    m = re.search(r"@(-?\d+\.\d+),(-?\d+\.\d+)", url)
    if m:
        lat, lon = float(m.group(1)), float(m.group(2))

    if lat is None:
        m = re.search(r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)", url)
        if m:
            lat, lon = float(m.group(1)), float(m.group(2))

    if lat is None:
        # Try to pull a place name out of a /place/<name>/ segment and geocode it instead
        m = re.search(r"/place/([^/@]+)", url)
        if m:
            place_name = m.group(1).replace("+", " ")
            logger.info(f"No coordinates in URL; geocoding extracted place name: '{place_name}'")
            site = resolve_from_place_name(place_name, radius_m=radius_m)
            site.source_method = "google_maps_url_place_name"
            site.raw_input = url
            return site

    if lat is None:
        raise ValueError(
            "Could not extract coordinates or a place name from this Google Maps URL. "
            "Try copying the URL from the address bar after opening the location on maps.google.com, "
            "or use the lat/lon or place-name input methods instead."
        )

    site = resolve_from_latlon(lat, lon, name="Site (from Google Maps URL)", radius_m=radius_m)
    site.source_method = "google_maps_url_coords"
    site.raw_input = url
    return site


def resolve_from_uploaded_boundary(file_path: str, name: str = "Uploaded Site") -> Site:
    '''Reads a GeoJSON, KML, or Shapefile (.zip or .shp) boundary upload.'''
    if not gpd:
        raise RuntimeError("geopandas is required to read uploaded boundary files but is not available.")

    ext = os.path.splitext(file_path)[1].lower()
    try:
        if ext == ".kml":
            gpd.io.file.fiona.drvsupport.supported_drivers["KML"] = "rw"
            gdf = gpd.read_file(file_path, driver="KML")
        else:
            gdf = gpd.read_file(file_path)  # handles .geojson, .json, .shp, .zip (shapefile bundle)
    except Exception as e:
        raise ValueError(f"Could not read boundary file '{file_path}': {e}")

    if gdf.crs is None:
        logger.warning("Uploaded boundary has no CRS defined; assuming EPSG:4326 (WGS84).")
        gdf = gdf.set_crs("EPSG:4326")
    elif gdf.crs.to_string() != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")

    # Dissolve multiple features into a single boundary polygon if needed
    geometry = gdf.geometry.unary_union
    centroid = geometry.centroid
    bbox = geometry.bounds

    return Site(
        name=name,
        centroid=(centroid.y, centroid.x),
        boundary=geometry,
        bbox=bbox,
        source_method="uploaded_boundary",
        raw_input=file_path,
        notes=[f"Boundary read from uploaded file with {len(gdf)} feature(s), dissolved into one polygon."],
    )


### Choose your input method

Set `INPUT_METHOD` below to one of `"place_name"`, `"google_maps_url"`, `"lat_lon"`, or `"upload"`, fill in the matching value, then run the cell.


In [ ]:
# @title Site input — edit and run
INPUT_METHOD = "lat_lon"  # @param ["place_name", "google_maps_url", "lat_lon", "upload"]

# --- Fill in the value(s) relevant to your chosen method ---
PLACE_NAME = "Al Safa Park 2, Dubai, UAE"
GOOGLE_MAPS_URL = "https://www.google.com/maps/place/Al+Safa+Park+2/@25.1558375,55.2191895,934m"
LATITUDE = 25.156090048103597
LONGITUDE = 55.22182877301442
UPLOADED_BOUNDARY_PATH = "/content/site_boundary.geojson"  # used only if INPUT_METHOD == "upload"

SITE_NAME = "Al Safa Park 2 Site"          # display name, used in reports/dashboards
SITE_RADIUS_M = 500                    # 500m = ~5 min walk, standard walkable-catchment radius for park accessibility analysis


def resolve_site() -> Site:
    if INPUT_METHOD == "place_name":
        return resolve_from_place_name(PLACE_NAME, radius_m=SITE_RADIUS_M)
    elif INPUT_METHOD == "google_maps_url":
        return resolve_from_google_maps_url(GOOGLE_MAPS_URL, radius_m=SITE_RADIUS_M)
    elif INPUT_METHOD == "lat_lon":
        return resolve_from_latlon(LATITUDE, LONGITUDE, name=SITE_NAME, radius_m=SITE_RADIUS_M)
    elif INPUT_METHOD == "upload":
        if not os.path.exists(UPLOADED_BOUNDARY_PATH):
            raise FileNotFoundError(
                f"'{UPLOADED_BOUNDARY_PATH}' not found. In Colab, use the Files panel (left sidebar) "
                "to upload your GeoJSON/KML/Shapefile, or run:\n"
                "  from google.colab import files\n  files.upload()\n"
                "then update UPLOADED_BOUNDARY_PATH to match the uploaded filename."
            )
        return resolve_from_uploaded_boundary(UPLOADED_BOUNDARY_PATH, name=SITE_NAME)
    else:
        raise ValueError(f"Unknown INPUT_METHOD: {INPUT_METHOD}")


SITE = resolve_site()
if SITE_NAME and INPUT_METHOD != "upload":
    SITE.name = SITE_NAME

print(f"Site resolved via: {SITE.source_method}")
area = SITE.area_m2()
if area:
    print(f"Approx. site area: {area:,.0f} m² ({area/10_000:,.2f} ha)")
SITE.summary()


Site resolved via: lat_lon
Approx. site area: 784,137 m² (78.41 ha)


,name,centroid_lat,centroid_lon,bbox,crs,source_method
0,Al Safa Park 2 Site,25.15609,55.221829,"(55.21687, 25.151577, 55.226788, 25.160603)",EPSG:4326,lat_lon


In [ ]:
# @title Quick visual check — plot the resolved site boundary
if folium and SITE.boundary is not None:
    m = folium.Map(location=SITE.centroid, zoom_start=16, tiles="CartoDB positron")
    try:
        folium.GeoJson(
            data=gpd.GeoSeries([SITE.boundary]).__geo_interface__ if gpd else None,
            style_function=lambda x: {"color": "#2E7D32", "weight": 3, "fillOpacity": 0.15},
        ).add_to(m)
    except Exception as e:
        logger.warning(f"Could not draw boundary geometry, falling back to a marker: {e}")
        folium.Marker(SITE.centroid, popup=SITE.name).add_to(m)
    folium.Marker(SITE.centroid, tooltip="Site centroid", icon=folium.Icon(color="green")).add_to(m)
    display(m)
else:
    print(f"Site centroid: {SITE.centroid} — install folium/geopandas for an interactive preview map.")


---
## AI-Assisted Site Boundary Resolver (optional alternative path)

Everything above (place name / Google Maps URL / lat-lon / uploaded boundary) remains the primary,
default way to set `SITE`, unchanged. This section is an **optional alternative**: instead of manually
supplying a boundary or accepting an auto-generated circular buffer, it searches OpenStreetMap for real
candidate polygons around a resolved location, scores them deterministically, and lets you approve one
(or fall back to manual lat/lon/radius) before it overwrites `SITE`.

**Source-first, not AI-first.** Per the design brief this implements: existing GIS/OSM polygons are always
preferred over anything AI-generated, every candidate keeps full source/confidence metadata, and a
low-confidence result is never auto-accepted — you always approve or reject before it's used.

**On the AI vision step specifically**: the spec calls for an AI model to visually compare candidate
polygons against satellite imagery. There is no genuinely free-of-charge vision-analysis API that performs
this specific reasoning task (general computer-vision APIs like Google/Azure/AWS Vision do object/label
detection, not "does this polygon match the visible site edge" reasoning, and are free-tier-then-paid,
not free). Rather than fake this with something that doesn't really work, the AI step is implemented as a
**working, pluggable interface** (`ai_boundary_analysis()`) that returns a clearly-labeled "not configured"
result by default. If you later add a vision-capable API key, this one function is the only thing that
needs to be filled in — nothing else in the pipeline changes.

**On interactive editing**: true draw/edit/delete polygon support needs `ipyleaflet` with a `DrawControl`,
which can be finicky in Colab. For now this section supports **approve / reject a candidate**, or **manual
lat/lon/radius entry** as the fallback - simpler and more reliable than in-notebook polygon drawing.


In [ ]:
# @title Boundary resolver configuration
SITE_BOUNDARY_CONFIG = {
    "search_radius_m": 500,
    "max_candidates": 10,

    "enable_osm": True,
    "enable_esri_imagery": True,   # imagery retrieval only - no AI analysis of it by default, see above
    "enable_ai": True,             # kept False by default since no free vision API is wired in; see ai_boundary_analysis()

    "require_user_confirmation": True,

    "high_confidence_threshold": 0.85,
    "medium_confidence_threshold": 0.65,

    # Ranking weights - must sum to 1.0. Configurable, not hard-coded into the scoring function itself.
    "weights": {
        "source_reliability": 0.35,
        "name_match": 0.20,
        "geometry_quality": 0.15,
        "center_proximity": 0.20,
        "landuse_agreement": 0.10,
        # satellite_agreement and road_alignment from the original spec are omitted from the default
        # weights (rather than given a weight that always contributes 0) since they depend on the AI
        # vision step, which is off by default - see enable_ai above. If you wire in a real vision API,
        # add those two keys back in here and rebalance the others to keep the total at 1.0.
    },
}

# Source reliability weights - initial, illustrative ranking values, not measured probabilities.
SOURCE_RELIABILITY = {
    "government_cadastral": 1.00,
    "official_gis": 0.95,
    "osm_boundary": 0.80,
    "commercial_gis": 0.85,
    "satellite_segmentation": 0.70,
    "ai_interpretation": 0.55,
}

assert abs(sum(SITE_BOUNDARY_CONFIG["weights"].values()) - 1.0) < 1e-6, "SITE_BOUNDARY_CONFIG weights must sum to 1.0"
print("Boundary resolver configuration loaded.")


Boundary resolver configuration loaded.


In [ ]:
# @title Geometry utilities

def normalize_geometry(geom):
    '''Repairs minor invalidities (self-intersections etc.) via a zero-width buffer trick,
    a standard Shapely idiom - returns None if the geometry still can't be made valid.'''
    if geom is None or not shapely:
        return None
    try:
        if not geom.is_valid:
            geom = geom.buffer(0)
        return geom if geom.is_valid and not geom.is_empty else None
    except Exception:
        return None


def validate_geometry(geom) -> bool:
    return geom is not None and hasattr(geom, "is_valid") and geom.is_valid and not geom.is_empty


def calculate_area_m2(geom, source_crs="EPSG:4326") -> Optional[float]:
    '''Computes a geometry's area in real square meters, regardless of whether geom is already in
    a projected (meters-based) CRS or a geographic (degrees-based) one like WGS84. Only estimates
    and reprojects to a UTM zone when the input is geographic - calling estimate_utm_crs() on data
    that's already projected treats meter-scale coordinates as if they were degrees, producing a
    nonsensical CRS estimate and silently wrong (often near-zero) areas. This bug was caught via a
    real Colab run where Module 10's zone areas came back as ~1e-7 m2.'''
    if geom is None or not gpd:
        return None
    try:
        gdf = gpd.GeoDataFrame({"geometry": [geom]}, crs=source_crs)
        if gdf.crs is not None and gdf.crs.is_geographic:
            utm_crs = gdf.estimate_utm_crs()
            return float(gdf.to_crs(utm_crs).area.iloc[0])
        return float(gdf.area.iloc[0])  # already projected - area is already in real units, no reprojection needed
    except Exception:
        return None


def calculate_centroid(geom):
    return geom.centroid if geom is not None else None


def calculate_distance_m(geom, lat, lon, source_crs="EPSG:4326") -> Optional[float]:
    '''Distance from a candidate geometry's centroid to a reference point, in meters.'''
    if geom is None or not gpd or not shapely:
        return None
    try:
        gdf = gpd.GeoDataFrame({"geometry": [geom.centroid]}, crs=source_crs)
        utm_crs = gdf.estimate_utm_crs()
        centroid_utm = gdf.to_crs(utm_crs).geometry.iloc[0]
        ref_gdf = gpd.GeoDataFrame({"geometry": [shapely.geometry.Point(lon, lat)]}, crs=source_crs)
        ref_utm = ref_gdf.to_crs(utm_crs).geometry.iloc[0]
        return float(centroid_utm.distance(ref_utm))
    except Exception:
        return None


def polygon_to_geojson(geom) -> Optional[dict]:
    if geom is None or not shapely:
        return None
    try:
        return shapely.geometry.mapping(geom)
    except Exception:
        return None


print("Geometry utilities loaded: normalize_geometry, validate_geometry, calculate_area_m2, "
      "calculate_centroid, calculate_distance_m, polygon_to_geojson")


Geometry utilities loaded: normalize_geometry, validate_geometry, calculate_area_m2, calculate_centroid, calculate_distance_m, polygon_to_geojson


### Step 1 — Resolve the site location (reuses the resolvers already defined above)

In [ ]:
# @title Resolve site location for the boundary resolver
USE_EXISTING_SITE_COORDS = True  # @param {type:"boolean"}
# When True (default), reuses SITE.centroid from Step 1's primary input method above instead of
# re-geocoding by name - geocoders (especially free fallbacks like Nominatim) can resolve the
# same place name to a noticeably different point than a manually verified lat/lon, which then
# centers the entire OSM candidate search on the wrong location. Set this to False only if you
# want the boundary resolver to search around a DIFFERENT location than the one already loaded
# into SITE, and provide that location in BOUNDARY_SITE_INPUT below.
BOUNDARY_SITE_INPUT = "Al Safa Park 2, Dubai, UAE"  # @param {type:"string"}
BOUNDARY_SEARCH_RADIUS_M = SITE_BOUNDARY_CONFIG["search_radius_m"]

def resolve_site_for_boundary_search(site_input: str, use_existing: bool = True) -> dict:
    '''Thin wrapper standardizing whichever existing resolver applies (place name, URL, or raw
    "lat,lon") into the dict shape the rest of this pipeline expects. Reuses the exact same
    resolve_from_place_name / resolve_from_google_maps_url / resolve_from_latlon functions
    already defined above - no duplicate geocoding logic.

    If use_existing is True and SITE already has a resolved centroid (from Step 1 above), that
    is used directly rather than re-geocoding site_input by name - avoids a mismatch between a
    manually verified location and whatever a free-tier geocoder resolves the same name to.'''
    site_input = site_input.strip()

    if use_existing and SITE is not None and SITE.centroid is not None:
        lat, lon = SITE.centroid
        return {
            "input": site_input, "name": SITE.name, "latitude": lat, "longitude": lon,
            "source": f"reused_from_existing_SITE ({SITE.source_method})",
            "resolution_type": "existing_site_coords", "site_obj": SITE,
        }

    latlon_match = re.match(r"^(-?\d+\.?\d*),\s*(-?\d+\.?\d*)$", site_input)
    if latlon_match:
        lat, lon = float(latlon_match.group(1)), float(latlon_match.group(2))
        site_obj = resolve_from_latlon(lat, lon, name=site_input, radius_m=BOUNDARY_SEARCH_RADIUS_M)
        method = "exact_coordinate"
    elif "google.com/maps" in site_input or "maps.app.goo.gl" in site_input:
        site_obj = resolve_from_google_maps_url(site_input, radius_m=BOUNDARY_SEARCH_RADIUS_M)
        method = "google_maps_url"
    else:
        site_obj = resolve_from_place_name(site_input, radius_m=BOUNDARY_SEARCH_RADIUS_M)
        method = "named_location"

    return {
        "input": site_input, "name": site_obj.name, "latitude": site_obj.centroid[0],
        "longitude": site_obj.centroid[1], "source": site_obj.source_method,
        "resolution_type": method, "site_obj": site_obj,
    }


RESOLVED_BOUNDARY_SITE = resolve_site_for_boundary_search(BOUNDARY_SITE_INPUT, use_existing=USE_EXISTING_SITE_COORDS)
print(f"Resolved '{RESOLVED_BOUNDARY_SITE['input']}' -> "
      f"({RESOLVED_BOUNDARY_SITE['latitude']:.6f}, {RESOLVED_BOUNDARY_SITE['longitude']:.6f}) "
      f"via {RESOLVED_BOUNDARY_SITE['resolution_type']} ({RESOLVED_BOUNDARY_SITE['source']})")


Resolved 'Al Safa Park 2, Dubai, UAE' -> (25.156090, 55.221829) via existing_site_coords (reused_from_existing_SITE (lat_lon))


### Step 2 — Search OpenStreetMap for candidate boundary polygons

In [ ]:
# @title Search OSM for candidate boundaries around the resolved location

# Extensible - not restricted to the site-analysis POI category list (per the design brief).
# Add more OSM tag combinations here as needed; each becomes its own set of candidates.
BOUNDARY_SEARCH_TAGS = {
    "leisure_park": {"leisure": ["park", "garden", "nature_reserve", "recreation_ground"]},
    "landuse": {"landuse": True},
    "boundary": {"boundary": True},
    "amenity_institutional": {"amenity": ["school", "hospital", "university", "college"]},
    "tourism": {"tourism": True},
    "building": {"building": True},
    "place": {"place": True},
    "natural": {"natural": True},
}


def search_osm_boundaries(latitude, longitude, radius_m=500, max_candidates=10) -> list:
    '''Returns a list of raw (geometry, tags, source) tuples from OSM, not yet scored/normalized.'''
    if not osmnx or not gpd:
        logger.warning("osmnx/geopandas not available - cannot search OSM for boundary candidates.")
        return []

    point_gdf = gpd.GeoDataFrame({"geometry": [shapely.geometry.Point(longitude, latitude)]}, crs="EPSG:4326")
    utm_crs = point_gdf.estimate_utm_crs()
    search_area = point_gdf.to_crs(utm_crs).buffer(radius_m).to_crs("EPSG:4326").iloc[0]

    raw_candidates = []
    for tag_group_name, tags in BOUNDARY_SEARCH_TAGS.items():
        try:
            features = _osmnx_features(search_area, tags)
        except HardTimeoutError:
            logger.warning(f"OSM boundary search for '{tag_group_name}' timed out - skipping.")
            continue
        except Exception as e:
            logger.warning(f"OSM boundary search for '{tag_group_name}' failed: {e}")
            continue
        if features is None or len(features) == 0:
            continue
        for idx, row in features.iterrows():
            geom = row.geometry
            if geom is None or geom.geom_type not in ("Polygon", "MultiPolygon"):
                continue  # only polygon-shaped features are usable as boundary candidates
            raw_candidates.append({
                "geometry": geom, "tags": {k: v for k, v in row.items() if k != "geometry" and pd.notna(v)},
                "source_group": tag_group_name,
            })
        if len(raw_candidates) >= max_candidates * 2:  # gather a bit extra pre-dedup, then trim after normalization
            break

    return raw_candidates[: max_candidates * 2]


RAW_OSM_CANDIDATES = search_osm_boundaries(
    RESOLVED_BOUNDARY_SITE["latitude"], RESOLVED_BOUNDARY_SITE["longitude"],
    radius_m=BOUNDARY_SEARCH_RADIUS_M, max_candidates=SITE_BOUNDARY_CONFIG["max_candidates"],
)
print(f"Found {len(RAW_OSM_CANDIDATES)} raw polygon candidate(s) from OSM.")


Found 20 raw polygon candidate(s) from OSM.


### Step 3 — Normalize candidates into a common structure

In [ ]:
# @title Normalize raw candidates into the common candidate schema

@dataclass
class BoundaryCandidate:
    id: str
    geometry: Any                    # Shapely Polygon/MultiPolygon, WGS84
    source: str                        # "osm_boundary" | "official_gis" | "satellite_segmentation" | "ai_interpretation" | ...
    source_type: str = "polygon"
    name: str = ""
    tags: dict = field(default_factory=dict)
    distance_to_center_m: Optional[float] = None
    area_m2: Optional[float] = None
    score: float = 0.0
    score_breakdown: dict = field(default_factory=dict)

    def to_geojson_feature(self) -> Optional[dict]:
        geom_json = polygon_to_geojson(self.geometry)
        if geom_json is None:
            return None
        return {
            "type": "Feature", "geometry": geom_json,
            "properties": {"id": self.id, "source": self.source, "name": self.name,
                            "score": round(self.score, 3), "area_m2": self.area_m2,
                            "distance_to_center_m": self.distance_to_center_m},
        }


def normalize_candidates(raw_candidates: list, center_lat: float, center_lon: float) -> list:
    normalized = []
    for i, raw in enumerate(raw_candidates):
        geom = normalize_geometry(raw["geometry"])
        if not validate_geometry(geom):
            continue
        tags = raw["tags"]
        name = tags.get("name") or tags.get("name:en") or ""
        candidate = BoundaryCandidate(
            id=f"osm_{i}", geometry=geom, source="osm_boundary", name=name, tags=tags,
            distance_to_center_m=calculate_distance_m(geom, center_lat, center_lon),
            area_m2=calculate_area_m2(geom),
        )
        normalized.append(candidate)
    return normalized


BOUNDARY_CANDIDATES = normalize_candidates(
    RAW_OSM_CANDIDATES, RESOLVED_BOUNDARY_SITE["latitude"], RESOLVED_BOUNDARY_SITE["longitude"]
)
print(f"{len(BOUNDARY_CANDIDATES)} candidate(s) passed geometry validation.")


20 candidate(s) passed geometry validation.


### Step 4 — AI-assisted interpretation (Gemini vision, free tier)

In [ ]:
# @title Satellite image fetch for AI analysis (ESRI World Imagery, free, no key)

def get_satellite_image_for_boundary(lat: float, lon: float, radius_m: int = 500,
                                       image_size_px: int = 640) -> Optional[dict]:
    '''Fetches a satellite image crop centered on (lat, lon) via ESRI World Imagery's public
    export endpoint (same free, no-key source already used for basemap imagery in Module 02).
    Returns an image_context dict with the image bytes plus the exact geographic bounds the
    image covers, so it can be related back to coordinates later - matching the design brief's
    image_context schema.'''
    if not requests:
        return None
    try:
        # Convert a meter radius to an approximate degree bbox around the point
        dlat = radius_m / 111_320
        dlon = radius_m / (111_320 * math.cos(math.radians(lat)) or 1e-6)
        bbox = (lon - dlon, lat - dlat, lon + dlon, lat + dlat)  # (west, south, east, north)

        url = ("https://services.arcgisonline.com/arcgis/rest/services/World_Imagery/MapServer/export")
        params = {
            "bbox": f"{bbox[0]},{bbox[1]},{bbox[2]},{bbox[3]}", "bboxSR": "4326",
            "size": f"{image_size_px},{image_size_px}", "imageSR": "4326",
            "format": "png", "f": "image",
        }
        with hard_timeout(30):
            resp = requests.get(url, params=params, timeout=25)
        if resp.status_code != 200 or not resp.headers.get("content-type", "").startswith("image"):
            logger.warning(f"Satellite image fetch failed: HTTP {resp.status_code}")
            return None
        return {
            "image_bytes": resp.content,
            "bounds": {"west": bbox[0], "south": bbox[1], "east": bbox[2], "north": bbox[3]},
            "width": image_size_px, "height": image_size_px, "crs": "EPSG:4326", "provider": "esri_world_imagery",
        }
    except HardTimeoutError:
        logger.warning("Satellite image fetch timed out.")
        return None
    except Exception as e:
        logger.warning(f"Satellite image fetch failed: {e}")
        return None


BOUNDARY_IMAGE_CONTEXT = None
if SITE_BOUNDARY_CONFIG.get("enable_ai", False):
    BOUNDARY_IMAGE_CONTEXT = get_satellite_image_for_boundary(
        RESOLVED_BOUNDARY_SITE["latitude"], RESOLVED_BOUNDARY_SITE["longitude"], radius_m=BOUNDARY_SEARCH_RADIUS_M
    )
    print(f"Satellite image {'fetched' if BOUNDARY_IMAGE_CONTEXT else 'unavailable'} for AI analysis.")
else:
    print("AI analysis disabled (SITE_BOUNDARY_CONFIG['enable_ai'] = False) - skipping satellite image fetch.")


Satellite image fetched for AI analysis.


In [ ]:
# @title AI boundary analysis via Gemini (free tier: pluggable, off by default)

GEMINI_MODEL_NAME = "gemini-3.6-flash"  # Current stable vision-capable Flash model.
                                        # If this model is retired, check:
                                        # https://ai.google.dev/gemini-api/docs/models
                                        # and update this constant.

_GEMINI_CLIENT_CACHE = {"attempted": False, "client": None}


def _get_gemini_client():
    """Creates (and caches) a Gemini client from GEMINI_API_KEY.

    Returns None if the key is missing, the google-genai package is unavailable,
    or client creation fails. Callers should treat None as
    "AI analysis unavailable this run" and fall back gracefully.
    """
    if _GEMINI_CLIENT_CACHE["attempted"]:
        return _GEMINI_CLIENT_CACHE["client"]

    _GEMINI_CLIENT_CACHE["attempted"] = True

    if not genai or not API_KEYS.get("gemini_api_key"):
        return None

    try:
        from google import genai as _genai_root

        client = _genai_root.Client(
            api_key=API_KEYS["gemini_api_key"]
        )

        _GEMINI_CLIENT_CACHE["client"] = client
        return client

    except Exception as e:
        logger.warning(f"Could not create Gemini client: {e}")
        return None


def ai_boundary_analysis(
    image_context: Optional[dict],
    candidates: list,
    site_context: dict
) -> dict:
    """Analyze existing boundary candidates using Gemini vision.

    AI is used only to choose among and critique existing candidate polygons.
    It never invents new coordinates.

    The function gracefully falls back to a clearly-labelled "not evaluated"
    result if AI analysis is unavailable.
    """

    not_configured = {
        "selected_candidate_id": None,
        "confidence": None,
        "boundary_quality": "not_evaluated",
        "reasoning": [],
        "issues": [],
        "requires_refinement": False,
        "ai_used": False,
    }

    # -------------------------------------------------------------------------
    # 1. Check whether AI analysis is enabled
    # -------------------------------------------------------------------------
    if not SITE_BOUNDARY_CONFIG.get("enable_ai", False):
        not_configured["reasoning"] = [
            "AI vision analysis is disabled "
            "(SITE_BOUNDARY_CONFIG['enable_ai'] = False)."
        ]
        return not_configured

    # -------------------------------------------------------------------------
    # 2. Get Gemini client
    # -------------------------------------------------------------------------
    client = _get_gemini_client()

    if client is None:
        not_configured["reasoning"] = [
            "Gemini client unavailable - either GEMINI_API_KEY is not "
            "configured in Colab Secrets, or the google-genai package "
            "failed to install/import. Add GEMINI_API_KEY via the "
            "🔑 Secrets panel (never paste a key directly into a cell) "
            "to enable this."
        ]
        return not_configured

    # -------------------------------------------------------------------------
    # 3. Check required inputs
    # -------------------------------------------------------------------------
    if not image_context or not PIL_Image or not candidates:
        not_configured["reasoning"] = [
            "No satellite image or no candidates available to analyze this run."
        ]
        return not_configured

    try:
        import io as _io
        from google.genai import types

        # ---------------------------------------------------------------------
        # 4. Load satellite image
        # ---------------------------------------------------------------------
        pil_image = PIL_Image.open(
            _io.BytesIO(image_context["image_bytes"])
        )

        # ---------------------------------------------------------------------
        # 5. Build compact candidate description
        # ---------------------------------------------------------------------
        candidates_desc = "\n".join(
            f"- id={c.id}, "
            f"name='{c.name or 'unnamed'}', "
            f"area={c.area_m2:.0f} m², "
            f"distance_from_center={c.distance_to_center_m:.0f} m, "
            f"tags={dict(list(c.tags.items())[:5])}"
            for c in candidates[:8]
        )

        # ---------------------------------------------------------------------
        # 6. Build Gemini prompt
        # ---------------------------------------------------------------------
        prompt = f"""
You are analyzing a satellite image to help identify the boundary of a named site.

Site name:
{site_context.get("name", site_context.get("input", "unknown"))}

Resolved center:
{site_context["latitude"]:.6f}, {site_context["longitude"]:.6f}

Image covers:
west={image_context["bounds"]["west"]:.6f},
south={image_context["bounds"]["south"]:.6f},
east={image_context["bounds"]["east"]:.6f},
north={image_context["bounds"]["north"]:.6f}

Candidate boundary polygons already found from OpenStreetMap:

{candidates_desc}

IMPORTANT RULES:

1. DO NOT invent new coordinates.
2. DO NOT create a new polygon.
3. ONLY choose among the candidate IDs provided above.
4. If none of the candidates reasonably represents the site, return null.
5. Base your decision on visible physical features such as:
   - walls
   - fences
   - paths
   - vegetation edges
   - parking areas
   - land-use transitions
   - buildings
   - roads
   - other visible site boundaries
6. Prefer an existing candidate over saying none fit when there is reasonable
   visual evidence supporting that candidate.

Determine which candidate ID most likely represents the actual site boundary.

Return ONLY a JSON object with exactly these fields:

{{
    "selected_candidate_id": "<id or null>",
    "confidence": <number between 0.0 and 1.0>,
    "boundary_quality": "<high|medium|low>",
    "reasoning": ["<short reason>", "..."],
    "issues": ["<short issue>", "..."],
    "requires_refinement": <true|false>
}}
"""

        # ---------------------------------------------------------------------
        # 7. Gemini vision request
        #
        # AFC is explicitly disabled because this analysis does not use
        # function calling. JSON output is also requested at the API level.
        # ---------------------------------------------------------------------
        with hard_timeout(45):
            response = client.models.generate_content(
                model=GEMINI_MODEL_NAME,
                contents=[
                    prompt,
                    pil_image,
                ],
                config=types.GenerateContentConfig(
                    automatic_function_calling=(
                        types.AutomaticFunctionCallingConfig(
                            disable=True
                        )
                    ),
                    response_mime_type="application/json",
                ),
            )

        # ---------------------------------------------------------------------
        # 8. Parse JSON response
        # ---------------------------------------------------------------------
        raw_text = response.text.strip()

        result = json.loads(raw_text)

        # ---------------------------------------------------------------------
        # 9. Mark AI as successfully used
        # ---------------------------------------------------------------------
        result["ai_used"] = True

        return result

    # -------------------------------------------------------------------------
    # 10. Graceful error handling
    # -------------------------------------------------------------------------
    except HardTimeoutError as e:
        not_configured["reasoning"] = [
            f"Gemini API call timed out: {e}"
        ]
        return not_configured

    except json.JSONDecodeError as e:
        not_configured["reasoning"] = [
            f"Gemini returned non-JSON output that could not be parsed: {e}"
        ]
        return not_configured

    except Exception as e:
        not_configured["reasoning"] = [
            f"Gemini API call failed: "
            f"{e.__class__.__name__}: {str(e)[:200]}"
        ]
        return not_configured


# =============================================================================
# Run AI boundary analysis
# =============================================================================

AI_ANALYSIS_RESULT = ai_boundary_analysis(
    image_context=BOUNDARY_IMAGE_CONTEXT,
    candidates=BOUNDARY_CANDIDATES,
    site_context=RESOLVED_BOUNDARY_SITE,
)


# =============================================================================
# Display result
# =============================================================================

print(
    f"AI analysis: "
    f"ai_used={AI_ANALYSIS_RESULT['ai_used']}, "
    f"quality={AI_ANALYSIS_RESULT['boundary_quality']}"
)

if AI_ANALYSIS_RESULT["reasoning"]:
    for reason in AI_ANALYSIS_RESULT["reasoning"]:
        print(f"  - {reason}")

if AI_ANALYSIS_RESULT.get("selected_candidate_id") is not None:
    print(
        f"Selected candidate: "
        f"{AI_ANALYSIS_RESULT['selected_candidate_id']}"
    )

if AI_ANALYSIS_RESULT.get("confidence") is not None:
    print(
        f"Confidence: "
        f"{AI_ANALYSIS_RESULT['confidence']:.2f}"
    )

if AI_ANALYSIS_RESULT.get("issues"):
    print("Issues:")
    for issue in AI_ANALYSIS_RESULT["issues"]:
        print(f"  - {issue}")

AI analysis: ai_used=True, quality=high
  - Candidate osm_0 translates directly to 'Safa 2 Park' (حديقة الصفا 2).
  - The distance from the resolved center is only 47 meters, matching the green park visible near the center of the satellite image.
Selected candidate: osm_0
Confidence: 0.92


### Step 5 — Deterministic candidate scoring

In [ ]:
# @title Score and rank all candidates

def score_candidate(candidate: BoundaryCandidate, site_context: dict, config: dict) -> BoundaryCandidate:
    weights = config["weights"]
    breakdown = {}

    breakdown["source_reliability"] = SOURCE_RELIABILITY.get(candidate.source, 0.5)

    query_name = site_context["input"].lower()
    cand_name = candidate.name.lower()
    if cand_name and cand_name in query_name or query_name in cand_name:
        breakdown["name_match"] = 1.0
    elif cand_name and any(word in cand_name for word in query_name.split() if len(word) > 3):
        breakdown["name_match"] = 0.5
    else:
        breakdown["name_match"] = 0.0

    # Geometry quality: penalize implausibly small (<50 m2, likely a mapping error) or implausibly
    # large (>0.5 km2, likely an administrative boundary rather than a specific site) polygons.
    if candidate.area_m2 is None:
        breakdown["geometry_quality"] = 0.3
    elif candidate.area_m2 < 50:
        breakdown["geometry_quality"] = 0.2
    elif candidate.area_m2 > 500_000:
        breakdown["geometry_quality"] = 0.4
    else:
        breakdown["geometry_quality"] = 1.0

    if candidate.distance_to_center_m is None:
        breakdown["center_proximity"] = 0.3
    else:
        # 1.0 at 0m, decaying to 0.0 at the search radius
        breakdown["center_proximity"] = max(0.0, 1.0 - candidate.distance_to_center_m / config["search_radius_m"])

    landuse_relevant_tags = {"leisure", "landuse", "amenity", "tourism"}
    breakdown["landuse_agreement"] = 1.0 if landuse_relevant_tags & set(candidate.tags.keys()) else 0.3

    total_score = sum(breakdown[k] * weights[k] for k in weights if k in breakdown)
    candidate.score = round(total_score, 3)
    candidate.score_breakdown = breakdown
    return candidate


def rank_boundary_candidates(candidates: list, site_context: dict, config: dict) -> list:
    scored = [score_candidate(c, site_context, config) for c in candidates]
    return sorted(scored, key=lambda c: -c.score)


RANKED_CANDIDATES = rank_boundary_candidates(BOUNDARY_CANDIDATES, RESOLVED_BOUNDARY_SITE, SITE_BOUNDARY_CONFIG)

print(f"Ranked {len(RANKED_CANDIDATES)} candidate(s):\n")
for c in RANKED_CANDIDATES[:5]:
    print(f"  {c.id}: score={c.score:.3f}  name='{c.name or '(unnamed)'}' "
          f"area={c.area_m2:.0f}m² dist={c.distance_to_center_m:.0f}m  tags_group={c.tags.get('source_group', '')}")


Ranked 20 candidate(s):

  osm_0: score=0.711  name='حديقة الصفا 2' area=14740m² dist=47m  tags_group=
  osm_10: score=0.622  name='(unnamed)' area=2593m² dist=96m  tags_group=
  osm_9: score=0.597  name='(unnamed)' area=6555m² dist=158m  tags_group=
  osm_5: score=0.571  name='(unnamed)' area=116091m² dist=399m  tags_group=
  osm_4: score=0.530  name='(unnamed)' area=2745m² dist=507m  tags_group=


### Step 6 — Confidence classification & map display

In [ ]:
# @title Combine Gemini's selection with deterministic scoring, then classify confidence + map

def select_final_candidate(ranked_candidates: list, ai_result: dict, config: dict) -> dict:
    '''Combines the AI's selection with the deterministic geometric/tag score into one final
    decision, rather than trusting either signal alone:

    - AI agrees with the top-ranked candidate -> strong agreement, confidence gets a boost
    - AI picks a DIFFERENT candidate than the top-ranked one -> AI's pick wins (a named-place
      match visible in real imagery is stronger evidence than geometry/tag heuristics alone),
      but flagged as "AI override" so it's visible in the output, not silently swapped in
    - AI was not run or found no match -> falls back to the deterministic top-ranked candidate
      alone, exactly as before

    Returns {"candidate": BoundaryCandidate, "confidence_label": str, "agreement": str,
    "combined_score": float}.'''
    if not ranked_candidates:
        return {"candidate": None, "confidence_label": None, "agreement": "no_candidates", "combined_score": 0.0}

    top_ranked = ranked_candidates[0]
    ai_used = ai_result.get("ai_used", False)
    ai_selected_id = ai_result.get("selected_candidate_id")
    ai_confidence = ai_result.get("confidence")

    if not ai_used or ai_selected_id is None:
        return {"candidate": top_ranked, "confidence_label": classify_confidence(top_ranked.score, config),
                "agreement": "ai_not_used", "combined_score": top_ranked.score}

    ai_candidate = next((c for c in ranked_candidates if c.id == ai_selected_id), None)
    if ai_candidate is None:
        # AI referenced a candidate ID that isn't in our ranked list (shouldn't normally happen
        # since the prompt only offers real candidate IDs, but handle it defensively)
        return {"candidate": top_ranked, "confidence_label": classify_confidence(top_ranked.score, config),
                "agreement": "ai_selection_not_found", "combined_score": top_ranked.score}

    if ai_candidate.id == top_ranked.id:
        # Agreement: both signals point to the same candidate - blend the two confidences,
        # weighted toward the higher one, since independent agreement is strong evidence.
        combined = max(top_ranked.score, ai_confidence or 0) * 0.7 + min(top_ranked.score, ai_confidence or 0) * 0.3
        combined = min(1.0, combined + 0.05)  # small agreement bonus
        return {"candidate": top_ranked, "confidence_label": classify_confidence(combined, config),
                "agreement": "agree", "combined_score": round(combined, 3)}
    else:
        # Disagreement: AI picked a different candidate than pure geometric/tag scoring.
        # Trust the AI's pick (it has visual + name-matching evidence the scorer doesn't),
        # but keep the disagreement visible rather than hiding it.
        combined = ai_confidence if ai_confidence is not None else ai_candidate.score
        return {"candidate": ai_candidate, "confidence_label": classify_confidence(combined, config),
                "agreement": "ai_override", "combined_score": round(combined, 3)}


def classify_confidence(score: float, config: dict) -> str:
    if score >= config["high_confidence_threshold"]:
        return "high"
    elif score >= config["medium_confidence_threshold"]:
        return "medium"
    return "low"


_SELECTION = select_final_candidate(RANKED_CANDIDATES, AI_ANALYSIS_RESULT, SITE_BOUNDARY_CONFIG)
BEST_CANDIDATE = _SELECTION["candidate"]
BEST_CANDIDATE_CONFIDENCE = _SELECTION["confidence_label"]

if BEST_CANDIDATE:
    agreement_msg = {
        "agree": "Gemini and the deterministic scorer AGREE on this candidate.",
        "ai_override": "Gemini selected a DIFFERENT candidate than the top deterministic score - "
                        "trusting Gemini's pick since it has visual/name-matching evidence the "
                        "scorer alone doesn't.",
        "ai_not_used": "AI analysis was not used this run - relying on deterministic score alone.",
        "ai_selection_not_found": "Gemini's selection didn't match a known candidate ID - falling "
                                    "back to the deterministic top score.",
    }.get(_SELECTION["agreement"], "")
    print(f"Final selection: {BEST_CANDIDATE.id} ('{BEST_CANDIDATE.name or 'unnamed'}')")
    print(f"  Combined confidence: {_SELECTION['combined_score']:.3f} ({BEST_CANDIDATE_CONFIDENCE})")
    print(f"  {agreement_msg}")
else:
    print("No usable candidates found - fall back to manual lat/lon/radius entry (Step 7 below).")

if folium and gpd and RANKED_CANDIDATES:
    m = folium.Map(location=[RESOLVED_BOUNDARY_SITE["latitude"], RESOLVED_BOUNDARY_SITE["longitude"]], zoom_start=16)
    folium.Marker([RESOLVED_BOUNDARY_SITE["latitude"], RESOLVED_BOUNDARY_SITE["longitude"]],
                   tooltip="Resolved center", icon=folium.Icon(color="red")).add_to(m)

    # Only the final selected candidate is drawn solid/filled; everything else is a thin,
    # low-opacity outline so the actual answer is visually unambiguous rather than five
    # similarly-styled boxes competing for attention.
    for c in RANKED_CANDIDATES[:5]:
        is_selected = BEST_CANDIDATE is not None and c.id == BEST_CANDIDATE.id
        try:
            folium.GeoJson(
                gpd.GeoSeries([c.geometry]).__geo_interface__,
                style_function=lambda x, sel=is_selected: {
                    "color": "#2E7D32" if sel else "#9E9E9E",
                    "weight": 4 if sel else 1,
                    "fillOpacity": 0.35 if sel else 0.03,
                    "dashArray": None if sel else "4,4",
                },
                tooltip=f"{'★ SELECTED - ' if is_selected else ''}{c.id}: score={c.score:.2f} ({c.name or 'unnamed'})",
            ).add_to(m)
        except Exception as e:
            logger.warning(f"Could not draw candidate {c.id}: {e}")

    display(m)


Final selection: osm_0 ('حديقة الصفا 2')
  Combined confidence: 0.907 (high)
  Gemini and the deterministic scorer AGREE on this candidate.


### Step 7 — Approve, reject, or fall back to manual entry

Set `BOUNDARY_DECISION` below based on what you saw on the map above, then run the cell.
- `"approve"` — accepts `BEST_CANDIDATE` and overwrites `SITE` with it
- `"reject"` — keeps whatever `SITE` was set to by the primary input method earlier in this module
- `"manual"` — ignores all candidates and builds `SITE` from `MANUAL_LAT`/`MANUAL_LON`/`MANUAL_RADIUS_M` instead


In [ ]:
# @title Approve / reject / manual fallback
BOUNDARY_DECISION = "approve"  # @param ["approve", "reject", "manual"]

MANUAL_LAT = RESOLVED_BOUNDARY_SITE["latitude"]
MANUAL_LON = RESOLVED_BOUNDARY_SITE["longitude"]
MANUAL_RADIUS_M = BOUNDARY_SEARCH_RADIUS_M


def apply_boundary_decision(decision: str):
    global SITE
    if decision == "approve":
        if BEST_CANDIDATE is None:
            print("No candidate available to approve - SITE left unchanged.")
            return
        if SITE_BOUNDARY_CONFIG["require_user_confirmation"] and BEST_CANDIDATE_CONFIDENCE == "low":
            print(f"⚠️  Best candidate has LOW confidence ({BEST_CANDIDATE.score:.2f}) - not auto-applying. "
                  f"Review the map above carefully, or switch BOUNDARY_DECISION to 'manual' instead.")
            return
        centroid = BEST_CANDIDATE.geometry.centroid
        SITE = Site(
            name=BEST_CANDIDATE.name or RESOLVED_BOUNDARY_SITE["name"],
            centroid=(centroid.y, centroid.x), boundary=BEST_CANDIDATE.geometry,
            bbox=BEST_CANDIDATE.geometry.bounds, source_method="ai_assisted_boundary_resolver",
            notes=[f"Approved OSM candidate {BEST_CANDIDATE.id}, score={BEST_CANDIDATE.score:.3f} "
                   f"(confidence={BEST_CANDIDATE_CONFIDENCE}). Detected, not a legal/cadastral boundary."],
        )
        print(f"✓ SITE updated from approved candidate {BEST_CANDIDATE.id} "
              f"(confidence={BEST_CANDIDATE_CONFIDENCE}, area={BEST_CANDIDATE.area_m2:,.0f} m²).")

    elif decision == "manual":
        SITE = resolve_from_latlon(MANUAL_LAT, MANUAL_LON, name=RESOLVED_BOUNDARY_SITE["name"], radius_m=MANUAL_RADIUS_M)
        SITE.source_method = "boundary_resolver_manual_fallback"
        print(f"✓ SITE updated from manual lat/lon/radius entry.")

    else:  # "reject"
        print("Boundary candidates rejected - SITE left as set by the primary input method above.")


apply_boundary_decision(BOUNDARY_DECISION)
print(f"\nActive SITE: name='{SITE.name}', source_method='{SITE.source_method}', "
      f"centroid={SITE.centroid}")


✓ SITE updated from approved candidate osm_0 (confidence=high, area=14,740 m²).

Active SITE: name='حديقة الصفا 2', source_method='ai_assisted_boundary_resolver', centroid=(25.155681554488865, 55.22193546902516)


### Step 8 — Override with the real Annex-1 CAD boundary (recommended)

Everything above (place name / URL / lat-lon / uploaded boundary / OSM candidate resolver) still works exactly as before and is unchanged. This step is an **additional, optional override**: if you have the Scope of Work's Annex-1 as-built drawing as an ASCII DXF export, this parses it directly (pure Python, no ezdxf/GDAL required) and georeferences the real property boundary using Dubai Municipality's own Dubai Local Transverse Mercator grid (EPSG:3997) — replacing the OSM-guessed or buffer-based boundary with the real, dimensioned site footprint. Every render, cost estimate, and score elsewhere in this notebook is only as good as `SITE.boundary`, so it's worth running this before anything downstream.

In [ ]:
# =============================================================================
# MODULE 01 ADDITION — Annex-1 CAD Boundary Import (real site footprint,
#                       replacing the OSM-guessed boundary used everywhere else)
# =============================================================================
# Paste this AFTER Module 01's existing "AI-assisted Site Boundary Resolver"
# section (after its Step 7 "Approve / reject / manual fallback" cell), before
# Module 02 begins. It works on the ASCII-DXF export of the Scope of Work's
# Annex-1 as-built drawing — no ezdxf/GDAL/network install required for the
# DXF parsing itself (pure Python).
#
# What this does, and why it's needed: every render, cost estimate, and score
# produced elsewhere in this notebook is only as good as SITE.boundary. Until
# now that boundary was either an OSM-guessed polygon or a circular buffer —
# neither is the real Al Safa 2 Park footprint.
#
# METHOD (revised after a real, visually-caught positioning bug): the DXF is
# trusted for what it's actually reliable for — precise SHAPE, DIMENSIONS, and
# internal proportions (validated: the extracted boundary's area matches both
# the Scope of Work's ~15,000 m2 and the drawing's own "15,001 SQ.M" label).
# An earlier version of this cell tried to derive real-world POSITION from
# survey gridline labels found elsewhere in the drawing (e.g. "488650E") —
# that produced a real ~360m positional error, because that label cluster
# turned out to sit at a different location within the DXF than the boundary
# polygon itself (almost certainly a separate index/key-plan inset, not
# co-located with the main site plan in the same local coordinate frame).
#
# The fix: get POSITION and ROTATION from Module 01's own OSM-based boundary
# resolver instead — real, independently-traced geometry for this exact park,
# reusing already-validated infrastructure rather than a fragile from-scratch
# geodetic derivation. The DXF shape is rotated and translated to match the
# OSM polygon's own principal-axis orientation and centroid.
#
# Honesty notes: the DXF appears to be a PDF-traced/auto-vectorized drawing
# (layer names like PDF_Geometry/PDF_Text), not a clean native survey — trust
# the simple closed boundary polygon (a straightforward quadrilateral) far
# more than any organic/curved entity. And this approach is only as good as
# the OSM anchor polygon — ALWAYS visually sanity-check the result against
# satellite imagery (the last cell in this section does exactly that) before
# trusting it for final submission.
# =============================================================================
# @title Annex-1 configuration
DXF_ANNEX1_PATH = "/content/ACAD-Al_Safa_Park_2_Plan.dxf"   # re-upload the DXF into this Colab session and update if needed
USE_DXF_BOUNDARY = True             # set False to skip this section entirely and keep whatever Module 01 already set
TARGET_SITE_AREA_M2 = 15000.0        # from the Scope of Work; used to auto-identify the boundary polygon among all DXF entities
AREA_MATCH_TOLERANCE = 0.30           # accept candidates within +/-30% of the target area

import os as _os_dxf

if USE_DXF_BOUNDARY and not _os_dxf.path.exists(DXF_ANNEX1_PATH):
    print(f"⚠️  DXF not found at {DXF_ANNEX1_PATH} this session — re-upload it (Colab Files panel or "
          f"`from google.colab import files; files.upload()`), update DXF_ANNEX1_PATH above, and re-run. "
          f"Skipping this section for now; SITE keeps whatever Module 01 already resolved.")
    USE_DXF_BOUNDARY = False
else:
  print(f"DXF file found at {DXF_ANNEX1_PATH} in this session.")

DXF file found at /content/ACAD-Al_Safa_Park_2_Plan.dxf in this session.


In [ ]:
# @title Pure-Python ASCII DXF reader (no ezdxf/GDAL needed) — polylines + text labels

def _dxf_shoelace_area(vertices):
    n = len(vertices)
    if n < 3:
        return 0.0
    area = sum(vertices[i][0]*vertices[(i+1) % n][1] - vertices[(i+1) % n][0]*vertices[i][1] for i in range(n))
    return abs(area) / 2.0


def dxf_extract_polylines(path, layer_filter=None):
    '''Extracts every LWPOLYLINE in the ENTITIES section as a proper list of vertices
    (not a flat point cloud) using only Python's standard library. Returns a list of
    dicts: layer, closed, n, area_m2, bbox, vertices.'''
    polylines = []
    current_section = None; awaiting_section_name = False
    current_entity = None; current_layer = None
    current_vertices = []; current_closed = None
    pending_x = None

    def flush():
        if current_entity == "LWPOLYLINE" and len(current_vertices) >= 3 and current_section == "ENTITIES":
            if layer_filter is None or current_layer == layer_filter:
                xs = [v[0] for v in current_vertices]; ys = [v[1] for v in current_vertices]
                polylines.append({
                    "layer": current_layer, "closed": current_closed, "n": len(current_vertices),
                    "area_m2": _dxf_shoelace_area(current_vertices),
                    "bbox": (min(xs), min(ys), max(xs), max(ys)),
                    "vertices": list(current_vertices),
                })

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        code = None
        for raw in f:
            line = raw.rstrip("\r\n")
            if code is None:
                code = line.strip(); continue
            value = line.strip()
            if code == "0":
                flush(); current_vertices = []; current_closed = None
                if value == "SECTION":
                    awaiting_section_name = True
                elif value == "ENDSEC":
                    current_section = None
                current_entity = value if (value.replace("_", "").isalnum() and value.isupper()) else None
                current_layer = None
            elif code == "2" and awaiting_section_name:
                current_section = value; awaiting_section_name = False
            elif code == "8" and current_section == "ENTITIES":
                current_layer = value
            elif code == "70" and current_entity == "LWPOLYLINE":
                try:
                    current_closed = (int(value) & 1) == 1
                except ValueError:
                    pass
            elif code == "10" and current_section == "ENTITIES" and current_entity == "LWPOLYLINE":
                try:
                    pending_x = float(value)
                except ValueError:
                    pending_x = None
            elif code == "20" and current_section == "ENTITIES" and current_entity == "LWPOLYLINE":
                if pending_x is not None:
                    try:
                        current_vertices.append((pending_x, float(value)))
                    except ValueError:
                        pass
                    pending_x = None
            code = None
        flush()
    return polylines


def dxf_extract_text_labels(path):
    '''Extracts MTEXT/TEXT content with insertion-point positions, stripping DXF's
    inline formatting codes (\\P, \\A1;, etc). Used to find the survey grid gridline
    labels (e.g. "488650E", "2783420N") for georeferencing.'''
    import re
    labels = []
    current_section = None; awaiting_section_name = False
    current_entity = None; pending_x = pending_y = None
    text_buffer = []

    def flush():
        if current_entity in ("MTEXT", "TEXT") and pending_x is not None and text_buffer:
            raw_txt = "".join(text_buffer)
            clean = re.sub(r'\\[A-Za-z][^;]*;?', '', raw_txt).replace("\\P", " ").strip()
            if clean:
                labels.append((pending_x, pending_y, clean))

    with open(path, "r", encoding="utf-8", errors="replace") as f:
        code = None
        for raw in f:
            line = raw.rstrip("\r\n")
            if code is None:
                code = line.strip(); continue
            value = line.strip()
            if code == "0":
                flush(); text_buffer = []; pending_x = pending_y = None
                if value == "SECTION":
                    awaiting_section_name = True
                elif value == "ENDSEC":
                    current_section = None
                current_entity = value if value.isupper() else None
            elif code == "2" and awaiting_section_name:
                current_section = value; awaiting_section_name = False
            elif current_section == "ENTITIES" and current_entity in ("MTEXT", "TEXT"):
                if code == "10":
                    try: pending_x = float(value)
                    except ValueError: pass
                elif code == "20":
                    try: pending_y = float(value)
                    except ValueError: pass
                elif code in ("1", "3"):
                    text_buffer.append(value)
            code = None
        flush()
    return labels

In [ ]:
# @title Boundary polygon auto-detection, plus a demoted diagnostic-only grid-label tool
#
# find_boundary_candidate() is the primary, trusted path — it identifies the
# boundary polygon purely from its AREA (validated against the SOW's known
# ~15,000 m2), independent of any position/rotation assumption.
#
# derive_grid_offset() below is kept only as an optional manual diagnostic —
# it is NOT called in the primary flow anymore. It was the original approach
# for deriving real-world position directly from survey gridline labels found
# in the drawing, but that produced a real ~360m positional error (see the
# note at the top of this section) because the label cluster it found sits at
# a different location in the DXF than the boundary polygon itself. Retained
# here in case a future, more carefully-verified reading of those labels (or
# a different DXF revision) makes it useful again — but do not wire it back
# into the primary path without independently re-verifying the label
# positions against the actual boundary location first.

def find_boundary_candidate(polylines, target_area_m2, tolerance=0.30, layer="0"):
    '''Picks the closed layer-'0' polygon whose area is closest to target_area_m2,
    among those within +/-tolerance. Returns None if nothing qualifies — callers
    must treat that as "could not auto-identify the boundary", not silently proceed
    with a wrong polygon.'''
    candidates = [p for p in polylines if p["layer"] == layer and p["closed"]
                  and abs(p["area_m2"] - target_area_m2) <= target_area_m2 * tolerance]
    if not candidates:
        return None
    return min(candidates, key=lambda p: abs(p["area_m2"] - target_area_m2))


def derive_grid_offset(labels):
    '''Looks for Dubai Municipality-style survey gridline labels ("488650E",
    "2783420N") among the extracted text labels, and derives the DXF-local ->
    DLTM offset by averaging every matched pair of same-value labels (a gridline
    is typically labeled at both ends of the sheet, giving 2+ independent estimates
    of the same offset — averaging them is a real, if simple, precision check, not
    just a single guess). Returns (offset_E, offset_N, diagnostic_dict) or
    (None, None, diagnostic_dict) if fewer than 2 distinct values were found on
    either axis (not enough to fix an offset with any confidence).'''
    import re
    e_labels = [(x, int(m.group(1))) for x, y, t in labels if (m := re.fullmatch(r"(\d{5,7})\s*E", t))]
    n_labels = [(y, int(m.group(1))) for x, y, t in labels if (m := re.fullmatch(r"(\d{6,8})\s*N", t))]
    # re-extract with y for E and x for N properly:
    e_labels = [(x, int(re.fullmatch(r"(\d{5,7})\s*E", t).group(1))) for x, y, t in labels if re.fullmatch(r"(\d{5,7})\s*E", t)]
    n_labels = [(y, int(re.fullmatch(r"(\d{6,8})\s*N", t).group(1))) for x, y, t in labels if re.fullmatch(r"(\d{6,8})\s*N", t)]

    diag = {"n_easting_labels": len(e_labels), "n_northing_labels": len(n_labels)}
    if len(set(v for _, v in e_labels)) < 2 or len(set(v for _, v in n_labels)) < 2:
        diag["reason"] = "fewer than 2 distinct gridline values found on one or both axes"
        return None, None, diag

    # Group by real-world value (each real value should have ~2 label instances at
    # different sheet positions for the same gridline) and average the implied offset.
    e_offsets = [real - dxf for dxf, real in e_labels]
    n_offsets = [real - dxf for dxf, real in n_labels]
    offset_E = sum(e_offsets) / len(e_offsets)
    offset_N = sum(n_offsets) / len(n_offsets)
    diag["offset_E_spread"] = max(e_offsets) - min(e_offsets)
    diag["offset_N_spread"] = max(n_offsets) - min(n_offsets)
    diag["offset_E_values"] = e_offsets
    diag["offset_N_values"] = n_offsets
    return offset_E, offset_N, diag

In [ ]:
# @title Run the extraction + georeferencing, and override SITE if successful
#
# REVISED APPROACH (v2): the original version derived a DXF-local -> DLTM offset
# from survey gridline labels found in the drawing's PDF_Text layer. On review,
# that gridline-label cluster sits at a different DXF-local location than the
# boundary polygon itself (X~4058-4322 vs the boundary's X~3742-3910) — almost
# certainly a separate index/key-plan inset elsewhere on the sheet, not
# co-located with the main site plan in the same local frame. Applying that
# offset produced a real, visually-confirmed ~360m positional error.
#
# Fix: trust the DXF for what it's actually reliable for — precise SHAPE,
# DIMENSIONS, and internal proportions (validated: 14,841 m2 against the SOW's
# ~15,000 m2 and the drawing's own "15,001 SQ.M" label) — and get POSITION and
# ROTATION from Module 01's own OSM-based boundary resolver instead, which is
# already-validated, real, independently-traced geometry for this exact park
# (and in this session, already approved into SITE.boundary before this cell
# runs). This reuses trusted infrastructure instead of a fragile from-scratch
# geodetic derivation, consistent with this notebook's "source-first" principle.

import math as _math_dxf

def _polygon_principal_axis_angle_deg(coords):
    '''Angle (degrees, standard math convention from +X axis) of a polygon's
    longest edge — a simple, robust orientation proxy for a near-rectangular
    shape. Normalized to [0, 180) since a line's orientation is ambiguous by
    180 degrees (this matters when matching two independently-drawn rectangles
    that might be traced starting from different corners).'''
    n = len(coords)
    best_len, best_angle = -1, 0.0
    for i in range(n):
        x1, y1 = coords[i]; x2, y2 = coords[(i + 1) % n]
        dx, dy = x2 - x1, y2 - y1
        length = _math_dxf.hypot(dx, dy)
        if length > best_len:
            best_len, best_angle = length, _math_dxf.degrees(_math_dxf.atan2(dy, dx))
    return best_angle % 180.0, best_len


def _rotate_translate_polygon(coords, rotate_deg, target_center):
    '''Re-centers coords on their own centroid, rotates by rotate_deg, then
    translates so the shape's centroid lands at target_center. All in the
    same consistent local-meters CRS as coords and target_center.'''
    cx = sum(c[0] for c in coords) / len(coords)
    cy = sum(c[1] for c in coords) / len(coords)
    theta = _math_dxf.radians(rotate_deg)
    cos_t, sin_t = _math_dxf.cos(theta), _math_dxf.sin(theta)
    out = []
    for x, y in coords:
        rx, ry = x - cx, y - cy
        nx = rx * cos_t - ry * sin_t
        ny = rx * sin_t + ry * cos_t
        out.append((nx + target_center[0], ny + target_center[1]))
    return out


if USE_DXF_BOUNDARY:
    print("Parsing Annex-1 DXF (pure Python, no external CAD library)...")
    _polylines = dxf_extract_polylines(DXF_ANNEX1_PATH)
    print(f"  {len(_polylines)} LWPOLYLINE entities found.")

    _boundary_candidate = find_boundary_candidate(_polylines, TARGET_SITE_AREA_M2, AREA_MATCH_TOLERANCE)

    if _boundary_candidate is None:
        print(f"⚠️  Could not auto-identify a closed layer-'0' polygon within "
              f"+/-{AREA_MATCH_TOLERANCE*100:.0f}% of {TARGET_SITE_AREA_M2:,.0f} m2. "
              f"Inspect `_polylines` manually, or widen AREA_MATCH_TOLERANCE above.")
    else:
        print(f"  Boundary candidate: {_boundary_candidate['area_m2']:,.0f} m2 "
              f"(target {TARGET_SITE_AREA_M2:,.0f} m2, "
              f"{abs(_boundary_candidate['area_m2']-TARGET_SITE_AREA_M2)/TARGET_SITE_AREA_M2*100:.1f}% off)")

        # --- Determine the anchor: prefer the already-resolved OSM boundary in
        # this session (SITE.boundary, if Module 01's resolver was approved —
        # check source_method rather than assuming), else fall back to a fresh
        # targeted OSM search, else fall back to the geocoded point alone with
        # rotation left unknown (loudly flagged, not silently defaulted).
        _RELIABLE_ANCHOR_METHODS = {
            "ai_assisted_boundary_resolver", "boundary_resolver_manual_fallback",
            "google_geocoding", "nominatim_osm", "uploaded_boundary",
        }
        _anchor_polygon_wgs84 = None
        _anchor_source = None

        if "SITE" in dir() and SITE.boundary is not None and SITE.source_method in _RELIABLE_ANCHOR_METHODS \
           and getattr(SITE.boundary, "geom_type", None) in ("Polygon", "MultiPolygon"):
            _anchor_polygon_wgs84 = SITE.boundary
            _anchor_source = f"existing SITE.boundary (source_method={SITE.source_method})"
        else:
            print("  No already-resolved polygon boundary found in this session — searching OSM directly...")
            try:
                _lat, _lon = (SITE.centroid if "SITE" in dir() and SITE.centroid else (LATITUDE, LONGITUDE))
                _raw = search_osm_boundaries(_lat, _lon, radius_m=200, max_candidates=5)
                _norm = normalize_candidates(_raw, _lat, _lon)
                if _norm:
                    _ranked = rank_boundary_candidates(_norm, {"input": SITE_NAME if "SITE_NAME" in dir() else "", "name": ""}, SITE_BOUNDARY_CONFIG)
                    _anchor_polygon_wgs84 = _ranked[0].geometry
                    _anchor_source = f"fresh OSM search (candidate score={_ranked[0].score:.2f})"
            except Exception as e:
                print(f"  Fresh OSM search failed: {e}")

        if _anchor_polygon_wgs84 is None:
            print(f"⚠️  No OSM polygon anchor available — cannot reliably determine rotation. The DXF "
                  f"polygon's SHAPE and AREA are trustworthy; its real-world POSITION and ORIENTATION "
                  f"are NOT without an anchor. Not overriding SITE. Run Module 01's AI-assisted boundary "
                  f"resolver and approve a candidate first, then re-run this cell.")
        else:
            try:
                # Compute a local UTM CRS from the anchor polygon itself, rather than depending on
                # Module 03's SITE_UTM_CRS — that variable does not exist yet this early in the
                # notebook (Step 8 runs within Module 01, well before Module 03), which is exactly
                # what caused the NameError. estimate_utm_crs() needs a geographic (EPSG:4326)
                # GeoDataFrame to pick the right UTM zone, so we build that first, then reproject.
                _anchor_gdf_wgs84 = gpd.GeoDataFrame({"geometry": [_anchor_polygon_wgs84]}, crs="EPSG:4326")
                _local_utm_crs = _anchor_gdf_wgs84.estimate_utm_crs()
                _anchor_utm = _anchor_gdf_wgs84.to_crs(_local_utm_crs).geometry.iloc[0]
                _anchor_coords = list(_anchor_utm.exterior.coords)[:-1]
                _anchor_angle, _ = _polygon_principal_axis_angle_deg(_anchor_coords)
                _anchor_centroid = (_anchor_utm.centroid.x, _anchor_utm.centroid.y)

                _dxf_angle, _dxf_long_edge = _polygon_principal_axis_angle_deg(_boundary_candidate["vertices"])
                _rotate_by = _anchor_angle - _dxf_angle

                print(f"  Anchor: {_anchor_source}")
                print(f"    anchor principal-axis angle: {_anchor_angle:.1f} deg, centroid (UTM): "
                      f"({_anchor_centroid[0]:.1f}, {_anchor_centroid[1]:.1f})")
                print(f"  DXF boundary principal-axis angle: {_dxf_angle:.1f} deg (longest edge "
                      f"{_dxf_long_edge:.1f}m — should be close to the ~163m long side found earlier)")
                print(f"  Rotating DXF shape by {_rotate_by:.1f} deg and anchoring to the OSM centroid.")

                _final_utm_coords = _rotate_translate_polygon(
                    _boundary_candidate["vertices"], _rotate_by, _anchor_centroid)

                from shapely.geometry import Polygon as _ShapelyPolygon
                _final_utm_polygon = _ShapelyPolygon(_final_utm_coords)
                _final_gdf = gpd.GeoDataFrame({"geometry": [_final_utm_polygon]}, crs=_local_utm_crs).to_crs("EPSG:4326")
                _dxf_boundary_polygon = _final_gdf.geometry.iloc[0]
                _centroid = _dxf_boundary_polygon.centroid

                SITE = Site(
                    name=SITE.name if "SITE" in dir() else "Al Safa Park 2 (Annex-1)",
                    centroid=(_centroid.y, _centroid.x),
                    boundary=_dxf_boundary_polygon,
                    bbox=_dxf_boundary_polygon.bounds,
                    source_method="annex1_dxf_shape_osm_anchored_position",
                    notes=[f"Real as-built shape/dimensions from Annex-1 DXF ({_boundary_candidate['area_m2']:,.0f} m2, "
                           f"SOW states ~15,000 m2), positioned and oriented using {_anchor_source} as the "
                           f"real-world anchor (rotated {_rotate_by:.1f} deg to match its principal axis). "
                           f"VISUALLY VERIFY against satellite imagery below before final submission."],
                )
                print(f"\n✓ SITE overridden: DXF shape + OSM-anchored position/rotation.")
                print(f"  New centroid: {SITE.centroid}")
                print(f"  Real area: {SITE.area_m2():,.0f} m2")
            except Exception as e:
                print(f"⚠️  Anchoring transform failed ({e.__class__.__name__}: {str(e)[:200]}) — "
                      f"SITE keeps whatever Module 01 already resolved.")
else:
    print("DXF boundary import skipped (USE_DXF_BOUNDARY=False or file not found this session).")

Parsing Annex-1 DXF (pure Python, no external CAD library)...
  92 LWPOLYLINE entities found.
  Boundary candidate: 14,841 m2 (target 15,000 m2, 1.1% off)
  Anchor: existing SITE.boundary (source_method=ai_assisted_boundary_resolver)
    anchor principal-axis angle: 53.9 deg, centroid (UTM): (320784.3, 2783368.6)
  DXF boundary principal-axis angle: 54.4 deg (longest edge 163.4m — should be close to the ~163m long side found earlier)
  Rotating DXF shape by -0.5 deg and anchoring to the OSM centroid.

✓ SITE overridden: DXF shape + OSM-anchored position/rotation.
  New centroid: (25.155682532451443, 55.22193659898959)
  Real area: 14,841 m2


In [ ]:
# @title Quick visual check — plot the Annex-1 boundary if it was just applied
if USE_DXF_BOUNDARY and globals().get("_dxf_boundary_polygon") is not None and folium:
    _m = folium.Map(location=SITE.centroid, zoom_start=17, tiles="Esri.WorldImagery")
    folium.GeoJson(
        gpd.GeoSeries([SITE.boundary]).__geo_interface__ if gpd else None,
        style_function=lambda x: {"color": "#FF6D00", "weight": 3, "fillOpacity": 0.15},
        name="Annex-1 real boundary",
    ).add_to(_m)
    folium.Marker(SITE.centroid, tooltip="Georeferenced centroid", icon=folium.Icon(color="orange")).add_to(_m)
    display(_m)
    print("Compare this orange outline against the satellite imagery underneath — it should sit "
          "directly on the park's visible tree canopy/paths. If it's offset, the georeference "
          "needs correction before anything downstream trusts it.")

Compare this orange outline against the satellite imagery underneath — it should sit directly on the park's visible tree canopy/paths. If it's offset, the georeference needs correction before anything downstream trusts it.


---
# Module 02 — Data Collection

Retrieves GIS and environmental data for the resolved `SITE` from every configured source, and stores each
result as a **tagged `DataLayer`** — never merged across sources. This keeps every downstream score traceable
back to exactly which source produced the number that drove it, and lets the scoring engine (Module 06)
choose a source-priority order per criterion rather than silently blending data of different quality.

### Sources implemented in this module

| Theme | Sources tried (in order added to the store) | Needs a key? |
|---|---|---|
| Basemap imagery | Google Static Maps, ESRI World Imagery, OSM/Contextily | Google: yes. Others: no |
| Roads / sidewalks / bike routes / transit | OSMnx (OSM), Google Roads/Places | OSM: no. Google: yes |
| Buildings (footprints) | OSM, ESRI/ArcGIS Living Atlas | No / Optional |
| Land cover / vegetation / tree canopy | Google Earth Engine (Sentinel-2/Dynamic World), ESRI Land Cover | Yes (GEE project) / Optional |
| Elevation / slope / aspect | SRTM via the `elevation` package (free), GEE (if available) | No / Optional |
| Hydrology (rivers, lakes, flood zones) | OSM (rivers/lakes), FEMA/OpenFEMA where in the US, GEE JRC Global Surface Water | No / No / Optional |
| Points of interest | OSM (Overpass via OSMnx), Google Places | No / Optional |
| Climate (temp, rainfall, wind, solar, humidity) | Open-Meteo (free, no key), NASA POWER (free, no key) | No |

Every connector function follows the same contract: it takes the `Site`, tries its primary source,
catches failures, tries the next fallback, and always returns a `DataLayer` — even an empty one with
`status="unavailable"` — so the pipeline never breaks on a missing dataset; it just lowers confidence later.


In [ ]:
# @title Data layer & store structures

@dataclass
class DataLayer:
    theme: str                     # e.g. "roads", "tree_canopy", "climate"
    source: str                     # e.g. "osm", "google_maps", "esri", "gee", "open_meteo"
    status: str                      # "ok" | "empty" | "unavailable" | "error"
    data: Any = None                  # GeoDataFrame, dict, DataFrame, raster array, etc. — theme-dependent
    fetched_at: str = field(default_factory=lambda: dt.datetime.now(dt.timezone.utc).isoformat())
    notes: list = field(default_factory=list)
    error: Optional[str] = None

    def is_usable(self) -> bool:
        return self.status == "ok" and self.data is not None


class DataStore:
    '''Keyed by (theme, source). Never overwrites; each source's result for a theme is kept separately.'''

    def __init__(self):
        self._layers: dict[tuple[str, str], DataLayer] = {}

    def add(self, layer: DataLayer):
        self._layers[(layer.theme, layer.source)] = layer
        icon = {"ok": "✓", "empty": "·", "unavailable": "–", "error": "✗"}.get(layer.status, "?")
        msg = f"  {icon} [{layer.theme:22s}] {layer.source:14s} -> {layer.status}"
        if layer.error:
            msg += f" ({layer.error})"
        print(msg)

    def get(self, theme: str, source: str) -> Optional[DataLayer]:
        return self._layers.get((theme, source))

    def get_best(self, theme: str, source_priority: list[str]) -> Optional[DataLayer]:
        '''Returns the first usable layer for theme, walking source_priority in order.'''
        for source in source_priority:
            layer = self.get(theme, source)
            if layer and layer.is_usable():
                return layer
        return None

    def themes(self) -> list[str]:
        return sorted(set(t for t, _ in self._layers.keys()))

    def sources_for(self, theme: str) -> list[str]:
        return [s for t, s in self._layers.keys() if t == theme]

    def summary(self) -> pd.DataFrame:
        rows = []
        for (theme, source), layer in sorted(self._layers.items()):
            detail = layer.error if layer.status == "error" and layer.error else (
                "; ".join(layer.notes) if layer.notes else "")
            rows.append({
                "theme": theme,
                "source": source,
                "status": layer.status,
                "notes": detail,
            })
        return pd.DataFrame(rows)

    def all(self):
        return list(self._layers.values())


STORE = DataStore()

# OSM_FETCH_TIMEOUT_S, the osmnx.settings configuration, and _osmnx_features() are now defined
# in Module 00 (Setup) instead of here, so Module 01's boundary resolver can also use them
# regardless of cell execution order. They behave identically to before - only the location moved.


## Basemap imagery

In [ ]:
# @title Connector: basemap imagery (Google Static Maps -> ESRI World Imagery -> OSM/Contextily)
import requests
import time

def fetch_basemap(site: Site, store: DataStore, zoom: int = 18, size: str = "640x640"):
    theme = "basemap_imagery"

    # 1. Google Static Maps (best quality, needs key)
    if API_KEYS.get("google_maps"):
        try:
            params = {
                "center": f"{site.centroid[0]},{site.centroid[1]}",
                "zoom": zoom,
                "size": size,
                "maptype": "satellite",
                "key": API_KEYS["google_maps"],
            }
            resp = requests.get("https://maps.googleapis.com/maps/api/staticmap", params=params, timeout=15)
            if resp.status_code == 200 and resp.headers.get("content-type", "").startswith("image"):
                store.add(DataLayer(theme, "google_static_maps", "ok", data=resp.content,
                                     notes=[f"zoom={zoom}, size={size}"]))
            else:
                store.add(DataLayer(theme, "google_static_maps", "error", error=f"HTTP {resp.status_code}"))
        except Exception as e:
            store.add(DataLayer(theme, "google_static_maps", "error", error=str(e)))
    else:
        store.add(DataLayer(theme, "google_static_maps", "unavailable", notes=["No GOOGLE_MAPS_API_KEY configured"]))

    # 2. ESRI World Imagery (free, no key, via contextily basemap provider)
    if contextily:
        try:
            store.add(DataLayer(theme, "esri_world_imagery", "ok",
                                 data="esri_world_imagery_provider",  # actual tile fetch happens on-demand when plotting
                                 notes=["Tiles fetched lazily via contextily when maps are rendered."]))
        except Exception as e:
            store.add(DataLayer(theme, "esri_world_imagery", "error", error=str(e)))
    else:
        store.add(DataLayer(theme, "esri_world_imagery", "unavailable", notes=["contextily not installed"]))

    # 3. OSM raster tiles (always-available fallback, no key)
    if contextily:
        store.add(DataLayer(theme, "osm_tiles", "ok", data="osm_provider",
                             notes=["Tiles fetched lazily via contextily when maps are rendered."]))
    else:
        store.add(DataLayer(theme, "osm_tiles", "unavailable", notes=["contextily not installed"]))


fetch_basemap(SITE, STORE)


  ✓ [basemap_imagery       ] google_static_maps -> ok
  ✓ [basemap_imagery       ] esri_world_imagery -> ok
  ✓ [basemap_imagery       ] osm_tiles      -> ok


## Transportation network (roads, sidewalks, bike routes, transit)

In [ ]:
# @title Connector: transportation network (OSMnx primary; Google Roads/Places as supplement)

def fetch_transportation(site: Site, store: DataStore):
    theme_roads = "roads"
    theme_bike = "bike_routes"
    theme_transit = "transit"

    # --- OSM via OSMnx: free, no key, primary source for network geometry ---
    if osmnx and site.boundary is not None:
        try:
            polygon = site.boundary
            with hard_timeout(OSM_FETCH_TIMEOUT_S):
                graph = osmnx.graph_from_polygon(polygon, network_type="all", retain_all=True, truncate_by_edge=True)
                edges = osmnx.graph_to_gdfs(graph, nodes=False)
            store.add(DataLayer(theme_roads, "osm", "ok", data=edges,
                                 notes=[f"{len(edges)} road/path segments from OSM within site boundary"]))

            bike_mask = pd.Series(False, index=edges.index)
            if "highway" in edges.columns:
                bike_mask = bike_mask | edges["highway"].astype(str).str.contains("cycle", na=False)
            if "bicycle" in edges.columns:
                bike_mask = bike_mask | edges["bicycle"].notna()
            bike_edges = edges[bike_mask]
            store.add(DataLayer(theme_bike, "osm", "ok" if len(bike_edges) else "empty", data=bike_edges,
                                 notes=[f"{len(bike_edges)} cycle-tagged segments"]))
        except HardTimeoutError as e:
            msg = f"{e} (Overpass API may be unreachable from this network)"
            store.add(DataLayer(theme_roads, "osm", "error", error=msg))
            store.add(DataLayer(theme_bike, "osm", "error", error=msg))
        except Exception as e:
            store.add(DataLayer(theme_roads, "osm", "error", error=str(e)))
            store.add(DataLayer(theme_bike, "osm", "error", error=str(e)))
    else:
        reason = "osmnx not installed" if not osmnx else "site boundary unavailable"
        store.add(DataLayer(theme_roads, "osm", "unavailable", notes=[reason]))
        store.add(DataLayer(theme_bike, "osm", "unavailable", notes=[reason]))

    # --- OSM public transit stops (from POI-style tags via OSMnx features) ---
    if osmnx and site.boundary is not None:
        try:
            transit_tags = {"public_transport": True, "railway": "station", "highway": "bus_stop"}
            transit = _osmnx_features(site.boundary, transit_tags)
            store.add(DataLayer(theme_transit, "osm", "ok" if len(transit) else "empty", data=transit,
                                 notes=[f"{len(transit)} transit-related features"]))
        except HardTimeoutError as e:
            store.add(DataLayer(theme_transit, "osm", "error", error=f"{e} (Overpass API may be unreachable)"))
        except Exception as e:
            store.add(DataLayer(theme_transit, "osm", "error", error=str(e)))
    else:
        store.add(DataLayer(theme_transit, "osm", "unavailable", notes=["osmnx not installed or no boundary"]))

    # --- Google Roads/Places supplement (only if key present; used for validation/POI richness later) ---
    if API_KEYS.get("google_maps"):
        store.add(DataLayer(theme_roads, "google_maps", "empty",
                             notes=["Google Roads API requires paths snapped to existing GPS traces; "
                                    "not applicable for area-based road extraction. Use Places API for "
                                    "road-adjacent POIs instead (see POI connector)."]))
    else:
        store.add(DataLayer(theme_roads, "google_maps", "unavailable", notes=["No GOOGLE_MAPS_API_KEY configured"]))


fetch_transportation(SITE, STORE)


  ✓ [roads                 ] osm            -> ok
  · [bike_routes           ] osm            -> empty
  · [transit               ] osm            -> empty
  · [roads                 ] google_maps    -> empty


## Buildings & urban fabric

In [ ]:
# @title Connector: building footprints (OSM primary; ESRI/ArcGIS Living Atlas optional)

def fetch_buildings(site: Site, store: DataStore):
    theme = "buildings"

    if osmnx and site.boundary is not None:
        try:
            buildings = _osmnx_features(site.boundary, {"building": True})
            store.add(DataLayer(theme, "osm", "ok" if len(buildings) else "empty", data=buildings,
                                 notes=[f"{len(buildings)} building footprints from OSM"]))
        except HardTimeoutError as e:
            store.add(DataLayer(theme, "osm", "error", error=f"{e} (Overpass API may be unreachable)"))
        except Exception as e:
            store.add(DataLayer(theme, "osm", "error", error=str(e)))
    else:
        store.add(DataLayer(theme, "osm", "unavailable", notes=["osmnx not installed or no boundary"]))

    # ESRI / ArcGIS Living Atlas building layers (optional, needs arcgis package + auth for premium layers;
    # some Living Atlas layers are public and queryable without login via REST)
    if arcgis:
        try:
            from arcgis.gis import GIS
            with hard_timeout(15):
                gis = GIS()  # anonymous session - works for public layers only
            store.add(DataLayer(theme, "esri_living_atlas", "empty",
                                 notes=["Anonymous ArcGIS session created. Querying specific Living Atlas "
                                        "building layers requires a layer item ID - wire up a specific "
                                        "Living Atlas dataset URL here if your project needs it."]))
        except HardTimeoutError as e:
            store.add(DataLayer(theme, "esri_living_atlas", "error",
                                 error=f"{e} (ArcGIS Online may be unreachable from this network)"))
        except Exception as e:
            store.add(DataLayer(theme, "esri_living_atlas", "error",
                                 error=f"{e.__class__.__name__}: {str(e)[:150]} "
                                       f"(ArcGIS Online anonymous session failed - check network access to arcgis.com)"))
    else:
        store.add(DataLayer(theme, "esri_living_atlas", "unavailable", notes=["arcgis package not installed"]))


fetch_buildings(SITE, STORE)


  ✓ [buildings             ] osm            -> ok
  · [buildings             ] esri_living_atlas -> empty


## Land cover, vegetation & tree canopy

In [ ]:
# @title Connector: land cover / vegetation / tree canopy (Earth Engine primary; OSM green-space fallback)

_GEE_INIT_CACHE = {"attempted": False, "success": False}

def _gee_initialize():
    '''Attempts Earth Engine init, then interactive auth as a fallback (works in a real
    Colab UI where ee.Authenticate() can open an OAuth flow). Result is cached after the
    first attempt so repeated connector calls do not re-trigger a doomed auth flow in
    non-interactive contexts. Returns True on success.'''
    if _GEE_INIT_CACHE["attempted"]:
        return _GEE_INIT_CACHE["success"]
    _GEE_INIT_CACHE["attempted"] = True

    if not ee:
        return False

    project = API_KEYS.get("gee_project_id")
    try:
        ee.Initialize(project=project) if project else ee.Initialize()
        _GEE_INIT_CACHE["success"] = True
        return True
    except Exception:
        try:
            ee.Authenticate()
            ee.Initialize(project=project) if project else ee.Initialize()
            _GEE_INIT_CACHE["success"] = True
            return True
        except Exception as e:
            logger.warning(f"Earth Engine authentication/initialization failed ({e.__class__.__name__}): "
                            f"{str(e)[:150]}. Earth-Engine-backed layers will be marked unavailable for "
                            f"the rest of this run.")
            _GEE_INIT_CACHE["success"] = False
            return False


def fetch_land_cover(site: Site, store: DataStore):
    theme_lc = "land_cover"
    theme_canopy = "tree_canopy"
    theme_impervious = "impervious_surfaces"

    gee_ready = _gee_initialize() if ee else False

    if gee_ready and site.boundary is not None:
        try:
            coords = list(site.boundary.exterior.coords) if hasattr(site.boundary, "exterior") else None
            if coords:
                aoi = ee.Geometry.Polygon([coords])
                # Dynamic World V1 gives near-real-time 10m land cover with a "trees" probability band
                dw = (ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
                        .filterBounds(aoi)
                        .filterDate(ee.Date(dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")).advance(-1, "year"),
                                    ee.Date(dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")))
                        .median())
                store.add(DataLayer(theme_lc, "gee_dynamic_world", "ok", data={"image": dw, "aoi": aoi},
                                     notes=["Dynamic World V1 median composite, last 12 months, 10m resolution"]))
                store.add(DataLayer(theme_canopy, "gee_dynamic_world", "ok",
                                     data={"image": dw.select("trees"), "aoi": aoi},
                                     notes=["'trees' probability band from Dynamic World"]))
                store.add(DataLayer(theme_impervious, "gee_dynamic_world", "ok",
                                     data={"image": dw.select("built"), "aoi": aoi},
                                     notes=["'built' probability band from Dynamic World, used as impervious proxy"]))
            else:
                raise ValueError("Site boundary is not a simple polygon; cannot build an Earth Engine AOI.")
        except Exception as e:
            store.add(DataLayer(theme_lc, "gee_dynamic_world", "error", error=str(e)))
            store.add(DataLayer(theme_canopy, "gee_dynamic_world", "error", error=str(e)))
            store.add(DataLayer(theme_impervious, "gee_dynamic_world", "error", error=str(e)))
    else:
        reason = "Earth Engine not authenticated/available" if not gee_ready else "no site boundary"
        for theme in (theme_lc, theme_canopy, theme_impervious):
            store.add(DataLayer(theme, "gee_dynamic_world", "unavailable", notes=[reason]))

    # Free fallback: OSM tagged green space / natural=wood / leisure=park as a coarse canopy/vegetation proxy
    if osmnx and site.boundary is not None:
        try:
            green_tags = {"landuse": ["forest", "grass", "meadow"], "natural": ["wood", "tree_row", "scrub"],
                          "leisure": ["park", "garden"]}
            green = _osmnx_features(site.boundary, green_tags)
            store.add(DataLayer(theme_canopy, "osm_greenspace_proxy", "ok" if len(green) else "empty", data=green,
                                 notes=[f"{len(green)} green-space features from OSM tags (coarse proxy, not true canopy %)"]))
        except HardTimeoutError as e:
            store.add(DataLayer(theme_canopy, "osm_greenspace_proxy", "error", error=f"{e} (Overpass API may be unreachable)"))
        except Exception as e:
            store.add(DataLayer(theme_canopy, "osm_greenspace_proxy", "error", error=str(e)))
    else:
        store.add(DataLayer(theme_canopy, "osm_greenspace_proxy", "unavailable", notes=["osmnx not installed or no boundary"]))


fetch_land_cover(SITE, STORE)


  ✓ [land_cover            ] gee_dynamic_world -> ok
  ✓ [tree_canopy           ] gee_dynamic_world -> ok
  ✓ [impervious_surfaces   ] gee_dynamic_world -> ok
  ✓ [tree_canopy           ] osm_greenspace_proxy -> ok


## Terrain: elevation, slope, aspect

In [ ]:
!apt-get install -y -qq gdal-bin

Selecting previously unselected package python3-numpy.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../python3-numpy_1%3a1.21.5-1ubuntu22.04.1_amd64.deb ...
Unpacking python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Selecting previously unselected package python3-gdal.
Preparing to unpack .../python3-gdal_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking python3-gdal (3.8.4+dfsg-1~jammy0) ...
Selecting previously unselected package gdal-bin.
Preparing to unpack .../gdal-bin_3.8.4+dfsg-1~jammy0_amd64.deb ...
Unpacking gdal-bin (3.8.4+dfsg-1~jammy0) ...
Setting up python3-numpy (1:1.21.5-1ubuntu22.04.1) ...
Setting up python3-gdal (3.8.4+dfsg-1~jammy0) ...
Setting up gdal-bin (3.8.4+dfsg-1~jammy0) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
!make --version

GNU Make 4.3
Built for x86_64-pc-linux-gnu
Copyright (C) 1988-2020 Free Software Foundation, Inc.
License GPLv3+: GNU GPL version 3 or later <http://gnu.org/licenses/gpl.html>
This is free software: you are free to change and redistribute it.
There is NO WARRANTY, to the extent permitted by law.


In [ ]:
# @title Connector: terrain (SRTM via `elevation` package primary; GEE DEM optional)
# Note: uses the shared hard_timeout() helper defined in the Setup module - it guards
# against the elevation package's underlying curl/wget calls hanging indefinitely with
# no usable timeout of their own (a real risk on slow or restricted networks).

def fetch_terrain(site: Site, store: DataStore, download_timeout_s: int = 45):
    theme_elev = "elevation"
    theme_slope = "slope"
    theme_aspect = "aspect"

    # 1. SRTM via the `elevation` package - free, no key, downloads a clipped DEM tile.
    # Wrapped in a hard timeout since the underlying curl/wget calls can hang indefinitely
    # on restricted or slow networks rather than failing fast.
    try:
        import elevation as elevation_pkg
        minx, miny, maxx, maxy = site.bbox
        dem_path = "/tmp/site_dem.tif"
        with hard_timeout(download_timeout_s):
            elevation_pkg.clip(bounds=(minx, miny, maxx, maxy), output=dem_path, product="SRTM1")
        store.add(DataLayer(theme_elev, "srtm", "ok", data=dem_path,
                             notes=["SRTM 30m DEM clipped to site bbox via the `elevation` package"]))

        if rasterio:
            with rasterio.open(dem_path) as src:
                elev_arr = src.read(1).astype(float)
                px_size_x, px_size_y = src.res
                gy, gx = np.gradient(elev_arr, px_size_y, px_size_x)
                slope_arr = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))
                aspect_arr = np.degrees(np.arctan2(-gx, gy))
                aspect_arr = np.where(aspect_arr < 0, 90 - aspect_arr, np.where(aspect_arr > 90, 360 - aspect_arr + 90, 90 - aspect_arr))
                store.add(DataLayer(theme_slope, "srtm_derived", "ok", data=slope_arr,
                                     notes=["Slope in degrees, derived from SRTM DEM via numpy gradient"]))
                store.add(DataLayer(theme_aspect, "srtm_derived", "ok", data=aspect_arr,
                                     notes=["Aspect in degrees (0=N), derived from SRTM DEM"]))
        else:
            store.add(DataLayer(theme_slope, "srtm_derived", "unavailable", notes=["rasterio not installed"]))
            store.add(DataLayer(theme_aspect, "srtm_derived", "unavailable", notes=["rasterio not installed"]))
    except HardTimeoutError as e:
        msg = f"{e} (SRTM download host may be unreachable from this network)"
        store.add(DataLayer(theme_elev, "srtm", "error", error=msg))
        store.add(DataLayer(theme_slope, "srtm_derived", "error", error=msg))
        store.add(DataLayer(theme_aspect, "srtm_derived", "error", error=msg))
    except Exception as e:
        store.add(DataLayer(theme_elev, "srtm", "error", error=str(e)))
        store.add(DataLayer(theme_slope, "srtm_derived", "error", error=str(e)))
        store.add(DataLayer(theme_aspect, "srtm_derived", "error", error=str(e)))

    # 1b. Fallbacks: pure-HTTP elevation grid APIs (free, no key, no local GDAL/make
    # toolchain required). Used only if the `elevation` package's local DEM build failed
    # above, which can happen in containerized environments missing system `make`/GDAL
    # command-line tools. We sample a small grid across the site bbox to approximate a
    # coarse DEM. Two providers are tried in turn since public demo instances of either
    # can rate-limit or block traffic unpredictably.

    def _sample_elevation_grid(url_builder, source_name, grid_n=6):
        '''Tries one HTTP elevation grid provider. Returns True if it succeeded (and adds
        the elevation/slope/aspect layers to the store), False otherwise (adds nothing,
        caller can try the next provider).'''
        minx, miny, maxx, maxy = site.bbox
        lats = np.linspace(miny, maxy, grid_n)
        lons = np.linspace(minx, maxx, grid_n)
        try:
            with hard_timeout(30):
                elevations = url_builder(lats, lons)
            if elevations is None:
                return False
            elev_grid = np.array(elevations).reshape(grid_n, grid_n)
            store.add(DataLayer(theme_elev, source_name, "ok", data=elev_grid,
                                 notes=[f"{grid_n}x{grid_n} coarse elevation grid via {source_name} "
                                        f"(SRTM-derived point samples) - fallback used because the local "
                                        f"`elevation` package DEM build failed"]))
            dlat_m = (maxy - miny) / (grid_n - 1) * 111_320
            dlon_m = (maxx - minx) / (grid_n - 1) * 111_320 * math.cos(math.radians((miny + maxy) / 2))
            gy, gx = np.gradient(elev_grid, dlat_m, dlon_m)
            slope_grid = np.degrees(np.arctan(np.sqrt(gx**2 + gy**2)))
            aspect_grid = np.degrees(np.arctan2(-gx, gy))
            aspect_grid = np.where(aspect_grid < 0, 90 - aspect_grid,
                                    np.where(aspect_grid > 90, 360 - aspect_grid + 90, 90 - aspect_grid))
            store.add(DataLayer(theme_slope, f"{source_name}_derived", "ok", data=slope_grid,
                                 notes=[f"Slope in degrees, derived from a coarse {grid_n}x{grid_n} "
                                        f"{source_name} grid - lower resolution than true SRTM raster"]))
            store.add(DataLayer(theme_aspect, f"{source_name}_derived", "ok", data=aspect_grid,
                                 notes=["Aspect in degrees (0=N), derived from the same coarse grid"]))
            return True
        except HardTimeoutError as e:
            store.add(DataLayer(theme_elev, source_name, "error", error=f"{e} ({source_name} may be unreachable)"))
            return False
        except Exception as e:
            store.add(DataLayer(theme_elev, source_name, "error", error=str(e)))
            return False

    def _opentopodata_fetch(lats, lons):
        locations = "|".join(f"{lat},{lon}" for lat in lats for lon in lons)
        resp = requests.post("https://api.opentopodata.org/v1/srtm30m", json={"locations": locations}, timeout=25)
        if resp.status_code != 200:
            raise RuntimeError(f"HTTP {resp.status_code}")
        payload = resp.json()
        if payload.get("status") != "OK":
            raise RuntimeError(f"API status: {payload.get('status')}")
        return [r["elevation"] for r in payload["results"]]

    def _openelevation_fetch(lats, lons):
        locations = [{"latitude": float(lat), "longitude": float(lon)} for lat in lats for lon in lons]
        resp = requests.post("https://api.open-elevation.com/api/v1/lookup", json={"locations": locations}, timeout=25)
        if resp.status_code != 200:
            raise RuntimeError(f"HTTP {resp.status_code}")
        payload = resp.json()
        return [r["elevation"] for r in payload["results"]]

    if not store.get(theme_elev, "srtm") or not store.get(theme_elev, "srtm").is_usable():
        if not _sample_elevation_grid(_opentopodata_fetch, "opentopodata"):
            _sample_elevation_grid(_openelevation_fetch, "open_elevation")

    # 2. GEE DEM (SRTM or higher-res, if you have Earth Engine access) - optional cross-check / alternative
    gee_ready = _gee_initialize() if ee else False
    if gee_ready and site.boundary is not None:
        try:
            coords = list(site.boundary.exterior.coords) if hasattr(site.boundary, "exterior") else None
            aoi = ee.Geometry.Polygon([coords]) if coords else None
            dem = ee.Image("USGS/SRTMGL1_003")
            store.add(DataLayer(theme_elev, "gee_srtm", "ok", data={"image": dem, "aoi": aoi},
                                 notes=["USGS SRTM GL1 30m via Earth Engine (kept separate from the downloaded-tile version)"]))
        except Exception as e:
            store.add(DataLayer(theme_elev, "gee_srtm", "error", error=str(e)))
    else:
        store.add(DataLayer(theme_elev, "gee_srtm", "unavailable",
                             notes=["Earth Engine not authenticated/available" if not gee_ready else "no boundary"]))


fetch_terrain(SITE, STORE)


  ✓ [elevation             ] srtm           -> ok
  ✓ [slope                 ] srtm_derived   -> ok
  ✓ [aspect                ] srtm_derived   -> ok
  ✓ [elevation             ] gee_srtm       -> ok


## Hydrology

In [ ]:
# @title Connector: hydrology (OSM rivers/lakes; GEE JRC surface water; FEMA flood zones where in the US)

def fetch_hydrology(site: Site, store: DataStore):
    theme_water = "water_bodies"
    theme_flood = "flood_zones"

    if osmnx and site.boundary is not None:
        try:
            water_tags = {"natural": ["water", "wetland"], "waterway": True}
            water = _osmnx_features(site.boundary, water_tags)
            store.add(DataLayer(theme_water, "osm", "ok" if len(water) else "empty", data=water,
                                 notes=[f"{len(water)} water-related features from OSM"]))
        except HardTimeoutError as e:
            store.add(DataLayer(theme_water, "osm", "error", error=f"{e} (Overpass API may be unreachable)"))
        except Exception as e:
            store.add(DataLayer(theme_water, "osm", "error", error=str(e)))
    else:
        store.add(DataLayer(theme_water, "osm", "unavailable", notes=["osmnx not installed or no boundary"]))

    gee_ready = _gee_initialize() if ee else False
    if gee_ready and site.boundary is not None:
        try:
            coords = list(site.boundary.exterior.coords) if hasattr(site.boundary, "exterior") else None
            aoi = ee.Geometry.Polygon([coords]) if coords else None
            gsw = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")
            store.add(DataLayer(theme_water, "gee_jrc_gsw", "ok", data={"image": gsw, "aoi": aoi},
                                 notes=["JRC Global Surface Water occurrence layer, 1984-2021, 30m"]))
        except Exception as e:
            store.add(DataLayer(theme_water, "gee_jrc_gsw", "error", error=str(e)))
    else:
        store.add(DataLayer(theme_water, "gee_jrc_gsw", "unavailable",
                             notes=["Earth Engine not authenticated/available" if not gee_ready else "no boundary"]))

    # FEMA flood zones - free, no key, but US-only. We attempt it and mark unavailable outside the US automatically
    # via an empty response rather than trying to geo-detect country client-side.
    try:
        minx, miny, maxx, maxy = site.bbox
        fema_url = "https://hazards.fema.gov/gis/nfhl/rest/services/public/NFHL/MapServer/28/query"
        params = {
            "geometry": f"{minx},{miny},{maxx},{maxy}",
            "geometryType": "esriGeometryEnvelope",
            "inSR": "4326",
            "spatialRel": "esriSpatialRelIntersects",
            "outFields": "FLD_ZONE,ZONE_SUBTY",
            "f": "geojson",
        }
        resp = requests.get(fema_url, params=params, timeout=15)
        if resp.status_code == 200 and resp.json().get("features"):
            store.add(DataLayer(theme_flood, "fema_nfhl", "ok", data=resp.json(),
                                 notes=[f"{len(resp.json()['features'])} FEMA flood zone features (US only)"]))
        else:
            store.add(DataLayer(theme_flood, "fema_nfhl", "empty",
                                 notes=["No FEMA flood zone features returned - site may be outside the US or in an unmapped area"]))
    except Exception as e:
        store.add(DataLayer(theme_flood, "fema_nfhl", "error", error=str(e)))


fetch_hydrology(SITE, STORE)


  ✗ [water_bodies          ] osm            -> error (HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7a538cea4410>: Failed to establish a new connection: [Errno 111] Connection refused')))
  ✓ [water_bodies          ] gee_jrc_gsw    -> ok
  · [flood_zones           ] fema_nfhl      -> empty


## Points of interest

In [ ]:
# @title Connector: points of interest (OSM primary; Google Places optional supplement)

POI_CATEGORIES_OSM = {
    # Merged from the original 6-category set + a richer 32-category taxonomy provided by
    # the user, now fully included (23 categories). The original 10 "generic urban fabric"
    # categories (shopping, markets, fuel_and_ev, tourism, hotels, financial_services,
    # personal_services, libraries, museums, public_open_space) are collected and
    # preprocessed every run as available inventory for future scoring demand, even though
    # they aren't yet wired into Module 05/06 metrics. Categories already covered by a
    # separate Module 02 theme under a different name (natural/water features ->
    # `water_bodies`; public_transport -> `transit`) are still left out to avoid duplicate
    # collection.

    "schools": {
        "amenity": ["school", "kindergarten", "college", "university", "research_institute",
                     "language_school", "music_school", "driving_school"],
        "education": ["school", "kindergarten", "college", "university"],
    },
    "hospitals": {
        "amenity": ["hospital", "clinic", "doctors", "dentist", "pharmacy", "veterinary"],
        "healthcare": ["hospital", "clinic", "doctor", "dentist", "pharmacy", "physiotherapist",
                        "psychotherapist", "optometrist", "alternative", "laboratory"],
    },
    "restaurants": {
        "amenity": ["restaurant", "cafe", "fast_food", "food_court", "ice_cream", "biergarten",
                     "pub", "bar", "bbq", "bistro"],
    },
    "parks": {
        "leisure": ["park", "garden", "playground", "nature_reserve", "common", "recreation_ground",
                     "village_green", "grass", "meadow"],
    },
    "sports_facilities": {
        "leisure": ["sports_centre", "stadium", "fitness_centre", "pitch", "track", "swimming_pool",
                     "ice_rink", "golf_course", "miniature_golf", "horse_riding", "sports_hall"],
    },
    "cultural_buildings": {
        "amenity": ["theatre", "arts_centre", "community_centre", "social_centre", "library", "cinema"],
        "tourism": ["museum", "gallery"],
    },

    # --- New additions from the uploaded taxonomy, park-relevant ---
    "playgrounds": {
        "leisure": ["playground", "nature_play"],
    },
    "religious_buildings": {
        "amenity": ["place_of_worship"],
    },
    "government": {
        "amenity": ["townhall", "courthouse", "embassy", "police", "fire_station", "post_office"],
        "office": ["government"],
    },
    "emergency_services": {
        "amenity": ["police", "fire_station", "ambulance_station"],
    },
    "childcare": {
        "amenity": ["kindergarten", "childcare", "nursery"],
    },
    "community_facilities": {
        "amenity": ["community_centre", "social_centre", "shelter", "food_bank", "social_facility"],
    },
    "accessibility_amenities": {
        # Renamed from the uploaded taxonomy's "accessibility" to avoid confusion with the
        # unrelated `universal_accessibility_score` metric in Module 05 (sidewalk/curb-cut
        # accessibility) - this category is about park comfort amenities (benches, toilets,
        # drinking water), not mobility accessibility.
        "amenity": ["bench", "drinking_water", "toilets", "shelter", "shower"],
    },

    # --- Added on request: collected and preprocessed every run for future scoring demand,
    # but not yet wired into Module 05/06 metrics or scoring. Originally left out as "generic
    # urban fabric, not directly park-relevant" - kept here as available inventory instead. ---
    "shopping": {
        "shop": ["supermarket", "convenience", "department_store", "mall", "general", "clothes",
                  "shoes", "bakery", "butcher", "greengrocer", "electronics", "furniture",
                  "hardware", "books", "doityourself", "beauty", "chemist"],
    },
    "markets": {
        "amenity": ["marketplace"],
        "shop": ["supermarket", "greengrocer", "convenience"],
    },
    "fuel_and_ev": {
        "amenity": ["fuel", "charging_station"],
    },
    "tourism": {
        "tourism": ["attraction", "museum", "gallery", "viewpoint", "zoo", "theme_park", "aquarium",
                     "information", "hotel", "motel", "guest_house", "hostel"],
    },
    "hotels": {
        "tourism": ["hotel", "motel", "guest_house", "hostel", "apartment", "resort"],
    },
    "financial_services": {
        "amenity": ["bank", "atm", "bureau_de_change"],
    },
    "personal_services": {
        "shop": ["hairdresser", "beauty", "laundry", "tailor", "dry_cleaning"],
    },
    "libraries": {
        "amenity": ["library"],
    },
    "museums": {
        "tourism": ["museum"],
    },
    "public_open_space": {
        "leisure": ["park", "garden", "common", "recreation_ground", "village_green", "nature_reserve"],
        "place": ["square"],
    },
}

def fetch_poi(site: Site, store: DataStore, buffer_m: int = 800):
    '''POIs are searched in a wider buffer than the site boundary itself (POIs around a park matter for access).'''
    theme_prefix = "poi_"

    search_area = site.boundary
    if gpd and shapely and site.boundary is not None:
        try:
            gdf = gpd.GeoDataFrame({"geometry": [site.boundary]}, crs="EPSG:4326")
            utm_crs = gdf.estimate_utm_crs()
            search_area = gdf.to_crs(utm_crs).buffer(buffer_m).to_crs("EPSG:4326").iloc[0]
        except Exception as e:
            logger.warning(f"Could not buffer site for POI search, using raw boundary: {e}")

    for i, (category, tags) in enumerate(POI_CATEGORIES_OSM.items()):
        theme = f"{theme_prefix}{category}"
        if i > 0:
            # Overpass's public instance rate-limits rapid sequential queries from the same
            # client; a short pause between categories measurably reduces timeout/429 errors
            # compared to firing all 6 queries back-to-back.
            time.sleep(5)
        if osmnx and search_area is not None:
            try:
                pois = _osmnx_features(search_area, tags)
                store.add(DataLayer(theme, "osm", "ok" if len(pois) else "empty", data=pois,
                                     notes=[f"{len(pois)} '{category}' POIs within {buffer_m}m of site"]))
            except HardTimeoutError:
                # Overpass timeouts are often transient (temporary congestion/rate-limiting)
                # rather than a genuinely unreachable host - one retry after a short cooldown
                # recovers a meaningful fraction of these without much added worst-case time.
                logger.info(f"'{category}' POI query timed out, retrying once after a cooldown...")
                time.sleep(5)
                try:
                    pois = _osmnx_features(search_area, tags)
                    store.add(DataLayer(theme, "osm", "ok" if len(pois) else "empty", data=pois,
                                         notes=[f"{len(pois)} '{category}' POIs within {buffer_m}m of site (succeeded on retry)"]))
                except HardTimeoutError as e:
                    store.add(DataLayer(theme, "osm", "error", error=f"{e} (Overpass API may be unreachable, failed twice)"))
                except Exception as e:
                    store.add(DataLayer(theme, "osm", "error", error=f"{str(e)} (failed on retry)"))
            except Exception as e:
                store.add(DataLayer(theme, "osm", "error", error=str(e)))
        else:
            store.add(DataLayer(theme, "osm", "unavailable", notes=["osmnx not installed or no search area"]))

        # Google Places supplement (optional, needs key) - richer metadata (ratings, hours) where OSM lacks it
        if API_KEYS.get("google_maps"):
            try:
                gmaps_type_map = {
                    "schools": "school", "hospitals": "hospital", "restaurants": "restaurant",
                    "parks": "park", "sports_facilities": "gym", "cultural_buildings": "museum",
                    "playgrounds": "park",  # Google Places has no dedicated "playground" type; park is the closest match
                    "religious_buildings": "church",  # Google's type list is religion-specific (church/mosque/synagogue/hindu_temple);
                                                          # "church" is used as a default and will under-count non-Christian sites -
                                                          # OSM's place_of_worship tag (used for this category) has no such bias
                    "government": "local_government_office",
                    "emergency_services": "fire_station",  # closest single valid type; police/ambulance aren't separately queryable here
                    "childcare": "school",  # Google Places has no dedicated childcare/kindergarten type
                    "community_facilities": "local_government_office",  # no direct match; closest available
                    "accessibility_amenities": None,  # no Google Places equivalent at all (benches/toilets aren't a place type)
                    # --- new inventory categories ---
                    "shopping": "shopping_mall",  # closest single type for a broad retail category; under-counts standalone shops
                    "markets": "supermarket",
                    "fuel_and_ev": "gas_station",  # covers fuel but not EV charging specifically - Google has no dedicated "charging_station" legacy type
                    "tourism": "tourist_attraction",
                    "hotels": "lodging",
                    "financial_services": "bank",  # under-counts standalone ATMs, which aren't a separate legacy type
                    "personal_services": "hair_care",  # closest single type; under-counts laundry/tailor which have no dedicated type
                    "libraries": "library",
                    "museums": "museum",
                    "public_open_space": "park",
                }
                gmaps_type = gmaps_type_map.get(category, category)
                if gmaps_type is None:
                    store.add(DataLayer(theme, "google_places", "unavailable",
                                         notes=["No corresponding Google Places type exists for this category"]))
                    continue
                resp = requests.get(
                    "https://maps.googleapis.com/maps/api/place/nearbysearch/json",
                    params={
                        "location": f"{site.centroid[0]},{site.centroid[1]}",
                        "radius": buffer_m,
                        "type": gmaps_type,
                        "key": API_KEYS["google_maps"],
                    },
                    timeout=15,
                )
                data = resp.json()
                if data.get("status") == "OK":
                    store.add(DataLayer(theme, "google_places", "ok", data=data["results"],
                                         notes=[f"{len(data['results'])} results from Google Places Nearby Search"]))
                else:
                    store.add(DataLayer(theme, "google_places", "empty", notes=[f"status={data.get('status')}"]))
            except Exception as e:
                store.add(DataLayer(theme, "google_places", "error", error=str(e)))
        else:
            store.add(DataLayer(theme, "google_places", "unavailable", notes=["No GOOGLE_MAPS_API_KEY configured"]))


fetch_poi(SITE, STORE)


  ✗ [poi_schools           ] osm            -> error (HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7a538d336990>: Failed to establish a new connection: [Errno 111] Connection refused')))
  ✓ [poi_schools           ] google_places  -> ok
  ✗ [poi_hospitals         ] osm            -> error (HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7a538cea4690>: Failed to establish a new connection: [Errno 111] Connection refused')))
  ✓ [poi_hospitals         ] google_places  -> ok
  ✗ [poi_restaurants       ] osm            -> error (HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7a538cea4e10>: F

## Climate

In [ ]:
# @title Connector: climate (Open-Meteo + NASA POWER, both free/no-key)

def fetch_climate(site: Site, store: DataStore, years_back: int = 5):
    theme = "climate"
    lat, lon = site.centroid
    end_date = dt.date.today()
    start_date = end_date.replace(year=end_date.year - years_back)

    # 1. Open-Meteo climate/historical API - free, no key, good global coverage
    try:
        resp = requests.get(
            "https://archive-api.open-meteo.com/v1/archive",
            params={
                "latitude": lat, "longitude": lon,
                "start_date": start_date.isoformat(), "end_date": end_date.isoformat(),
                "daily": "temperature_2m_mean,precipitation_sum,windspeed_10m_max,shortwave_radiation_sum",
                "timezone": "auto",
            },
            timeout=20,
        )
        if resp.status_code == 200:
            payload = resp.json()
            daily = payload.get("daily", {})
            if daily:
                df = pd.DataFrame(daily)
                summary = {
                    "avg_temp_c": df["temperature_2m_mean"].mean() if "temperature_2m_mean" in df else None,
                    "total_precip_mm_annualized": (df["precipitation_sum"].sum() / years_back) if "precipitation_sum" in df else None,
                    "avg_max_windspeed_kmh": df["windspeed_10m_max"].mean() if "windspeed_10m_max" in df else None,
                    "avg_solar_radiation_mj_m2": df["shortwave_radiation_sum"].mean() if "shortwave_radiation_sum" in df else None,
                }
                store.add(DataLayer(theme, "open_meteo", "ok", data={"daily_df": df, "summary": summary},
                                     notes=[f"{years_back}-year daily archive, {len(df)} days"]))
            else:
                store.add(DataLayer(theme, "open_meteo", "empty", notes=["No daily data returned"]))
        else:
            store.add(DataLayer(theme, "open_meteo", "error", error=f"HTTP {resp.status_code}"))
    except Exception as e:
        store.add(DataLayer(theme, "open_meteo", "error", error=str(e)))

    # 2. NASA POWER - free, no key, good for solar radiation and humidity specifically
    try:
        resp = requests.get(
            "https://power.larc.nasa.gov/api/temporal/climatology/point",
            params={
                "latitude": lat, "longitude": lon,
                "community": "RE",
                "parameters": "T2M,PRECTOTCORR,WS10M,ALLSKY_SFC_SW_DWN,RH2M",
                "format": "JSON",
            },
            timeout=20,
        )
        if resp.status_code == 200:
            payload = resp.json()
            params_data = payload.get("properties", {}).get("parameter", {})
            if params_data:
                summary = {k: (v.get("ANN") if isinstance(v, dict) else None) for k, v in params_data.items()}
                store.add(DataLayer(theme, "nasa_power", "ok", data={"summary": summary},
                                     notes=["Climatology annual averages (multi-year), includes humidity + solar radiation"]))
            else:
                store.add(DataLayer(theme, "nasa_power", "empty", notes=["No parameter data returned"]))
        else:
            store.add(DataLayer(theme, "nasa_power", "error", error=f"HTTP {resp.status_code}"))
    except Exception as e:
        store.add(DataLayer(theme, "nasa_power", "error", error=str(e)))


fetch_climate(SITE, STORE)


  ✓ [climate               ] open_meteo     -> ok
  ✓ [climate               ] nasa_power     -> ok


## Collection summary

In [ ]:
# @title Data collection summary
summary_df = STORE.summary()
print(f"Collected {len(summary_df)} (theme, source) layers across {summary_df['theme'].nunique()} themes.")
print(f"  ok: {(summary_df.status=='ok').sum()}  |  empty: {(summary_df.status=='empty').sum()}  |  "
      f"unavailable: {(summary_df.status=='unavailable').sum()}  |  error: {(summary_df.status=='error').sum()}")
summary_df


Collected 68 (theme, source) layers across 37 themes.
  ok: 33  |  empty: 10  |  unavailable: 1  |  error: 24


,theme,source,status,notes
0,aspect,srtm_derived,ok,"Aspect in degrees (0=N), derived from SRTM DEM"
1,basemap_imagery,esri_world_imagery,ok,Tiles fetched lazily via contextily when maps ...
2,basemap_imagery,google_static_maps,ok,"zoom=18, size=640x640"
3,basemap_imagery,osm_tiles,ok,Tiles fetched lazily via contextily when maps ...
4,bike_routes,osm,empty,0 cycle-tagged segments
...,...,...,...,...
63,transit,osm,empty,0 transit-related features
64,tree_canopy,gee_dynamic_world,ok,'trees' probability band from Dynamic World
65,tree_canopy,osm_greenspace_proxy,ok,1 green-space features from OSM tags (coarse p...
66,water_bodies,gee_jrc_gsw,ok,"JRC Global Surface Water occurrence layer, 198..."


---
# Module 03 — GIS Preprocessing

Takes every raw `DataLayer` collected in Module 02 and brings it into a **consistent, analysis-ready form**:

- **Vector layers** (OSM GeoDataFrames: roads, buildings, POIs, water, transit) → reprojected from WGS84 to a
  local UTM CRS (so areas/lengths/distances downstream are in real meters, not degrees), then clipped/tagged
  against both the site boundary and the wider POI search buffer.
- **Earth Engine layers** (land cover, canopy, impervious surfaces, water occurrence, SRTM elevation) → these
  are still *lazy references* (an `ee.Image` + `ee.Geometry`, nothing has been downloaded yet). This module
  materializes them into local numpy arrays clipped to the site, with georeferencing metadata attached.
- **Already-local raster grids** (the Open-Topo-Data elevation/slope/aspect fallback) → wrapped in the same
  `RasterLayer` interface as the Earth Engine outputs, so Module 05 never needs to know which source produced
  a given raster.
- **Non-spatial layers** (climate summaries) → passed through unchanged; there's nothing to reproject or clip.

This keeps the same "never merge across sources" discipline from Module 02 — preprocessing transforms each
layer in place (new CRS, clipped extent, materialized pixels) but never blends two sources' data together.
The result is a new `ProcessedStore`, keyed the same way as `DataStore` (`(theme, source)`), so the scoring
engine in Module 06 can apply the same source-priority-list pattern on top of processed data.


In [ ]:
# @title Processed layer structures

@dataclass
class RasterLayer:
    '''Common interface for any raster-like result, regardless of whether it came from
    Earth Engine, Open-Topo-Data, or a downloaded SRTM tile. Always a 2D numpy array plus
    enough georeferencing to know what each cell covers.'''
    array: Any                  # 2D numpy array
    bounds: tuple                 # (minx, miny, maxx, maxy) in `crs`
    crs: str
    band_name: str = "value"
    pixel_size_m: Optional[float] = None   # approximate ground resolution, if known
    source_resolution_note: str = ""         # e.g. "10m Sentinel-2 derived" vs "coarse 6x6 point grid"

    def stats(self) -> dict:
        arr = np.asarray(self.array, dtype=float)
        valid = arr[~np.isnan(arr)] if np.issubdtype(arr.dtype, np.floating) else arr.flatten()
        if valid.size == 0:
            return {"min": None, "max": None, "mean": None, "std": None, "n_cells": 0}
        return {
            "min": float(np.nanmin(valid)), "max": float(np.nanmax(valid)),
            "mean": float(np.nanmean(valid)), "std": float(np.nanstd(valid)),
            "n_cells": int(valid.size),
        }


@dataclass
class ProcessedLayer:
    theme: str
    source: str
    kind: str                    # "vector" | "raster" | "scalar"
    status: str                   # "ok" | "empty" | "skipped" | "error"
    data: Any = None               # GeoDataFrame (vector) | RasterLayer (raster) | dict (scalar)
    crs: Optional[str] = None
    notes: list = field(default_factory=list)
    error: Optional[str] = None
    source_layer_status: str = ""   # the original DataLayer.status, kept for the audit trail

    def is_usable(self) -> bool:
        return self.status == "ok" and self.data is not None


class ProcessedStore:
    '''Mirrors DataStore's (theme, source) keying and never-merge discipline - see Module 02.'''

    def __init__(self):
        self._layers: dict[tuple[str, str], ProcessedLayer] = {}

    def add(self, layer: ProcessedLayer):
        self._layers[(layer.theme, layer.source)] = layer
        icon = {"ok": "✓", "empty": "·", "skipped": "–", "error": "✗"}.get(layer.status, "?")
        msg = f"  {icon} [{layer.theme:22s}] {layer.source:14s} -> {layer.status} ({layer.kind})"
        if layer.error:
            msg += f"  [{layer.error}]"
        print(msg)

    def get(self, theme: str, source: str) -> Optional[ProcessedLayer]:
        return self._layers.get((theme, source))

    def get_best(self, theme: str, source_priority: list[str]) -> Optional[ProcessedLayer]:
        for source in source_priority:
            layer = self.get(theme, source)
            if layer and layer.is_usable():
                return layer
        return None

    def summary(self) -> pd.DataFrame:
        rows = []
        for (theme, source), layer in sorted(self._layers.items()):
            detail = layer.error if layer.status == "error" and layer.error else (
                "; ".join(layer.notes) if layer.notes else "")
            rows.append({"theme": theme, "source": source, "kind": layer.kind,
                         "status": layer.status, "notes": detail})
        return pd.DataFrame(rows)

    def all(self):
        return list(self._layers.values())


PROCESSED = ProcessedStore()

# Determine the site's local UTM CRS once - every vector/raster reprojection in this
# module targets this same CRS so all downstream metric calculations are consistent.
SITE_UTM_CRS = None
if gpd and SITE.boundary is not None:
    try:
        _site_gdf = gpd.GeoDataFrame({"geometry": [SITE.boundary]}, crs="EPSG:4326")
        SITE_UTM_CRS = _site_gdf.estimate_utm_crs()
        print(f"Site UTM CRS: {SITE_UTM_CRS}")
    except Exception as e:
        logger.warning(f"Could not determine a local UTM CRS for the site ({e}); vector layers will remain in WGS84.")
else:
    logger.warning("No site boundary or geopandas available; vector layers will remain in WGS84.")


Site UTM CRS: EPSG:32640


## Vector layers: reproject + clip

In [ ]:
# @title Preprocess vector layers (roads, bike routes, transit, buildings, water, POIs)

# Themes whose primary geometry should be clipped to the site boundary itself (buildings,
# water, land features physically on/at the site). POI themes are intentionally NOT clipped
# to the boundary - they're valuable precisely because they're in the surrounding area, so
# we keep them at their original search-buffer extent and just reproject.
CLIP_TO_SITE_BOUNDARY = {"roads", "bike_routes", "transit", "buildings", "water_bodies", "tree_canopy"}

VECTOR_THEMES = (
    ["roads", "bike_routes", "transit", "buildings", "water_bodies", "tree_canopy"] +
    [f"poi_{cat}" for cat in POI_CATEGORIES_OSM.keys()]
)

def _reproject_and_clip_vector(gdf, theme: str) -> tuple:
    '''Returns (processed_gdf, notes_list). Reprojects to SITE_UTM_CRS if available,
    clips to the site boundary for themes in CLIP_TO_SITE_BOUNDARY.'''
    notes = []
    if gdf is None or len(gdf) == 0:
        return gdf, notes

    working = gdf.copy()
    if working.crs is None:
        working = working.set_crs("EPSG:4326")
        notes.append("Source layer had no CRS defined; assumed EPSG:4326")

    if SITE_UTM_CRS is not None:
        working = working.to_crs(SITE_UTM_CRS)
        notes.append(f"Reprojected to {SITE_UTM_CRS}")

    if theme in CLIP_TO_SITE_BOUNDARY and SITE.boundary is not None:
        try:
            boundary_gdf = gpd.GeoDataFrame({"geometry": [SITE.boundary]}, crs="EPSG:4326")
            if SITE_UTM_CRS is not None:
                boundary_gdf = boundary_gdf.to_crs(SITE_UTM_CRS)
            clipped = gpd.clip(working, boundary_gdf)
            notes.append(f"Clipped to site boundary ({len(working)} -> {len(clipped)} features)")
            working = clipped
        except Exception as e:
            notes.append(f"Clip to site boundary failed, keeping unclipped features: {e}")

    return working, notes


# Sources that are inherently raster-shaped even when their theme also has vector-shaped
# sources under the same name (mirror of VECTOR_ONLY_SOURCES above).
RASTER_ONLY_SOURCES = {"gee_dynamic_world", "gee_jrc_gsw", "gee_srtm", "srtm", "srtm_derived",
                        "opentopodata", "opentopodata_derived", "open_elevation", "open_elevation_derived"}

def preprocess_vector_layers(store: DataStore, processed: ProcessedStore):
    if not gpd:
        for theme in VECTOR_THEMES:
            for source in store.sources_for(theme):
                if source in RASTER_ONLY_SOURCES:
                    continue
                processed.add(ProcessedLayer(theme, source, "vector", "skipped",
                                              notes=["geopandas not available"]))
        return

    for theme in VECTOR_THEMES:
        for source in store.sources_for(theme):
            if source in RASTER_ONLY_SOURCES:
                continue  # owned by preprocess_raster_layers instead
            raw_layer = store.get(theme, source)
            if raw_layer is None:
                continue
            if not raw_layer.is_usable():
                processed.add(ProcessedLayer(theme, source, "vector",
                                              "empty" if raw_layer.status == "empty" else "skipped",
                                              source_layer_status=raw_layer.status,
                                              notes=[f"Source layer status was '{raw_layer.status}'"]))
                continue

            data = raw_layer.data
            # Google Places results arrive as a list of dicts, not a GeoDataFrame - convert first
            if source == "google_places" and isinstance(data, list):
                try:
                    records = []
                    for r in data:
                        loc = r.get("geometry", {}).get("location", {})
                        if "lat" in loc and "lng" in loc:
                            records.append({
                                "name": r.get("name"), "rating": r.get("rating"),
                                "geometry": shapely.geometry.Point(loc["lng"], loc["lat"]) if shapely else None,
                            })
                    data = gpd.GeoDataFrame(records, crs="EPSG:4326") if records and shapely else None
                except Exception as e:
                    processed.add(ProcessedLayer(theme, source, "vector", "error",
                                                  source_layer_status=raw_layer.status, error=str(e)))
                    continue

            if not isinstance(data, gpd.GeoDataFrame) or len(data) == 0:
                processed.add(ProcessedLayer(theme, source, "vector", "empty",
                                              source_layer_status=raw_layer.status,
                                              notes=["No usable geometry after conversion"]))
                continue

            try:
                clean_gdf, notes = _reproject_and_clip_vector(data, theme)
                status = "ok" if len(clean_gdf) > 0 else "empty"
                processed.add(ProcessedLayer(theme, source, "vector", status, data=clean_gdf,
                                              crs=str(SITE_UTM_CRS) if SITE_UTM_CRS else "EPSG:4326",
                                              source_layer_status=raw_layer.status, notes=notes))
            except Exception as e:
                processed.add(ProcessedLayer(theme, source, "vector", "error",
                                              source_layer_status=raw_layer.status, error=str(e)))


preprocess_vector_layers(STORE, PROCESSED)


  ✓ [roads                 ] osm            -> ok (vector)
  · [roads                 ] google_maps    -> empty (vector)
  · [bike_routes           ] osm            -> empty (vector)
  · [transit               ] osm            -> empty (vector)
  ✓ [buildings             ] osm            -> ok (vector)
  · [buildings             ] esri_living_atlas -> empty (vector)
  – [water_bodies          ] osm            -> skipped (vector)
  ✓ [tree_canopy           ] osm_greenspace_proxy -> ok (vector)
  – [poi_schools           ] osm            -> skipped (vector)
  ✓ [poi_schools           ] google_places  -> ok (vector)
  – [poi_hospitals         ] osm            -> skipped (vector)
  ✓ [poi_hospitals         ] google_places  -> ok (vector)
  – [poi_restaurants       ] osm            -> skipped (vector)
  ✓ [poi_restaurants       ] google_places  -> ok (vector)
  – [poi_parks             ] osm            -> skipped (vector)
  ✓ [poi_parks             ] google_places  -> ok (vector)
  – [poi_s

## Raster layers: materialize Earth Engine images + wrap local grids

In [ ]:
# @title Preprocess raster layers (elevation/slope/aspect, land cover, canopy, impervious, water occurrence)

RASTER_THEMES = ["elevation", "slope", "aspect", "land_cover", "tree_canopy", "impervious_surfaces", "water_bodies"]

# Earth Engine images are materialized via sampleRectangle, which has a hard server-side
# limit of 262,144 total pixels per band (a documented constraint, not a bug on our side).
# We start at this preferred resolution but auto-coarsen the scale if the site's bbox would
# exceed that pixel budget, rather than hardcoding one scale regardless of site size.
GEE_SAMPLE_SCALE_M = 10
GEE_MAX_PIXELS = 262_144

def _scale_for_pixel_budget(bbox, preferred_scale_m):
    '''Returns a scale (meters/pixel) that keeps the bbox's pixel count under GEE_MAX_PIXELS,
    coarsening from preferred_scale_m only if necessary.'''
    minx, miny, maxx, maxy = bbox
    mid_lat = (miny + maxy) / 2
    width_m = (maxx - minx) * 111_320 * math.cos(math.radians(mid_lat))
    height_m = (maxy - miny) * 111_320
    n_pixels_at_preferred = (width_m / preferred_scale_m) * (height_m / preferred_scale_m)
    if n_pixels_at_preferred <= GEE_MAX_PIXELS:
        return preferred_scale_m
    scale_factor = math.sqrt(n_pixels_at_preferred / GEE_MAX_PIXELS)
    return preferred_scale_m * scale_factor * 1.05  # small safety margin

def _materialize_gee_image(image_and_aoi: dict, band_name: str, source_note: str) -> tuple:
    '''Downloads a small Earth Engine image clip as a local numpy array via sampleRectangle.
    Returns (RasterLayer or None, notes_list, error_or_None).

    Uses SITE.bbox (a clean axis-aligned rectangle we already have in plain Python) to build
    the sampling region, rather than the original AOI polygon. sampleRectangle documentation
    describes it as extracting a *rectangular* pixel region - a many-vertex, non-axis-aligned
    boundary polygon (e.g. a ~65-point buffered-circle boundary) can trigger obscure server-side
    JSON parsing errors when Earth Engine tries to rasterize that shape into a bounding grid.
    A plain Rectangle sidesteps that failure mode entirely and is simpler to reason about.'''
    if not ee:
        return None, [], "Earth Engine not available"
    image = image_and_aoi.get("image")
    if image is None:
        return None, [], "Missing image reference"
    # NaN is not valid JSON (no such literal exists in the JSON spec) - Earth Engine's
    # request serialization emits a bare `NaN` token for np.nan, which its server-side
    # JSON parser then rejects with an opaque "Invalid JSON payload" error. We use a
    # concrete numeric sentinel instead and convert it back to NaN locally afterward.
    MASKED_SENTINEL = -9999.0  # standard GIS nodata convention; fits comfortably within
                                 # int16 range (unlike -999999.0, which overflows the typical
                                 # int16 band used for elevation/occurrence data and was
                                 # rejected by sampleRectangle's defaultValue type/range check)
    try:
        minx, miny, maxx, maxy = SITE.bbox
        scale_m = _scale_for_pixel_budget(SITE.bbox, GEE_SAMPLE_SCALE_M)
        with hard_timeout(60):
            rect = ee.Geometry.Rectangle([minx, miny, maxx, maxy])
            # Select the band first, then reproject just that single band - reprojecting
            # a whole multi-band image (some sources mix native resolutions/projections
            # across bands) can behave unpredictably. sampleRectangle has no `scale`
            # parameter of its own; resolution is controlled via this reproject step.
            first_band_name = image.bandNames().get(0)
            single_band = image.select([first_band_name])
            clipped = single_band.clip(rect).reproject(crs="EPSG:4326", scale=scale_m)
            sampled = clipped.sampleRectangle(region=rect, defaultValue=MASKED_SENTINEL, properties=[])
            band_data = sampled.get(first_band_name).getInfo()
        arr = np.array(band_data, dtype=float)
        arr[arr == MASKED_SENTINEL] = np.nan
        raster = RasterLayer(
            array=arr, bounds=(minx, miny, maxx, maxy), crs="EPSG:4326",
            band_name=band_name, pixel_size_m=scale_m, source_resolution_note=source_note,
        )
        note = f"Materialized {arr.shape[0]}x{arr.shape[1]} px via Earth Engine sampleRectangle at {scale_m:.1f}m/px (sampled over the site bbox, not the full boundary polygon)"
        if scale_m > GEE_SAMPLE_SCALE_M * 1.01:
            note += f" - coarsened from the preferred {GEE_SAMPLE_SCALE_M}m/px to stay under sampleRectangle's {GEE_MAX_PIXELS:,}-pixel limit"
        return raster, [note], None
    except HardTimeoutError as e:
        return None, [], f"{e} (Earth Engine sampleRectangle call may be slow/unreachable)"
    except Exception as e:
        return None, [], f"{e.__class__.__name__}: {str(e)[:300]}"


# Sources that are inherently vector-shaped even when their theme also has raster-shaped
# sources under the same name (e.g. tree_canopy: gee_dynamic_world is raster, but
# osm_greenspace_proxy is vector). The raster preprocessor skips these entirely and lets
# the vector preprocessor own them, rather than logging a confusing "wrong shape" skip.
VECTOR_ONLY_SOURCES = {"osm", "osm_greenspace_proxy", "google_places", "google_maps", "esri_living_atlas"}

def preprocess_raster_layers(store: DataStore, processed: ProcessedStore):
    for theme in RASTER_THEMES:
        for source in store.sources_for(theme):
            if source in VECTOR_ONLY_SOURCES:
                continue  # owned by preprocess_vector_layers instead
            raw_layer = store.get(theme, source)
            if raw_layer is None:
                continue
            if not raw_layer.is_usable():
                processed.add(ProcessedLayer(theme, source, "raster",
                                              "empty" if raw_layer.status == "empty" else "skipped",
                                              source_layer_status=raw_layer.status,
                                              notes=[f"Source layer status was '{raw_layer.status}'"]))
                continue

            data = raw_layer.data

            # Case 1: Earth Engine lazy reference {"image": ee.Image, "aoi": ee.Geometry}
            if isinstance(data, dict) and "image" in data:
                raster, notes, err = _materialize_gee_image(data, band_name=theme, source_note=raw_layer.notes[0] if raw_layer.notes else "")
                if raster is not None:
                    processed.add(ProcessedLayer(theme, source, "raster", "ok", data=raster,
                                                  crs=raster.crs, source_layer_status=raw_layer.status, notes=notes))
                else:
                    processed.add(ProcessedLayer(theme, source, "raster", "error",
                                                  source_layer_status=raw_layer.status, error=err))
                continue

            # Case 2: already a local numpy array (Open-Topo-Data / Open-Elevation grids)
            if isinstance(data, np.ndarray):
                raster = RasterLayer(
                    array=data, bounds=tuple(SITE.bbox), crs="EPSG:4326", band_name=theme,
                    pixel_size_m=None,
                    source_resolution_note=raw_layer.notes[0] if raw_layer.notes else "coarse point-sampled grid",
                )
                processed.add(ProcessedLayer(theme, source, "raster", "ok", data=raster, crs="EPSG:4326",
                                              source_layer_status=raw_layer.status,
                                              notes=["Wrapped existing local grid in RasterLayer (no re-download needed)"]))
                continue

            # Case 3: a downloaded DEM file path (the `elevation` package's local GeoTIFF) -
            # not expected to reach here often since that path errors out before this module
            # in the current environment, but handled for completeness / other environments.
            if isinstance(data, str) and rasterio and os.path.exists(data):
                try:
                    with rasterio.open(data) as src:
                        arr = src.read(1).astype(float)
                        bounds = src.bounds
                        src_crs = str(src.crs)
                    raster = RasterLayer(array=arr, bounds=(bounds.left, bounds.bottom, bounds.right, bounds.top),
                                          crs=src_crs, band_name=theme,
                                          source_resolution_note="Local SRTM GeoTIFF")
                    processed.add(ProcessedLayer(theme, source, "raster", "ok", data=raster, crs=raster.crs,
                                                  source_layer_status=raw_layer.status,
                                                  notes=["Loaded local GeoTIFF via rasterio"]))
                except Exception as e:
                    processed.add(ProcessedLayer(theme, source, "raster", "error",
                                                  source_layer_status=raw_layer.status, error=str(e)))
                continue

            processed.add(ProcessedLayer(theme, source, "raster", "skipped",
                                          source_layer_status=raw_layer.status,
                                          notes=[f"Unrecognized raster data shape: {type(data).__name__}"]))


preprocess_raster_layers(STORE, PROCESSED)


  ✓ [elevation             ] srtm           -> ok (raster)
  ✓ [elevation             ] gee_srtm       -> ok (raster)
  ✓ [slope                 ] srtm_derived   -> ok (raster)
  ✓ [aspect                ] srtm_derived   -> ok (raster)
  ✓ [land_cover            ] gee_dynamic_world -> ok (raster)
  ✓ [tree_canopy           ] gee_dynamic_world -> ok (raster)
  ✓ [impervious_surfaces   ] gee_dynamic_world -> ok (raster)
  ✗ [water_bodies          ] gee_jrc_gsw    -> error (raster)  [EEException: Image.sampleRectangle: Default value -9999.000000 is incompatible with band 'occurrence'.]


## Scalar/non-spatial layers: pass through

In [ ]:
# @title Preprocess scalar layers (climate) - no reprojection or clipping needed

SCALAR_THEMES = ["climate"]

def preprocess_scalar_layers(store: DataStore, processed: ProcessedStore):
    for theme in SCALAR_THEMES:
        for source in store.sources_for(theme):
            raw_layer = store.get(theme, source)
            if raw_layer is None:
                continue
            if not raw_layer.is_usable():
                processed.add(ProcessedLayer(theme, source, "scalar",
                                              "empty" if raw_layer.status == "empty" else "skipped",
                                              source_layer_status=raw_layer.status,
                                              notes=[f"Source layer status was '{raw_layer.status}'"]))
                continue
            processed.add(ProcessedLayer(theme, source, "scalar", "ok", data=raw_layer.data,
                                          source_layer_status=raw_layer.status,
                                          notes=["Pass-through, no spatial preprocessing applicable"]))


preprocess_scalar_layers(STORE, PROCESSED)


  ✓ [climate               ] open_meteo     -> ok (scalar)
  ✓ [climate               ] nasa_power     -> ok (scalar)


## Preprocessing summary

In [ ]:
# @title Preprocessing summary
processed_summary_df = PROCESSED.summary()
print(f"Processed {len(processed_summary_df)} (theme, source) layers.")
print(f"  ok: {(processed_summary_df.status=='ok').sum()}  |  empty: {(processed_summary_df.status=='empty').sum()}  |  "
      f"skipped: {(processed_summary_df.status=='skipped').sum()}  |  error: {(processed_summary_df.status=='error').sum()}")
processed_summary_df


Processed 64 (theme, source) layers.
  ok: 29  |  empty: 9  |  skipped: 25  |  error: 1


,theme,source,kind,status,notes
0,aspect,srtm_derived,raster,ok,Wrapped existing local grid in RasterLayer (no...
1,bike_routes,osm,vector,empty,Source layer status was 'empty'
2,buildings,esri_living_atlas,vector,empty,Source layer status was 'empty'
3,buildings,osm,vector,ok,Reprojected to EPSG:32640; Clipped to site bou...
4,climate,nasa_power,scalar,ok,"Pass-through, no spatial preprocessing applicable"
...,...,...,...,...,...
59,transit,osm,vector,empty,Source layer status was 'empty'
60,tree_canopy,gee_dynamic_world,raster,ok,Materialized 20x20 px via Earth Engine sampleR...
61,tree_canopy,osm_greenspace_proxy,vector,ok,Reprojected to EPSG:32640; Clipped to site bou...
62,water_bodies,gee_jrc_gsw,raster,error,EEException: Image.sampleRectangle: Default va...


In [ ]:
# @title Quick check: site area and a sample raster's stats, to sanity-check the pipeline
if SITE_UTM_CRS is not None:
    print(f"Site area (from boundary, in {SITE_UTM_CRS}): {SITE.area_m2():,.0f} m²")

# Show stats for whichever elevation raster actually succeeded, walking the same
# source-priority pattern the scoring engine will use later.
elev_priority = ["srtm", "gee_srtm", "opentopodata", "open_elevation"]
best_elev = PROCESSED.get_best("elevation", elev_priority)
if best_elev:
    print(f"\nBest available elevation layer: source='{best_elev.source}'")
    print(best_elev.data.stats())
else:
    print("\nNo usable elevation layer found across any source.")


Site area (from boundary, in EPSG:32640): 784,137 m²

Best available elevation layer: source='gee_srtm'
{'min': -30.0, 'max': 49.0, 'mean': 2.3062171081973064, 'std': 5.780356351413967, 'n_cells': 11211}


---
# Module 05 — Spatial Analysis

Computes **explainable metrics** from every usable layer in `PROCESSED` (Module 03's output). Each metric
is a `Metric` object, not a bare number — it always carries the data source(s) used, the formula/method,
and a confidence level, per the "no score without explanation" requirement from the project brief.

### Scope
Three tiers of metric, all computed here:

1. **Direct metrics** — computed straight from real GIS data (e.g. tree canopy %, road density, POI counts,
   mean slope). These have high confidence and are the most defensible.
2. **Proxy/estimated metrics** — existing-condition stand-ins for design-dependent KPIs from the analysis
   framework's Section 6 (e.g. "Recreation diversity" isn't measurable until a design exists, but the mix
   of existing recreation-related POIs nearby is a reasonable proxy for what the site's context already
   offers). These are flagged with lower confidence and an explicit note that they're proxies, not the
   final KPI.
3. **Placeholders** — KPIs that are genuinely only computable once a design exists (e.g. "Play area coverage"
   needs a proposed playground footprint). These appear with `value=None` and a note explaining what design
   input a future design-evaluation pass would need to compute them for real.

### Output organization
Every metric is computed once, then indexed two ways for Module 06 to consume however it prefers:
- `METRICS_BY_KPI_CATEGORY` — grouped under the analysis document's Section 6 headings (Accessibility,
  Recreation, Social, Environmental, Safety, Sustainability, Smart City) plus a `Site Context` group for
  Section 1 existing-conditions metrics that aren't themselves KPIs but feed into them.
- `METRICS_BY_DATA_THEME` — grouped by the same theme names used in Modules 02/03 (roads, buildings,
  tree_canopy, etc.), useful for auditing which raw data produced which metrics.


In [ ]:
# @title Metric structure

@dataclass
class Metric:
    name: str
    value: Any                    # float | int | dict | None (None = placeholder, not yet computable)
    unit: str = ""
    tier: str = "direct"            # "direct" | "proxy" | "placeholder"
    confidence: str = "medium"       # "high" | "medium" | "low" | "none"
    data_sources: list = field(default_factory=list)   # [(theme, source), ...] actually used
    formula: str = ""
    assumptions: list = field(default_factory=list)
    notes: str = ""

    def explain(self) -> str:
        lines = [f"{self.name}: {self.value}{(' ' + self.unit) if self.unit else ''}  [{self.tier}, confidence={self.confidence}]"]
        if self.formula:
            lines.append(f"  formula: {self.formula}")
        if self.data_sources:
            src_str = ", ".join(f"{t}/{s}" for t, s in self.data_sources)
            lines.append(f"  sources: {src_str}")
        if self.assumptions:
            lines.append(f"  assumptions: {'; '.join(self.assumptions)}")
        if self.notes:
            lines.append(f"  notes: {self.notes}")
        return "\n".join(lines)


class MetricRegistry:
    '''Flat store of every computed Metric, plus the two grouped views Module 06 will consume.'''

    def __init__(self):
        self._metrics: dict[str, Metric] = {}
        self._kpi_category_of: dict[str, str] = {}
        self._data_theme_of: dict[str, list] = {}

    def add(self, metric: Metric, kpi_category: str, data_themes: list):
        self._metrics[metric.name] = metric
        self._kpi_category_of[metric.name] = kpi_category
        self._data_theme_of[metric.name] = data_themes
        icon = {"direct": "●", "proxy": "◐", "placeholder": "○"}.get(metric.tier, "?")
        val_str = f"{metric.value}{(' ' + metric.unit) if metric.unit and metric.value is not None else ''}"
        print(f"  {icon} [{kpi_category:15s}] {metric.name:32s} = {val_str}")

    def get(self, name: str) -> Optional[Metric]:
        return self._metrics.get(name)

    def by_kpi_category(self) -> dict:
        groups = {}
        for name, metric in self._metrics.items():
            cat = self._kpi_category_of[name]
            groups.setdefault(cat, {})[name] = metric
        return groups

    def by_data_theme(self) -> dict:
        groups = {}
        for name, metric in self._metrics.items():
            for theme in self._data_theme_of[name]:
                groups.setdefault(theme, {})[name] = metric
        return groups

    def summary_df(self) -> pd.DataFrame:
        rows = []
        for name, metric in self._metrics.items():
            rows.append({
                "kpi_category": self._kpi_category_of[name],
                "metric": name,
                "value": metric.value,
                "unit": metric.unit,
                "tier": metric.tier,
                "confidence": metric.confidence,
                "sources": ", ".join(f"{t}/{s}" for t, s in metric.data_sources),
            })
        return pd.DataFrame(rows)


METRICS = MetricRegistry()

# KPI category labels, matching Section 6 of the analysis document plus a Site Context
# group for Section 1 existing-conditions metrics that feed into (but aren't themselves) KPIs.
KPI_ACCESSIBILITY = "Accessibility"
KPI_RECREATION = "Recreation"
KPI_SOCIAL = "Social"
KPI_ENVIRONMENTAL = "Environmental"
KPI_SAFETY = "Safety"
KPI_SUSTAINABILITY = "Sustainability"
KPI_SMART_CITY = "Smart City"
KPI_SITE_CONTEXT = "Site Context"


## Site Context metrics (Section 1: existing conditions)

In [ ]:
# @title Site geometry and urban context metrics

def compute_site_context_metrics():
    area_m2 = SITE.area_m2()
    METRICS.add(Metric(
        name="site_area", value=round(area_m2, 0) if area_m2 else None, unit="m2", tier="direct",
        confidence="high" if area_m2 else "none",
        data_sources=[], formula="Site boundary polygon area, reprojected to local UTM",
        notes="Site boundary as resolved in Module 01 (may be an auto-generated buffer if no real boundary was uploaded).",
    ), KPI_SITE_CONTEXT, ["site_geometry"])

    buildings = PROCESSED.get_best("buildings", ["osm", "esri_living_atlas"])
    if buildings and area_m2:
        n_buildings = len(buildings.data)
        density = n_buildings / (area_m2 / 10_000)
        METRICS.add(Metric(
            name="building_density", value=round(density, 2), unit="buildings/ha", tier="direct",
            confidence="medium", data_sources=[("buildings", buildings.source)],
            formula="count(buildings clipped to site boundary) / (site_area_m2 / 10000)",
            assumptions=["OSM building tagging completeness varies by region"],
        ), KPI_SITE_CONTEXT, ["buildings"])
    else:
        METRICS.add(Metric(
            name="building_density", value=None, unit="buildings/ha", tier="direct", confidence="none",
            notes="No usable building layer or site area available.",
        ), KPI_SITE_CONTEXT, ["buildings"])

    roads = PROCESSED.get_best("roads", ["osm", "google_maps"])
    if roads and area_m2:
        try:
            total_length_m = roads.data.geometry.length.sum()
            density_km_per_km2 = (total_length_m / 1000) / (area_m2 / 1_000_000)
            METRICS.add(Metric(
                name="road_network_density", value=round(density_km_per_km2, 2), unit="km/km2", tier="direct",
                confidence="medium", data_sources=[("roads", roads.source)],
                formula="sum(road segment lengths clipped to site) / site_area_km2",
                assumptions=["Includes all OSM-tagged highway types, not filtered by hierarchy class"],
            ), KPI_SITE_CONTEXT, ["roads"])
        except Exception as e:
            METRICS.add(Metric(name="road_network_density", value=None, tier="direct", confidence="none",
                                notes=f"Computation failed: {e}"), KPI_SITE_CONTEXT, ["roads"])
    else:
        METRICS.add(Metric(name="road_network_density", value=None, unit="km/km2", tier="direct", confidence="none",
                            notes="No usable road layer or site area available."), KPI_SITE_CONTEXT, ["roads"])

    canopy = PROCESSED.get_best("tree_canopy", ["osm_greenspace_proxy"])
    if canopy and area_m2:
        try:
            green_area_m2 = canopy.data.geometry.area.sum()
            ratio = min(green_area_m2 / area_m2, 1.0)
            METRICS.add(Metric(
                name="green_area_ratio_vector", value=round(ratio * 100, 1), unit="%", tier="direct",
                confidence="low", data_sources=[("tree_canopy", canopy.source)],
                formula="sum(OSM green-space polygon areas clipped to site) / site_area_m2",
                assumptions=["OSM green-space tagging is coarse and often incomplete; treat as a lower bound"],
            ), KPI_ENVIRONMENTAL, ["tree_canopy"])
        except Exception as e:
            METRICS.add(Metric(name="green_area_ratio_vector", value=None, tier="direct", confidence="none",
                                notes=f"Computation failed: {e}"), KPI_ENVIRONMENTAL, ["tree_canopy"])
    else:
        METRICS.add(Metric(name="green_area_ratio_vector", value=None, unit="%", tier="direct", confidence="none",
                            notes="No usable OSM green-space layer or site area available."), KPI_ENVIRONMENTAL, ["tree_canopy"])


compute_site_context_metrics()


  ● [Site Context   ] site_area                        = 14841.0 m2
  ● [Site Context   ] building_density                 = 2.7 buildings/ha
  ● [Site Context   ] road_network_density             = 155.11 km/km2
  ● [Environmental  ] green_area_ratio_vector          = 97.9 %


## Environmental metrics (Section 1 Environmental Context + Section 6 Environmental KPIs)

In [ ]:
# @title Terrain, canopy, land cover, and climate metrics

def compute_environmental_metrics():
    slope = PROCESSED.get_best("slope", ["srtm_derived", "opentopodata_derived"])
    if slope:
        stats = slope.data.stats()
        METRICS.add(Metric(
            name="mean_slope", value=round(stats["mean"], 2) if stats["mean"] is not None else None,
            unit="degrees", tier="direct",
            confidence="high" if slope.source == "srtm_derived" else "low",
            data_sources=[("slope", slope.source)],
            formula="mean(slope raster values over site bbox)",
            assumptions=(["Coarse 6x6 point-sampled grid, not a true derived-from-DEM raster"]
                         if "opentopodata" in slope.source else []),
            notes=slope.data.source_resolution_note,
        ), KPI_SITE_CONTEXT, ["slope"])
    else:
        METRICS.add(Metric(name="mean_slope", value=None, unit="degrees", tier="direct", confidence="none",
                            notes="No usable slope layer available."), KPI_SITE_CONTEXT, ["slope"])

    canopy_raster = PROCESSED.get_best("tree_canopy", ["gee_dynamic_world"])
    if canopy_raster:
        stats = canopy_raster.data.stats()
        coverage_pct = stats["mean"] * 100 if stats["mean"] is not None else None
        METRICS.add(Metric(
            name="tree_canopy_coverage", value=round(coverage_pct, 1) if coverage_pct is not None else None,
            unit="%", tier="direct", confidence="high",
            data_sources=[("tree_canopy", "gee_dynamic_world")],
            formula="mean(Dynamic World 'trees' probability band over site bbox) * 100",
            assumptions=["Dynamic World probability is a model estimate, not a direct canopy measurement"],
        ), KPI_ENVIRONMENTAL, ["tree_canopy"])
    else:
        METRICS.add(Metric(name="tree_canopy_coverage", value=None, unit="%", tier="direct", confidence="none",
                            notes="Earth Engine tree canopy layer not available this run; see green_area_ratio_vector "
                                  "for a lower-confidence OSM-based alternative."), KPI_ENVIRONMENTAL, ["tree_canopy"])

    impervious = PROCESSED.get_best("impervious_surfaces", ["gee_dynamic_world"])
    if impervious:
        stats = impervious.data.stats()
        pct = stats["mean"] * 100 if stats["mean"] is not None else None
        METRICS.add(Metric(
            name="impervious_surface_ratio", value=round(pct, 1) if pct is not None else None,
            unit="%", tier="direct", confidence="high",
            data_sources=[("impervious_surfaces", "gee_dynamic_world")],
            formula="mean(Dynamic World 'built' probability band over site bbox) * 100",
        ), KPI_SUSTAINABILITY, ["impervious_surfaces"])
    else:
        METRICS.add(Metric(name="impervious_surface_ratio", value=None, unit="%", tier="direct", confidence="none",
                            notes="Earth Engine impervious-surface layer not available this run."),
                    KPI_SUSTAINABILITY, ["impervious_surfaces"])

    water_raster = PROCESSED.get_best("water_bodies", ["gee_jrc_gsw"])
    water_vector = PROCESSED.get_best("water_bodies", ["osm"])
    if water_raster:
        stats = water_raster.data.stats()
        METRICS.add(Metric(
            name="water_occurrence_pct", value=round(stats["mean"], 1) if stats["mean"] is not None else 0.0,
            unit="%", tier="direct", confidence="high",
            data_sources=[("water_bodies", "gee_jrc_gsw")],
            formula="mean(JRC Global Surface Water 'occurrence' band, 1984-2021, over site bbox)",
            notes="0% is a valid, expected result for sites with no historical surface water (e.g. arid/desert regions).",
        ), KPI_SUSTAINABILITY, ["water_bodies"])
    elif water_vector:
        n_features = len(water_vector.data)
        METRICS.add(Metric(
            name="water_occurrence_pct", value=None, tier="proxy", confidence="low",
            data_sources=[("water_bodies", "osm")],
            formula="count(OSM water-tagged features) as a presence/absence proxy",
            notes=f"Earth Engine water layer unavailable this run; OSM found {n_features} water feature(s) as a fallback signal only.",
        ), KPI_SUSTAINABILITY, ["water_bodies"])
    else:
        METRICS.add(Metric(name="water_occurrence_pct", value=None, tier="direct", confidence="none",
                            notes="No usable water layer from any source this run."), KPI_SUSTAINABILITY, ["water_bodies"])

    climate = PROCESSED.get_best("climate", ["open_meteo", "nasa_power"])
    if climate:
        summary = climate.data.get("summary", {}) if isinstance(climate.data, dict) else {}
        METRICS.add(Metric(
            name="climate_summary", value=summary, unit="", tier="direct", confidence="high",
            data_sources=[("climate", climate.source)],
            formula="Multi-year daily/annual climatology averages from the source API",
            notes="Structured dict (avg_temp_c, precipitation, wind, solar radiation) rather than a single number - "
                  "see individual sub-values for use in specific KPI formulas.",
        ), KPI_SITE_CONTEXT, ["climate"])
    else:
        METRICS.add(Metric(name="climate_summary", value=None, tier="direct", confidence="none",
                            notes="No usable climate data from any source this run."), KPI_SITE_CONTEXT, ["climate"])


compute_environmental_metrics()


  ● [Site Context   ] mean_slope                       = 89.99 degrees
  ● [Environmental  ] tree_canopy_coverage             = 3.6 %
  ● [Sustainability ] impervious_surface_ratio         = 65.0 %
  ● [Sustainability ] water_occurrence_pct             = None
  ● [Site Context   ] climate_summary                  = {'avg_temp_c': np.float64(28.05873015873016), 'total_precip_mm_annualized': np.float64(177.88000000000002), 'avg_max_windspeed_kmh': np.float64(22.010618500273672), 'avg_solar_radiation_mj_m2': np.float64(20.902873563218392)}


## Accessibility metrics (Section 6 Accessibility KPIs)

In [ ]:
# @title Walking distance, connectivity, and access-point metrics

def compute_accessibility_metrics():
    roads = PROCESSED.get_best("roads", ["osm", "google_maps"])
    if roads and SITE.boundary is not None and gpd:
        try:
            boundary_gdf = gpd.GeoDataFrame({"geometry": [SITE.boundary]}, crs="EPSG:4326")
            if SITE_UTM_CRS is not None:
                boundary_gdf = boundary_gdf.to_crs(SITE_UTM_CRS)
            boundary_line = boundary_gdf.geometry.iloc[0].boundary
            touching = roads.data[roads.data.geometry.intersects(boundary_line)]
            METRICS.add(Metric(
                name="potential_access_points", value=len(touching), unit="segments", tier="proxy",
                confidence="medium", data_sources=[("roads", roads.source)],
                formula="count(road segments intersecting the site boundary line)",
                assumptions=["Each intersecting road segment is treated as one potential access point; "
                             "does not account for pedestrian-only access or pathways not tagged as roads"],
            ), KPI_ACCESSIBILITY, ["roads"])
        except Exception as e:
            METRICS.add(Metric(name="potential_access_points", value=None, tier="proxy", confidence="none",
                                notes=f"Computation failed: {e}"), KPI_ACCESSIBILITY, ["roads"])
    else:
        METRICS.add(Metric(name="potential_access_points", value=None, unit="segments", tier="proxy", confidence="none",
                            notes="No usable road layer or boundary available."), KPI_ACCESSIBILITY, ["roads"])

    transit = PROCESSED.get_best("transit", ["osm"])
    if transit and len(transit.data) > 0 and SITE_UTM_CRS is not None:
        try:
            site_centroid_utm = gpd.GeoSeries([shapely.geometry.Point(SITE.centroid[1], SITE.centroid[0])],
                                               crs="EPSG:4326").to_crs(SITE_UTM_CRS).iloc[0]
            distances = transit.data.geometry.distance(site_centroid_utm)
            nearest_m = float(distances.min())
            METRICS.add(Metric(
                name="nearest_transit_distance", value=round(nearest_m, 0), unit="m", tier="direct",
                confidence="medium", data_sources=[("transit", "osm")],
                formula="min(distance from site centroid to each transit feature)",
            ), KPI_ACCESSIBILITY, ["transit"])
        except Exception as e:
            METRICS.add(Metric(name="nearest_transit_distance", value=None, tier="direct", confidence="none",
                                notes=f"Computation failed: {e}"), KPI_ACCESSIBILITY, ["transit"])
    else:
        METRICS.add(Metric(name="nearest_transit_distance", value=None, unit="m", tier="direct", confidence="none",
                            notes="No transit features found within the search buffer this run."),
                    KPI_ACCESSIBILITY, ["transit"])

    METRICS.add(Metric(
        name="universal_accessibility_score", value=None, tier="placeholder", confidence="none",
        notes="Requires sidewalk-level data (curb cuts, surface material, ramp gradients) not covered by "
              "any currently-integrated source. Would need a dedicated accessibility audit or a source like "
              "AccessMap/Project Sidewalk if regional coverage exists.",
    ), KPI_ACCESSIBILITY, [])


compute_accessibility_metrics()


  ◐ [Accessibility  ] potential_access_points          = 4 segments
  ● [Accessibility  ] nearest_transit_distance         = None
  ○ [Accessibility  ] universal_accessibility_score    = None


## Recreation, Social & Safety metrics (Section 6, using nearby POIs as proxies)

In [ ]:
# @title POI-based proxy metrics for design-dependent KPIs

def compute_poi_proxy_metrics():
    recreation_themes = ["poi_parks", "poi_sports_facilities"]
    counts = {}
    sources_used = []
    for theme in recreation_themes:
        layer = PROCESSED.get_best(theme, ["osm", "google_places"])
        if layer:
            counts[theme] = len(layer.data)
            sources_used.append((theme, layer.source))
        else:
            counts[theme] = 0
    diversity_categories_present = sum(1 for v in counts.values() if v > 0)
    METRICS.add(Metric(
        name="recreation_diversity_proxy", value=diversity_categories_present,
        unit=f"of {len(recreation_themes)} categories present",
        tier="proxy", confidence="low", data_sources=sources_used,
        formula="count(POI categories among [parks, sports_facilities] with >=1 feature within 800m)",
        assumptions=["Existing nearby amenities are not the same as what a new design will provide; "
                     "this describes recreational context/demand, not the site's own future offering"],
        notes=f"Raw counts: {counts}",
    ), KPI_RECREATION, recreation_themes)

    sports = PROCESSED.get_best("poi_sports_facilities", ["osm", "google_places"])
    METRICS.add(Metric(
        name="active_recreation_opportunities_proxy", value=len(sports.data) if sports else 0,
        unit="facilities within 800m", tier="proxy", confidence="low",
        data_sources=[("poi_sports_facilities", sports.source)] if sports else [],
        formula="count(sports_facilities POIs within 800m search buffer)",
        assumptions=["Proxy for existing context, not the design's own active-recreation provision"],
    ), KPI_RECREATION, ["poi_sports_facilities"])

    # play_area_coverage: now a real proxy (existing nearby playgrounds) rather than a pure
    # placeholder, using the new poi_playgrounds category. Still clearly not the same thing as
    # a proposed design's own playground footprint, so kept at "proxy" tier with low confidence.
    playgrounds = PROCESSED.get_best("poi_playgrounds", ["osm", "google_places"])
    if playgrounds:
        METRICS.add(Metric(
            name="play_area_coverage", value=len(playgrounds.data), unit="existing playgrounds within 800m",
            tier="proxy", confidence="low", data_sources=[("poi_playgrounds", playgrounds.source)],
            formula="count(playgrounds POIs within 800m search buffer)",
            assumptions=["Existing nearby playgrounds are context, not a substitute for evaluating a "
                         "proposed design's own play-area footprint and coverage"],
        ), KPI_RECREATION, ["poi_playgrounds"])
    else:
        METRICS.add(Metric(
            name="play_area_coverage", value=None, tier="placeholder", confidence="none",
            notes="No existing playground POIs found nearby this run, and a proposed design's own "
                  "playground footprint would be needed for a real coverage measurement.",
        ), KPI_RECREATION, ["poi_playgrounds"])

    # Existing comfort amenities (benches, toilets, drinking water) - maps to the analysis
    # document's Section 3 "Comfort" program heading, tracked here as existing-context signal.
    amenities = PROCESSED.get_best("poi_accessibility_amenities", ["osm"])
    METRICS.add(Metric(
        name="existing_comfort_amenities_proxy", value=len(amenities.data) if amenities else 0,
        unit="benches/toilets/water POIs within 800m", tier="proxy", confidence="low",
        data_sources=[("poi_accessibility_amenities", amenities.source)] if amenities else [],
        formula="count(bench/drinking_water/toilets/shelter/shower POIs within 800m search buffer)",
        assumptions=["OSM tagging of street furniture (benches, taps) is often incomplete; treat as a "
                     "lower bound, not a comprehensive amenity count"],
    ), KPI_RECREATION, ["poi_accessibility_amenities"])

    # Social: cultural buildings + the new community_facilities category together
    cultural = PROCESSED.get_best("poi_cultural_buildings", ["osm", "google_places"])
    community = PROCESSED.get_best("poi_community_facilities", ["osm", "google_places"])
    cultural_n = len(cultural.data) if cultural else 0
    community_n = len(community.data) if community else 0
    social_sources = [s for s in [("poi_cultural_buildings", cultural.source) if cultural else None,
                                    ("poi_community_facilities", community.source) if community else None] if s]
    METRICS.add(Metric(
        name="community_gathering_proxy", value=cultural_n + community_n,
        unit="cultural/community POIs within 800m", tier="proxy", confidence="low",
        data_sources=social_sources,
        formula="count(cultural_buildings POIs) + count(community_facilities POIs) within 800m search buffer",
        assumptions=["Proxy for existing community infrastructure context, not the design's own gathering spaces"],
        notes=f"cultural_buildings: {cultural_n}, community_facilities: {community_n}",
    ), KPI_SOCIAL, ["poi_cultural_buildings", "poi_community_facilities"])

    METRICS.add(Metric(name="seating_capacity", value=None, tier="placeholder", confidence="none",
                        notes="Requires proposed seating layout/counts from a design."), KPI_SOCIAL, [])
    METRICS.add(Metric(name="event_capacity", value=None, tier="placeholder", confidence="none",
                        notes="Requires a proposed event-lawn/plaza footprint and assumed occupancy density."),
                KPI_SOCIAL, [])

    # Safety: hospitals remains the primary distance metric; emergency_services (police/fire) added
    # as a second, separate proxy rather than merged into one number, since response capability for
    # medical vs. fire/police emergencies are genuinely different KPIs, not interchangeable.
    hospitals = PROCESSED.get_best("poi_hospitals", ["osm", "google_places"])
    if hospitals and len(hospitals.data) > 0 and SITE_UTM_CRS is not None and gpd and shapely:
        try:
            site_centroid_utm = gpd.GeoSeries([shapely.geometry.Point(SITE.centroid[1], SITE.centroid[0])],
                                               crs="EPSG:4326").to_crs(SITE_UTM_CRS).iloc[0]
            nearest_m = float(hospitals.data.geometry.distance(site_centroid_utm).min())
            METRICS.add(Metric(
                name="nearest_hospital_distance", value=round(nearest_m, 0), unit="m", tier="proxy",
                confidence="medium", data_sources=[("poi_hospitals", hospitals.source)],
                formula="min(distance from site centroid to each hospital POI)",
                assumptions=["Straight-line distance, not actual road/emergency-vehicle travel distance or time"],
            ), KPI_SAFETY, ["poi_hospitals"])
        except Exception as e:
            METRICS.add(Metric(name="nearest_hospital_distance", value=None, tier="proxy", confidence="none",
                                notes=f"Computation failed: {e}"), KPI_SAFETY, ["poi_hospitals"])
    else:
        METRICS.add(Metric(name="nearest_hospital_distance", value=None, unit="m", tier="proxy", confidence="none",
                            notes="No hospital POIs found within the search buffer this run."), KPI_SAFETY, ["poi_hospitals"])

    emergency = PROCESSED.get_best("poi_emergency_services", ["osm", "google_places"])
    if emergency and len(emergency.data) > 0 and SITE_UTM_CRS is not None and gpd and shapely:
        try:
            site_centroid_utm = gpd.GeoSeries([shapely.geometry.Point(SITE.centroid[1], SITE.centroid[0])],
                                               crs="EPSG:4326").to_crs(SITE_UTM_CRS).iloc[0]
            nearest_m = float(emergency.data.geometry.distance(site_centroid_utm).min())
            METRICS.add(Metric(
                name="nearest_emergency_services_distance", value=round(nearest_m, 0), unit="m", tier="proxy",
                confidence="medium", data_sources=[("poi_emergency_services", emergency.source)],
                formula="min(distance from site centroid to each police/fire/ambulance station POI)",
                assumptions=["Straight-line distance, not actual road/emergency-vehicle travel distance or time"],
            ), KPI_SAFETY, ["poi_emergency_services"])
        except Exception as e:
            METRICS.add(Metric(name="nearest_emergency_services_distance", value=None, tier="proxy", confidence="none",
                                notes=f"Computation failed: {e}"), KPI_SAFETY, ["poi_emergency_services"])
    else:
        METRICS.add(Metric(name="nearest_emergency_services_distance", value=None, unit="m", tier="proxy", confidence="none",
                            notes="No police/fire/ambulance POIs found within the search buffer this run."),
                    KPI_SAFETY, ["poi_emergency_services"])

    for name, note in [
        ("lighting_coverage", "Requires proposed or existing lighting fixture locations - not covered by OSM tagging reliably enough to trust, and no dedicated source is integrated."),
        ("visibility_score", "Requires sightline/visibility analysis against a proposed design's landscaping and structures."),
        ("wayfinding_effectiveness", "Requires proposed signage/wayfinding element placements."),
    ]:
        METRICS.add(Metric(name=name, value=None, tier="placeholder", confidence="none", notes=note), KPI_SAFETY, [])

    # Civic infrastructure context (religious buildings, government offices, childcare) -
    # existing-context signal for Section 1 "Urban Context", not a Section 6 KPI on its own.
    civic_themes = ["poi_religious_buildings", "poi_government", "poi_childcare"]
    civic_counts = {}
    civic_sources = []
    for theme in civic_themes:
        layer = PROCESSED.get_best(theme, ["osm", "google_places"])
        if layer:
            civic_counts[theme] = len(layer.data)
            civic_sources.append((theme, layer.source))
        else:
            civic_counts[theme] = 0
    METRICS.add(Metric(
        name="civic_infrastructure_proxy", value=sum(civic_counts.values()),
        unit="religious/government/childcare POIs within 800m", tier="proxy", confidence="low",
        data_sources=civic_sources,
        formula="count(religious_buildings) + count(government) + count(childcare) POIs within 800m",
        assumptions=["Existing-context signal only, informs Site Context rather than any single KPI"],
        notes=f"Raw counts: {civic_counts}",
    ), KPI_SITE_CONTEXT, civic_themes)


compute_poi_proxy_metrics()


  ◐ [Recreation     ] recreation_diversity_proxy       = 2 of 2 categories present
  ◐ [Recreation     ] active_recreation_opportunities_proxy = 3 facilities within 800m
  ◐ [Recreation     ] play_area_coverage               = 3 existing playgrounds within 800m
  ◐ [Recreation     ] existing_comfort_amenities_proxy = 0 benches/toilets/water POIs within 800m
  ◐ [Social         ] community_gathering_proxy        = 5 cultural/community POIs within 800m
  ○ [Social         ] seating_capacity                 = None
  ○ [Social         ] event_capacity                   = None
  ◐ [Safety         ] nearest_hospital_distance        = 128.0 m
  ◐ [Safety         ] nearest_emergency_services_distance = None
  ○ [Safety         ] lighting_coverage                = None
  ○ [Safety         ] visibility_score                 = None
  ○ [Safety         ] wayfinding_effectiveness         = None
  ◐ [Site Context   ] civic_infrastructure_proxy       = 21 religious/government/childcare POIs within 80

## Sustainability & Smart City metrics (Section 4 & 6, mostly placeholders)

In [ ]:
# @title Remaining KPI placeholders with explicit data-gap explanations

def compute_remaining_placeholders():
    sustainability_placeholders = [
        ("carbon_reduction", "Requires a proposed design's planting palette and materials to estimate sequestration/embodied carbon versus baseline."),
        ("irrigation_efficiency", "Requires a proposed irrigation system design."),
        ("maintenance_cost", "Requires a proposed design's material and planting choices plus regional labor/material cost data."),
    ]
    for name, note in sustainability_placeholders:
        METRICS.add(Metric(name=name, value=None, tier="placeholder", confidence="none", notes=note),
                    KPI_SUSTAINABILITY, [])

    climate_metric = METRICS.get("climate_summary")
    impervious_metric = METRICS.get("impervious_surface_ratio")
    if climate_metric and climate_metric.value and impervious_metric and impervious_metric.value is not None:
        avg_temp = climate_metric.value.get("avg_temp_c")
        METRICS.add(Metric(
            name="climate_resilience_proxy", value={"avg_temp_c": avg_temp, "impervious_pct": impervious_metric.value},
            tier="proxy", confidence="low",
            data_sources=[("climate", "open_meteo"), ("impervious_surfaces", "gee_dynamic_world")],
            formula="Combines mean annual temperature with impervious surface % as a coarse heat-exposure indicator",
            assumptions=["Not a validated climate-resilience index; a simple two-factor proxy only"],
        ), KPI_SUSTAINABILITY, ["climate", "impervious_surfaces"])
    else:
        METRICS.add(Metric(name="climate_resilience_proxy", value=None, tier="proxy", confidence="none",
                            notes="Requires both climate_summary and impervious_surface_ratio to be available."),
                    KPI_SUSTAINABILITY, [])

    smart_city_placeholders = [
        ("sensor_coverage", "Requires a proposed sensor/monitoring deployment plan."),
        ("smart_management_capability", "Requires a proposed smart-systems design (irrigation control, lighting automation, etc.)."),
        ("digital_interaction_opportunities", "Requires a proposed digital installation/interactive-feature design."),
    ]
    for name, note in smart_city_placeholders:
        METRICS.add(Metric(name=name, value=None, tier="placeholder", confidence="none", notes=note),
                    KPI_SMART_CITY, [])


compute_remaining_placeholders()


  ○ [Sustainability ] carbon_reduction                 = None
  ○ [Sustainability ] irrigation_efficiency            = None
  ○ [Sustainability ] maintenance_cost                 = None
  ◐ [Sustainability ] climate_resilience_proxy         = {'avg_temp_c': np.float64(28.05873015873016), 'impervious_pct': 65.0}
  ○ [Smart City     ] sensor_coverage                  = None
  ○ [Smart City     ] smart_management_capability      = None
  ○ [Smart City     ] digital_interaction_opportunities = None


## Grouped output views

In [ ]:
# @title Build the two grouped views for Module 06

METRICS_BY_KPI_CATEGORY = METRICS.by_kpi_category()
METRICS_BY_DATA_THEME = METRICS.by_data_theme()

print(f"Computed {len(METRICS._metrics)} metrics across {len(METRICS_BY_KPI_CATEGORY)} KPI categories "
      f"and {len(METRICS_BY_DATA_THEME)} data themes.\n")

tier_counts = pd.Series([m.tier for m in METRICS._metrics.values()]).value_counts()
print("By tier:")
for tier, count in tier_counts.items():
    print(f"  {tier}: {count}")


Computed 32 metrics across 8 KPI categories and 20 data themes.

By tier:
  placeholder: 12
  direct: 10
  proxy: 10


In [ ]:
# @title Metrics summary table
metrics_summary_df = METRICS.summary_df()
metrics_summary_df


,kpi_category,metric,value,unit,tier,confidence,sources
0,Site Context,site_area,784137.0,m2,direct,high,
1,Site Context,building_density,10.32,buildings/ha,direct,medium,buildings/osm
2,Site Context,road_network_density,55.34,km/km2,direct,medium,roads/osm
3,Environmental,green_area_ratio_vector,1.9,%,direct,low,tree_canopy/osm_greenspace_proxy
4,Site Context,mean_slope,1.36,degrees,direct,low,slope/opentopodata_derived
5,Environmental,tree_canopy_coverage,3.1,%,direct,high,tree_canopy/gee_dynamic_world
6,Sustainability,impervious_surface_ratio,60.7,%,direct,high,impervious_surfaces/gee_dynamic_world
7,Sustainability,water_occurrence_pct,None,,direct,none,
8,Site Context,climate_summary,"{'avg_temp_c': 28.057307060755335, 'total_prec...",,direct,high,climate/open_meteo
9,Accessibility,potential_access_points,66,segments,proxy,medium,roads/osm


In [ ]:
# @title Example: full explanation for a few representative metrics
for name in ["tree_canopy_coverage", "water_occurrence_pct", "recreation_diversity_proxy", "play_area_coverage"]:
    m = METRICS.get(name)
    if m:
        print(m.explain())
        print()


tree_canopy_coverage: 3.6 %  [direct, confidence=high]
  formula: mean(Dynamic World 'trees' probability band over site bbox) * 100
  sources: tree_canopy/gee_dynamic_world
  assumptions: Dynamic World probability is a model estimate, not a direct canopy measurement

water_occurrence_pct: None  [direct, confidence=none]
  notes: No usable water layer from any source this run.

recreation_diversity_proxy: 2 of 2 categories present  [proxy, confidence=low]
  formula: count(POI categories among [parks, sports_facilities] with >=1 feature within 800m)
  sources: poi_parks/google_places, poi_sports_facilities/google_places
  assumptions: Existing nearby amenities are not the same as what a new design will provide; this describes recreational context/demand, not the site's own future offering
  notes: Raw counts: {'poi_parks': 3, 'poi_sports_facilities': 3}

play_area_coverage: 3 existing playgrounds within 800m  [proxy, confidence=low]
  formula: count(playgrounds POIs within 800m search bu

---
# Module 06 — Scoring Engine

Converts Module 05's raw `Metric` objects into **normalized, weighted, fully explained scores**.

### Design approach
You told us you're not an urban-design SME, so rather than inventing plausible-sounding thresholds, every
normalization curve below is grounded in a **cited, real-world planning/design standard** — not an arbitrary
guess. Sources are named inline in each criterion's `notes`. Where no defensible universal standard exists
(e.g. "visibility score", "wayfinding effectiveness"), the criterion is left as a placeholder with 0 weight
rather than scored against a made-up number.

**This is a starting point, not gospel.** If a landscape architect or urban planner reviews this later, every
threshold in `SCORING_CONFIG` below is a plain Python dict — editable in one place, with the source of each
number documented right next to it, so a subject-matter expert can override any of it without touching code.

### Scoring scale
Every metric is normalized to **0-100** (0 = worst, 100 = best against the cited standard), then rolled up
into a weighted average per KPI category and an overall weighted score - while the underlying raw value,
unit, and source are always kept alongside the normalized score, so nothing is lost in the rollup.

### Cited standards used below
- **Tree canopy**: American Forests / US Forest Service (Nowak & Greenfield) - climate-adjusted baseline
  targets: ~15% for desert cities, ~20% grassland, 40-60% forested cities under ideal conditions.
- **3-30-300 rule** (Konijnendijk, 2022) - 30% neighborhood canopy; <=300m to nearest park/green space.
- **Walkable catchment**: 400m / 5-minute walk is the standard "pedestrian shed" threshold used across
  urban planning literature (Clarence Perry, 1929; widely reconfirmed since); 800m / 10-minute walk is the
  more lenient secondary threshold used for transit access specifically.
- **Impervious surface / watershed health**: Arnold & Gibbons (1996), Booth & Jackson (1997) - measurable
  stream/watershed degradation begins around 10% impervious cover; >25% is considered highly impacted.


In [ ]:
# @title Score structure

@dataclass
class Score:
    metric_name: str
    raw_value: Any
    raw_unit: str
    normalized_score: Optional[float]
    weight: float
    confidence: str
    standard_used: str
    notes: str = ""

    def explain(self) -> str:
        lines = [f"{self.metric_name}: raw={self.raw_value}{(' ' + self.raw_unit) if self.raw_unit else ''} "
                 f"-> score={self.normalized_score if self.normalized_score is not None else 'N/A'}/100 "
                 f"(weight={self.weight}, confidence={self.confidence})"]
        if self.standard_used:
            lines.append(f"  standard: {self.standard_used}")
        if self.notes:
            lines.append(f"  notes: {self.notes}")
        return "\n".join(lines)


## Scoring configuration

Editable in one place. Each entry maps a Module 05 metric name to:
- `normalize`: a function `(raw_value) -> 0-100 score`, or `None` for un-scorable placeholders
- `weight`: relative weight *within its KPI category* (weights are renormalized to sum to 1 per category
  automatically, so you don't need to hand-balance them - just set relative importance)
- `standard`: a one-line citation for the threshold used, shown in every score's explanation


In [ ]:
# @title SCORING_CONFIG - edit thresholds/weights here

def _linear_score(value, worst, best):
    '''Linearly maps value to 0-100 between worst (->0) and best (->100). Clamps outside the range.'''
    if value is None:
        return None
    if worst == best:
        return 100.0 if value >= best else 0.0
    pct = (value - worst) / (best - worst)
    return round(max(0.0, min(1.0, pct)) * 100, 1)


def _inverse_linear_score(value, best, worst):
    '''Like _linear_score but lower raw values are better (e.g. distance, impervious %).'''
    return _linear_score(value, worst, best)


SCORING_CONFIG = {
    "tree_canopy_coverage": {
        "normalize": lambda v: _linear_score(v, worst=0, best=15),
        "weight": 1.0,
        "standard": "American Forests / US Forest Service (Nowak & Greenfield): ~15% is a realistic baseline "
                     "canopy target for desert-climate cities (vs 20% grassland, 40-60% forested cities). "
                     "0% -> 0, 15%+ -> 100. Adjust `best` upward if scoring a non-arid-climate site.",
    },
    "green_area_ratio_vector": {
        "normalize": lambda v: _linear_score(v, worst=0, best=30),
        "weight": 0.4,
        "standard": "3-30-300 rule (Konijnendijk, 2022): 30% neighborhood green space as the benchmark. "
                     "Lower-confidence than tree_canopy_coverage since it's OSM-tag-based, not imagery-derived.",
    },

    "impervious_surface_ratio": {
        "normalize": lambda v: _inverse_linear_score(v, best=10, worst=50),
        "weight": 1.0,
        "standard": "Arnold & Gibbons (1996) / Booth & Jackson (1997): measurable watershed degradation begins "
                     "around 10% impervious cover; >25% is considered highly impacted, >50% severely so. "
                     "<=10% -> 100, >=50% -> 0.",
    },
    "water_occurrence_pct": {
        "normalize": None,
        "weight": 0.0,
        "standard": "Not scored: surface water occurrence is site context (e.g. 0% is entirely expected and "
                     "not a deficiency in an arid-climate site), not a design-quality indicator on its own.",
    },
    "climate_resilience_proxy": {
        "normalize": None,
        "weight": 0.0,
        "standard": "Not scored: this is a coarse two-factor proxy (temperature + impervious %), not a "
                     "validated resilience index. Kept for reference, not included in the weighted rollup.",
    },

    "nearest_transit_distance": {
        "normalize": lambda v: _inverse_linear_score(v, best=400, worst=800),
        "weight": 1.0,
        "standard": "400m (5-min walk) is the standard 'pedestrian shed' walkable-access threshold used "
                     "across urban planning literature; 800m (10-min) is the widely-used lenient secondary "
                     "threshold, especially common for transit access specifically. <=400m -> 100, >=800m -> 0.",
    },
    "potential_access_points": {
        "normalize": lambda v: _linear_score(v, worst=0, best=4),
        "weight": 0.5,
        "standard": "No universal standard for this proxy metric; a modest 0-4 scale is used since most "
                     "small-to-mid park sites reasonably have 1-4 distinct street-facing access points. "
                     "Treat this as illustrative, not authoritative - the SME override is especially "
                     "encouraged here.",
    },
    "universal_accessibility_score": {
        "normalize": None, "weight": 0.0,
        "standard": "Not scored: no sidewalk-level accessibility data source is integrated (see Module 05 notes).",
    },

    "recreation_diversity_proxy": {
        "normalize": lambda v: _linear_score(v, worst=0, best=2),
        "weight": 0.5,
        "standard": "No universal standard; scored against the 2 categories tracked (parks, sports_facilities) "
                     "as existing-context signal only, not a design-quality measure.",
    },
    "active_recreation_opportunities_proxy": {
        "normalize": lambda v: _linear_score(v, worst=0, best=5),
        "weight": 0.5,
        "standard": "No universal standard; 5+ nearby sports facilities within 800m treated as a reasonably "
                     "well-served context based on general POI-density practice, not a formal benchmark.",
    },
    "play_area_coverage": {
        "normalize": lambda v: _linear_score(v, worst=0, best=2),
        "weight": 0.3,  # lower weight: existing nearby playgrounds are a weak proxy for the site's own future play provision
        "standard": "No universal standard; illustrative 0-2 scale (existing nearby playgrounds as context "
                     "only, not a substitute for a proposed design's own play-area coverage).",
    },
    "existing_comfort_amenities_proxy": {
        "normalize": lambda v: _linear_score(v, worst=0, best=10),
        "weight": 0.2,
        "standard": "No universal standard; illustrative 0-10 scale based on general POI-density practice "
                     "for benches/toilets/drinking water as existing park-adjacent comfort infrastructure.",
    },

    "community_gathering_proxy": {
        "normalize": lambda v: _linear_score(v, worst=0, best=3),
        "weight": 1.0,
        "standard": "No universal standard; illustrative 0-3 scale based on general POI-density practice.",
    },
    "seating_capacity": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
    "event_capacity": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},

    "nearest_hospital_distance": {
        "normalize": lambda v: _inverse_linear_score(v, best=800, worst=3000),
        "weight": 1.0,
        "standard": "No universal emergency-response distance standard found in general planning literature; "
                     "800m-3000m range chosen as a reasonable illustrative scale (closer is better for "
                     "general safety/emergency-access context). Straight-line distance only, not actual "
                     "travel time - treat as a rough proxy.",
    },
    "nearest_emergency_services_distance": {
        "normalize": lambda v: _inverse_linear_score(v, best=800, worst=3000),
        "weight": 1.0,
        "standard": "Same illustrative 800m-3000m scale as nearest_hospital_distance; no universal standard "
                     "found for police/fire response distance either. Straight-line distance only.",
    },
    "civic_infrastructure_proxy": {
        "normalize": None,  # informational context, not a KPI in its own right
        "weight": 0.0,
        "standard": "Not scored: existing civic infrastructure (religious buildings, government offices, "
                     "childcare) is Site Context information, not a design-quality indicator.",
    },
    "lighting_coverage": {"normalize": None, "weight": 0.0, "standard": "Not scored: no data source integrated."},
    "visibility_score": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires design-stage sightline analysis."},
    "wayfinding_effectiveness": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires proposed signage design."},

    "site_area": {"normalize": None, "weight": 0.0, "standard": "Informational only, not a scored KPI."},
    "building_density": {"normalize": None, "weight": 0.0, "standard": "Informational only, not a scored KPI."},
    "road_network_density": {"normalize": None, "weight": 0.0, "standard": "Informational only, not a scored KPI."},
    "mean_slope": {"normalize": None, "weight": 0.0, "standard": "Informational only, not a scored KPI."},
    "climate_summary": {"normalize": None, "weight": 0.0, "standard": "Informational only, not a scored KPI."},

    "carbon_reduction": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
    "irrigation_efficiency": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
    "maintenance_cost": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
    "sensor_coverage": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
    "smart_management_capability": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
    "digital_interaction_opportunities": {"normalize": None, "weight": 0.0, "standard": "Not scored: requires a proposed design."},
}

print(f"Scoring config covers {len(SCORING_CONFIG)} metrics; "
      f"{sum(1 for c in SCORING_CONFIG.values() if c['normalize'] is not None)} are actively scored, "
      f"{sum(1 for c in SCORING_CONFIG.values() if c['normalize'] is None)} are informational/placeholder only.")


Scoring config covers 32 metrics; 12 are actively scored, 20 are informational/placeholder only.


## Compute scores

In [ ]:
# @title Score every metric using SCORING_CONFIG

def compute_scores(metrics_registry, scoring_config) -> dict:
    scores = {}
    for name, metric in metrics_registry._metrics.items():
        config = scoring_config.get(name)
        if config is None:
            logger.warning(f"No scoring config entry for metric '{name}' - skipping. Add an entry to "
                            f"SCORING_CONFIG if this metric should be scored.")
            continue

        normalize_fn = config["normalize"]
        raw_value = metric.value

        if normalize_fn is None or raw_value is None:
            norm_score = None
        else:
            try:
                if isinstance(raw_value, dict):
                    norm_score = None
                else:
                    norm_score = normalize_fn(raw_value)
            except Exception as e:
                logger.warning(f"Normalization failed for '{name}': {e}")
                norm_score = None

        scores[name] = Score(
            metric_name=name, raw_value=raw_value, raw_unit=metric.unit,
            normalized_score=norm_score, weight=config["weight"], confidence=metric.confidence,
            standard_used=config["standard"],
            notes=metric.notes,
        )
    return scores


SCORES = compute_scores(METRICS, SCORING_CONFIG)
print(f"Computed {len(SCORES)} scores.")


Computed 32 scores.


## Category and overall rollup

In [ ]:
# @title Weighted rollup per KPI category + overall

def rollup_category(score_dict: dict, metric_names: list) -> Optional[dict]:
    '''Weighted average of normalized_score, plus a completeness indicator: what fraction of
    the metrics that COULD have been scored this run (weight > 0 in SCORING_CONFIG) actually
    were. This distinguishes "this category has few scorable metrics by design" (e.g. Safety,
    where most KPIs are genuinely design-dependent placeholders) from "data was missing this
    run for a metric that should have scored" (e.g. transit distance failing due to an empty
    Overpass result) - the latter should visibly lower confidence in the category score, not
    just silently vanish from the weighted average.'''
    scorable_by_design = [n for n in metric_names if n in score_dict and score_dict[n].weight > 0]
    usable = [(score_dict[n].normalized_score, score_dict[n].weight)
              for n in scorable_by_design if score_dict[n].normalized_score is not None]

    completeness = (len(usable) / len(scorable_by_design)) if scorable_by_design else 0.0

    if not usable:
        return {
            "score": None, "n_metrics_scored": 0, "n_metrics_total": len(metric_names),
            "n_scorable_by_design": len(scorable_by_design), "completeness": completeness,
        }
    total_weight = sum(w for _, w in usable)
    if total_weight == 0:
        return {
            "score": None, "n_metrics_scored": 0, "n_metrics_total": len(metric_names),
            "n_scorable_by_design": len(scorable_by_design), "completeness": completeness,
        }
    weighted_sum = sum(s * w for s, w in usable)
    return {
        "score": round(weighted_sum / total_weight, 1),
        "n_metrics_scored": len(usable),
        "n_metrics_total": len(metric_names),
        "n_scorable_by_design": len(scorable_by_design),
        "completeness": round(completeness, 2),
    }


CATEGORY_ROLLUPS = {}
for category, metrics_in_category in METRICS_BY_KPI_CATEGORY.items():
    metric_names = list(metrics_in_category.keys())
    result = rollup_category(SCORES, metric_names)
    CATEGORY_ROLLUPS[category] = result
    if result and result["score"] is not None:
        flag = "" if result["completeness"] >= 0.99 else f"  ⚠ only {result['completeness']*100:.0f}% of scorable metrics had data this run"
        print(f"  {category:15s}: {result['score']:5.1f}/100  "
              f"({result['n_metrics_scored']}/{result['n_scorable_by_design']} scorable-by-design metrics had data)"
              f"{flag}")
    else:
        print(f"  {category:15s}: not scorable (no metrics with weight > 0 and a numeric score)")

CATEGORY_WEIGHTS = {cat: 1.0 for cat in CATEGORY_ROLLUPS if CATEGORY_ROLLUPS[cat] and CATEGORY_ROLLUPS[cat]["score"] is not None}

scorable_categories = [(cat, CATEGORY_ROLLUPS[cat]["score"], CATEGORY_WEIGHTS.get(cat, 1.0))
                        for cat in CATEGORY_ROLLUPS if CATEGORY_ROLLUPS[cat] and CATEGORY_ROLLUPS[cat]["score"] is not None]
if scorable_categories:
    total_w = sum(w for _, _, w in scorable_categories)
    OVERALL_SCORE = round(sum(s * w for _, s, w in scorable_categories) / total_w, 1)
    print(f"\nOverall weighted score: {OVERALL_SCORE}/100  "
          f"(across {len(scorable_categories)} scorable KPI categories, equally weighted by default)")
else:
    OVERALL_SCORE = None
    print("\nOverall score: not computable (no KPI category has a numeric rollup).")


  Site Context   : not scorable (no metrics with weight > 0 and a numeric score)
  Environmental  :  45.7/100  (2/2 scorable-by-design metrics had data)
  Sustainability :   0.0/100  (1/1 scorable-by-design metrics had data)
  Accessibility  : 100.0/100  (1/2 scorable-by-design metrics had data)  ⚠ only 50% of scorable metrics had data this run
  Recreation     :  73.3/100  (4/4 scorable-by-design metrics had data)
  Social         : 100.0/100  (1/1 scorable-by-design metrics had data)
  Safety         : 100.0/100  (1/2 scorable-by-design metrics had data)  ⚠ only 50% of scorable metrics had data this run
  Smart City     : not scorable (no metrics with weight > 0 and a numeric score)

Overall weighted score: 69.8/100  (across 6 scorable KPI categories, equally weighted by default)


## Scores summary table

In [ ]:
# @title Full scores table
rows = []
for name, score in SCORES.items():
    rows.append({
        "kpi_category": next((cat for cat, ms in METRICS_BY_KPI_CATEGORY.items() if name in ms), ""),
        "metric": name,
        "raw_value": score.raw_value if not isinstance(score.raw_value, dict) else str(score.raw_value)[:60],
        "unit": score.raw_unit,
        "normalized_score": score.normalized_score,
        "weight": score.weight,
        "confidence": score.confidence,
    })
scores_df = pd.DataFrame(rows).sort_values(["kpi_category", "metric"]).reset_index(drop=True)
scores_df


,kpi_category,metric,raw_value,unit,normalized_score,weight,confidence
0,Accessibility,nearest_transit_distance,None,m,NaN,1.0,none
1,Accessibility,potential_access_points,4,segments,100.0,0.5,medium
2,Accessibility,universal_accessibility_score,None,,NaN,0.0,none
3,Environmental,green_area_ratio_vector,97.9,%,100.0,0.4,low
4,Environmental,tree_canopy_coverage,3.6,%,24.0,1.0,high
5,Recreation,active_recreation_opportunities_proxy,3,facilities within 800m,60.0,0.5,low
6,Recreation,existing_comfort_amenities_proxy,0,benches/toilets/water POIs within 800m,0.0,0.2,low
7,Recreation,play_area_coverage,3,existing playgrounds within 800m,100.0,0.3,low
8,Recreation,recreation_diversity_proxy,2,of 2 categories present,100.0,0.5,low
9,Safety,lighting_coverage,None,,NaN,0.0,none


In [ ]:
# @title Category rollup table
category_rows = []
for cat, result in CATEGORY_ROLLUPS.items():
    category_rows.append({
        "kpi_category": cat,
        "score": result["score"] if result else None,
        "metrics_with_data": result["n_metrics_scored"] if result else 0,
        "metrics_scorable_by_design": result.get("n_scorable_by_design", 0) if result else 0,
        "metrics_total_in_category": result["n_metrics_total"] if result else len(METRICS_BY_KPI_CATEGORY.get(cat, {})),
        "data_completeness": f"{result['completeness']*100:.0f}%" if result and result.get("completeness") is not None else "",
    })
category_rollup_df = pd.DataFrame(category_rows).sort_values("kpi_category").reset_index(drop=True)
print(f"Overall score: {OVERALL_SCORE}/100" if OVERALL_SCORE is not None else "Overall score: not computable")
print("\nNote: 'data_completeness' < 100% means at least one metric that COULD have been scored this run "
      "(has weight > 0 in SCORING_CONFIG) was missing data - e.g. an Overpass query returned nothing this "
      "run. A low completeness % means the category score above is resting on fewer data points than it "
      "was designed to use, and should be treated with more caution. This is separate from "
      "'metrics_total_in_category', which also counts intentional placeholders (design-dependent KPIs with "
      "weight=0) that were never meant to be scored from GIS data alone.")
category_rollup_df


Overall score: 69.8/100

Note: 'data_completeness' < 100% means at least one metric that COULD have been scored this run (has weight > 0 in SCORING_CONFIG) was missing data - e.g. an Overpass query returned nothing this run. A low completeness % means the category score above is resting on fewer data points than it was designed to use, and should be treated with more caution. This is separate from 'metrics_total_in_category', which also counts intentional placeholders (design-dependent KPIs with weight=0) that were never meant to be scored from GIS data alone.


,kpi_category,score,metrics_with_data,metrics_scorable_by_design,metrics_total_in_category,data_completeness
0,Accessibility,100.0,1,2,3,50%
1,Environmental,45.7,2,2,2,100%
2,Recreation,73.3,4,4,4,100%
3,Safety,100.0,1,2,5,50%
4,Site Context,NaN,0,0,6,0%
5,Smart City,NaN,0,0,3,0%
6,Social,100.0,1,1,3,100%
7,Sustainability,0.0,1,1,6,100%


In [ ]:
# @title Example: full explanation for a few representative scores
for name in ["tree_canopy_coverage", "impervious_surface_ratio", "nearest_transit_distance"]:
    if name in SCORES:
        print(SCORES[name].explain())
        print()


tree_canopy_coverage: raw=3.6 % -> score=24.0/100 (weight=1.0, confidence=high)
  standard: American Forests / US Forest Service (Nowak & Greenfield): ~15% is a realistic baseline canopy target for desert-climate cities (vs 20% grassland, 40-60% forested cities). 0% -> 0, 15%+ -> 100. Adjust `best` upward if scoring a non-arid-climate site.

impervious_surface_ratio: raw=65.0 % -> score=0.0/100 (weight=1.0, confidence=high)
  standard: Arnold & Gibbons (1996) / Booth & Jackson (1997): measurable watershed degradation begins around 10% impervious cover; >25% is considered highly impacted, >50% severely so. <=10% -> 100, >=50% -> 0.

nearest_transit_distance: raw=None m -> score=N/A/100 (weight=1.0, confidence=none)
  standard: 400m (5-min walk) is the standard 'pedestrian shed' walkable-access threshold used across urban planning literature; 800m (10-min) is the widely-used lenient secondary threshold, especially common for transit access specifically. <=400m -> 100, >=800m -> 0.
  note

# Module 06B — 9-Domain Park Assessment Engine

*Implements `AI_Park_Analysis_Scoring_Weighting_Methodology.md` (rationale) and `AI_Park_Analysis_Scoring_Weighting_techniques.md` (implementation spec).*

This is an **addition alongside** the original Module 06 above, not a replacement of it — the original `SCORES`/`CATEGORY_ROLLUPS`/`OVERALL_SCORE` variables are left untouched so Modules 07-09 below keep working exactly as before. This section builds a richer, separately-tracked assessment: 9 domains (typology-adjusted weights for a neighbourhood park), confidence and evidence-coverage kept strictly separate from performance score, critical-deficiency flagging, a genuinely AI-scored Placemaking domain (Gemini vision, not a hand-written heuristic), and a cross-map into Dubai Municipality's own Neighbourhood Parks Manual rubric. It also plugs directly into Module 10's `DESIGN_CONCEPTS`, giving the what-if rescoring a properly structured backbone instead of the four ad-hoc metrics in Module 10's original Stage 4.

**Still pending**: migrating Modules 07 (Dashboard) and 08 (AI Recommendations) to consume this engine's output instead of (or alongside) the original Module 06's — for now they continue to run against the original scoring path only.

In [ ]:
# =============================================================================
# MODULE 06 (REPLACEMENT) — 9-Domain Park Assessment Engine
# =============================================================================
# Implements AI_Park_Analysis_Scoring_Weighting_Methodology.md (rationale) and
# AI_Park_Analysis_Scoring_Weighting_techniques.md (implementation spec).
#
# Paste this in AFTER Module 05 (Spatial Analysis) and Module 10 (Design
# Concepts), since it consumes both METRICS (Module 05) and DESIGN_CONCEPTS
# (Module 10) as raw inputs. It does not delete Module 06/08 — it supersedes
# their SCORING role. Module 08's recommendation text can stay; feed it
# ASSESSMENTS[...] instead of the old SCORES dict if you want the richer
# language later.
#
# Design rules enforced throughout (from both source docs):
#   1. Never manufacture precision — an indicator with no real evidence is
#      recorded as evidence_type="unknown", confidence=0.0, score=None —
#      never silently defaulted to 0 or to a guessed number.
#   2. Performance, Confidence and Coverage are three separate numbers,
#      never blended into one another.
#   3. Domain weights use the NEIGHBORHOOD PARK typology profile (Section 42
#      of the methodology doc), not the generic baseline — Al Safa 2 is
#      explicitly a neighborhood park per both the Scope of Work and the
#      Dubai Municipality Neighborhood Parks Manual. The generic baseline is
#      kept alongside it for transparency (Section 41's "always show
#      default vs. project-adjusted weights" rule).
#   4. A domain scoring < 40 raises a critical flag independent of the
#      weighted average, so a strong ecology score can never quietly paper
#      over a failing accessibility score.
# =============================================================================
# @title Domain Assessment Engine — data model
from dataclasses import dataclass, field
from typing import Optional, Any
import math

@dataclass
class IndicatorResult:
    id: str
    name: str
    domain: str
    raw_value: Optional[float] = None
    unit: str = ""
    score: Optional[float] = None            # 0-100, None = not computed (never 0-by-default)
    weight: float = 1.0                       # intra-domain weight (equal by default — see note below)
    evidence_type: str = "unknown"            # measured|derived|observed|ai_inferred|external_data|estimated|unknown
    confidence: float = 0.0                   # 0-1, independent of score
    formula: str = ""
    notes: str = ""

    def explain(self) -> str:
        val = f"{self.raw_value:.2f}{self.unit}" if isinstance(self.raw_value, (int, float)) else "n/a"
        sc = f"{self.score:.0f}/100" if self.score is not None else "not computed"
        return (f"[{self.domain}] {self.id} {self.name}: raw={val} -> {sc} "
                f"(evidence={self.evidence_type}, confidence={self.confidence:.2f})\n"
                f"  formula: {self.formula}\n  notes: {self.notes}")


@dataclass
class DomainResult:
    id: str
    name: str
    weight: float
    indicators: list = field(default_factory=list)   # list[IndicatorResult]
    score: Optional[float] = None
    confidence: Optional[float] = None
    coverage: Optional[float] = None
    critical_flag: bool = False

    def rollup(self):
        '''Weighted average over indicators that actually have a score. Confidence and
        coverage are computed independently — never multiplied into the score itself.'''
        scored = [(i.score, i.weight, i.confidence) for i in self.indicators if i.score is not None]
        applicable = len(self.indicators)
        if not scored:
            self.score, self.confidence, self.coverage = None, 0.0, 0.0
            self.critical_flag = False
            return
        total_w = sum(w for _, w, _ in scored)
        self.score = round(sum(s * w for s, w, _ in scored) / total_w, 1) if total_w else None
        self.confidence = round(sum(c * w for _, w, c in scored) / total_w, 2) if total_w else 0.0
        self.coverage = round(100 * len(scored) / applicable, 1) if applicable else 0.0
        self.critical_flag = self.score is not None and self.score < CRITICAL_THRESHOLD


@dataclass
class ParkAssessment:
    label: str                                        # e.g. "Baseline (existing site)" or concept name
    domains: dict = field(default_factory=dict)       # domain_id -> DomainResult
    overall_score: Optional[float] = None
    overall_confidence: Optional[float] = None
    overall_coverage: Optional[float] = None
    critical_flags: list = field(default_factory=list)

    def rollup(self):
        for d in self.domains.values():
            d.rollup()
        scored = [(d.score, ACTIVE_DOMAIN_WEIGHTS[d.id], d.confidence, d.coverage)
                   for d in self.domains.values() if d.score is not None]
        if not scored:
            self.overall_score = self.overall_confidence = self.overall_coverage = None
        else:
            total_w = sum(w for _, w, _, _ in scored)
            self.overall_score = round(sum(s * w for s, w, _, _ in scored) / total_w, 1)
            self.overall_confidence = round(sum(c * w for _, w, c, _ in scored) / total_w, 2)
            self.overall_coverage = round(sum(cov * w for _, w, _, cov in scored) / total_w, 1)
        self.critical_flags = [d.name for d in self.domains.values() if d.critical_flag]

In [ ]:
# @title Universal normalization functions (Section 2/12 of the techniques doc)

def clamp(x: float) -> float:
    return max(0.0, min(100.0, x))

def higher_is_better(x, L, U):
    if x is None:
        return None
    if U == L:
        return 100.0 if x >= U else 0.0
    return clamp(100 * (x - L) / (U - L))

def lower_is_better(x, L, U):
    if x is None:
        return None
    if U == L:
        return 100.0 if x <= L else 0.0
    return clamp(100 * (U - x) / (U - L))

def target_range(x, low, high, decay_fraction=0.6):
    '''100 within [low, high]; decays linearly outside the range over a band
    decay_fraction * (high-low) wide, per Section 5.3 of the methodology doc.'''
    if x is None:
        return None
    if low <= x <= high:
        return 100.0
    span = max(high - low, 1e-9) * decay_fraction
    if x < low:
        return clamp(100 * (1 - (low - x) / span))
    return clamp(100 * (1 - (x - high) / span))

def presence_over_target(count, target):
    if count is None or not target:
        return None
    return clamp(100 * count / target)

def binary_score(present):
    if present is None:
        return None
    return 100.0 if present else 0.0

def simpson_diversity_score(counts: dict, expected_categories: int):
    '''D = 1 - sum(p_i^2), normalized against the max diversity achievable with
    expected_categories categories, per Section 6 (Activities) / Section 7 (Ecology).'''
    total = sum(counts.values())
    if total == 0 or expected_categories <= 1:
        return 0.0
    p = [c / total for c in counts.values() if c > 0]
    d = 1 - sum(pi ** 2 for pi in p)
    max_d = 1 - 1 / expected_categories
    return clamp(100 * d / max_d) if max_d > 0 else 0.0

In [ ]:
# @title Domain weight configuration — typology-adjusted (neighborhood park)

DOMAIN_NAMES = {
    "accessibility": "Accessibility & Connectivity",
    "safety": "Safety & Security",
    "climate": "Thermal Comfort & Climate Resilience",
    "activities": "Activities & Recreation",
    "ecology": "Nature & Biodiversity",
    "amenities": "Amenities & Infrastructure",
    "placemaking": "Spatial Quality & Placemaking",
    "inclusion": "Inclusion & Universal Design",
    "management": "Management & Maintenance",
}

# Generic baseline from the techniques doc — kept only for transparency/comparison.
DOMAIN_WEIGHTS_GENERIC_BASELINE = {
    "accessibility": 0.15, "safety": 0.12, "climate": 0.15, "activities": 0.12,
    "ecology": 0.12, "amenities": 0.10, "placemaking": 0.10, "inclusion": 0.08, "management": 0.06,
}

# Neighborhood-park typology profile (methodology doc, Section 42) — ACTIVE for this project,
# since Al Safa 2 is explicitly a neighborhood park per the SOW and the DM Neighborhood Parks Manual.
DOMAIN_WEIGHTS_NEIGHBORHOOD_PARK = {
    "accessibility": 0.18, "safety": 0.13, "climate": 0.15, "activities": 0.14,
    "ecology": 0.10, "amenities": 0.09, "placemaking": 0.09, "inclusion": 0.08, "management": 0.04,
}

ACTIVE_DOMAIN_WEIGHTS = DOMAIN_WEIGHTS_NEIGHBORHOOD_PARK
assert abs(sum(ACTIVE_DOMAIN_WEIGHTS.values()) - 1.0) < 1e-6, "Domain weights must sum to 1.0"
assert abs(sum(DOMAIN_WEIGHTS_GENERIC_BASELINE.values()) - 1.0) < 1e-6

CRITICAL_THRESHOLD = 40.0   # domain score below this raises a flag, independent of the weighted average

print("Active domain weights (neighborhood-park typology, vs. generic baseline):")
for k in ACTIVE_DOMAIN_WEIGHTS:
    print(f"  {DOMAIN_NAMES[k]:42s} {ACTIVE_DOMAIN_WEIGHTS[k]*100:4.0f}%   (generic baseline: {DOMAIN_WEIGHTS_GENERIC_BASELINE[k]*100:.0f}%)")

Active domain weights (neighborhood-park typology, vs. generic baseline):
  Accessibility & Connectivity                 18%   (generic baseline: 15%)
  Safety & Security                            13%   (generic baseline: 12%)
  Thermal Comfort & Climate Resilience         15%   (generic baseline: 15%)
  Activities & Recreation                      14%   (generic baseline: 12%)
  Nature & Biodiversity                        10%   (generic baseline: 12%)
  Amenities & Infrastructure                    9%   (generic baseline: 10%)
  Spatial Quality & Placemaking                 9%   (generic baseline: 10%)
  Inclusion & Universal Design                  8%   (generic baseline: 8%)
  Management & Maintenance                      4%   (generic baseline: 6%)


In [ ]:
# @title Small helpers bridging this engine to the existing notebook's data (Modules 02/05/10)

def _metric_value(name):
    '''Reads a raw value out of Module 05's METRICS registry, if it exists in this session.'''
    registry = globals().get("METRICS")
    if registry is None:
        return None
    m = registry.get(name)
    return m.value if (m is not None and not isinstance(m.value, dict)) else None

def _site_area_m2():
    site = globals().get("SITE")
    try:
        return site.area_m2() if site else None
    except Exception:
        return None

def _processed_layer(theme, sources):
    proc = globals().get("PROCESSED")
    if proc is None:
        return None
    return proc.get_best(theme, sources)

def _concept_zone_area(concept, purpose):
    if not concept:
        return 0.0
    return sum((z.actual_area_m2 or 0.0) for z in concept.zones if z.purpose == purpose and z.geometry is not None)

def _concept_zone_count(concept, purpose):
    if not concept:
        return 0
    return sum(1 for z in concept.zones if z.purpose == purpose and z.geometry is not None)

def unknown_indicator(id_, name, domain, formula, notes):
    '''Explicit placeholder for an indicator the current data pipeline cannot support yet —
    recorded so the assessment schema stays complete and auditable, per the "never silently
    drop an indicator, mark it unknown instead" rule.'''
    return IndicatorResult(id=id_, name=name, domain=domain, score=None,
                            evidence_type="unknown", confidence=0.0, formula=formula, notes=notes)

def _registry_available():
    return globals().get("METRICS") is not None

def _processed_available():
    return globals().get("PROCESSED") is not None

def _combine_existing_and_proposed(metric_name, concept, purpose):
    '''The single most important guard in this file. Returns (value, known):
      - known=False, value=None  ->  neither a real existing-metric reading nor a design
        concept were available, so this indicator genuinely has NO evidence. Callers must
        pass this straight through as score=None / evidence_type="unknown" — never coerce
        it to a scored 0. This is the exact failure mode Section 38 of the methodology doc
        calls out: "Never convert unknown to 0."
      - known=True, value=<float>  ->  at least one real source exists (an existing METRICS
        reading, and/or a design concept whose zones can genuinely be counted, even if that
        count is legitimately zero). A real, evidenced zero is fine to score; a missing-data
        zero is not.'''
    existing = _metric_value(metric_name)          # None only if the metric truly isn't available
    existing_known = existing is not None
    concept_known = concept is not None
    if not existing_known and not concept_known:
        return None, False
    total = (existing or 0.0) + (float(_concept_zone_count(concept, purpose)) if concept_known else 0.0)
    return total, True

In [ ]:
# @title Domain 1 — Accessibility & Connectivity (18% typology-adjusted / 15% generic)

def assess_accessibility(concept=None) -> DomainResult:
    inds = []
    site_area = _site_area_m2()

    road_density = _metric_value("road_network_density")   # km/km2
    inds.append(IndicatorResult(
        id="A1", name="Walkable network permeability", domain="accessibility",
        raw_value=road_density, unit=" km/km2",
        score=target_range(road_density, 8, 20) if road_density is not None else None,
        evidence_type="derived" if road_density is not None else "unknown",
        confidence=0.55 if road_density is not None else 0.0,
        formula="target_range(road/path network density within site, 8-20 km/km2 preferred)",
        notes="Proxy for A1 (true metric needs a population-weighted 400/800m network isochrone, "
              "not yet run). Road density is a reasonable stand-in for how permeable the walking "
              "network around the site is."))

    access_points = _metric_value("potential_access_points") or 0
    new_entrances = _concept_zone_count(concept, "entrance_plaza")
    total_entrances = access_points + new_entrances
    density = (total_entrances / (site_area / 10_000)) if site_area else None
    inds.append(IndicatorResult(
        id="A2", name="Entrance density", domain="accessibility",
        raw_value=density, unit=" entrances/ha",
        score=target_range(density, 2, 6) if density is not None else None,
        evidence_type="derived" if density is not None else "unknown",
        confidence=0.6 if density is not None else 0.0,
        formula="target_range(entrances / hectare, 2-6 preferred)",
        notes=f"{total_entrances} entrance(s) counted ({access_points} existing-context road/boundary "
              f"intersections + {new_entrances} proposed entrance plaza(s))."))

    transit_dist = _metric_value("nearest_transit_distance")
    inds.append(IndicatorResult(
        id="A9", name="Transit access", domain="accessibility",
        raw_value=transit_dist, unit=" m",
        score=lower_is_better(transit_dist, 400, 800) if transit_dist is not None else None,
        evidence_type="measured" if transit_dist is not None else "unknown",
        confidence=0.75 if transit_dist is not None else 0.0,
        formula="lower_is_better(distance to nearest transit stop, best<=400m, worst>=800m)",
        notes="From OSM-tagged transit features within the POI search buffer."))

    bike_layer = _processed_layer("bike_routes", ["osm"])
    proc_available = _processed_available()
    has_bike = bool(bike_layer and bike_layer.is_usable() and len(bike_layer.data) > 0)
    new_dropoff = _concept_zone_count(concept, "bicycle_dropoff") > 0
    bike_known = proc_available or (concept is not None)
    inds.append(IndicatorResult(
        id="A10", name="Bicycle access", domain="accessibility",
        raw_value=(1.0 if (has_bike or new_dropoff) else 0.0) if bike_known else None,
        score=binary_score(has_bike or new_dropoff) if bike_known else None,
        evidence_type=("measured" if proc_available else "derived") if bike_known else "unknown",
        confidence=0.6 if bike_known else 0.0,
        formula="binary(cycle-tagged infrastructure present, or a bicycle drop-off zone proposed)",
        notes=("OSM cycle-way tagging near this site is often sparse — treat a 0 here cautiously; "
               "it may reflect missing map data rather than a real absence of cycling access."
               if bike_known else
               "Neither Module 02's data collection nor a design concept were available this run, "
               "so this cannot be distinguished from 'not measured' and is left unknown.")))

    inds.append(unknown_indicator(
        "A7", "Accessible primary routes", "accessibility",
        "accessible primary routes / total primary routes x 100",
        "Requires ramp-gradient / path-width survey data (sidewalk-level accessibility dataset). "
        "Not available for this site from OSM/GIS alone — matches the existing "
        "universal_accessibility_score placeholder already flagged in Module 05."))
    inds.append(unknown_indicator(
        "A8", "Safe crossings", "accessibility",
        "safe crossings / required crossings x 100",
        "Requires a crossing-by-crossing street-level audit; no such dataset is wired in yet."))

    d = DomainResult(id="accessibility", name=DOMAIN_NAMES["accessibility"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["accessibility"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 2 — Safety & Security (13% / 12%)

def assess_safety(concept=None) -> DomainResult:
    inds = []

    emergency_dist = _metric_value("nearest_emergency_services_distance")
    inds.append(IndicatorResult(
        id="S7", name="Emergency access", domain="safety",
        raw_value=emergency_dist, unit=" m",
        score=lower_is_better(emergency_dist, 800, 3000) if emergency_dist is not None else None,
        evidence_type="derived" if emergency_dist is not None else "unknown",
        confidence=0.6 if emergency_dist is not None else 0.0,
        formula="lower_is_better(distance to nearest police/fire/ambulance station, best<=800m, worst>=3000m)",
        notes="Straight-line distance proxy, not routed emergency-vehicle travel time."))

    hospital_dist = _metric_value("nearest_hospital_distance")
    inds.append(IndicatorResult(
        id="S7b", name="Medical emergency access", domain="safety",
        raw_value=hospital_dist, unit=" m",
        score=lower_is_better(hospital_dist, 800, 3000) if hospital_dist is not None else None,
        evidence_type="derived" if hospital_dist is not None else "unknown",
        confidence=0.6 if hospital_dist is not None else 0.0,
        formula="lower_is_better(distance to nearest hospital/clinic, best<=800m, worst>=3000m)",
        notes="Kept separate from S7 since medical vs. fire/police response are different capabilities."))

    # S5 Active edges — approximate using building density near the boundary as a proxy for
    # "eyes on the park" / natural surveillance from surrounding frontage.
    building_density = _metric_value("building_density")
    inds.append(IndicatorResult(
        id="S5", name="Active edges (surveillance proxy)", domain="safety",
        raw_value=building_density, unit=" bldgs/ha",
        score=target_range(building_density, 15, 60) if building_density is not None else None,
        evidence_type="estimated" if building_density is not None else "unknown",
        confidence=0.35 if building_density is not None else 0.0,
        formula="target_range(surrounding building density, 15-60 bldgs/ha) as a crude natural-surveillance proxy",
        notes="Weak proxy — real S5 needs frontage-facing-park analysis (which buildings actually "
              "have windows/entries onto the park), not just density. Confidence intentionally low."))

    for id_, name, formula in [
        ("S1", "Natural surveillance (sightlines)", "visible/overlooked edge length / total edge length"),
        ("S3", "Lighting coverage", "lit path length / total path length"),
        ("S4", "Concealed areas", "concealed area / usable area (lower is better)"),
        ("S9", "Traffic conflict points", "conflict points / total crossings (lower is better)"),
        ("S10", "Safety perception", "community survey score / 5"),
    ]:
        inds.append(unknown_indicator(id_, name, "safety", formula,
            "Requires street-level imagery, a lighting-fixture inventory, or a community survey — "
            "none of which are wired into the current GIS pipeline. Per both source docs, this must "
            "stay UNKNOWN rather than being scored 0 or guessed."))

    d = DomainResult(id="safety", name=DOMAIN_NAMES["safety"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["safety"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 3 — Thermal Comfort & Climate Resilience (15% / 15%) — highest-confidence domain

def _shaded_path_ratio(concept):
    '''Best-effort geometric approximation of C3 (shaded path ratio): buffers the road/path
    network by a walking-width margin and intersects it with any tree_canopy_expansion zones
    proposed in a design concept, plus the existing canopy vector layer if present. Returns
    None (not a fabricated 0) if neither a path layer nor a canopy layer is available.'''
    if not (globals().get("gpd") and globals().get("shapely")):
        return None, "geopandas/shapely not available"
    roads = _processed_layer("roads", ["osm"])
    if roads is None or not roads.is_usable():
        return None, "no road/path layer available this run"
    try:
        path_buffer = roads.data.geometry.buffer(1.5)  # ~3m wide walking path corridor
        path_union = path_buffer.unary_union
        total_len = roads.data.geometry.length.sum()
        if total_len <= 0:
            return None, "path network has zero length"
        canopy_geoms = []
        canopy_vec = _processed_layer("tree_canopy", ["osm_greenspace_proxy"])
        if canopy_vec is not None and canopy_vec.is_usable():
            canopy_geoms.extend(canopy_vec.data.geometry.tolist())
        if concept is not None:
            canopy_geoms.extend(z.geometry for z in concept.zones
                                  if z.purpose == "tree_canopy_expansion" and z.geometry is not None)
        if not canopy_geoms:
            return 0.0, "path network found but no canopy geometry (existing or proposed) to intersect"
        import shapely.ops as ops
        canopy_union = ops.unary_union(canopy_geoms)
        shaded_len = path_union.intersection(canopy_union).length if not canopy_union.is_empty else 0.0
        # length of a buffered polygon isn't path length directly; approximate via area/width instead
        shaded_area = path_union.intersection(canopy_union).area if not canopy_union.is_empty else 0.0
        total_area = path_union.area
        ratio = shaded_area / total_area if total_area > 0 else 0.0
        return clamp(ratio * 100), "computed from path-corridor / canopy-polygon overlap area"
    except Exception as e:
        return None, f"computation failed: {e}"


def assess_climate(concept=None) -> DomainResult:
    inds = []

    canopy_pct = _metric_value("tree_canopy_coverage")
    canopy_zone_area = _concept_zone_area(concept, "tree_canopy_expansion")
    site_area = _site_area_m2()
    if canopy_zone_area and site_area:
        added_pct = (canopy_zone_area / site_area) * 100 * 0.6   # maturity-discount, matches Module 10
        canopy_pct = min(100.0, (canopy_pct or 0) + added_pct)
    inds.append(IndicatorResult(
        id="C1", name="Tree canopy coverage", domain="climate",
        raw_value=canopy_pct, unit="%",
        score=higher_is_better(canopy_pct, 0, 30) if canopy_pct is not None else None,
        evidence_type="measured" if canopy_pct is not None else "unknown",
        confidence=0.85 if canopy_pct is not None else 0.0,
        formula="higher_is_better(canopy % over site, 0-30% scale; 30% ~ 3-30-300 rule neighbourhood target)",
        notes="From Earth Engine Dynamic World 'trees' band; includes proposed canopy zones at 60% "
              "assumed maturity discount if a design concept was passed in."))

    impervious_pct = _metric_value("impervious_surface_ratio")
    permeable_zone_area = _concept_zone_area(concept, "permeable_surface_conversion")
    if permeable_zone_area and site_area and impervious_pct is not None:
        reduced_pct = (permeable_zone_area / site_area) * 100 * 0.85
        impervious_pct = max(0.0, impervious_pct - reduced_pct)
    inds.append(IndicatorResult(
        id="C6/C7", name="Heat-exposed hardscape / permeability", domain="climate",
        raw_value=impervious_pct, unit="%",
        score=lower_is_better(impervious_pct, 10, 50) if impervious_pct is not None else None,
        evidence_type="measured" if impervious_pct is not None else "unknown",
        confidence=0.85 if impervious_pct is not None else 0.0,
        formula="lower_is_better(impervious surface %, best<=10%, worst>=50%; Arnold & Gibbons 1996 threshold)",
        notes="From Earth Engine Dynamic World 'built' band; combines C6 (exposed hardscape) and the "
              "inverse of C7 (permeability) since both derive from the same raster."))

    shaded_path_pct, shaded_path_note = _shaded_path_ratio(concept)
    inds.append(IndicatorResult(
        id="C3", name="Shaded path ratio", domain="climate",
        raw_value=shaded_path_pct, unit="%",
        score=higher_is_better(shaded_path_pct, 0, 70) if shaded_path_pct is not None else None,
        evidence_type="derived" if shaded_path_pct is not None else "unknown",
        confidence=0.4 if shaded_path_pct is not None else 0.0,
        formula="shaded path corridor area / total path corridor area x 100 (geometric overlap, not true solar-position shading)",
        notes=f"{shaded_path_note}. This is the key metric both source docs flag as essential — a park "
              f"can have high canopy % (see C1) while its paths and activity nodes remain fully exposed; "
              f"do not read C1 as a proxy for this."))

    water_pct = _metric_value("water_occurrence_pct")
    inds.append(unknown_indicator(
        "C9", "Water-efficient landscape", "climate",
        "water-efficient planting area / total planted area x 100",
        f"Historical surface-water occurrence measured at {water_pct}% (context only, not a design "
        f"metric) — actual planting-palette water efficiency needs the planting plan, which doesn't "
        f"exist until landscape design development." if water_pct is not None else
        "No water-occurrence data available and no planting plan exists yet to assess."))

    climate_summary = _metric_value("climate_summary")
    if isinstance(climate_summary, dict) and climate_summary.get("avg_temp_c") is not None:
        avg_temp = climate_summary["avg_temp_c"]
        inds.append(IndicatorResult(
            id="C10", name="Climate stress baseline", domain="climate",
            raw_value=avg_temp, unit="°C avg",
            score=None,   # informational, not itself a 0-100 performance indicator
            evidence_type="external_data", confidence=0.9,
            formula="Open-Meteo 5-year daily archive mean temperature",
            notes="Context indicator only (not scored) — confirms this is a hot-arid site where the "
                  "climate domain's elevated 15% weight and the C1/C3/C6 indicators above matter most."))

    d = DomainResult(id="climate", name=DOMAIN_NAMES["climate"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["climate"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 4 — Activities & Recreation (14% / 12%)

def assess_activities(concept=None) -> DomainResult:
    inds = []

    # R1 Activity diversity — Simpson index over POI categories + proposed zone categories.
    # Only computed if we actually have a way to observe categories (a live PROCESSED store to
    # query, or a design concept whose zones we can inspect) — otherwise "no categories found"
    # would be indistinguishable from "genuinely zero activity diversity", which is exactly the
    # confusion Section 38 forbids.
    diversity_known = _processed_available() or (concept is not None)
    if diversity_known:
        categories = ["poi_parks", "poi_sports_facilities", "poi_playgrounds", "poi_community_facilities"]
        counts = {}
        for theme in categories:
            layer = _processed_layer(theme, ["osm", "google_places"])
            counts[theme] = len(layer.data) if layer and layer.is_usable() else 0
        for purpose in ["play_area", "sports_recreation_zone", "community_gathering_plaza", "picnic_area"]:
            counts[f"zone_{purpose}"] = counts.get(f"zone_{purpose}", 0) + _concept_zone_count(concept, purpose)
        diversity = simpson_diversity_score(counts, expected_categories=len(counts))
        inds.append(IndicatorResult(
            id="R1", name="Activity diversity", domain="activities",
            raw_value=diversity, unit=" (Simpson index, 0-100 scaled)",
            score=diversity,
            evidence_type="derived", confidence=0.55 if _processed_available() else 0.35,
            formula="Simpson diversity D = 1 - sum(p_i^2) over existing POI categories + proposed zone categories",
            notes=f"Category counts: {counts}"))
    else:
        inds.append(unknown_indicator("R1", "Activity diversity", "activities",
            "Simpson diversity D = 1 - sum(p_i^2) over existing POI categories + proposed zone categories",
            "Neither Module 02's data collection nor a design concept were available this run."))

    total_play, play_known = _combine_existing_and_proposed("play_area_coverage", concept, "play_area")
    inds.append(IndicatorResult(
        id="R2", name="Children's facilities", domain="activities",
        raw_value=total_play, unit=" play area(s) (existing nearby + proposed)",
        score=presence_over_target(total_play, 2) if play_known else None,
        evidence_type="derived" if play_known else "unknown",
        confidence=0.5 if play_known else 0.0,
        formula="presence_over_target(existing nearby + proposed playgrounds, target=2)",
        notes="Existing count is context (nearby playgrounds within 800m); proposed count is this "
              "design's own." if play_known else
              "Neither Module 05's METRICS registry nor a design concept were available this run."))

    total_sports, sports_known = _combine_existing_and_proposed(
        "active_recreation_opportunities_proxy", concept, "sports_recreation_zone")
    inds.append(IndicatorResult(
        id="R3", name="Sports facilities", domain="activities",
        raw_value=total_sports, unit=" facilities (existing nearby + proposed)",
        score=presence_over_target(total_sports, 5) if sports_known else None,
        evidence_type="derived" if sports_known else "unknown",
        confidence=0.5 if sports_known else 0.0,
        formula="presence_over_target(existing nearby + proposed sports/fitness facilities, target=5)",
        notes="" if sports_known else
              "Neither Module 05's METRICS registry nor a design concept were available this run."))

    total_social, social_known = _combine_existing_and_proposed(
        "community_gathering_proxy", concept, "community_gathering_plaza")
    inds.append(IndicatorResult(
        id="R4", name="Social/gathering spaces", domain="activities",
        raw_value=total_social, unit=" (existing context + proposed)",
        score=presence_over_target(total_social, 3) if social_known else None,
        evidence_type="derived" if social_known else "unknown",
        confidence=0.5 if social_known else 0.0,
        formula="presence_over_target(existing cultural/community context + proposed gathering plazas, target=3)",
        notes="" if social_known else
              "Neither Module 05's METRICS registry nor a design concept were available this run."))

    n_picnic = _concept_zone_count(concept, "picnic_area")
    inds.append(IndicatorResult(
        id="R7", name="Family/picnic facilities", domain="activities",
        raw_value=n_picnic, unit=" proposed picnic zone(s)",
        score=presence_over_target(n_picnic, 1) if concept else None,
        evidence_type="derived" if concept else "unknown",
        confidence=0.5 if concept else 0.0,
        formula="presence_over_target(proposed picnic zones, target=1)",
        notes="Only scorable once a design concept exists; the current park has no picnic-area POI tag to check against."))

    for id_, name, formula in [
        ("R8", "Senior-friendly activities", "senior-friendly facilities / target"),
        ("R10", "Activity spatial distribution", "% activity nodes adequately distributed"),
    ]:
        inds.append(unknown_indicator(id_, name, "activities", formula,
            "No age-specific facility tagging or spatial-evenness analysis run yet."))

    d = DomainResult(id="activities", name=DOMAIN_NAMES["activities"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["activities"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 5 — Nature & Biodiversity (10% / 12%)

def assess_ecology(concept=None) -> DomainResult:
    inds = []

    green_pct = _metric_value("green_area_ratio_vector")
    canopy_zone_area = _concept_zone_area(concept, "tree_canopy_expansion") + _concept_zone_area(concept, "green_space_expansion")
    site_area = _site_area_m2()
    if canopy_zone_area and site_area:
        green_pct = min(100.0, (green_pct or 0) + (canopy_zone_area / site_area) * 100)
    inds.append(IndicatorResult(
        id="E1", name="Green coverage", domain="ecology",
        raw_value=green_pct, unit="%",
        score=higher_is_better(green_pct, 0, 40) if green_pct is not None else None,
        evidence_type="measured" if green_pct is not None else "unknown",
        confidence=0.5 if green_pct is not None else 0.0,
        formula="higher_is_better(green area % of site, 0-40% scale)",
        notes="OSM-tagged green space is a coarser, lower-confidence source than the canopy raster used in C1 — kept separate deliberately."))

    permeable_pct = _metric_value("impervious_surface_ratio")
    permeable_pct = (100 - permeable_pct) if permeable_pct is not None else None
    inds.append(IndicatorResult(
        id="E6", name="Permeable soil", domain="ecology",
        raw_value=permeable_pct, unit="%",
        score=higher_is_better(permeable_pct, 50, 90) if permeable_pct is not None else None,
        evidence_type="derived" if permeable_pct is not None else "unknown",
        confidence=0.8 if permeable_pct is not None else 0.0,
        formula="higher_is_better(100 - impervious surface %, 50-90% scale)",
        notes="Direct inverse of C6/C7 in the climate domain — intentionally reused, since permeable "
              "soil is both a thermal-comfort and an ecological indicator in the source docs."))

    inds.append(unknown_indicator(
        "E9", "NDVI (vegetation vigor)", "ecology", "mean/median NDVI over site",
        "No Sentinel-2/Landsat NDVI band is pulled in the current Earth Engine connector (it uses "
        "Dynamic World's categorical 'trees'/'built' bands, not raw NIR/Red reflectance). Adding an "
        "NDVI pull to Module 02's land-cover connector would close this gap directly."))
    inds.append(unknown_indicator(
        "E5", "Native/adapted planting", "ecology", "adapted planting area / total planting area x 100",
        "Requires a species-level planting inventory; doesn't exist until the landscape planting plan is developed."))
    inds.append(unknown_indicator(
        "E7", "Ecological connectivity", "ecology", "connected habitat / habitat area",
        "Requires a citywide green-patch connectivity graph (least-cost-path analysis) beyond this single-site pipeline."))

    d = DomainResult(id="ecology", name=DOMAIN_NAMES["ecology"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["ecology"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 6 — Amenities & Infrastructure (9% / 10%)

def assess_amenities(concept=None) -> DomainResult:
    inds = []

    total, comfort_known = _combine_existing_and_proposed(
        "existing_comfort_amenities_proxy", concept, "seating_comfort_area")
    inds.append(IndicatorResult(
        id="I1/I3/I4", name="Seating / drinking water / toilets (combined)", domain="amenities",
        raw_value=total, unit=" amenity POIs (existing context) + proposed zones",
        score=presence_over_target(total, 10) if comfort_known else None,
        evidence_type="derived" if comfort_known else "unknown",
        confidence=0.45 if comfort_known else 0.0,
        formula="presence_over_target(bench+drinking_water+toilets+shelter+shower POIs + proposed comfort zones, target=10)",
        notes=("OSM tags these together (I1/I3/I4 can't be cleanly split from the current query); "
               "treat this as a combined comfort-infrastructure indicator, not three independent ones."
               if comfort_known else
               "Neither Module 05's METRICS registry nor a design concept were available this run.")))

    n_restrooms = _concept_zone_count(concept, "restrooms_and_services")
    inds.append(IndicatorResult(
        id="I4b", name="Restrooms & services (proposed)", domain="amenities",
        raw_value=n_restrooms, unit=" proposed zone(s)",
        score=presence_over_target(n_restrooms, 1) if concept else None,
        evidence_type="derived" if concept else "unknown",
        confidence=0.5 if concept else 0.0,
        formula="presence_over_target(proposed restroom/service zones, target=1)", notes=""))

    sports_total, sports_infra_known = _combine_existing_and_proposed(
        "active_recreation_opportunities_proxy", concept, "sports_recreation_zone")
    inds.append(IndicatorResult(
        id="I9", name="Sports infrastructure", domain="amenities",
        raw_value=sports_total, unit=" facilities",
        score=presence_over_target(sports_total, 3) if sports_infra_known else None,
        evidence_type="derived" if sports_infra_known else "unknown",
        confidence=0.5 if sports_infra_known else 0.0,
        formula="presence_over_target(sports facilities existing + proposed, target=3)",
        notes="" if sports_infra_known else
              "Neither Module 05's METRICS registry nor a design concept were available this run."))

    for id_, name, formula in [
        ("I2", "Shaded seating", "shaded seats / total seats x 100"),
        ("I7", "Wayfinding", "signed decision points / total decision points x 100"),
        ("I8", "Bicycle facilities", "bike facilities / target"),
    ]:
        inds.append(unknown_indicator(id_, name, "amenities", formula,
            "Requires furniture/signage-level field or high-res imagery survey not available from GIS/OSM alone."))

    d = DomainResult(id="amenities", name=DOMAIN_NAMES["amenities"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["amenities"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 7 — Spatial Quality & Placemaking (9% / 10%) — the genuinely AI-scored domain
#
# Per both source docs, this domain is qualitative and should use a structured vision-language
# assessment rather than a single "is this beautiful?" prompt, with a mandatory written
# justification per indicator (never a bare number). Reuses Module 01's Gemini client and
# satellite-image fetch helper — no new API surface is introduced.

PLACEMAKING_INDICATORS = [
    ("P1", "Spatial hierarchy", "Primary/secondary/tertiary space organization"),
    ("P2", "Legibility", "Ease of understanding the park's structure at a glance"),
    ("P3", "Spatial variety", "Variety of distinct spatial experiences across the site"),
    ("P4", "Enclosure", "Quality/balance of enclosed vs. open spaces"),
    ("P5", "Focal points", "Strength and distribution of landmarks/focal features"),
    ("P6", "Human scale", "Relationship between built/landscape elements and people"),
    ("P7", "Visual coherence", "Coherence between landscape and architectural elements"),
    ("P8", "Identity", "Distinctiveness / sense of place"),
    ("P9", "Spatial sequence", "Quality of the movement experience through the site"),
    ("P10", "Flexibility", "Apparent ability of the space to support changing uses"),
]

def assess_placemaking_via_ai(image_context=None, concept=None) -> DomainResult:
    '''Calls Gemini vision (reusing the Module 01 client) to score P1-P10 with mandatory
    evidence text per indicator. Falls back to all-unknown if no client/image is available —
    this domain is never silently scored without a real model call behind it.'''
    inds = []
    client_fn = globals().get("_get_gemini_client")
    client = client_fn() if callable(client_fn) else None

    if client is None or image_context is None or not globals().get("PIL_Image"):
        for id_, name, desc in PLACEMAKING_INDICATORS:
            inds.append(unknown_indicator(id_, name, "placemaking", f"AI vision-language assessment: {desc}",
                "Gemini client or satellite image not available this run — call "
                "get_satellite_image_for_boundary() (Module 01) and pass its result as image_context."))
        d = DomainResult(id="placemaking", name=DOMAIN_NAMES["placemaking"],
                          weight=ACTIVE_DOMAIN_WEIGHTS["placemaking"], indicators=inds)
        d.rollup()
        return d

    try:
        import io as _io
        pil_image = globals()["PIL_Image"].open(_io.BytesIO(image_context["image_bytes"]))
        indicator_list = "\n".join(f"- {id_}: {name} — {desc}" for id_, name, desc in PLACEMAKING_INDICATORS)
        concept_note = f"\nA proposed redesign concept named '{concept.name}' should also be considered qualitatively if described." if concept else ""
        prompt = f'''You are assessing the SPATIAL QUALITY of a park site from a satellite image, using a
structured framework — not a generic beauty judgement.

Score each of the following 10 indicators from 0-100, and give a short (1-2 sentence) EVIDENCE
justification citing what you actually see in the image for each score:
{indicator_list}{concept_note}

Respond with ONLY a JSON object of exactly this shape, no other text:
{{"P1": {{"score": <0-100>, "evidence": "<short text>"}}, "P2": {{...}}, ... "P10": {{...}}}}'''

        hard_timeout_fn = globals().get("hard_timeout")
        ctx = hard_timeout_fn(45) if callable(hard_timeout_fn) else _no_op_ctx()
        with ctx:
            response = client.models.generate_content(
                model=globals().get("GEMINI_MODEL_NAME", "gemini-3.6-flash"),
                contents=[prompt, pil_image])
        raw_text = response.text.strip()
        if raw_text.startswith("```"):
            raw_text = raw_text.split("```")[1]
            if raw_text.startswith("json"):
                raw_text = raw_text[4:]
        import json as _json
        result = _json.loads(raw_text.strip())

        for id_, name, desc in PLACEMAKING_INDICATORS:
            entry = result.get(id_, {})
            score = entry.get("score")
            evidence = entry.get("evidence", "")
            inds.append(IndicatorResult(
                id=id_, name=name, domain="placemaking",
                raw_value=score, unit=" (0-100 AI-scored)",
                score=clamp(float(score)) if score is not None else None,
                evidence_type="ai_inferred", confidence=0.65 if score is not None else 0.0,
                formula=f"Gemini vision-language structured assessment: {desc}",
                notes=evidence or "No evidence text returned."))
    except Exception as e:
        for id_, name, desc in PLACEMAKING_INDICATORS:
            inds.append(unknown_indicator(id_, name, "placemaking", f"AI vision-language assessment: {desc}",
                f"Gemini call failed this run: {e.__class__.__name__}: {str(e)[:150]}"))

    d = DomainResult(id="placemaking", name=DOMAIN_NAMES["placemaking"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["placemaking"], indicators=inds)
    d.rollup()
    return d

class _no_op_ctx:
    def __enter__(self): return self
    def __exit__(self, *a): return False

In [ ]:
# @title Domain 8 — Inclusion & Universal Design (8% / 8%)

def assess_inclusion(concept=None) -> DomainResult:
    inds = []
    civic_proxy = _metric_value("civic_infrastructure_proxy")
    inds.append(IndicatorResult(
        id="U8", name="Distribution equity (context proxy)", domain="inclusion",
        raw_value=civic_proxy, unit=" civic POIs within 800m",
        score=presence_over_target(civic_proxy, 5) if civic_proxy is not None else None,
        evidence_type="estimated" if civic_proxy is not None else "unknown",
        confidence=0.3 if civic_proxy is not None else 0.0,
        formula="presence_over_target(religious/government/childcare POIs nearby, target=5) — weak proxy for equitable civic access",
        notes="This does NOT measure demographic equity of park access itself (which needs a "
              "population-grid cross-tabulation per Section 5.5 of the framework doc) — it only "
              "checks whether civic infrastructure exists nearby. Confidence kept deliberately low."))

    for id_, name, formula in [
        ("U1", "Accessible entrances", "accessible entrances / total entrances x 100"),
        ("U2", "Accessible paths", "accessible path length / total primary path length x 100"),
        ("U3", "Accessible play", "accessible play elements / total play elements x 100"),
        ("U4", "Accessible seating", "accessible seating / required seating x 100"),
        ("U5", "Accessible toilets", "accessible toilets / required toilets x 100"),
        ("U7", "Age inclusivity", "age groups adequately served / target groups"),
    ]:
        inds.append(unknown_indicator(id_, name, "inclusion", formula,
            "Requires ADA/local-code-equivalent dimensioned survey data (ramp gradients, path widths, "
            "tactile paving) — matches the existing universal_accessibility_score gap already flagged "
            "in Module 05. This should be prioritized once the AutoCAD as-built site drawing (Annex-1 "
            "of the Scope of Work) is available."))

    d = DomainResult(id="inclusion", name=DOMAIN_NAMES["inclusion"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["inclusion"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Domain 9 — Management & Maintenance (4% / 6%)

def assess_management(concept=None) -> DomainResult:
    '''Deliberately near-empty for a pre-construction design concept: maintenance condition
    doesn't exist until something is built. Kept in the schema (rather than deleted) so the
    9-domain structure stays complete and this gap stays visible, per both source docs'
    "mark unknown, don't drop the indicator" rule. Populate this domain for real once the
    Al Safa 2 as-built condition survey (Annex-1) or site-visit photos are available.'''
    inds = []
    for id_, name, formula in [
        ("M1", "Cleanliness", "clean zones / inspected zones"),
        ("M2", "Vegetation condition", "healthy vegetation / total vegetation"),
        ("M4", "Path condition", "good-condition paths / total paths"),
        ("M6", "Waste management", "adequate bins per zone / required"),
        ("M8", "Safety hazards", "hazard points / hectare (lower is better)"),
    ]:
        inds.append(unknown_indicator(id_, name, "management", formula,
            "Not applicable to a pre-construction design concept, and no site-visit/condition survey "
            "exists for the current park yet. Populate this domain when Annex-1 as-built condition "
            "data or site photos become available."))
    d = DomainResult(id="management", name=DOMAIN_NAMES["management"],
                      weight=ACTIVE_DOMAIN_WEIGHTS["management"], indicators=inds)
    d.rollup()
    return d

In [ ]:
# @title Full assessment runner

def assess_park(label: str, concept=None, placemaking_image_context=None) -> ParkAssessment:
    assessment = ParkAssessment(label=label)
    assessment.domains["accessibility"] = assess_accessibility(concept)
    assessment.domains["safety"] = assess_safety(concept)
    assessment.domains["climate"] = assess_climate(concept)
    assessment.domains["activities"] = assess_activities(concept)
    assessment.domains["ecology"] = assess_ecology(concept)
    assessment.domains["amenities"] = assess_amenities(concept)
    assessment.domains["placemaking"] = assess_placemaking_via_ai(placemaking_image_context, concept)
    assessment.domains["inclusion"] = assess_inclusion(concept)
    assessment.domains["management"] = assess_management(concept)
    assessment.rollup()
    return assessment


def priority_label(domain: DomainResult) -> str:
    '''Section 46/59: Priority = f(gap, weight) — not itself a performance number.'''
    if domain.score is None:
        return "Unknown"
    gap = 100 - domain.score
    impact = gap * domain.weight
    if impact >= 8 or domain.critical_flag:
        return "High"
    if impact >= 4:
        return "Medium"
    return "Low"


def print_triplet_dashboard(assessment: ParkAssessment):
    '''Performance + Confidence + Priority, per the techniques doc's closing recommendation.'''
    print(f"\n=== {assessment.label} ===")
    if assessment.overall_score is not None:
        print(f"Overall Performance: {assessment.overall_score}/100   "
              f"Evidence Confidence: {assessment.overall_confidence*100:.0f}%   "
              f"Evidence Coverage: {assessment.overall_coverage:.0f}%")
    else:
        print("Overall Performance: not computable this run (no domain produced a score)")
    if assessment.critical_flags:
        print(f"CRITICAL FLAGS (score < {CRITICAL_THRESHOLD:.0f}): {', '.join(assessment.critical_flags)}")
    print()
    for domain_id, d in assessment.domains.items():
        score_str = f"{d.score:.0f}/100" if d.score is not None else "n/a"
        conf_str = f"{d.confidence*100:.0f}%" if d.confidence is not None else "n/a"
        cov_str = f"{d.coverage:.0f}%" if d.coverage is not None else "n/a"
        flag = "  [CRITICAL]" if d.critical_flag else ""
        print(f"  {d.name:42s} {score_str:>8s}  ·  confidence {conf_str:>4s}  ·  coverage {cov_str:>4s}  "
              f"·  priority {priority_label(d):6s}{flag}")

In [ ]:
# @title Radar chart (baseline vs. each design concept, same style as Module 07)

def render_domain_radar(assessments: list):
    try:
        import plotly.graph_objects as go
    except ImportError:
        print("plotly not available - skipping radar chart.")
        return
    domain_ids = list(ACTIVE_DOMAIN_WEIGHTS.keys())
    labels = [DOMAIN_NAMES[d] for d in domain_ids] + [DOMAIN_NAMES[domain_ids[0]]]
    fig = go.Figure()
    for a in assessments:
        values = [(a.domains[d].score if a.domains[d].score is not None else 0) for d in domain_ids]
        values = values + [values[0]]
        fig.add_trace(go.Scatterpolar(r=values, theta=labels, fill="toself", name=a.label, opacity=0.6))
    fig.update_layout(polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
                        title="9-Domain Park Assessment — Baseline vs. Design Concepts", height=550)
    fig.show()

In [ ]:
# @title Cross-map to Dubai Municipality's own Neighborhood Parks Manual rubric
#
# DM's manual scores three categories (Zoning; Activities & Amenities; Events & Experiences) on a
# 60=good / 80=excellent / 100=exceptional band. This is a mapping/display layer only — it does
# NOT replace the 9-domain engine above, it re-presents the same evidence in the jury's own
# internal vocabulary.

DM_MANUAL_CROSS_MAP = {
    "Zoning (Accessibility to the park, Walkability, Landscapes, Zones quality & use)":
        ["accessibility", "placemaking"],
    "Activities & Amenities (Playscapes, Active/Passive recreation, Commercial, Wayfinding, Services)":
        ["activities", "amenities", "inclusion"],
    "Events & Experiences (Events, Attractiveness)":
        ["placemaking", "activities"],
}

def print_dm_manual_view(assessment: ParkAssessment):
    print(f"\n=== {assessment.label} — mapped to DM Neighborhood Parks Manual categories ===")
    for dm_category, domain_ids in DM_MANUAL_CROSS_MAP.items():
        scores = [assessment.domains[d].score for d in domain_ids if assessment.domains[d].score is not None]
        avg = round(sum(scores) / len(scores), 1) if scores else None
        band = "exceptional" if (avg or 0) >= 100 else "excellent" if (avg or 0) >= 80 else \
               "good" if (avg or 0) >= 60 else "below DM's 'good' threshold" if avg is not None else "not computable"
        print(f"  {dm_category}\n    -> {avg if avg is not None else 'n/a'}/100  ({band})")


# =============================================================================
# USAGE (paste as the final cell after Modules 05 and 10 have run):
#
#   image_ctx = get_satellite_image_for_boundary(SITE.centroid[0], SITE.centroid[1],
#                                                 radius_m=SITE_RADIUS_M)  # from Module 01
#   BASELINE = assess_park("Baseline (existing site)", concept=None,
#                           placemaking_image_context=image_ctx)
#   print_triplet_dashboard(BASELINE)
#   print_dm_manual_view(BASELINE)
#
#   CONCEPT_ASSESSMENTS = []
#   for c in DESIGN_CONCEPTS:
#       a = assess_park(c.name, concept=c, placemaking_image_context=image_ctx)
#       print_triplet_dashboard(a)
#       CONCEPT_ASSESSMENTS.append(a)
#
#   render_domain_radar([BASELINE] + CONCEPT_ASSESSMENTS)
# =============================================================================

---

---
# Module 07 — Dashboard

Builds the visual analysis dashboard from Module 06's `SCORES` and `CATEGORY_ROLLUPS`:

- **Radar chart per KPI category** with a data-completeness indicator, plus one overall radar
- **KPI summary cards** - at-a-glance score + confidence + completeness per category
- **Bar chart** comparing all category scores side by side
- **Heatmap** of every individual metric's normalized score, grouped by category
- **Interactive GIS map** showing the site boundary plus collected point/line/polygon layers
- **Opportunities & constraints matrix** - automatically derived from which metrics scored high vs. low

Only KPI categories with `CATEGORY_ROLLUPS[cat]["score"] is not None` are plotted in the radar/bar
charts (`Site Context` and `Smart City` are informational/placeholder-only this run and are listed
separately rather than plotted as a misleading zero).


In [ ]:
# @title Dashboard setup
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

DASHBOARD_CATEGORIES = [cat for cat in CATEGORY_ROLLUPS if CATEGORY_ROLLUPS[cat] and CATEGORY_ROLLUPS[cat]["score"] is not None]
INFORMATIONAL_CATEGORIES = [cat for cat in CATEGORY_ROLLUPS if cat not in DASHBOARD_CATEGORIES]

print(f"Scorable categories to plot: {DASHBOARD_CATEGORIES}")
print(f"Informational-only categories (not plotted as scores): {INFORMATIONAL_CATEGORIES}")


Scorable categories to plot: ['Environmental', 'Sustainability', 'Accessibility', 'Recreation', 'Social', 'Safety']
Informational-only categories (not plotted as scores): ['Site Context', 'Smart City']


## Overall radar chart

In [ ]:
# @title Overall radar chart across all scorable KPI categories
categories_for_radar = DASHBOARD_CATEGORIES + [DASHBOARD_CATEGORIES[0]]
scores_for_radar = [CATEGORY_ROLLUPS[c]["score"] for c in DASHBOARD_CATEGORIES] + [CATEGORY_ROLLUPS[DASHBOARD_CATEGORIES[0]]["score"]]

fig_overall_radar = go.Figure()
fig_overall_radar.add_trace(go.Scatterpolar(
    r=scores_for_radar, theta=categories_for_radar, fill="toself", name="Score",
    line=dict(color="#2E7D32", width=2), fillcolor="rgba(46, 125, 50, 0.25)",
))
fig_overall_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
    title=f"Overall Site Score: {OVERALL_SCORE}/100" if OVERALL_SCORE is not None else "Overall Site Score",
    showlegend=False, height=500,
)
fig_overall_radar.show()


## Radar chart per KPI category (individual metrics within each)

In [ ]:
# @title Per-category radar charts showing individual metric contributions
def category_radar(category_name):
    metrics_in_cat = METRICS_BY_KPI_CATEGORY.get(category_name, {})
    names, values = [], []
    for name in metrics_in_cat:
        if name in SCORES and SCORES[name].normalized_score is not None and SCORES[name].weight > 0:
            names.append(name.replace("_", " "))
            values.append(SCORES[name].normalized_score)
    if not names:
        return None
    names_closed = names + [names[0]]
    values_closed = values + [values[0]]
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(r=values_closed, theta=names_closed, fill="toself",
                                    line=dict(color="#1565C0", width=2), fillcolor="rgba(21, 101, 192, 0.25)"))
    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 100])),
        title=f"{category_name}: {CATEGORY_ROLLUPS[category_name]['score']}/100",
        showlegend=False, height=400,
    )
    return fig


for cat in DASHBOARD_CATEGORIES:
    fig = category_radar(cat)
    if fig:
        fig.show()


## KPI summary cards

In [ ]:
# @title KPI summary cards (HTML/CSS grid, renders inline in Colab)
from IPython.display import HTML, display

def _card_color(score):
    if score is None:
        return "#9E9E9E"
    if score >= 70:
        return "#2E7D32"
    if score >= 40:
        return "#F9A825"
    return "#C62828"


def render_kpi_cards():
    cards_html = []
    all_categories = list(CATEGORY_ROLLUPS.keys())
    for cat in sorted(all_categories):
        result = CATEGORY_ROLLUPS[cat]
        score = result["score"] if result else None
        completeness = result["completeness"] * 100 if result and result.get("completeness") is not None else None
        color = _card_color(score)
        score_str = f"{score:.1f}" if score is not None else "N/A"
        completeness_str = f"{completeness:.0f}% data completeness" if completeness is not None else "informational only"
        cards_html.append(f'''
        <div style="display:inline-block; width:150px; margin:8px; padding:14px; border-radius:10px;
                    background:#1e1e1e; border:2px solid {color}; text-align:center; font-family:sans-serif;">
            <div style="font-size:13px; color:#ccc; margin-bottom:6px;">{cat}</div>
            <div style="font-size:28px; font-weight:bold; color:{color};">{score_str}</div>
            <div style="font-size:11px; color:#888; margin-top:4px;">{completeness_str}</div>
        </div>
        ''')
    display(HTML(f'''
    <div style="padding:10px;">
        <div style="font-size:16px; color:#eee; font-family:sans-serif; margin-bottom:10px;">
            Overall Score: <span style="font-size:22px; font-weight:bold; color:{_card_color(OVERALL_SCORE)};">
            {OVERALL_SCORE if OVERALL_SCORE is not None else 'N/A'}</span>/100
        </div>
        {''.join(cards_html)}
    </div>
    '''))


render_kpi_cards()


## Category score bar chart

In [ ]:
# @title Bar chart comparing all category scores
bar_df = pd.DataFrame([
    {"category": cat, "score": CATEGORY_ROLLUPS[cat]["score"]}
    for cat in DASHBOARD_CATEGORIES
]).sort_values("score", ascending=True)

fig_bar = px.bar(bar_df, x="score", y="category", orientation="h", range_x=[0, 100],
                   color="score", color_continuous_scale=["#C62828", "#F9A825", "#2E7D32"], range_color=[0, 100],
                   title="KPI Category Scores")
fig_bar.update_layout(height=400, showlegend=False, coloraxis_showscale=False)
fig_bar.show()


## Metric-level heatmap

In [ ]:
# @title Heatmap of every individual metric's normalized score, grouped by category
heatmap_rows = []
for cat, metrics_in_cat in METRICS_BY_KPI_CATEGORY.items():
    for name in metrics_in_cat:
        if name in SCORES and SCORES[name].normalized_score is not None:
            heatmap_rows.append({"category": cat, "metric": name.replace("_", " "), "score": SCORES[name].normalized_score})

if heatmap_rows:
    heatmap_df = pd.DataFrame(heatmap_rows)
    pivot = heatmap_df.pivot_table(index="metric", columns="category", values="score", aggfunc="first")
    fig_heatmap = px.imshow(pivot, color_continuous_scale=["#C62828", "#F9A825", "#2E7D32"], range_color=[0, 100],
                              aspect="auto", title="Metric Scores by Category", labels=dict(color="Score"))
    fig_heatmap.update_layout(height=max(400, 30 * len(pivot)))
    fig_heatmap.show()
else:
    print("No scored metrics available to build a heatmap this run.")


## Interactive site map

In [ ]:
# @title Interactive map: site boundary + collected layers
def build_site_map():
    if not folium or SITE.boundary is None:
        print("folium not available or no site boundary - skipping interactive map.")
        return None

    m = folium.Map(location=SITE.centroid, zoom_start=16, tiles="CartoDB positron")

    try:
        folium.GeoJson(
            gpd.GeoSeries([SITE.boundary]).__geo_interface__ if gpd else None,
            style_function=lambda x: {"color": "#2E7D32", "weight": 3, "fillOpacity": 0.1},
            name="Site boundary",
        ).add_to(m)
    except Exception as e:
        logger.warning(f"Could not draw site boundary on map: {e}")

    roads = PROCESSED.get_best("roads", ["osm", "google_maps"])
    if roads and gpd:
        try:
            roads_wgs84 = roads.data.to_crs("EPSG:4326")
            folium.GeoJson(roads_wgs84.__geo_interface__,
                            style_function=lambda x: {"color": "#616161", "weight": 1.5},
                            name="Roads").add_to(m)
        except Exception as e:
            logger.warning(f"Could not draw roads on map: {e}")

    buildings = PROCESSED.get_best("buildings", ["osm", "esri_living_atlas"])
    if buildings and gpd:
        try:
            buildings_wgs84 = buildings.data.to_crs("EPSG:4326")
            folium.GeoJson(buildings_wgs84.__geo_interface__,
                            style_function=lambda x: {"color": "#8D6E63", "weight": 0.5, "fillOpacity": 0.4},
                            name="Buildings").add_to(m)
        except Exception as e:
            logger.warning(f"Could not draw buildings on map: {e}")

    poi_colors = {
        "poi_schools": "blue", "poi_hospitals": "red", "poi_restaurants": "orange",
        "poi_parks": "green", "poi_sports_facilities": "purple", "poi_cultural_buildings": "cadetblue",
        "poi_playgrounds": "lightgreen", "poi_religious_buildings": "gray", "poi_government": "darkblue",
        "poi_emergency_services": "darkred", "poi_childcare": "pink", "poi_community_facilities": "beige",
        "poi_accessibility_amenities": "lightgray",
    }
    for theme, color in poi_colors.items():
        layer = PROCESSED.get_best(theme, ["osm", "google_places"])
        if layer and gpd and len(layer.data) > 0:
            try:
                layer_wgs84 = layer.data.to_crs("EPSG:4326")
                fg = folium.FeatureGroup(name=theme.replace("poi_", "").replace("_", " ").title())
                for _, row in layer_wgs84.iterrows():
                    geom = row.geometry
                    pt = geom.centroid if geom.geom_type != "Point" else geom
                    folium.CircleMarker(location=[pt.y, pt.x], radius=4, color=color, fill=True,
                                          fill_opacity=0.8).add_to(fg)
                fg.add_to(m)
            except Exception as e:
                logger.warning(f"Could not draw {theme} POIs on map: {e}")

    folium.LayerControl(collapsed=False).add_to(m)
    return m


site_map = build_site_map()
if site_map:
    display(site_map)


## Opportunities & constraints matrix

In [ ]:
# @title Auto-derived opportunities/constraints from metric scores

def build_opportunities_constraints():
    '''An "opportunity" is a metric scoring well (>=70) with usable confidence; a "constraint"
    is one scoring poorly (<40). Metrics with no score, or "none" confidence, are excluded from
    both lists rather than risk overstating a weak-evidence signal as a firm finding.'''
    opportunities, constraints = [], []
    for name, score_obj in SCORES.items():
        if score_obj.normalized_score is None or score_obj.weight == 0:
            continue
        if score_obj.confidence == "none":
            continue
        entry = {
            "metric": name.replace("_", " "),
            "score": score_obj.normalized_score,
            "raw": f"{score_obj.raw_value}{(' ' + score_obj.raw_unit) if score_obj.raw_unit else ''}",
            "confidence": score_obj.confidence,
        }
        if score_obj.normalized_score >= 70:
            opportunities.append(entry)
        elif score_obj.normalized_score < 40:
            constraints.append(entry)
    return (sorted(opportunities, key=lambda x: -x["score"]),
            sorted(constraints, key=lambda x: x["score"]))


OPPORTUNITIES, CONSTRAINTS = build_opportunities_constraints()

opp_df = pd.DataFrame(OPPORTUNITIES) if OPPORTUNITIES else pd.DataFrame(columns=["metric", "score", "raw", "confidence"])
con_df = pd.DataFrame(CONSTRAINTS) if CONSTRAINTS else pd.DataFrame(columns=["metric", "score", "raw", "confidence"])

print(f"Opportunities (score >= 70, confidence >= low): {len(OPPORTUNITIES)}")
print(f"Constraints (score < 40, confidence >= low): {len(CONSTRAINTS)}")


Opportunities (score >= 70, confidence >= low): 6
Constraints (score < 40, confidence >= low): 3


In [ ]:
# @title Opportunities table
opp_df


,metric,score,raw,confidence
0,potential access points,100.0,66 segments,medium
1,recreation diversity proxy,100.0,2 of 2 categories present,low
2,active recreation opportunities proxy,100.0,13 facilities within 800m,low
3,play area coverage,100.0,5 existing playgrounds within 800m,low
4,community gathering proxy,100.0,6 cultural/community POIs within 800m,low
5,nearest hospital distance,100.0,366.0 m,medium
6,nearest transit distance,89.5,442.0 m,medium


In [ ]:
# @title Constraints table
con_df


,metric,score,raw,confidence
0,impervious surface ratio,0.0,60.7 %,high
1,existing comfort amenities proxy,0.0,0 benches/toilets/water POIs within 800m,low
2,green area ratio vector,6.3,1.9 %,low
3,tree canopy coverage,20.7,3.1 %,high


---
# Module 08 — AI Recommendations

Turns Module 06/07's scores into **written analysis**: strengths, weaknesses, opportunities, risks, and
concrete design recommendations. Every statement here is generated by a rule-based engine walking the
actual `SCORES` and `CATEGORY_ROLLUPS` data - not a free-form language-model narrative - so every claim
traces back to a specific metric, its raw value, and (where applicable) the same cited planning standard
used to score it in Module 06. This keeps the output explainable and reproducible: the same data always
produces the same recommendations, and every recommendation can be traced to its source.

### Categories produced
- **Strengths** - metrics scoring well (>=70) with usable confidence
- **Weaknesses** - metrics scoring poorly (<40) with usable confidence
- **Opportunities** - specific, actionable improvements derived from each weakness, grounded in the same
  standard used to score that metric where one exists
- **Risks** - data gaps and low-confidence findings that should be verified before being treated as fact
  (distinct from weaknesses: a risk is about *uncertainty*, not a confirmed poor score)
- **Design recommendations** - a prioritized list combining the above into concrete next steps


In [ ]:
# @title Recommendation knowledge base

def _unit_str(unit: str) -> str:
    '''Formats a unit string for concatenation after a raw value - adds a leading space
    unless the unit is empty or already starts with one (e.g. some units read naturally
    with no space, like "%", while others like "m" or "segments" need one).'''
    if not unit:
        return ""
    return unit if unit.startswith(" ") or unit.startswith("%") else f" {unit}"


RECOMMENDATION_KB = {
    "tree_canopy_coverage": "Increase tree planting to close the gap toward the climate-adjusted canopy "
        "target. Prioritize native/climate-adapted species suited to the site's conditions (see climate_summary) "
        "to maximize survival and long-term canopy establishment.",
    "green_area_ratio_vector": "Expand vegetated/green surface area within the site boundary - convert "
        "underused hardscape or low-value paved areas to planted beds, lawns, or naturalized zones.",
    "impervious_surface_ratio": "Reduce impervious surface coverage through permeable paving, expanded "
        "planting beds, or bioswales, particularly in high-traffic paved zones. This also directly improves "
        "stormwater performance and reduces urban heat island effect.",
    "nearest_transit_distance": "Coordinate with local transit authorities on a stop closer to the site, "
        "or improve signed pedestrian routing from the nearest existing stop to reduce effective walking distance.",
    "potential_access_points": "Add or improve street-facing entrances - even modest, clearly marked "
        "secondary access points meaningfully improve perceived and actual accessibility for surrounding residents.",
    "recreation_diversity_proxy": "Introduce recreation typologies not already well-represented in the "
        "surrounding area, to complement (rather than duplicate) existing nearby offerings.",
    "active_recreation_opportunities_proxy": "Consider adding active-recreation infrastructure (fitness "
        "stations, sport courts) if the surrounding area is underserved relative to demand.",
    "play_area_coverage": "If existing nearby playgrounds are sparse, prioritize a well-equipped, "
        "age-differentiated play area in the design; if nearby playgrounds are already plentiful, consider "
        "a differentiated offering (e.g. nature play, sensory play) instead of duplicating what exists.",
    "existing_comfort_amenities_proxy": "Add basic comfort infrastructure - seating, shade, drinking water, "
        "and restrooms - particularly if existing nearby provision is sparse; these have an outsized impact "
        "on how long and how comfortably visitors use a park.",
    "community_gathering_proxy": "Introduce or strengthen community-oriented gathering space (plaza, "
        "flexible lawn, event space) especially if the site's context has limited existing cultural/community "
        "infrastructure nearby.",
    "nearest_hospital_distance": "No direct design mitigation - this is a fixed locational fact. Ensure the "
        "design supports clear, fast emergency vehicle access to compensate for distance where relevant.",
    "nearest_emergency_services_distance": "No direct design mitigation - this is a fixed locational fact. "
        "Ensure clear sightlines and signage support faster on-site emergency response.",
}

DATA_QUALITY_RISK_NOTES = {
    "low_completeness": "One or more KPI categories are scored from fewer data points than they were "
        "designed to use this run (see the completeness column in Module 06). Treat scores in those "
        "categories as provisional until a data-complete re-run confirms them.",
    "proxy_heavy": "Several scores rely on proxy metrics (existing nearby amenities standing in for "
        "design-dependent KPIs) rather than direct measurements. These describe the site's context, not "
        "its own future performance, and should not be read as predictions about a completed design.",
}


## Strengths & weaknesses

In [ ]:
# @title Generate strengths and weaknesses from SCORES

def generate_strengths_weaknesses():
    strengths, weaknesses = [], []
    for name, score_obj in SCORES.items():
        if score_obj.normalized_score is None or score_obj.weight == 0 or score_obj.confidence == "none":
            continue
        category = next((cat for cat, ms in METRICS_BY_KPI_CATEGORY.items() if name in ms), "")
        entry = {
            "metric": name, "category": category, "score": score_obj.normalized_score,
            "raw_value": score_obj.raw_value, "unit": score_obj.raw_unit,
            "confidence": score_obj.confidence, "standard": score_obj.standard_used,
        }
        if score_obj.normalized_score >= 70:
            strengths.append(entry)
        elif score_obj.normalized_score < 40:
            weaknesses.append(entry)
    return (sorted(strengths, key=lambda x: -x["score"]), sorted(weaknesses, key=lambda x: x["score"]))


STRENGTHS, WEAKNESSES = generate_strengths_weaknesses()

print(f"Strengths: {len(STRENGTHS)}")
for s in STRENGTHS:
    print(f"  + [{s['category']}] {s['metric'].replace('_',' ')}: {s['raw_value']}{_unit_str(s['unit'])} -> {s['score']}/100")

print(f"\nWeaknesses: {len(WEAKNESSES)}")
for w in WEAKNESSES:
    print(f"  - [{w['category']}] {w['metric'].replace('_',' ')}: {w['raw_value']}{_unit_str(w['unit'])} -> {w['score']}/100")


Strengths: 6
  + [Environmental] green area ratio vector: 97.9% -> 100.0/100
  + [Accessibility] potential access points: 4 segments -> 100.0/100
  + [Recreation] recreation diversity proxy: 2 of 2 categories present -> 100.0/100
  + [Recreation] play area coverage: 3 existing playgrounds within 800m -> 100.0/100
  + [Social] community gathering proxy: 5 cultural/community POIs within 800m -> 100.0/100
  + [Safety] nearest hospital distance: 128.0 m -> 100.0/100

Weaknesses: 3
  - [Sustainability] impervious surface ratio: 65.0% -> 0.0/100
  - [Recreation] existing comfort amenities proxy: 0 benches/toilets/water POIs within 800m -> 0.0/100
  - [Environmental] tree canopy coverage: 3.6% -> 24.0/100


## Opportunities (actionable, derived from weaknesses)

In [ ]:
# @title Generate design opportunities from weaknesses + the recommendation knowledge base

def generate_opportunities():
    opportunities = []
    for w in WEAKNESSES:
        recommendation = RECOMMENDATION_KB.get(w["metric"])
        if recommendation is None:
            continue
        opportunities.append({
            "category": w["category"],
            "based_on_metric": w["metric"],
            "current_value": f"{w['raw_value']}{_unit_str(w['unit'])}",
            "current_score": w["score"],
            "recommendation": recommendation,
            "standard_reference": w["standard"],
        })
    return opportunities


DESIGN_OPPORTUNITIES = generate_opportunities()

print(f"{len(DESIGN_OPPORTUNITIES)} design opportunities generated from {len(WEAKNESSES)} weaknesses "
      f"({len(WEAKNESSES) - len(DESIGN_OPPORTUNITIES)} weaknesses had no authored recommendation and were skipped).\n")
for o in DESIGN_OPPORTUNITIES:
    print(f"[{o['category']}] Based on {o['based_on_metric'].replace('_',' ')} "
          f"(currently {o['current_value']}, scoring {o['current_score']}/100):")
    print(f"    {o['recommendation']}")
    if o["standard_reference"]:
        print(f"    Reference: {o['standard_reference'][:120]}...")
    print()


3 design opportunities generated from 3 weaknesses (0 weaknesses had no authored recommendation and were skipped).

[Sustainability] Based on impervious surface ratio (currently 65.0%, scoring 0.0/100):
    Reduce impervious surface coverage through permeable paving, expanded planting beds, or bioswales, particularly in high-traffic paved zones. This also directly improves stormwater performance and reduces urban heat island effect.
    Reference: Arnold & Gibbons (1996) / Booth & Jackson (1997): measurable watershed degradation begins around 10% impervious cover; >...

[Recreation] Based on existing comfort amenities proxy (currently 0 benches/toilets/water POIs within 800m, scoring 0.0/100):
    Add basic comfort infrastructure - seating, shade, drinking water, and restrooms - particularly if existing nearby provision is sparse; these have an outsized impact on how long and how comfortably visitors use a park.
    Reference: No universal standard; illustrative 0-10 scale based on gen

## Risks (data-quality and confidence caveats)

In [ ]:
# @title Generate risk notes from data completeness and proxy-reliance patterns

def generate_risks():
    risks = []

    low_completeness_cats = [cat for cat, result in CATEGORY_ROLLUPS.items()
                              if result and result.get("completeness") is not None and result["completeness"] < 0.99]
    if low_completeness_cats:
        risks.append({
            "type": "data_completeness",
            "affected_categories": low_completeness_cats,
            "note": DATA_QUALITY_RISK_NOTES["low_completeness"],
        })

    proxy_heavy_cats = []
    for cat, metrics_in_cat in METRICS_BY_KPI_CATEGORY.items():
        scored_names = [n for n in metrics_in_cat if n in SCORES and SCORES[n].normalized_score is not None
                         and SCORES[n].weight > 0]
        if not scored_names:
            continue
        proxy_names = [n for n in scored_names if METRICS.get(n) and METRICS.get(n).tier == "proxy"]
        if len(proxy_names) == len(scored_names):
            proxy_heavy_cats.append(cat)
    if proxy_heavy_cats:
        risks.append({
            "type": "proxy_reliance",
            "affected_categories": proxy_heavy_cats,
            "note": DATA_QUALITY_RISK_NOTES["proxy_heavy"],
        })

    low_confidence_contributors = [
        name for name, s in SCORES.items()
        if s.confidence == "low" and s.normalized_score is not None and s.weight > 0
    ]
    if low_confidence_contributors:
        risks.append({
            "type": "low_confidence_metrics",
            "affected_categories": sorted(set(next((cat for cat, ms in METRICS_BY_KPI_CATEGORY.items() if n in ms), "")
                                                for n in low_confidence_contributors)),
            "note": f"{len(low_confidence_contributors)} metric(s) contributing to scores this run have "
                    f"'low' confidence (coarse proxies or low-resolution fallback data sources): "
                    f"{', '.join(n.replace('_',' ') for n in low_confidence_contributors)}.",
        })

    return risks


RISKS = generate_risks()

print(f"{len(RISKS)} risk note(s) generated:\n")
for r in RISKS:
    print(f"[{r['type']}] affects: {', '.join(r['affected_categories'])}")
    print(f"    {r['note']}")
    print()


3 risk note(s) generated:

[data_completeness] affects: Site Context, Accessibility, Safety, Smart City
    One or more KPI categories are scored from fewer data points than they were designed to use this run (see the completeness column in Module 06). Treat scores in those categories as provisional until a data-complete re-run confirms them.

[proxy_reliance] affects: Accessibility, Recreation, Social, Safety
    Several scores rely on proxy metrics (existing nearby amenities standing in for design-dependent KPIs) rather than direct measurements. These describe the site's context, not its own future performance, and should not be read as predictions about a completed design.

[low_confidence_metrics] affects: Environmental, Recreation, Social
    6 metric(s) contributing to scores this run have 'low' confidence (coarse proxies or low-resolution fallback data sources): green area ratio vector, recreation diversity proxy, active recreation opportunities proxy, play area coverage, existi

## Consolidated design recommendations

In [ ]:
# @title Prioritized design recommendations combining opportunities + risk-aware caveats

def generate_prioritized_recommendations():
    ranked = sorted(DESIGN_OPPORTUNITIES, key=lambda o: (o["current_score"],
                     {"high": 0, "medium": 1, "low": 2}.get(
                         SCORES[o["based_on_metric"]].confidence, 3)))
    return ranked


PRIORITIZED_RECOMMENDATIONS = generate_prioritized_recommendations()

print(f"=== Prioritized Design Recommendations ({len(PRIORITIZED_RECOMMENDATIONS)}) ===\n")
for i, o in enumerate(PRIORITIZED_RECOMMENDATIONS, 1):
    confidence = SCORES[o["based_on_metric"]].confidence
    print(f"{i}. [{o['category']}, confidence={confidence}] {o['recommendation']}")
    print(f"   Basis: {o['based_on_metric'].replace('_',' ')} = {o['current_value']} (scored {o['current_score']}/100)")
    print()

if RISKS:
    print("=== Caveats to keep in mind when acting on the above ===\n")
    for r in RISKS:
        print(f"- {r['note']}")


=== Prioritized Design Recommendations (3) ===

1. [Sustainability, confidence=high] Reduce impervious surface coverage through permeable paving, expanded planting beds, or bioswales, particularly in high-traffic paved zones. This also directly improves stormwater performance and reduces urban heat island effect.
   Basis: impervious surface ratio = 65.0% (scored 0.0/100)

2. [Recreation, confidence=low] Add basic comfort infrastructure - seating, shade, drinking water, and restrooms - particularly if existing nearby provision is sparse; these have an outsized impact on how long and how comfortably visitors use a park.
   Basis: existing comfort amenities proxy = 0 benches/toilets/water POIs within 800m (scored 0.0/100)

3. [Environmental, confidence=high] Increase tree planting to close the gap toward the climate-adjusted canopy target. Prioritize native/climate-adapted species suited to the site's conditions (see climate_summary) to maximize survival and long-term canopy establishmen

## Summary report table

In [ ]:
# @title Combined analysis summary table
analysis_rows = []
for s in STRENGTHS:
    analysis_rows.append({"type": "Strength", "category": s["category"], "metric": s["metric"],
                           "detail": f"{s['raw_value']}{_unit_str(s['unit'])} (score {s['score']}/100)"})
for w in WEAKNESSES:
    analysis_rows.append({"type": "Weakness", "category": w["category"], "metric": w["metric"],
                           "detail": f"{w['raw_value']}{_unit_str(w['unit'])} (score {w['score']}/100)"})
for o in DESIGN_OPPORTUNITIES:
    analysis_rows.append({"type": "Opportunity", "category": o["category"], "metric": o["based_on_metric"],
                           "detail": o["recommendation"][:100] + "..."})
for r in RISKS:
    analysis_rows.append({"type": "Risk", "category": ", ".join(r["affected_categories"]), "metric": r["type"],
                           "detail": r["note"][:100] + "..."})

analysis_summary_df = pd.DataFrame(analysis_rows)
analysis_summary_df


,type,category,metric,detail
0,Strength,Environmental,green_area_ratio_vector,97.9% (score 100.0/100)
1,Strength,Accessibility,potential_access_points,4 segments (score 100.0/100)
2,Strength,Recreation,recreation_diversity_proxy,2 of 2 categories present (score 100.0/100)
3,Strength,Recreation,play_area_coverage,3 existing playgrounds within 800m (score 100....
4,Strength,Social,community_gathering_proxy,5 cultural/community POIs within 800m (score 1...
5,Strength,Safety,nearest_hospital_distance,128.0 m (score 100.0/100)
6,Weakness,Sustainability,impervious_surface_ratio,65.0% (score 0.0/100)
7,Weakness,Recreation,existing_comfort_amenities_proxy,0 benches/toilets/water POIs within 800m (scor...
8,Weakness,Environmental,tree_canopy_coverage,3.6% (score 24.0/100)
9,Opportunity,Sustainability,impervious_surface_ratio,Reduce impervious surface coverage through per...


---
# Module 09 — Report Generation

Exports the complete analysis (site info, scores, category rollups, strengths/weaknesses/opportunities/
risks, recommendations) into four deliverable formats:

- **PDF report** - formatted, readable document for sharing with stakeholders/clients
- **Excel workbook** - multi-sheet, for further analysis or hand-off to a planning team
- **CSV** - raw scores table, for import into other tools
- **HTML dashboard** - self-contained interactive file (works offline, no server needed)

All four are generated from the same underlying data (`SCORES`, `CATEGORY_ROLLUPS`,
`PRIORITIZED_RECOMMENDATIONS`, etc.) so they never drift out of sync with each other or with what's
shown earlier in the notebook. Each generator degrades gracefully if its library isn't available,
consistent with the rest of this notebook.


In [ ]:
# @title Report output setup
import os as _os

REPORT_OUTPUT_DIR = "/content/park_analysis_reports" if _os.path.isdir("/content") else "/tmp/park_analysis_reports"
_os.makedirs(REPORT_OUTPUT_DIR, exist_ok=True)

REPORT_BASENAME = SITE.name.replace(" ", "_").replace("/", "-") if SITE.name else "park_site_analysis"
print(f"Reports will be written to: {REPORT_OUTPUT_DIR}")


Reports will be written to: /content/park_analysis_reports


## CSV export

In [ ]:
# @title Export scores to CSV
csv_path = _os.path.join(REPORT_OUTPUT_DIR, f"{REPORT_BASENAME}_scores.csv")
try:
    scores_df.to_csv(csv_path, index=False)
    print(f"✓ CSV written: {csv_path}")
except Exception as e:
    print(f"✗ CSV export failed: {e}")
    csv_path = None


✓ CSV written: /content/park_analysis_reports/حديقة_الصفا_2_scores.csv


## Excel workbook export

In [ ]:
# @title Export multi-sheet Excel workbook
xlsx_path = _os.path.join(REPORT_OUTPUT_DIR, f"{REPORT_BASENAME}_report.xlsx")

def export_excel():
    try:
        with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
            overview_df = pd.DataFrame([
                {"field": "Site name", "value": SITE.name},
                {"field": "Centroid (lat, lon)", "value": f"{SITE.centroid[0]:.6f}, {SITE.centroid[1]:.6f}"},
                {"field": "Site area (m2)", "value": SITE.area_m2()},
                {"field": "Overall score", "value": OVERALL_SCORE},
                {"field": "Report generated", "value": dt.datetime.now(dt.timezone.utc).isoformat() + " UTC"},
            ])
            overview_df.to_excel(writer, sheet_name="Overview", index=False)

            category_rollup_df.to_excel(writer, sheet_name="Category Scores", index=False)
            scores_df.to_excel(writer, sheet_name="Metric Scores", index=False)
            metrics_summary_df.to_excel(writer, sheet_name="Raw Metrics", index=False)
            analysis_summary_df.to_excel(writer, sheet_name="Strengths-Weaknesses-Risks", index=False)

            rec_rows = [{
                "priority": i, "category": o["category"], "metric": o["based_on_metric"],
                "current_value": o["current_value"], "current_score": o["current_score"],
                "confidence": SCORES[o["based_on_metric"]].confidence,
                "recommendation": o["recommendation"], "standard_reference": o["standard_reference"],
            } for i, o in enumerate(PRIORITIZED_RECOMMENDATIONS, 1)]
            pd.DataFrame(rec_rows).to_excel(writer, sheet_name="Recommendations", index=False)

            data_collection_df = STORE.summary()
            data_collection_df.to_excel(writer, sheet_name="Data Sources", index=False)

        print(f"✓ Excel workbook written: {xlsx_path}")
        return xlsx_path
    except Exception as e:
        print(f"✗ Excel export failed: {e}")
        return None


xlsx_path = export_excel()


✗ Excel export failed: name 'metrics_summary_df' is not defined


## PDF report export

In [ ]:
# @title Export formatted PDF report
pdf_path = _os.path.join(REPORT_OUTPUT_DIR, f"{REPORT_BASENAME}_report.pdf")

def export_pdf():
    try:
        from reportlab.lib.pagesizes import letter
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib import colors
        from reportlab.lib.units import inch
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak
    except ImportError:
        print("✗ reportlab not available - skipping PDF export.")
        return None

    try:
        doc = SimpleDocTemplate(pdf_path, pagesize=letter,
                                  topMargin=0.75*inch, bottomMargin=0.75*inch)
        styles = getSampleStyleSheet()
        title_style = ParagraphStyle("TitleCustom", parent=styles["Title"], fontSize=20, spaceAfter=6)
        h2_style = ParagraphStyle("H2Custom", parent=styles["Heading2"], spaceBefore=14, spaceAfter=6)
        body_style = styles["BodyText"]
        story = []

        story.append(Paragraph(f"Site Analysis Report: {SITE.name}", title_style))
        story.append(Paragraph(f"Generated {dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}", body_style))
        story.append(Spacer(1, 12))

        overall_str = f"{OVERALL_SCORE}/100" if OVERALL_SCORE is not None else "N/A"
        story.append(Paragraph(f"<b>Overall Score: {overall_str}</b>", h2_style))
        area = SITE.area_m2()
        if area:
            story.append(Paragraph(f"Site area: {area:,.0f} m2 ({area/10_000:.2f} ha)", body_style))
        story.append(Paragraph(f"Location: {SITE.centroid[0]:.6f}, {SITE.centroid[1]:.6f}", body_style))

        story.append(Paragraph("Category Scores", h2_style))
        cat_table_data = [["Category", "Score", "Data Completeness"]]
        for _, row in category_rollup_df.iterrows():
            score_str = f"{row['score']:.1f}" if pd.notna(row["score"]) else "N/A"
            cat_table_data.append([row["kpi_category"], score_str, row["data_completeness"] or "-"])
        cat_table = Table(cat_table_data, colWidths=[2.5*inch, 1.5*inch, 2*inch])
        cat_table.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#2E7D32")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F5F5F5")]),
        ]))
        story.append(cat_table)

        story.append(Paragraph("Strengths", h2_style))
        for s in STRENGTHS:
            story.append(Paragraph(f"- {s['metric'].replace('_',' ')}: {s['raw_value']}{_unit_str(s['unit'])} "
                                     f"(score {s['score']}/100)", body_style))

        story.append(Paragraph("Weaknesses", h2_style))
        for w in WEAKNESSES:
            story.append(Paragraph(f"- {w['metric'].replace('_',' ')}: {w['raw_value']}{_unit_str(w['unit'])} "
                                     f"(score {w['score']}/100)", body_style))

        story.append(PageBreak())
        story.append(Paragraph("Prioritized Design Recommendations", h2_style))
        for i, o in enumerate(PRIORITIZED_RECOMMENDATIONS, 1):
            story.append(Paragraph(f"<b>{i}. [{o['category']}]</b> {o['recommendation']}", body_style))
            story.append(Paragraph(f"<i>Basis: {o['based_on_metric'].replace('_',' ')} = {o['current_value']} "
                                     f"(scored {o['current_score']}/100)</i>", body_style))
            story.append(Spacer(1, 8))

        if RISKS:
            story.append(Paragraph("Risks & Caveats", h2_style))
            for r in RISKS:
                story.append(Paragraph(f"- {r['note']}", body_style))

        doc.build(story)
        print(f"✓ PDF written: {pdf_path}")
        return pdf_path
    except Exception as e:
        print(f"✗ PDF export failed: {e}")
        return None


pdf_path = export_pdf()


✓ PDF written: /content/park_analysis_reports/حديقة_الصفا_2_report.pdf


## HTML dashboard export

In [ ]:
# @title Export a self-contained interactive HTML dashboard
html_path = _os.path.join(REPORT_OUTPUT_DIR, f"{REPORT_BASENAME}_dashboard.html")

def export_html():
    try:
        radar_html = fig_overall_radar.to_html(full_html=False, include_plotlyjs="cdn")
        bar_html = fig_bar.to_html(full_html=False, include_plotlyjs=False)

        cat_rows_html = "".join(
            f"<tr><td>{row['kpi_category']}</td><td>{row['score'] if pd.notna(row['score']) else 'N/A'}</td>"
            f"<td>{row['data_completeness'] or '-'}</td></tr>"
            for _, row in category_rollup_df.iterrows()
        )
        rec_html = "".join(
            f"<li><b>[{o['category']}]</b> {o['recommendation']}<br>"
            f"<i>Basis: {o['based_on_metric'].replace('_',' ')} = {o['current_value']} "
            f"(scored {o['current_score']}/100)</i></li>"
            for o in PRIORITIZED_RECOMMENDATIONS
        )
        risk_html = "".join(f"<li>{r['note']}</li>" for r in RISKS)

        overall_str = f"{OVERALL_SCORE}/100" if OVERALL_SCORE is not None else "N/A"
        title_line = f"Site Analysis: {SITE.name}"

        html_parts = [
            "<!DOCTYPE html><html><head><meta charset='utf-8'>",
            f"<title>{title_line}</title>",
            "<style>",
            "body { font-family: -apple-system, sans-serif; max-width: 900px; margin: 40px auto; padding: 0 20px; background: #121212; color: #eee; }",
            "h1, h2 { color: #4CAF50; }",
            "table { border-collapse: collapse; width: 100%; margin: 16px 0; }",
            "th, td { border: 1px solid #444; padding: 8px 12px; text-align: left; }",
            "th { background: #2E7D32; color: white; }",
            "tr:nth-child(even) { background: #1e1e1e; }",
            "li { margin-bottom: 12px; }",
            "</style></head><body>",
            f"<h1>Site Analysis Report: {SITE.name}</h1>",
            f"<p>Generated {dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}</p>",
            f"<h2>Overall Score: {overall_str}</h2>",
            radar_html,
            "<h2>Category Scores</h2>",
            f"<table><tr><th>Category</th><th>Score</th><th>Data Completeness</th></tr>{cat_rows_html}</table>",
            bar_html,
            "<h2>Prioritized Recommendations</h2>",
            f"<ol>{rec_html}</ol>",
            "<h2>Risks &amp; Caveats</h2>",
            f"<ul>{risk_html}</ul>",
            "</body></html>",
        ]
        html_content = "\n".join(html_parts)

        with open(html_path, "w", encoding="utf-8") as f:
            f.write(html_content)
        print(f"✓ HTML dashboard written: {html_path}")
        return html_path
    except Exception as e:
        print(f"✗ HTML export failed: {e}")
        return None


html_path = export_html()


✓ HTML dashboard written: /content/park_analysis_reports/حديقة_الصفا_2_dashboard.html


## Export summary

In [ ]:
# @title Summary of generated files
exports = {"CSV": csv_path, "Excel": xlsx_path, "PDF": pdf_path, "HTML": html_path}
print("Report generation summary:")
for fmt, path in exports.items():
    if path and _os.path.exists(path):
        size_kb = _os.path.getsize(path) / 1024
        print(f"  ✓ {fmt:8s}: {path} ({size_kb:.1f} KB)")
    else:
        print(f"  ✗ {fmt:8s}: not generated")


Report generation summary:
  ✓ CSV     : /content/park_analysis_reports/حديقة_الصفا_2_scores.csv (2.0 KB)
  ✗ Excel   : not generated
  ✓ PDF     : /content/park_analysis_reports/حديقة_الصفا_2_report.pdf (4.8 KB)
  ✓ HTML    : /content/park_analysis_reports/حديقة_الصفا_2_dashboard.html (18.9 KB)


---
# Module 10 — AI-Generated Design Concepts

Turns Module 08's prioritized recommendations into **spatial design concepts** — alternative zoning
layouts placed within the real site boundary, delivered as vector (GeoJSON + SVG) and raster (PNG) files,
with an optional AI-illustrated concept render.

### Design principle: AI proposes intent, code owns geometry
Same discipline as Module 01's boundary resolver: the AI step never invents raw coordinates. Gemini
proposes a **zoning program** — named zones with a purpose, an approximate area share, and a rough
compass-direction location within the site — as structured JSON. All of that gets turned into actual,
non-overlapping Shapely polygons that are guaranteed to sit inside the real site boundary and avoid
existing buildings/water, by this notebook's own deterministic geometry code, not by the AI.

### What this module produces, per concept
1. A **full master-plan-style zoning program** by default: zones addressing current weaknesses
   (e.g. "Expanded tree canopy — north edge") PLUS standard park elements — entrance, circulation/paths,
   seating & comfort, restrooms/services, sports/fitness — drawn from this project's own analysis
   document's Park Program section, not proposed only when something scores poorly. (Set
   `DESIGN_GENERATOR_CONFIG["concept_scope"] = "weakness_driven"` to go back to a narrower,
   only-fix-what's-scored-poorly scope instead.)
2. GeoJSON (real geospatial vector) + SVG (styled plan) + PNG (same plan, raster) — all from one shared
   geometry, so the three formats can never disagree with each other
3. Optionally, an AI-illustrated "artist's impression" render (off by default — see the note in the AI
   image cell below on why this is opt-in)
4. A **what-if rescoring**: how much the site's score would move if this concept were built, computed only
   for the metrics a park design can actually influence — locational metrics (transit distance, hospital
   distance, road density, etc.) are explicitly held fixed and labeled as such, not silently ignored.
   Standard amenity zones (paths, entrance, restrooms) don't map to any scored metric and correctly
   contribute no score movement — they're about completeness of the plan, not measurable KPIs.

### Honesty about the geometry algorithm
Zone placement uses a simple 3×3 compass-direction grid allocator — a **schematic heuristic**, not a
professional site-planning algorithm. It produces plausible, non-overlapping, boundary-respecting starting
layouts for discussion. Treat zone shapes and exact positions as illustrative, not final design.


In [ ]:
# @title Design generator configuration
import os as _os_design
import math as _math_design

DESIGN_OUTPUT_DIR = _os_design.path.join(REPORT_OUTPUT_DIR, "design_concepts") if "REPORT_OUTPUT_DIR" in dir() \
                     else ("/content/design_concepts" if _os_design.path.isdir("/content") else "/tmp/design_concepts")
_os_design.makedirs(DESIGN_OUTPUT_DIR, exist_ok=True)

DESIGN_GENERATOR_CONFIG = {
    # "auto_priority_groups": derive 2-3 concepts automatically from the current recommendations'
    # categories. "manual": use MANUAL_RECOMMENDATION_GROUPS below instead.
    "grouping_mode": "auto_priority_groups",

    "enable_ai_zoning": True,     # Gemini text-only zoning proposal - free tier, same client as Module 01
    "enable_ai_render": True,      # Gemini image generation ("artist's impression") - now on by
                                     # default since it was explicitly and repeatedly requested;
                                     # uses your GEMINI_API_KEY's free-tier image quota per concept
                                     # generated. Set to False to skip this and save quota/time.

    # "weakness_driven": zones only for metrics currently scoring as weaknesses (narrow, minimal).
    # "full_master_plan": also include standard park-program elements (paths, seating, entrance,
    # restrooms, sports/fitness) regardless of current scores, for a fuller concept plan. This is
    # the default per the project's direction - see the markdown note above.
    "concept_scope": "full_master_plan",

    "max_zones_per_concept": 9,   # raised from the weakness-driven default (6) to comfortably fit
                                    # both priority (weakness-driven) zones and standard elements
    "max_total_zone_area_fraction": 0.65,  # a full master plan reasonably programs more of the site
                                              # than a narrow weakness-patching pass would; still
                                              # leaves ~35% for unprogrammed lawn/buffer/circulation
                                              # not explicitly zoned by this schematic tool
    "max_single_zone_area_fraction": 0.22,  # ALSO never let one individual zone claim more than ~22%
                                              # of the site on its own, even if it's the only zone in a
                                              # concept - avoids an unrealistic single oversized zone
                                              # when a concept only addresses one or two recommendations

    "grid_size": 3,  # 3x3 compass-direction grid for schematic zone placement (N/NE/E/SE/S/SW/W/NW/C)
}

# Only used when grouping_mode == "manual" - a list of lists of metric names (must match
# PRIORITIZED_RECOMMENDATIONS' based_on_metric values) defining exactly which recommendations
# feed each concept. Example: [["impervious_surface_ratio", "tree_canopy_coverage"], ["community_gathering_proxy"]]
MANUAL_RECOMMENDATION_GROUPS = []

print(f"Design outputs will be written to: {DESIGN_OUTPUT_DIR}")


Design outputs will be written to: /content/park_analysis_reports/design_concepts


In [ ]:
# @title Zone taxonomy, illustrative planning assumptions (clearly labeled, cited where possible)

# Maps a Module 08 recommendation's based_on_metric to the zone purpose that would address it.
# Only metrics with a genuinely areal/spatial design response are included - locational metrics
# (nearest_transit_distance, nearest_hospital_distance, etc.) have no zone response and are
# intentionally absent here; they're held fixed in the what-if rescoring below.
ZONE_PURPOSE_FOR_METRIC = {
    "tree_canopy_coverage": "tree_canopy_expansion",
    "green_area_ratio_vector": "green_space_expansion",
    "impervious_surface_ratio": "permeable_surface_conversion",
    "play_area_coverage": "play_area",
    "community_gathering_proxy": "community_gathering_plaza",
}

ZONE_PURPOSE_LABELS = {
    "tree_canopy_expansion": "Expanded Tree Canopy",
    "green_space_expansion": "Green Space Expansion",
    "permeable_surface_conversion": "Permeable Surface Conversion",
    "play_area": "Play Area",
    "community_gathering_plaza": "Community Gathering Plaza",
    "seating_comfort_area": "Seating & Comfort Area",
    "entrance_plaza": "Entrance Plaza",
    "circulation_paths": "Circulation & Path Network",
    "restrooms_and_services": "Restrooms & Services",
    "sports_recreation_zone": "Sports & Fitness Zone",
    "picnic_area": "Picnic Area",
}

ZONE_PURPOSE_COLORS = {
    "tree_canopy_expansion": "#2E7D32",
    "green_space_expansion": "#66BB6A",
    "permeable_surface_conversion": "#8D6E63",
    "play_area": "#FB8C00",
    "community_gathering_plaza": "#5E35B1",
    "seating_comfort_area": "#1E88E5",
    "entrance_plaza": "#FDD835",
    "circulation_paths": "#9E9E9E",
    "restrooms_and_services": "#546E7A",
    "sports_recreation_zone": "#00897B",
    "picnic_area": "#C0CA33",
}

# Standard park-program elements NOT tied to any specific scored metric - included to produce a
# fuller master-plan-style concept (paths, seating, entrance, services, sports/fitness) rather
# than only patching current weaknesses. Drawn from this project's own analysis document (Section
# 3: Arrival & Access, Movement Network, Sports & Recreation, Community Spaces, Comfort, Amenities)
# rather than an arbitrary list, so the taxonomy stays grounded in the project's own brief. Each
# has an illustrative default area fraction used by the non-AI fallback template below - these are
# reasonable planning proportions for a small urban park, NOT cited engineering standards like
# Module 06's thresholds; a landscape architect should treat them as a starting point to override.
STANDARD_PARK_ELEMENTS = {
    "entrance_plaza": 0.04,
    "circulation_paths": 0.08,   # a schematic area representing path/circulation network -
                                    # NOT a routed path centerline; this module doesn't do path
                                    # routing, only area allocation, consistent with everything
                                    # else in Module 10 being a schematic space-allocation tool.
    "seating_comfort_area": 0.05,
    "restrooms_and_services": 0.03,
    "sports_recreation_zone": 0.10,
    "picnic_area": 0.05,
}

ALL_ZONE_PURPOSES = set(ZONE_PURPOSE_FOR_METRIC.values()) | set(STANDARD_PARK_ELEMENTS.keys())


# --- Illustrative planning assumptions used in the what-if rescoring (Stage 4) ---
# These are NOT cited engineering standards like Module 06's Arnold & Gibbons / American Forests
# thresholds - they are reasonable, clearly-labeled estimates used only to translate a proposed
# zone's area into a projected metric value. A landscape architect should treat every number here
# as a starting assumption to override, not a fact.
CANOPY_MATURITY_COVERAGE_ASSUMPTION = 0.6   # a newly planted "tree canopy" zone is assumed to reach
                                              # ~60% canopy closure at maturity - a commonly-used rule
                                              # of thumb in planting design, not a rigid standard;
                                              # actual results depend heavily on species/spacing/years.
IMPERVIOUS_CONVERSION_EFFECTIVENESS = 0.85   # a "permeable conversion" zone is assumed to become 85%
                                               # pervious (not 100%) - accounts for paths/edges within
                                               # the zone that typically remain hard surface.
EVENT_SPACE_M2_PER_PERSON = 3.0   # general event/gathering-space planning guidance converges on
                                    # roughly 2-4 m² of usable space per person for casual gathering/
                                    # event use; 3.0 is the midpoint, used as a working estimate.
SEATING_SPACE_M2_PER_PERSON = 6.0  # casual park seating (benches, lawn seating with circulation) is
                                     # assumed sparser than a standing/gathering crowd - no single
                                     # authoritative citation for this figure; treat as illustrative.

print("Zone taxonomy and planning assumptions loaded.")


Zone taxonomy and planning assumptions loaded.


In [ ]:
# @title Design concept data structures

@dataclass
class DesignZone:
    id: str
    name: str
    purpose: str                      # key into ZONE_PURPOSE_LABELS/ZONE_PURPOSE_COLORS
    target_area_fraction: float          # AI-proposed or fallback-computed, 0-1 of site area
    location_hint: str                    # "N"|"NE"|"E"|"SE"|"S"|"SW"|"W"|"NW"|"C"
    addresses_metric: Optional[str] = None  # the Module 05/06 metric name this zone is meant to
                                               # improve, if any - standard amenity zones (paths,
                                               # entrance, restrooms) address no specific metric
    geometry: Any = None                    # Shapely polygon in SITE_UTM_CRS, filled in by Stage 2
    actual_area_m2: Optional[float] = None    # the real delivered area, may differ from the target
    shortfall_note: str = ""


@dataclass
class DesignConcept:
    id: str
    name: str
    source_recommendations: list             # list of based_on_metric names this concept addresses
    zones: list = field(default_factory=list)  # list[DesignZone]
    geojson_path: Optional[str] = None
    svg_path: Optional[str] = None
    png_path: Optional[str] = None
    ai_render_path: Optional[str] = None
    projected_scores: dict = field(default_factory=dict)      # metric_name -> new Score
    projected_category_rollups: dict = field(default_factory=dict)
    projected_overall_score: Optional[float] = None


print("DesignZone and DesignConcept data structures defined.")


DesignZone and DesignConcept data structures defined.


## Group recommendations into concept alternatives

In [ ]:
# @title Group PRIORITIZED_RECOMMENDATIONS into 2-3 concept alternatives (or use manual groups)

def _areal_recommendations() -> list:
    '''Only recommendations whose metric has a zone response (see ZONE_PURPOSE_FOR_METRIC) can
    drive a design concept - locational recommendations are filtered out here rather than
    silently producing an empty/meaningless zone later.'''
    return [o for o in PRIORITIZED_RECOMMENDATIONS if o["based_on_metric"] in ZONE_PURPOSE_FOR_METRIC]


def group_recommendations_for_concepts(config: dict) -> list:
    '''Returns a list of {"concept_name": str, "recommendations": [rec_dict, ...]} groups.'''
    areal_recs = _areal_recommendations()
    if not areal_recs:
        return []

    if config["grouping_mode"] == "manual" and MANUAL_RECOMMENDATION_GROUPS:
        groups = []
        for i, metric_list in enumerate(MANUAL_RECOMMENDATION_GROUPS, 1):
            recs = [o for o in areal_recs if o["based_on_metric"] in metric_list]
            if recs:
                groups.append({"concept_name": f"Manual Concept {i}", "recommendations": recs})
        return groups

    # auto_priority_groups: split by KPI category clusters, then add a "Balanced" concept
    # combining the single highest-priority recommendation from each cluster.
    environmental_cluster = {"tree_canopy_coverage", "green_area_ratio_vector", "impervious_surface_ratio"}
    social_cluster = {"play_area_coverage", "community_gathering_proxy"}

    env_recs = [o for o in areal_recs if o["based_on_metric"] in environmental_cluster]
    social_recs = [o for o in areal_recs if o["based_on_metric"] in social_cluster]

    groups = []
    if env_recs:
        groups.append({"concept_name": "Environmental & Sustainability Focus", "recommendations": env_recs})
    if social_recs:
        groups.append({"concept_name": "Recreation & Social Focus", "recommendations": social_recs})
    if env_recs and social_recs:
        groups.append({"concept_name": "Balanced", "recommendations": areal_recs})
    elif not groups:
        # neither cluster matched (shouldn't normally happen given ZONE_PURPOSE_FOR_METRIC's
        # coverage, but handle it rather than silently returning nothing)
        groups.append({"concept_name": "Balanced", "recommendations": areal_recs})

    return groups


RECOMMENDATION_GROUPS = group_recommendations_for_concepts(DESIGN_GENERATOR_CONFIG)

if not RECOMMENDATION_GROUPS:
    print("No design-addressable recommendations found this run (either no weaknesses were scored, "
          "or none map to a zone-able metric) - Module 10 has nothing to generate. This can happen "
          "on a site that's already performing well against the scored criteria.")
else:
    print(f"{len(RECOMMENDATION_GROUPS)} concept group(s) to generate:")
    for g in RECOMMENDATION_GROUPS:
        metric_list = ", ".join(o["based_on_metric"] for o in g["recommendations"])
        print(f"  - {g['concept_name']}: {metric_list}")


1 concept group(s) to generate:
  - Environmental & Sustainability Focus: impervious_surface_ratio, tree_canopy_coverage


## Stage 1 — AI-assisted zoning program (Gemini, text-only, free tier)

In [ ]:
# @title Generate a zoning program per concept group (Gemini, with a deterministic fallback)

def _existing_constraints_summary() -> str:
    '''Short text description of what's already on the site, for the AI prompt - buildings and
    water bodies the proposed zones should avoid overlapping.'''
    parts = []
    buildings = PROCESSED.get_best("buildings", ["osm", "esri_living_atlas"])
    if buildings and len(buildings.data) > 0:
        parts.append(f"{len(buildings.data)} existing building(s) on site")
    water = PROCESSED.get_best("water_bodies", ["osm"])
    if water and len(water.data) > 0:
        parts.append(f"{len(water.data)} existing water feature(s) on site")
    return "; ".join(parts) if parts else "no significant existing structures recorded on site"


def _fallback_zoning_program(recommendations: list, config: dict) -> list:
    '''Deterministic zone proposal used when AI zoning is disabled or fails. In "full_master_plan"
    scope (the default), combines priority zones addressing the group's actual weaknesses with a
    baseline set of standard park elements (paths, seating, entrance, restrooms, etc.) from
    STANDARD_PARK_ELEMENTS, so a missing/failed AI call still produces a fuller concept rather than
    just weakness patches. In "weakness_driven" scope, only the priority zones are included.'''
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "C"]
    zones = []

    # Priority zones: one per recommendation, sized larger than standard elements since these
    # address confirmed weaknesses
    n_priority = min(len(recommendations), config["max_zones_per_concept"])
    priority_budget = config["max_total_zone_area_fraction"] * 0.6  # majority of the budget to priority zones
    area_each_priority = min(priority_budget / max(n_priority, 1), config["max_single_zone_area_fraction"])
    for i, rec in enumerate(recommendations[:n_priority]):
        purpose = ZONE_PURPOSE_FOR_METRIC[rec["based_on_metric"]]
        zones.append({
            "name": ZONE_PURPOSE_LABELS.get(purpose, purpose), "purpose": purpose,
            "target_area_fraction": round(area_each_priority, 3), "location_hint": directions[i % len(directions)],
            "addresses_metric": rec["based_on_metric"],
        })

    if config.get("concept_scope", "full_master_plan") == "full_master_plan":
        remaining_slots = config["max_zones_per_concept"] - len(zones)
        remaining_budget = config["max_total_zone_area_fraction"] - sum(z["target_area_fraction"] for z in zones)
        standard_items = list(STANDARD_PARK_ELEMENTS.items())[:max(remaining_slots, 0)]
        for j, (purpose, default_fraction) in enumerate(standard_items):
            fraction = min(default_fraction, config["max_single_zone_area_fraction"], max(remaining_budget, 0))
            if fraction <= 0:
                break
            zones.append({
                "name": ZONE_PURPOSE_LABELS.get(purpose, purpose), "purpose": purpose,
                "target_area_fraction": round(fraction, 3),
                "location_hint": directions[(n_priority + j) % len(directions)],
                "addresses_metric": None,
            })
            remaining_budget -= fraction

    return zones


def generate_design_program(recommendations: list, site_context, config: dict) -> list:
    '''Returns a list of zone spec dicts: {"name","purpose","target_area_fraction","location_hint",
    "addresses_metric"}. Uses Gemini when enable_ai_zoning is True and a client is available;
    otherwise (or on any API failure) falls back to the deterministic template above - this module
    never fails outright just because AI zoning was unavailable this run.'''
    if not config.get("enable_ai_zoning", False):
        return _fallback_zoning_program(recommendations, config)

    client = _get_gemini_client()
    if client is None:
        logger.warning("Gemini client unavailable for zoning generation - using deterministic fallback.")
        return _fallback_zoning_program(recommendations, config)

    rec_lines = "\n".join(
        f"- metric: {r['based_on_metric']}, current value: {r['current_value']} (score {r['current_score']}/100), "
        f"recommendation: {r['recommendation']}"
        for r in recommendations
    )
    full_scope = config.get("concept_scope", "full_master_plan") == "full_master_plan"
    purpose_options = ", ".join(sorted(ALL_ZONE_PURPOSES if full_scope else ZONE_PURPOSE_FOR_METRIC.values()))
    scope_instruction = (
        "Propose a fuller, realistic small-park master plan: include zones that directly address the "
        "weaknesses below AND standard park elements appropriate for a small urban park (e.g. an entrance "
        "area, circulation/path space, seating/comfort areas, restrooms/services, sports or fitness space) "
        "even where the analysis didn't flag a specific weakness for them. Zones addressing an actual "
        "listed weakness below should generally be sized larger than purely standard/amenity zones."
        if full_scope else
        "Propose zones that directly address ONLY the weaknesses listed below - do not add zones for "
        "anything not explicitly listed."
    )
    prompt = f'''You are proposing a schematic zoning program for a park site.

{scope_instruction}

Weaknesses found by a GIS analysis this design should prioritize:
{rec_lines}

Site context: area is approximately {site_context.get("latitude", ""):.4f}, {site_context.get("longitude", ""):.4f}.
Existing constraints on site: {_existing_constraints_summary()}.

Propose up to {config["max_zones_per_concept"]} named zones total.
Each zone's purpose MUST be exactly one of: {purpose_options}.
Total target_area_fraction across all zones MUST NOT exceed {config["max_total_zone_area_fraction"]}.
No single zone's target_area_fraction should exceed {config["max_single_zone_area_fraction"]}, even if
only one or two zones are proposed - a realistic park design uses multiple modestly-sized zones, not one
zone dominating most of the site.
Each zone needs a location_hint that is exactly one of: N, NE, E, SE, S, SW, W, NW, C (compass
direction within the site, C = center). Do not invent coordinates - only choose a compass direction.
For each zone, set addresses_metric to the exact metric name from the weaknesses list above if that zone
directly targets one of them, or null if it's a standard park element not tied to a specific weakness.

Respond with ONLY a JSON array in exactly this shape, no other text:
[{{"name": "<short name>", "purpose": "<one of the allowed purposes>", "target_area_fraction": <0.0-1.0>, "location_hint": "<N|NE|E|SE|S|SW|W|NW|C>", "addresses_metric": "<metric name or null>"}}, ...]'''

    try:
        with hard_timeout(45):
            response = client.models.generate_content(model=GEMINI_MODEL_NAME, contents=prompt)
        raw_text = response.text.strip()
        if raw_text.startswith("```"):
            raw_text = raw_text.split("```")[1]
            if raw_text.startswith("json"):
                raw_text = raw_text[4:]
        zones = json.loads(raw_text.strip())

        # Validate and clamp - never trust AI-proposed numeric output blindly
        valid_purposes = ALL_ZONE_PURPOSES if full_scope else set(ZONE_PURPOSE_FOR_METRIC.values())
        zones = [z for z in zones if z.get("purpose") in valid_purposes
                  and z.get("location_hint") in ["N","NE","E","SE","S","SW","W","NW","C"]]
        zones = zones[: config["max_zones_per_concept"]]
        for z in zones:  # per-zone cap, enforced regardless of what the AI proposed
            z["target_area_fraction"] = min(max(0.0, z.get("target_area_fraction", 0)), config["max_single_zone_area_fraction"])
            if z.get("addresses_metric") not in ZONE_PURPOSE_FOR_METRIC:  # normalize any invalid/hallucinated metric name to None
                z["addresses_metric"] = None
        total_fraction = sum(z["target_area_fraction"] for z in zones)
        if total_fraction > config["max_total_zone_area_fraction"] and total_fraction > 0:
            scale = config["max_total_zone_area_fraction"] / total_fraction
            for z in zones:
                z["target_area_fraction"] = round(z["target_area_fraction"] * scale, 4)

        if not zones:
            logger.warning("Gemini zoning response had no valid zones after validation - using deterministic fallback.")
            return _fallback_zoning_program(recommendations, config)
        return zones
    except HardTimeoutError:
        logger.warning("Gemini zoning call timed out - using deterministic fallback.")
        return _fallback_zoning_program(recommendations, config)
    except Exception as e:
        logger.warning(f"Gemini zoning call failed ({e.__class__.__name__}: {str(e)[:150]}) - using deterministic fallback.")
        return _fallback_zoning_program(recommendations, config)


print("generate_design_program() defined.")


generate_design_program() defined.


## Stage 2 — Deterministic geometry construction

In [ ]:
# @title Compass-direction grid allocator (schematic heuristic, not a professional site-planning algorithm)
import shapely.ops
import shapely.affinity
from shapely.geometry import box
import numpy as _np_design

def _build_direction_grid(boundary_utm, n=3):
    '''Splits boundary_utm's bounding box into an n x n grid, clips each cell to the actual
    boundary shape, and labels cells by compass direction (only exact for n=3, which gives the
    standard N/NE/E/SE/S/SW/W/NW/C vocabulary this module uses throughout).'''
    minx, miny, maxx, maxy = boundary_utm.bounds
    x_edges = _np_design.linspace(minx, maxx, n + 1)
    y_edges = _np_design.linspace(miny, maxy, n + 1)
    row_labels = ["S", "", "N"]   # ascending south -> north
    col_labels = ["W", "", "E"]    # ascending west -> east

    cells_by_label = {}
    for ri in range(n):
        for ci in range(n):
            cell_box = box(x_edges[ci], y_edges[ri], x_edges[ci + 1], y_edges[ri + 1])
            label = (row_labels[ri] + col_labels[ci]) or "C"
            clipped = cell_box.intersection(boundary_utm)
            if not clipped.is_empty:
                cells_by_label.setdefault(label, []).append(clipped)

    return {label: shapely.ops.unary_union(geoms) for label, geoms in cells_by_label.items()}


_DIRECTION_NEIGHBORS = {
    "N": ["NW", "NE", "C"], "S": ["SW", "SE", "C"], "E": ["NE", "SE", "C"], "W": ["NW", "SW", "C"],
    "NE": ["N", "E", "C"], "NW": ["N", "W", "C"], "SE": ["S", "E", "C"], "SW": ["S", "W", "C"],
    "C": ["N", "S", "E", "W"],
}


def _shrink_to_area(geom, target_area_m2):
    '''Scales geom toward its centroid to approximately hit target_area_m2 - an approximation
    that works reasonably for roughly-convex shapes, not an exact area-matching algorithm.'''
    current_area = geom.area
    if current_area <= 0 or target_area_m2 <= 0 or current_area <= target_area_m2:
        return geom
    factor = _math_design.sqrt(target_area_m2 / current_area)
    return shapely.affinity.scale(geom, xfact=factor, yfact=factor, origin="centroid")


def allocate_zone_geometry(zone_spec: dict, available_area, direction_grid: dict, site_area_m2: float) -> tuple:
    '''Returns (geometry, actual_area_m2, shortfall_note). Consumes from (and does not mutate)
    available_area - the caller is responsible for subtracting the returned geometry from
    available_area before allocating the next zone, so zones never overlap.'''
    target_area_m2 = zone_spec["target_area_fraction"] * site_area_m2
    hint = zone_spec["location_hint"]

    candidate = direction_grid.get(hint)
    if candidate is None or candidate.is_empty:
        candidate_region = available_area
    else:
        candidate_region = candidate.intersection(available_area)

    # If the primary cell doesn't have enough available area, pull in neighboring cells
    for neighbor in _DIRECTION_NEIGHBORS.get(hint, []):
        if candidate_region.area >= target_area_m2 * 0.95:
            break
        neighbor_geom = direction_grid.get(neighbor)
        if neighbor_geom is not None and not neighbor_geom.is_empty:
            candidate_region = shapely.ops.unary_union([candidate_region, neighbor_geom.intersection(available_area)])

    if candidate_region.is_empty or candidate_region.area <= 0:
        return None, 0.0, "No available area found for this zone (fully constrained by existing buildings/water or other zones)."

    shortfall_note = ""
    if candidate_region.area > target_area_m2 * 1.05:
        final_geom = _shrink_to_area(candidate_region, target_area_m2)
        # Defensive re-intersection: scaling a non-convex candidate_region toward its centroid
        # (which can happen once neighbor-cell-widening pulls in an irregularly-shaped leftover
        # area around a previously-allocated zone) is not guaranteed to stay within the original
        # region - it can push the shrunk boundary back across a concave notch and overlap a
        # zone that's already been carved out. Re-intersecting with available_area guarantees no
        # overlap with any previously-allocated zone, regardless of how the scale operation behaved.
        final_geom = final_geom.intersection(available_area)
    else:
        final_geom = candidate_region
        if candidate_region.area < target_area_m2 * 0.95:
            pct = candidate_region.area / target_area_m2 * 100 if target_area_m2 > 0 else 0
            shortfall_note = f"Only {pct:.0f}% of the targeted area was available after accounting for existing constraints and other zones."

    final_geom = normalize_geometry(final_geom)
    if final_geom is None:
        return None, 0.0, "Resulting geometry was invalid after allocation."
    return final_geom, calculate_area_m2(final_geom, source_crs=str(SITE_UTM_CRS)) or final_geom.area, shortfall_note


### Which boundary to design within — important

`SITE.boundary` may be a **coarse walkable-catchment buffer** (e.g. the 500m-radius circle used for
POI/context analysis elsewhere in this notebook), not the park's actual footprint. Placing design zones
across that buffer would sprawl "park" zones across surrounding city blocks full of real buildings — not
a usable park design. This section resolves the **actual buildable park polygon** to design within,
preferring the AI-assisted boundary resolver's result from Module 01 (if that optional section was run)
over the coarser `SITE.boundary`, regardless of whether you formally approved it there.


In [ ]:
# @title Resolve the actual park polygon to design within (NOT necessarily SITE.boundary)
def resolve_park_design_boundary():
    '''Determines the actual buildable park polygon to design within - this is deliberately
    NOT always the same as SITE.boundary, which may be a coarse circular walkable-catchment
    buffer (e.g. the 500m radius used for POI/context analysis) rather than the park's real
    footprint. Placing design zones across that buffer would sprawl new "park" zones across
    surrounding city blocks full of existing buildings, which is exactly the bug this fixes.

    Preference order:
    1. The AI-assisted boundary resolver's best candidate from this session (Module 01's
       BEST_CANDIDATE), if it was run - used regardless of whether you chose "approve" for
       BOUNDARY_DECISION there, since that decision controls whether it overwrites the shared
       SITE object, not whether Module 10 can use it for its own purposes.
    2. SITE.boundary itself, if it was set from an uploaded real boundary file (not a
       auto-generated buffer).
    3. SITE.boundary as a last resort, with a loud warning - this is very likely a coarse
       circular buffer, not the real park shape, and design zones will not be realistic.

    Returns (boundary_geometry_wgs84, source_description).'''
    best_candidate = globals().get("BEST_CANDIDATE")
    if best_candidate is not None and getattr(best_candidate, "geometry", None) is not None:
        print(f"✓ Using the boundary resolver's best candidate ('{best_candidate.name or best_candidate.id}', "
              f"{best_candidate.area_m2:,.0f} m²) as the design canvas - NOT SITE.boundary, which may still be "
              f"a coarser walkable-catchment buffer used for other analysis.")
        # BEST_CANDIDATE.geometry comes straight from OSM via osmnx (Module 01's search_osm_boundaries),
        # which always returns geometries in EPSG:4326 (WGS84) - OSM/osmnx data is never in a projected
        # CRS. It is NOT in SITE_UTM_CRS. Returning it directly here, with no reprojection, since it's
        # already in this function's documented output CRS. (An earlier version of this function
        # incorrectly assumed UTM and reprojected it as such, which silently produced a near-zero-area
        # degenerate polygon - caught via a real Colab run producing ~1e-7 m² zones.)
        return best_candidate.geometry, f"boundary_resolver_candidate ({best_candidate.id})"

    is_buffer = any("auto-generated" in note or "buffer" in note.lower() for note in (SITE.notes or []))
    if not is_buffer:
        return SITE.boundary, f"SITE.boundary ({SITE.source_method})"

    print("⚠️  WARNING: no AI-assisted boundary resolver result found this session, and SITE.boundary "
          "appears to be an auto-generated circular buffer (e.g. from a walkable-catchment radius), not "
          "the park's real footprint. Design zones will be placed within that buffer and are likely to "
          "sprawl across surrounding buildings/streets, not just the park itself. For realistic design "
          "zones, run and use the AI-assisted boundary resolver section in Module 01 first (the "
          "'BEST_CANDIDATE' it finds will be used automatically here, whether or not you set "
          "BOUNDARY_DECISION='approve').")
    return SITE.boundary, f"SITE.boundary ({SITE.source_method}) - COARSE BUFFER, NOT A VERIFIED PARK BOUNDARY"


PARK_DESIGN_BOUNDARY, PARK_DESIGN_BOUNDARY_SOURCE = resolve_park_design_boundary()
print(f"Design boundary source: {PARK_DESIGN_BOUNDARY_SOURCE}")

# Module 10 must not assume the optional AI-assisted boundary resolver section in Module 01 was
# run - RESOLVED_BOUNDARY_SITE only exists if that section executed. Build an equivalent minimal
# dict here from SITE directly if it's missing, so this module works standalone either way.
if "RESOLVED_BOUNDARY_SITE" not in dir():
    RESOLVED_BOUNDARY_SITE = {"latitude": SITE.centroid[0], "longitude": SITE.centroid[1],
                               "name": SITE.name, "input": SITE.name}


✓ Using the boundary resolver's best candidate ('حديقة الصفا 2', 14,740 m²) as the design canvas - NOT SITE.boundary, which may still be a coarser walkable-catchment buffer used for other analysis.
Design boundary source: boundary_resolver_candidate (osm_0)


In [ ]:
# @title Compute the design boundary's area once, for reuse below
_design_boundary_area_m2 = calculate_area_m2(PARK_DESIGN_BOUNDARY, source_crs="EPSG:4326")
if _design_boundary_area_m2:
    print(f"Design canvas area: {_design_boundary_area_m2:,.0f} m² ({_design_boundary_area_m2/10_000:.2f} ha)")


Design canvas area: 14,740 m² (1.47 ha)


In [ ]:
# @title Zone geometry construction (uses PARK_DESIGN_BOUNDARY, not SITE.boundary directly)

def build_design_concept_geometry(zone_specs: list, config: dict, design_boundary_wgs84) -> list:
    '''Turns a list of zone spec dicts into a list of DesignZone objects with real, non-overlapping
    geometry, constrained to design_boundary_wgs84 (see resolve_park_design_boundary() above -
    this is the actual park footprint when available, not necessarily SITE.boundary). Returns an
    empty list (with a printed reason) if the boundary or UTM CRS aren't available - callers
    should treat that as "no geometry could be built this run", not an error.'''
    if not gpd or design_boundary_wgs84 is None or SITE_UTM_CRS is None:
        print("Cannot build zone geometry: missing geopandas, design boundary, or UTM CRS.")
        return []

    boundary_gdf = gpd.GeoDataFrame({"geometry": [design_boundary_wgs84]}, crs="EPSG:4326").to_crs(SITE_UTM_CRS)
    boundary_utm = boundary_gdf.geometry.iloc[0]
    site_area_m2 = boundary_utm.area

    available_area = boundary_utm
    buildings = PROCESSED.get_best("buildings", ["osm", "esri_living_atlas"])
    if buildings and len(buildings.data) > 0:
        try:
            buildings_union = shapely.ops.unary_union(buildings.data.geometry.tolist())
            available_area = available_area.difference(buildings_union)
        except Exception as e:
            logger.warning(f"Could not exclude existing buildings from available area: {e}")
    water = PROCESSED.get_best("water_bodies", ["osm"])
    if water and len(water.data) > 0:
        try:
            water_union = shapely.ops.unary_union(water.data.geometry.tolist())
            available_area = available_area.difference(water_union)
        except Exception as e:
            logger.warning(f"Could not exclude existing water from available area: {e}")

    direction_grid = _build_direction_grid(boundary_utm, n=config["grid_size"])

    zones = []
    for i, spec in enumerate(zone_specs):
        geom, actual_area, note = allocate_zone_geometry(spec, available_area, direction_grid, site_area_m2)
        zone = DesignZone(
            id=f"zone_{i}", name=spec["name"], purpose=spec["purpose"],
            addresses_metric=spec.get("addresses_metric"), target_area_fraction=spec["target_area_fraction"],
            location_hint=spec["location_hint"], geometry=geom, actual_area_m2=actual_area, shortfall_note=note,
        )
        zones.append(zone)
        if geom is not None:
            available_area = available_area.difference(geom)  # prevent later zones from overlapping this one

    return zones


print("Geometry allocation functions defined.")


Geometry allocation functions defined.


## Stage 3 — Render to GeoJSON, SVG, and PNG (from one shared geometry)

In [ ]:
# @title Render a design concept to GeoJSON + SVG + PNG
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    from matplotlib.patches import Polygon as MplPolygon
    _MATPLOTLIB_AVAILABLE = True
except ImportError:
    _MATPLOTLIB_AVAILABLE = False


def concept_to_geojson(concept: "DesignConcept") -> Optional[dict]:
    if not shapely or SITE_UTM_CRS is None or not gpd:
        return None
    features = []
    for z in concept.zones:
        if z.geometry is None:
            continue
        try:
            wgs84_geom = gpd.GeoSeries([z.geometry], crs=SITE_UTM_CRS).to_crs("EPSG:4326").iloc[0]
            features.append({
                "type": "Feature", "geometry": shapely.geometry.mapping(wgs84_geom),
                "properties": {"id": z.id, "name": z.name, "purpose": z.purpose,
                                "addresses_metric": z.addresses_metric, "area_m2": z.actual_area_m2,
                                "shortfall_note": z.shortfall_note},
            })
        except Exception as e:
            logger.warning(f"Could not convert zone {z.id} to GeoJSON: {e}")
    return {"type": "FeatureCollection", "properties": {"concept_name": concept.name}, "features": features}


def render_concept_plan(concept: "DesignConcept", output_dir: str) -> tuple:
    '''Draws the site boundary, existing buildings for context, and every zone color-coded by
    purpose, then saves the SAME matplotlib figure as both SVG (vector) and PNG (raster) - this
    guarantees the two formats can never show different content. Returns (svg_path, png_path),
    either of which may be None if matplotlib/geopandas aren't available.'''
    if not _MATPLOTLIB_AVAILABLE or not gpd or SITE_UTM_CRS is None:
        print(f"Skipping plan rendering for '{concept.name}' - matplotlib/geopandas/CRS unavailable.")
        return None, None

    fig, ax = plt.subplots(figsize=(10, 10))
    boundary_gdf = gpd.GeoDataFrame({"geometry": [PARK_DESIGN_BOUNDARY]}, crs="EPSG:4326").to_crs(SITE_UTM_CRS)
    boundary_utm = boundary_gdf.geometry.iloc[0]
    boundary_gdf.boundary.plot(ax=ax, color="#333333", linewidth=2)

    # Buildings from PROCESSED are clipped to SITE.boundary (which may be a much larger
    # walkable-catchment buffer than the actual park - see resolve_park_design_boundary() above),
    # so drawing all of them here can pull in hundreds of surrounding buildings, forcing
    # matplotlib's auto-scale to zoom out until the park itself is a barely-visible speck. Clip
    # to a modest buffer around the actual design boundary instead, for real plan-scale context.
    buildings = PROCESSED.get_best("buildings", ["osm", "esri_living_atlas"])
    if buildings and len(buildings.data) > 0:
        try:
            context_area = boundary_utm.buffer(30)  # a small margin around the park itself, not the whole catchment
            nearby_buildings = buildings.data[buildings.data.geometry.intersects(context_area)]
            if len(nearby_buildings) > 0:
                nearby_buildings.plot(ax=ax, color="#BDBDBD", alpha=0.6, edgecolor="#757575", linewidth=0.5)
        except Exception as e:
            logger.warning(f"Could not clip buildings context for plan rendering: {e}")

    used_purposes = set()
    for z in concept.zones:
        if z.geometry is None:
            continue
        color = ZONE_PURPOSE_COLORS.get(z.purpose, "#607D8B")
        gpd.GeoSeries([z.geometry]).plot(ax=ax, color=color, alpha=0.55, edgecolor=color, linewidth=1.5)
        used_purposes.add(z.purpose)
        centroid = z.geometry.centroid
        ax.annotate(z.name, (centroid.x, centroid.y), ha="center", fontsize=8, color="#1a1a1a",
                     bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.7))

    legend_handles = [mpatches.Patch(color=ZONE_PURPOSE_COLORS.get(p, "#607D8B"), label=ZONE_PURPOSE_LABELS.get(p, p))
                        for p in used_purposes]
    if legend_handles:
        ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1), fontsize=9)

    # Explicitly frame the view on the design boundary (with a small margin) regardless of what
    # else got drawn - this is the key fix: never let stray context data (e.g. distant buildings)
    # force the view to zoom out past the actual park being designed.
    minx, miny, maxx, maxy = boundary_utm.bounds
    margin = max((maxx - minx), (maxy - miny)) * 0.12
    ax.set_xlim(minx - margin, maxx + margin)
    ax.set_ylim(miny - margin, maxy + margin)

    ax.set_title(f"{SITE.name}\nDesign Concept: {concept.name}", fontsize=13)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.annotate("N ↑", xy=(0.95, 0.95), xycoords="axes fraction", fontsize=12, ha="center", fontweight="bold")
    fig.tight_layout()

    safe_name = concept.name.replace(" ", "_").replace("/", "-")
    svg_path = _os_design.path.join(output_dir, f"{safe_name}_plan.svg")
    png_path = _os_design.path.join(output_dir, f"{safe_name}_plan.png")
    try:
        fig.savefig(svg_path, format="svg", bbox_inches="tight")
        fig.savefig(png_path, format="png", dpi=150, bbox_inches="tight")
    except Exception as e:
        logger.warning(f"Could not save plan rendering for '{concept.name}': {e}")
        svg_path, png_path = None, None
    finally:
        plt.close(fig)

    return svg_path, png_path


print("Rendering functions defined.")


Rendering functions defined.


### AI-illustrated concept render (Gemini image generation)

This produces an **illustrative artist's impression**, not a geometrically accurate output — it's a
picture inspired by the zoning program's description, not a rendering of the actual computed geometry.
On by default (`enable_ai_render: True` above), uses your `GEMINI_API_KEY`'s free-tier image quota — one
image per generated concept. Set `enable_ai_render` to `False` above to skip this and save quota/time.

Uses `gemini-3.1-flash-image` (the current "Nano Banana 2" generation model as of this notebook's build).
Google discontinues/replaces these models faster than their announced timelines — if this specific model
name has been retired by the time you run this, check https://ai.google.dev/gemini-api/docs/models for
the current image-generation model and update the one constant below.


In [ ]:
# @title Optional: AI-illustrated concept render (Gemini image generation)
GEMINI_IMAGE_MODEL_NAME = "gemini-3.1-flash-image"

def render_ai_concept_image(concept: "DesignConcept", output_dir: str) -> Optional[str]:
    if not DESIGN_GENERATOR_CONFIG.get("enable_ai_render", False):
        return None
    client = _get_gemini_client()
    if client is None or not PIL_Image:
        logger.warning("Gemini client or PIL unavailable - skipping AI concept render.")
        return None

    zone_desc = "; ".join(f"{z.name} ({ZONE_PURPOSE_LABELS.get(z.purpose, z.purpose)}, "
                            f"~{z.actual_area_m2:.0f} m², {z.location_hint} area of the site)"
                            for z in concept.zones if z.geometry is not None)
    prompt = (f"A landscape architecture concept illustration, top-down aerial style, of a small urban "
              f"park redesign. The design includes: {zone_desc}. Clean, modern landscape architecture "
              f"rendering style, daytime, no text or labels in the image.")

    try:
        from google.genai import types as _genai_types
        with hard_timeout(45):
            response = client.models.generate_content(
                model=GEMINI_IMAGE_MODEL_NAME, contents=prompt,
                config=_genai_types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]),
            )
        for part in response.candidates[0].content.parts:
            if getattr(part, "inline_data", None) is not None:
                import io as _io_design
                image = PIL_Image.open(_io_design.BytesIO(part.inline_data.data))
                safe_name = concept.name.replace(" ", "_").replace("/", "-")
                path = _os_design.path.join(output_dir, f"{safe_name}_ai_render.png")
                image.save(path)
                return path
        logger.warning("Gemini image response contained no image data.")
        return None
    except HardTimeoutError:
        logger.warning(f"AI render for '{concept.name}' timed out.")
        return None
    except Exception as e:
        logger.warning(f"AI render for '{concept.name}' failed: {e.__class__.__name__}: {str(e)[:150]}")
        return None


print(f"AI concept render function defined (enabled={DESIGN_GENERATOR_CONFIG.get('enable_ai_render', False)}).")


AI concept render function defined (enabled=True).


## Stage 4 — What-if rescoring

Projects only the metrics a park design can actually influence. Everything locational (transit
distance, hospital distance, road density, slope, climate, etc.) is held at its baseline value and
explicitly labeled as unchanged — the projected score improvement is real but partial, not a full re-run.


In [ ]:
# @title What-if metric projection + rescoring

# Extra scoring entries for metrics that were pure placeholders in the baseline (Module 06)
# specifically because they need a proposed design as input - now that Module 10 provides one,
# they become genuinely scorable. Kept separate from the global SCORING_CONFIG dict rather than
# mutating it, so the baseline Module 06 run is never silently changed by running this module.
WHAT_IF_SCORING_ADDITIONS = {
    "seating_capacity": {
        "normalize": lambda v: _linear_score(v, worst=0, best=100),
        "weight": 0.5,
        "standard": f"Illustrative estimate: proposed seating/comfort zone area / {SEATING_SPACE_M2_PER_PERSON:.0f} m² "
                     f"per person (see planning assumptions above), scored on a 0-100 person illustrative scale.",
    },
    "event_capacity": {
        "normalize": lambda v: _linear_score(v, worst=0, best=150),
        "weight": 0.5,
        "standard": f"General event-space planning guidance (~2-4 m² per person for casual gathering/event "
                     f"use; {EVENT_SPACE_M2_PER_PERSON:.1f} m² midpoint used here), scored on a 0-150 person "
                     f"illustrative scale.",
    },
}


def project_metrics_for_concept(concept: "DesignConcept") -> dict:
    '''Returns a dict of {metric_name: Score} covering ONLY the metrics this concept's zones can
    plausibly affect - baseline SCORES entries for every other metric are left untouched when this
    is merged into a full projected-scores dict by rescore_concept() below.'''
    projected = {}
    site_area_m2 = SITE.area_m2()
    if not site_area_m2:
        return projected

    zones_by_purpose = {}
    for z in concept.zones:
        if z.geometry is not None:
            zones_by_purpose.setdefault(z.purpose, []).append(z)

    def _total_area(purpose):
        return sum(z.actual_area_m2 or 0 for z in zones_by_purpose.get(purpose, []))

    # tree_canopy_coverage: baseline + new canopy zones' area, discounted by the maturity assumption
    baseline_canopy = SCORES.get("tree_canopy_coverage")
    canopy_zone_area = _total_area("tree_canopy_expansion")
    if baseline_canopy is not None and canopy_zone_area > 0:
        baseline_val = baseline_canopy.raw_value if isinstance(baseline_canopy.raw_value, (int, float)) else 0
        added_pct = (canopy_zone_area / site_area_m2) * 100 * CANOPY_MATURITY_COVERAGE_ASSUMPTION
        new_val = min(100.0, baseline_val + added_pct)
        cfg = SCORING_CONFIG.get("tree_canopy_coverage", {})
        projected["tree_canopy_coverage"] = Score(
            "tree_canopy_coverage", round(new_val, 1), "%", cfg["normalize"](new_val) if cfg.get("normalize") else None,
            cfg.get("weight", 1.0), "projected", cfg.get("standard", ""),
            notes=f"Projected from baseline {baseline_val}% + {added_pct:.1f}pp from '{concept.name}' zones "
                  f"(illustrative maturity assumption, see Stage 4 notes above).")

    # green_area_ratio_vector: baseline + green_space_expansion AND tree_canopy_expansion area (tree
    # zones are also green space)
    baseline_green = SCORES.get("green_area_ratio_vector")
    green_zone_area = _total_area("green_space_expansion") + canopy_zone_area
    if baseline_green is not None and green_zone_area > 0:
        baseline_val = baseline_green.raw_value if isinstance(baseline_green.raw_value, (int, float)) else 0
        added_pct = (green_zone_area / site_area_m2) * 100
        new_val = min(100.0, baseline_val + added_pct)
        cfg = SCORING_CONFIG.get("green_area_ratio_vector", {})
        projected["green_area_ratio_vector"] = Score(
            "green_area_ratio_vector", round(new_val, 1), "%", cfg["normalize"](new_val) if cfg.get("normalize") else None,
            cfg.get("weight", 1.0), "projected", cfg.get("standard", ""),
            notes=f"Projected from baseline {baseline_val}% + {added_pct:.1f}pp from '{concept.name}' zones.")

    # impervious_surface_ratio: baseline - permeable conversion area, discounted by effectiveness
    baseline_imp = SCORES.get("impervious_surface_ratio")
    permeable_area = _total_area("permeable_surface_conversion")
    if baseline_imp is not None and permeable_area > 0:
        baseline_val = baseline_imp.raw_value if isinstance(baseline_imp.raw_value, (int, float)) else 0
        reduced_pct = (permeable_area / site_area_m2) * 100 * IMPERVIOUS_CONVERSION_EFFECTIVENESS
        new_val = max(0.0, baseline_val - reduced_pct)
        cfg = SCORING_CONFIG.get("impervious_surface_ratio", {})
        projected["impervious_surface_ratio"] = Score(
            "impervious_surface_ratio", round(new_val, 1), "%", cfg["normalize"](new_val) if cfg.get("normalize") else None,
            cfg.get("weight", 1.0), "projected", cfg.get("standard", ""),
            notes=f"Projected from baseline {baseline_val}% - {reduced_pct:.1f}pp from '{concept.name}' zones.")

    # play_area_coverage: baseline count (existing nearby) + new play zones proposed in this concept
    baseline_play = SCORES.get("play_area_coverage")
    n_play_zones = len(zones_by_purpose.get("play_area", []))
    if baseline_play is not None and n_play_zones > 0:
        baseline_val = baseline_play.raw_value if isinstance(baseline_play.raw_value, (int, float)) else 0
        new_val = baseline_val + n_play_zones
        cfg = SCORING_CONFIG.get("play_area_coverage", {})
        projected["play_area_coverage"] = Score(
            "play_area_coverage", new_val, baseline_play.raw_unit, cfg["normalize"](new_val) if cfg.get("normalize") else None,
            cfg.get("weight", 1.0), "projected", cfg.get("standard", ""),
            notes=f"Projected: baseline {baseline_val} existing nearby + {n_play_zones} new play zone(s) proposed.")

    # seating_capacity, event_capacity: newly scorable via WHAT_IF_SCORING_ADDITIONS (baseline was
    # a pure placeholder - None - since these need a proposed design, which this concept now is)
    seating_area = _total_area("seating_comfort_area")
    if seating_area > 0:
        capacity = seating_area / SEATING_SPACE_M2_PER_PERSON
        cfg = WHAT_IF_SCORING_ADDITIONS["seating_capacity"]
        projected["seating_capacity"] = Score(
            "seating_capacity", round(capacity, 0), "people", cfg["normalize"](capacity),
            cfg["weight"], "projected", cfg["standard"],
            notes=f"Estimated from {seating_area:.0f} m² of proposed seating/comfort zone area in '{concept.name}'.")

    event_area = _total_area("community_gathering_plaza")
    if event_area > 0:
        capacity = event_area / EVENT_SPACE_M2_PER_PERSON
        cfg = WHAT_IF_SCORING_ADDITIONS["event_capacity"]
        projected["event_capacity"] = Score(
            "event_capacity", round(capacity, 0), "people", cfg["normalize"](capacity),
            cfg["weight"], "projected", cfg["standard"],
            notes=f"Estimated from {event_area:.0f} m² of proposed community gathering/plaza zone area in '{concept.name}'.")

    return projected


def rescore_concept(concept: "DesignConcept") -> None:
    '''Fills in concept.projected_scores / projected_category_rollups / projected_overall_score
    in place. Every metric NOT touched by this concept's zones keeps its exact baseline Score
    object - the projection only ever improves (or leaves unchanged) the metrics a design can
    plausibly influence, never invents movement elsewhere.'''
    projected_deltas = project_metrics_for_concept(concept)
    projected_scores = dict(SCORES)  # start from baseline, override only the affected metrics
    projected_scores.update(projected_deltas)
    concept.projected_scores = projected_scores

    rollups = {}
    for cat, metrics_in_cat in METRICS_BY_KPI_CATEGORY.items():
        rollups[cat] = rollup_category(projected_scores, list(metrics_in_cat.keys()))
    concept.projected_category_rollups = rollups

    scorable = [(cat, r["score"]) for cat, r in rollups.items() if r and r.get("score") is not None]
    concept.projected_overall_score = round(sum(s for _, s in scorable) / len(scorable), 1) if scorable else None


print("What-if rescoring functions defined.")


What-if rescoring functions defined.


## Generate all concepts

In [ ]:
# @title Run the full pipeline for every recommendation group

DESIGN_CONCEPTS = []

for group in RECOMMENDATION_GROUPS:
    print(f"\n=== Generating: {group['concept_name']} ===")
    zone_specs = generate_design_program(group["recommendations"], RESOLVED_BOUNDARY_SITE, DESIGN_GENERATOR_CONFIG)
    zones = build_design_concept_geometry(zone_specs, DESIGN_GENERATOR_CONFIG, PARK_DESIGN_BOUNDARY)

    concept = DesignConcept(
        id=group["concept_name"].lower().replace(" ", "_").replace("&", "and"),
        name=group["concept_name"],
        source_recommendations=[o["based_on_metric"] for o in group["recommendations"]],
        zones=zones,
    )

    for z in zones:
        area_str = f"{z.actual_area_m2:.0f} m²" if z.actual_area_m2 else "no area allocated"
        note = f" — {z.shortfall_note}" if z.shortfall_note else ""
        print(f"  · {z.name} ({z.location_hint}): {area_str}{note}")

    geojson = concept_to_geojson(concept)
    if geojson:
        geojson_path = _os_design.path.join(DESIGN_OUTPUT_DIR, f"{concept.id}.geojson")
        with open(geojson_path, "w", encoding="utf-8") as f:
            json.dump(geojson, f, indent=2)
        concept.geojson_path = geojson_path

    concept.svg_path, concept.png_path = render_concept_plan(concept, DESIGN_OUTPUT_DIR)
    concept.ai_render_path = render_ai_concept_image(concept, DESIGN_OUTPUT_DIR)

    rescore_concept(concept)

    DESIGN_CONCEPTS.append(concept)
    print(f"  Projected overall score: {concept.projected_overall_score} "
          f"(baseline: {OVERALL_SCORE})")

print(f"\n{len(DESIGN_CONCEPTS)} design concept(s) generated.")



=== Generating: Environmental & Sustainability Focus ===
  · Northwest Native Canopy Grove (NW): 2288 m²
  · Eastern Shade Canopy Belt (E): 1714 m²
  · Central Permeable Plaza & Bioswale (C): 2064 m²
  · Northeast Lawn Expansion (NE): 1474 m²
  · South Main Entrance Plaza (S): 590 m²
  · Perimeter Promenade Loop (W): 590 m²
  · Southeast Shaded Seating Garden (SE): no area allocated — Resulting geometry was invalid after allocation.
  · Southwest Service & Restroom Pavilion (SW): 147 m²
  · North Play & Recreation Zone (N): 4 m²


  Projected overall score: 78.9 (baseline: 69.8)

1 design concept(s) generated.


## Comparison summary

In [ ]:
# @title Baseline vs. projected score comparison, per concept

comparison_rows = []
for concept in DESIGN_CONCEPTS:
    for cat in METRICS_BY_KPI_CATEGORY:
        baseline_score = CATEGORY_ROLLUPS.get(cat, {}).get("score") if CATEGORY_ROLLUPS.get(cat) else None
        projected_score = concept.projected_category_rollups.get(cat, {}).get("score") if concept.projected_category_rollups.get(cat) else None
        changed = (baseline_score != projected_score) if (baseline_score is not None and projected_score is not None) else False
        comparison_rows.append({
            "concept": concept.name, "kpi_category": cat,
            "baseline_score": baseline_score, "projected_score": projected_score,
            "delta": round(projected_score - baseline_score, 1) if changed else (0.0 if baseline_score is not None else None),
        })
    comparison_rows.append({
        "concept": concept.name, "kpi_category": "OVERALL",
        "baseline_score": OVERALL_SCORE, "projected_score": concept.projected_overall_score,
        "delta": round(concept.projected_overall_score - OVERALL_SCORE, 1)
                 if (concept.projected_overall_score is not None and OVERALL_SCORE is not None) else None,
    })

design_comparison_df = pd.DataFrame(comparison_rows)
print("Note: categories/metrics with delta=0 were not touched by that concept's zones and reflect "
      "the exact baseline (locational/unchanged) score, not a design improvement of zero.")
design_comparison_df


Note: categories/metrics with delta=0 were not touched by that concept's zones and reflect the exact baseline (locational/unchanged) score, not a design improvement of zero.


,concept,kpi_category,baseline_score,projected_score,delta
0,Environmental & Sustainability Focus,Site Context,NaN,NaN,NaN
1,Environmental & Sustainability Focus,Environmental,45.7,100.0,54.3
2,Environmental & Sustainability Focus,Sustainability,0.0,0.0,0.0
3,Environmental & Sustainability Focus,Accessibility,100.0,100.0,0.0
4,Environmental & Sustainability Focus,Recreation,73.3,73.3,0.0
5,Environmental & Sustainability Focus,Social,100.0,100.0,0.0
6,Environmental & Sustainability Focus,Safety,100.0,100.0,0.0
7,Environmental & Sustainability Focus,Smart City,NaN,NaN,NaN
8,Environmental & Sustainability Focus,OVERALL,69.8,78.9,9.1


In [ ]:
# @title Generated files summary
print("Design concept files generated:")
for concept in DESIGN_CONCEPTS:
    print(f"\n{concept.name}:")
    for label, path in [("GeoJSON", concept.geojson_path), ("SVG", concept.svg_path),
                          ("PNG", concept.png_path), ("AI render", concept.ai_render_path)]:
        if path and _os_design.path.exists(path):
            size_kb = _os_design.path.getsize(path) / 1024
            print(f"  ✓ {label:10s}: {path} ({size_kb:.1f} KB)")
        elif label == "AI render" and not DESIGN_GENERATOR_CONFIG.get("enable_ai_render", False):
            print(f"  – {label:10s}: disabled (DESIGN_GENERATOR_CONFIG['enable_ai_render'] = False)")
        else:
            print(f"  ✗ {label:10s}: not generated")


Design concept files generated:

Environmental & Sustainability Focus:
  ✓ GeoJSON   : /content/park_analysis_reports/design_concepts/environmental_and_sustainability_focus.geojson (11.5 KB)
  ✓ SVG       : /content/park_analysis_reports/design_concepts/Environmental_&_Sustainability_Focus_plan.svg (67.8 KB)
  ✓ PNG       : /content/park_analysis_reports/design_concepts/Environmental_&_Sustainability_Focus_plan.png (157.0 KB)
  ✗ AI render : not generated


# Module 10 Upgrade — Zone Taxonomy, Jogging Track, Multi-View Renders, Commercial Facilities Map

Extends (does not replace) everything above. Run once per session, after Module 10's original cells have already defined `ZONE_PURPOSE_LABELS`, `DesignZone`, `DESIGN_CONCEPTS`, etc.

Closes these gaps against the Scope of Work: a real routed jogging/walking track (honest about lap count on a site this size, not a fabricated single 1km loop); day AND night perspective renders at human-scale hero nodes, in addition to the existing aerial render; and the Commercial & Service Facilities Map, a named required deliverable that didn't exist before.

In [ ]:
# =============================================================================
# MODULE 10 UPGRADE — Zone Taxonomy, Jogging Track, Multi-View Renders,
#                      Cost Check, Commercial & Service Facilities Map
# =============================================================================
# Paste this AFTER Module 10's existing cells (it extends, not replaces, the
# ZONE_PURPOSE_* dicts and DesignConcept/DesignZone classes already defined
# there). Run it once per notebook session, before generating/rendering
# concepts, since add_jogging_track_to_concept() and the render/cost functions
# below assume the extended taxonomy is already in place.
#
# What this closes, against the Scope of Work:
#   - "Jogging track (approximately 1km or as appropriate to the site)" —
#     previously only area-allocated like every other zone; now actually
#     routed as a loop, honestly reporting lap count if the site is too
#     small for a single 1km loop rather than faking one.
#   - "High-quality visualizations... aerial perspectives, human-scale
#     perspectives, and key experiential views" + "Day and Night Perspectives"
#     — previously one static aerial image per concept; now a day/night pair
#     at each of up to 3 hero nodes, generated the same source-first way as
#     the existing boundary resolver (real zone geometry decides WHERE the
#     hero shot is; AI only renders WHAT it looks like).
#   - "Preliminary cost and resource considerations" against the AED 35M
#     budget — did not exist at all before this.
#   - "Commercial and Service Facilities Map" — a named required deliverable
#     that did not exist before this.
# =============================================================================
# @title Extend the zone taxonomy (run once, after Module 10's original taxonomy cell)
import os as _os_upg

ZONE_PURPOSE_LABELS.update({
    "jogging_track": "Jogging & Walking Track",
    "outdoor_fitness": "Outdoor Fitness & Wellness",
    "bicycle_dropoff": "Bicycle Parking & Drop-off",
    "commercial_kiosk": "Commercial / F&B Kiosk",
})
ZONE_PURPOSE_COLORS.update({
    "jogging_track": "#D84315",
    "outdoor_fitness": "#00ACC1",
    "bicycle_dropoff": "#6D4C41",
    "commercial_kiosk": "#F4511E",
})
# Area-fraction defaults for the generic fallback zoning template (Module 10's
# _fallback_zoning_program). jogging_track is deliberately excluded here — it's
# routed geometrically (see add_jogging_track_to_concept below), not
# area-allocated through the compass-grid system like every other zone.
STANDARD_PARK_ELEMENTS.update({
    "outdoor_fitness": 0.05,
    "bicycle_dropoff": 0.015,
    "commercial_kiosk": 0.02,
})
ALL_ZONE_PURPOSES.update(ZONE_PURPOSE_LABELS.keys())

print("Zone taxonomy extended:", list(ZONE_PURPOSE_LABELS.keys()))

Zone taxonomy extended: ['tree_canopy_expansion', 'green_space_expansion', 'permeable_surface_conversion', 'play_area', 'community_gathering_plaza', 'seating_comfort_area', 'entrance_plaza', 'circulation_paths', 'restrooms_and_services', 'sports_recreation_zone', 'picnic_area', 'jogging_track', 'outdoor_fitness', 'bicycle_dropoff', 'commercial_kiosk']


In [ ]:
# @title Jogging / walking track routing (real geometry, honest about lap count)

def build_jogging_track_route(boundary_utm, target_length_m=1000, path_width_m=2.5):
    '''Builds the track as an inward-offset ring of the site boundary rather than an
    arbitrary area allocation, since a real jogging track is a LOOP, not a blob.

    Honesty check: a 15,000 sqm neighbourhood park (~120m x 125m if roughly square) has
    an outer perimeter well under 1000m once you inset it enough to leave room for other
    zones, so a single lap essentially never reaches the SOW's "approximately 1km" figure.
    Rather than stretch/fabricate a route to hit 1000m, this returns the best achievable
    single-lap loop AND the number of laps needed to cover 1km, so the design narrative
    can honestly say e.g. "3 laps of the 340m perimeter track = ~1.0km" — a completely
    normal, common solution in small urban parks.

    Returns (track_polygon_utm, single_lap_length_m, laps_for_target) — track_polygon_utm
    is None if no valid inset ring could be built at any tested distance.'''
    best = None
    for inset in (3, 5, 8, 12, 16, 20, 25, 30, 35):
        try:
            ring = boundary_utm.buffer(-inset)
            if ring.is_empty or not hasattr(ring, "exterior"):
                continue
            best = (inset, ring.exterior, ring.exterior.length)
            break   # take the SMALLEST feasible inset — that maximizes loop length while
                    # staying safely clear of the boundary edge. Continuing to larger insets
                    # would shrink the loop toward the site's center for no benefit.
        except Exception:
            continue
    if best is None:
        return None, 0.0, None
    inset, loop_line, loop_length = best
    laps_for_target = max(1, round(target_length_m / loop_length)) if loop_length > 0 else None
    track_polygon = loop_line.buffer(path_width_m / 2)
    return track_polygon, round(loop_length, 1), laps_for_target


def add_jogging_track_to_concept(concept, boundary_utm=None):
    '''Appends a jogging_track DesignZone to an existing concept in place. Call this
    AFTER build_design_concept_geometry() has already allocated the concept's other
    zones, since the track is routed around the site boundary independent of the
    compass-grid allocator (it doesn't compete with other zones for area the same way).'''
    boundary_utm = boundary_utm or globals().get("_last_boundary_utm")
    if boundary_utm is None and globals().get("PARK_DESIGN_BOUNDARY") is not None and globals().get("SITE_UTM_CRS"):
        boundary_gdf = gpd.GeoDataFrame({"geometry": [PARK_DESIGN_BOUNDARY]}, crs="EPSG:4326").to_crs(SITE_UTM_CRS)
        boundary_utm = boundary_gdf.geometry.iloc[0]
    if boundary_utm is None:
        print(f"Could not add jogging track to '{concept.name}' — no boundary geometry available.")
        return concept

    track_polygon, lap_length, laps_needed = build_jogging_track_route(boundary_utm)
    if track_polygon is None:
        print(f"Could not fit a jogging track loop within '{concept.name}''s boundary.")
        return concept

    zone = DesignZone(
        id=f"zone_jogging_{len(concept.zones)}", name="Jogging & Walking Track",
        purpose="jogging_track", target_area_fraction=0.0, location_hint="C",
        addresses_metric=None, geometry=track_polygon,
        actual_area_m2=calculate_area_m2(track_polygon, source_crs=str(SITE_UTM_CRS)) or track_polygon.area,
        shortfall_note=(f"Single-lap loop length ~{lap_length:.0f}m; {laps_needed} lap(s) needed to "
                         f"cover the SOW's ~1km reference distance." if laps_needed else ""),
    )
    zone.loop_length_m = lap_length
    zone.laps_for_1km = laps_needed
    concept.zones.append(zone)
    print(f"Added jogging track to '{concept.name}': {lap_length:.0f}m per lap, "
          f"{laps_needed} lap(s) ≈ 1km total distance.")
    return concept

In [ ]:
# @title Multi-view AI render pipeline — aerial + human-scale hero nodes, day AND night
#
# Directly answers the "high-quality visualizations... aerial, human-scale, and key
# experiential views" + "Day and Night Perspectives" deliverables. Source-first, same
# principle as Module 01's boundary resolver and Module 10's original AI render: the
# hero node LOCATIONS come from the concept's own real zone geometry (centroid of an
# actual entrance/plaza/play zone) — Gemini only renders what that real location looks
# like, at a chosen time of day. It never invents where things are.

HERO_NODE_PRIORITY = ["entrance_plaza", "community_gathering_plaza", "play_area",
                       "jogging_track", "sports_recreation_zone", "seating_comfort_area"]

TIME_OF_DAY_PROMPTS = {
    "day": ("bright daytime, clear blue sky, strong overhead sun typical of Dubai, warm natural "
            "light and hard shadows, families and park users actively using the space"),
    "night": ("evening scene after sunset, warm architectural path and seating lighting, "
              "illuminated shade structures and planting uplights, an activated and welcoming "
              "after-dark atmosphere, a few people present, deep blue-purple night sky"),
}

def select_hero_zones(concept, max_nodes=3):
    picked = []
    for purpose in HERO_NODE_PRIORITY:
        match = next((z for z in concept.zones if z.purpose == purpose and z.geometry is not None), None)
        if match and match not in picked:
            picked.append(match)
        if len(picked) >= max_nodes:
            break
    return picked


def render_hero_view(concept, zone, time_of_day, output_dir):
    '''One Gemini image-gen call per (zone, time_of_day). Returns a file path, or None on
    any failure/unavailability — never raises, consistent with every other optional-AI
    path elsewhere in this notebook.'''
    client_fn = globals().get("_get_gemini_client")
    client = client_fn() if callable(client_fn) else None
    if client is None or not globals().get("PIL_Image"):
        return None

    label = ZONE_PURPOSE_LABELS.get(zone.purpose, zone.purpose)
    prompt = (
        f"A photorealistic human-eye-level landscape architecture rendering of the "
        f"'{label}' area within a redesigned neighbourhood park in Al Safa, Dubai, UAE. "
        f"{TIME_OF_DAY_PROMPTS[time_of_day]}. Design language: native and climate-adapted "
        f"planting providing deep shade, fabric shade sails or tree-canopy structures, "
        f"permeable light-toned paving, contemporary Gulf public-realm materials (light "
        f"stone, warm timber accents, minimal clean detailing), inclusive and family-"
        f"friendly character. No text, no logos, no watermarks, no readable signage."
    )
    try:
        from google.genai import types as _genai_types
        import io as _io
        hard_timeout_fn = globals().get("hard_timeout")
        ctx = hard_timeout_fn(45) if callable(hard_timeout_fn) else _no_op_ctx_upg()
        with ctx:
            response = client.models.generate_content(
                model=globals().get("GEMINI_IMAGE_MODEL_NAME", "gemini-3.1-flash-image"),
                contents=prompt,
                config=_genai_types.GenerateContentConfig(response_modalities=["TEXT", "IMAGE"]),
            )
        for part in response.candidates[0].content.parts:
            if getattr(part, "inline_data", None) is not None:
                image = globals()["PIL_Image"].open(_io.BytesIO(part.inline_data.data))
                safe_name = f"{concept.id}_{zone.purpose}_{time_of_day}"
                path = _os_upg.path.join(output_dir, f"{safe_name}.png")
                image.save(path)
                return path
    except Exception as e:
        logger.warning(f"Hero render failed for {zone.purpose}/{time_of_day}: "
                        f"{e.__class__.__name__}: {str(e)[:150]}")
    return None

class _no_op_ctx_upg:
    def __enter__(self): return self
    def __exit__(self, *a): return False


def render_multi_view_set(concept, output_dir, max_nodes=3):
    '''Generates the full day/night hero-shot set for a concept. Returns a dict of
    {"<purpose>_<day|night>": path_or_None} plus reuses Module 10's existing aerial
    plan render (render_concept_plan) so the full deliverable set for one concept is:
    1 schematic aerial plan + 1 AI aerial impression + up to 3 hero nodes x 2 times of
    day = up to 8 images total, all traceable to real zone geometry.'''
    hero_zones = select_hero_zones(concept, max_nodes=max_nodes)
    if not hero_zones:
        print(f"No hero-node zones found in '{concept.name}' to render (needs at least one of "
              f"{HERO_NODE_PRIORITY} with real geometry).")
        return {}
    renders = {}
    for zone in hero_zones:
        for tod in ("day", "night"):
            path = render_hero_view(concept, zone, tod, output_dir)
            renders[f"{zone.purpose}_{tod}"] = path
            status = "ok" if path else "unavailable"
            print(f"  {zone.purpose:28s} {tod:6s} -> {status}")
    return renders

In [ ]:
# @title Commercial & Service Facilities Map (named required deliverable)

COMMERCIAL_SERVICE_PURPOSES = {"commercial_kiosk", "restrooms_and_services"}

def render_commercial_facilities_map(concept, output_dir):
    if not globals().get("_MATPLOTLIB_AVAILABLE") or not gpd or SITE_UTM_CRS is None:
        print("matplotlib/geopandas/CRS unavailable — skipping commercial facilities map.")
        return None
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 9))
    boundary_gdf = gpd.GeoDataFrame({"geometry": [PARK_DESIGN_BOUNDARY]}, crs="EPSG:4326").to_crs(SITE_UTM_CRS)
    boundary_utm = boundary_gdf.geometry.iloc[0]
    boundary_gdf.boundary.plot(ax=ax, color="#333333", linewidth=2)

    for z in concept.zones:
        if z.geometry is not None and z.purpose not in COMMERCIAL_SERVICE_PURPOSES:
            gpd.GeoSeries([z.geometry]).plot(ax=ax, color="#EEEEEE", alpha=0.5, edgecolor="#CCCCCC", linewidth=0.5)

    found_any = False
    for z in concept.zones:
        if z.geometry is not None and z.purpose in COMMERCIAL_SERVICE_PURPOSES:
            found_any = True
            color = ZONE_PURPOSE_COLORS.get(z.purpose, "#F4511E")
            gpd.GeoSeries([z.geometry]).plot(ax=ax, color=color, alpha=0.85, edgecolor=color, linewidth=1.5)
            c = z.geometry.centroid
            ax.annotate(z.name, (c.x, c.y), ha="center", fontsize=8, color="#1a1a1a",
                         bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.85))

    minx, miny, maxx, maxy = boundary_utm.bounds
    margin = max((maxx - minx), (maxy - miny)) * 0.1
    ax.set_xlim(minx - margin, maxx + margin)
    ax.set_ylim(miny - margin, maxy + margin)
    ax.set_title(f"{SITE.name}\nCommercial & Service Facilities Map — {concept.name}", fontsize=12)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.tight_layout()

    path = _os_upg.path.join(output_dir, f"{concept.id}_commercial_services_map.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    if not found_any:
        print(f"Note: '{concept.name}' has no commercial_kiosk / restrooms_and_services zones yet — "
              f"map generated but will show only the empty site boundary. Add at least one of each "
              f"via the zoning program before final submission (both are required by the SOW).")
    return path


# =============================================================================
# USAGE (run once per concept, after Module 10's original geometry/render cells):
#
#   for concept in DESIGN_CONCEPTS:
#       add_jogging_track_to_concept(concept)                       # real routed loop
#       print_cost_check(concept)                                   # AED 35M sanity check
#       render_multi_view_set(concept, DESIGN_OUTPUT_DIR)           # day/night hero shots
#       render_commercial_facilities_map(concept, DESIGN_OUTPUT_DIR)
#
# Re-run rescore_concept(concept) afterwards (Module 10's original function) if you want
# the jogging track's added circulation area reflected in the what-if rescoring, and
# re-run assess_park(concept.name, concept=concept, ...) (Module 06) for the full
# 9-domain picture including the new zones.
# =============================================================================

## Full Project Cost Model (10 categories, replaces the simpler zone-only cost check)

The zone-only estimate this originally shipped with only covered direct visible construction — comparing that alone against the AED 35M implementation budget was misleading, since a real project budget also carries demolition/relocation, infrastructure/utilities, smart-tech/ICT, preliminaries, testing/commissioning, and professional fees/contingency. This section prices all 10, using the **225 real existing trees found in the Annex-1 DXF** for Category 2 rather than a generic guess, Dubai market-rate-derived unit costs for Category 1, and labeled general-industry-benchmark percentages (not Dubai-verified) for the categories with no site-specific figure available.

In [ ]:
# =============================================================================
# MODULE 10 UPGRADE — FULL PROJECT COST MODEL (replaces the zone-only estimate)
# =============================================================================
# The earlier estimate_concept_cost() only covered Category 1 below (direct
# visible zone construction) — that is NOT the same thing as the AED 35M total
# project budget envelope, and comparing one against the other was misleading.
# This replaces it with all 10 categories a real project budget needs, per a
# public-realm cost-plan structure (RIBA-stage-C style: preliminaries ~9% of
# construction, ~7.5% general contingency + ~5% existing-services relocation
# contingency — these are general international industry benchmarks, not a
# Dubai-verified rate card, and are labeled as such throughout).
#
# Design intent, unchanged from the earlier conversation: build the honest
# number FIRST, decide on signature/ambition features AFTER, not the reverse.
# =============================================================================
# @title Category-by-category rate basis (each with its own explicit source/rationale)

# --- Category 1: Direct landscape & park construction (unchanged from before) ---
UNIT_COST_AED_PER_SQM = {
    "tree_canopy_expansion": 130, "green_space_expansion": 90,
    "permeable_surface_conversion": 220, "play_area": 950, "sports_recreation_zone": 850,
    "outdoor_fitness": 700, "community_gathering_plaza": 380, "seating_comfort_area": 300,
    "entrance_plaza": 420, "circulation_paths": 280, "jogging_track": 260,
    "restrooms_and_services": 4200, "commercial_kiosk": 3600, "picnic_area": 260,
    "bicycle_dropoff": 200,
}
BASE_UNPROGRAMMED_RATE_AED_PER_SQM = 110

# --- Category 2: Existing site works — demolition, relocation, earthworks ---
DEMOLITION_CLEARANCE_RATE_AED_PER_SQM = 55       # general soft/hard landscape strip-out, order-of-magnitude
MATURE_TREE_RELOCATION_COST_AED = 5500            # per tree, professional root-ball relocation (hot-climate risk premium)
MATURE_TREE_PROTECTION_COST_AED = 900             # per tree, protect-in-place instead of relocating
EXISTING_TREE_COUNT_AL_SAFA_2 = 225               # REAL count, from the as-built DXF's "arboles" block INSERT entities
DEFAULT_TREE_RELOCATION_FRACTION = 0.35            # assume ~35% of existing trees need moving, rest protected in place

# --- Category 3: Infrastructure & utilities (irrigation, drainage, electrical, water) ---
INFRASTRUCTURE_PCT_OF_DIRECT_CONSTRUCTION = 0.15   # typical MEP/infra allowance for landscape-heavy public realm

# --- Category 4: Buildings & structures — signature elements beyond the base zone rates ---
SIGNATURE_SHADE_STRUCTURE_RATE_AED_PER_SQM = 11000  # architectural canopy, vs. ~basic shade sail already in zone rates
DEFAULT_SIGNATURE_SHADE_AREA_SQM = 0                 # 0 unless explicitly enabled — see enable_signature_shade below

# --- Category 5: Smart technology & ICT (not costed anywhere before this) ---
ICT_BACKBONE_BASE_COST_AED = 380000                  # fiber/conduit backbone + head-end/control room integration
SMART_PARK_DASHBOARD_PLATFORM_AED = 260000           # software integration, once, regardless of site size
SENSOR_UNIT_COSTS_AED = {
    "soil_moisture": 2800, "air_quality": 11000, "noise": 9500,
    "cctv_camera": 6500, "digital_kiosk": 55000, "smart_lighting_node": 3200,
}
SENSOR_TARGET_DENSITY_PER_HA = {   # nodes per hectare of site, illustrative deployment density
    "soil_moisture": 6, "air_quality": 1, "noise": 1,
    "cctv_camera": 4, "digital_kiosk": 1, "smart_lighting_node": 10,
}

# --- Category 6: Softscape & plant establishment (12-month post-planting warranty period) ---
PLANT_ESTABLISHMENT_PCT_OF_SOFTSCAPE = 0.15

# --- Category 7: Water feature / specialist systems (optional — off unless a concept proposes one) ---
WATER_FEATURE_RATE_AED_PER_SQM = 3200
DEFAULT_WATER_FEATURE_AREA_SQM = 0

# --- Category 8: Preliminaries, equipment, logistics, temporary works ---
PRELIMINARIES_PCT = 0.09    # matches the RIBA-stage-C public-realm precedent found via search

# --- Category 9: Testing, commissioning & integration ---
COMMISSIONING_PCT_OF_INFRA_AND_ICT = 0.05

# --- Category 10: Professional fees, risk & contingency ---
PROFESSIONAL_FEES_PCT = 0.10          # design + PM fees, general industry mid-range benchmark
GENERAL_CONTINGENCY_PCT = 0.075        # matches RIBA-stage-C precedent's "general design development / risk"
EXISTING_SERVICES_CONTINGENCY_PCT = 0.05  # matches RIBA-stage-C precedent's "relocation/diversion of existing services"

IMPLEMENTATION_BUDGET_AED = 35_000_000

In [ ]:
# @title Full 10-category project cost model

def estimate_full_project_cost(concept, site_area_m2=None, enable_signature_shade=False,
                                 signature_shade_area_sqm=None, enable_water_feature=False,
                                 water_feature_area_sqm=150, tree_relocation_fraction=DEFAULT_TREE_RELOCATION_FRACTION):
    '''Returns a dict with all 10 categories, each carrying its own AED value, basis/method
    string, and confidence label — so every number is traceable back to either a real
    site-specific figure (e.g. the 225 real trees from the DXF), a Dubai-market-rate-derived
    unit cost, or a labeled general industry benchmark percentage. Nothing here is a licensed
    QS estimate — see the caveat printed alongside the total.'''
    site_area_m2 = site_area_m2 or (SITE.area_m2() if globals().get("SITE") else None) or 15000.0
    site_area_ha = site_area_m2 / 10_000

    # --- Category 1: Direct landscape & park construction ---
    zone_cost = 0.0
    zoned_area = 0.0
    softscape_area = 0.0
    breakdown_c1 = {}
    for z in concept.zones:
        if z.geometry is None:
            continue
        rate = UNIT_COST_AED_PER_SQM.get(z.purpose, 250)
        cost = (z.actual_area_m2 or 0.0) * rate
        zone_cost += cost
        zoned_area += (z.actual_area_m2 or 0.0)
        breakdown_c1[z.name] = breakdown_c1.get(z.name, 0.0) + cost
        if z.purpose in ("tree_canopy_expansion", "green_space_expansion"):
            softscape_area += (z.actual_area_m2 or 0.0)
    unprogrammed_area = max(0.0, site_area_m2 - zoned_area)
    c1_total = zone_cost + unprogrammed_area * BASE_UNPROGRAMMED_RATE_AED_PER_SQM

    # --- Category 2: Existing site works — demolition, relocation, earthworks ---
    n_relocate = round(EXISTING_TREE_COUNT_AL_SAFA_2 * tree_relocation_fraction)
    n_protect = EXISTING_TREE_COUNT_AL_SAFA_2 - n_relocate
    c2_demolition = site_area_m2 * DEMOLITION_CLEARANCE_RATE_AED_PER_SQM
    c2_tree_relocation = n_relocate * MATURE_TREE_RELOCATION_COST_AED
    c2_tree_protection = n_protect * MATURE_TREE_PROTECTION_COST_AED
    c2_total = c2_demolition + c2_tree_relocation + c2_tree_protection

    # --- Category 3: Infrastructure & utilities ---
    c3_total = c1_total * INFRASTRUCTURE_PCT_OF_DIRECT_CONSTRUCTION

    # --- Category 4: Buildings & structures (signature shade, if enabled) ---
    shade_area = signature_shade_area_sqm if signature_shade_area_sqm is not None else DEFAULT_SIGNATURE_SHADE_AREA_SQM
    c4_total = (shade_area * SIGNATURE_SHADE_STRUCTURE_RATE_AED_PER_SQM) if enable_signature_shade else 0.0

    # --- Category 5: Smart technology & ICT ---
    c5_sensor_cost = 0.0
    sensor_breakdown = {}
    for sensor_type, unit_cost in SENSOR_UNIT_COSTS_AED.items():
        n_units = max(1, round(SENSOR_TARGET_DENSITY_PER_HA[sensor_type] * site_area_ha))
        cost = n_units * unit_cost
        c5_sensor_cost += cost
        sensor_breakdown[sensor_type] = {"units": n_units, "cost_aed": cost}
    c5_total = ICT_BACKBONE_BASE_COST_AED + SMART_PARK_DASHBOARD_PLATFORM_AED + c5_sensor_cost

    # --- Category 6: Softscape & plant establishment ---
    c6_total = softscape_area * PLANT_ESTABLISHMENT_PCT_OF_SOFTSCAPE * (
        UNIT_COST_AED_PER_SQM.get("tree_canopy_expansion", 130))  # 15% of the softscape's own construction value

    # --- Category 7: Water feature / specialist systems ---
    c7_total = (water_feature_area_sqm * WATER_FEATURE_RATE_AED_PER_SQM) if enable_water_feature else 0.0

    subtotal_direct_and_infra = c1_total + c2_total + c3_total + c4_total + c5_total + c6_total + c7_total

    # --- Category 8: Preliminaries, equipment, logistics, temporary works ---
    c8_total = subtotal_direct_and_infra * PRELIMINARIES_PCT

    # --- Category 9: Testing, commissioning & integration ---
    c9_total = (c3_total + c5_total) * COMMISSIONING_PCT_OF_INFRA_AND_ICT

    subtotal_before_fees_and_contingency = subtotal_direct_and_infra + c8_total + c9_total

    # --- Category 10: Professional fees, risk & contingency ---
    c10_fees = subtotal_before_fees_and_contingency * PROFESSIONAL_FEES_PCT
    c10_general_contingency = subtotal_before_fees_and_contingency * GENERAL_CONTINGENCY_PCT
    c10_existing_services_contingency = c2_total * EXISTING_SERVICES_CONTINGENCY_PCT
    c10_total = c10_fees + c10_general_contingency + c10_existing_services_contingency

    grand_total = subtotal_before_fees_and_contingency + c10_total

    return {
        "site_area_m2": site_area_m2,
        "categories": {
            "1. Direct landscape & park construction": {
                "aed": round(c1_total, 0),
                "basis": "Zone-by-zone area x Dubai-market-derived unit rate + unprogrammed base landscape rate",
                "detail": breakdown_c1,
            },
            "2. Existing site works, demolition, relocation, earthworks": {
                "aed": round(c2_total, 0),
                "basis": (f"Site-wide clearance @ AED {DEMOLITION_CLEARANCE_RATE_AED_PER_SQM}/m2 + "
                          f"{n_relocate} of {EXISTING_TREE_COUNT_AL_SAFA_2} REAL existing trees (from the as-built "
                          f"DXF) relocated @ AED {MATURE_TREE_RELOCATION_COST_AED:,}/tree, {n_protect} protected "
                          f"in place @ AED {MATURE_TREE_PROTECTION_COST_AED:,}/tree"),
                "detail": {"demolition_clearance": c2_demolition, "tree_relocation": c2_tree_relocation,
                           "tree_protection": c2_tree_protection, "trees_relocated": n_relocate, "trees_protected": n_protect},
            },
            "3. Infrastructure & utilities (irrigation, drainage, electrical)": {
                "aed": round(c3_total, 0),
                "basis": f"{INFRASTRUCTURE_PCT_OF_DIRECT_CONSTRUCTION*100:.0f}% of direct construction (Cat. 1) — "
                         f"general industry MEP allowance for landscape-heavy public realm, not Dubai-verified",
            },
            "4. Buildings & structures (signature elements)": {
                "aed": round(c4_total, 0),
                "basis": (f"{shade_area:.0f} m2 architectural shade structure @ AED "
                          f"{SIGNATURE_SHADE_STRUCTURE_RATE_AED_PER_SQM:,}/m2" if enable_signature_shade else
                          "Not enabled this run — restrooms/kiosk structures are already priced in Category 1's zone rates"),
            },
            "5. Smart technology & ICT": {
                "aed": round(c5_total, 0),
                "basis": (f"ICT backbone (AED {ICT_BACKBONE_BASE_COST_AED:,}) + dashboard platform "
                          f"(AED {SMART_PARK_DASHBOARD_PLATFORM_AED:,}) + sensor/device network sized to "
                          f"{site_area_ha:.2f} ha at illustrative deployment densities"),
                "detail": sensor_breakdown,
            },
            "6. Softscape & plant establishment (12-month warranty period)": {
                "aed": round(c6_total, 0),
                "basis": f"{PLANT_ESTABLISHMENT_PCT_OF_SOFTSCAPE*100:.0f}% of softscape construction value "
                         f"({softscape_area:.0f} m2 of canopy/green-space zones)",
            },
            "7. Water feature / specialist systems": {
                "aed": round(c7_total, 0),
                "basis": (f"{water_feature_area_sqm:.0f} m2 @ AED {WATER_FEATURE_RATE_AED_PER_SQM:,}/m2" if enable_water_feature
                          else "Not enabled this run"),
            },
            "8. Preliminaries, equipment, logistics, temporary works": {
                "aed": round(c8_total, 0),
                "basis": f"{PRELIMINARIES_PCT*100:.0f}% of Categories 1-7 subtotal — matches a public-realm "
                         f"feasibility cost-plan precedent (general industry benchmark, not Dubai-verified)",
            },
            "9. Testing, commissioning & integration": {
                "aed": round(c9_total, 0),
                "basis": f"{COMMISSIONING_PCT_OF_INFRA_AND_ICT*100:.0f}% of (Infrastructure + Smart Tech/ICT) subtotal",
            },
            "10. Professional fees, risk & contingency": {
                "aed": round(c10_total, 0),
                "basis": (f"Fees {PROFESSIONAL_FEES_PCT*100:.0f}% + general contingency "
                          f"{GENERAL_CONTINGENCY_PCT*100:.1f}% of pre-fee subtotal, + existing-services "
                          f"contingency {EXISTING_SERVICES_CONTINGENCY_PCT*100:.0f}% of Category 2 — same "
                          f"3-way split as the public-realm cost-plan precedent"),
                "detail": {"professional_fees": c10_fees, "general_contingency": c10_general_contingency,
                           "existing_services_contingency": c10_existing_services_contingency},
            },
        },
        "grand_total_aed": round(grand_total, 0),
        "budget_aed": IMPLEMENTATION_BUDGET_AED,
        "pct_of_budget": round(100 * grand_total / IMPLEMENTATION_BUDGET_AED, 1),
        "headroom_aed": round(IMPLEMENTATION_BUDGET_AED - grand_total, 0),
    }


def print_full_cost_breakdown(concept, **kwargs):
    result = estimate_full_project_cost(concept, **kwargs)
    print(f"\n=== Full Project Cost Model — {concept.name} ===")
    print(f"Site area: {result['site_area_m2']:,.0f} m2\n")
    for cat_name, cat in result["categories"].items():
        print(f"  {cat_name}")
        print(f"    AED {cat['aed']:>14,.0f}   ({cat['basis']})")
    print()
    print(f"  {'GRAND TOTAL':50s} AED {result['grand_total_aed']:>14,.0f}")
    print(f"  {'AED 35M IMPLEMENTATION BUDGET':50s} AED {result['budget_aed']:>14,.0f}")
    print(f"  -> {result['pct_of_budget']:.0f}% of budget used, "
          f"AED {result['headroom_aed']:,.0f} headroom remaining")
    print()
    print("  This is a high-level order-of-magnitude model across 10 cost categories, anchored where\n"
          "  possible to real site data (the 225 existing trees found in the as-built DXF) and Dubai\n"
          "  market rates, with international general-industry benchmark percentages (preliminaries,\n"
          "  contingency, professional fees) used where no Dubai-specific figure was available. It is\n"
          "  NOT a licensed quantity-surveyor estimate — treat it as a budget sanity-check to inform the\n"
          "  ambition-level decision, not as the number to put in front of a real cost consultant unchecked.")
    return result


# =============================================================================
# USAGE:
#   result = print_full_cost_breakdown(concept)                                    # baseline, no signature elements
#   result = print_full_cost_breakdown(concept, enable_signature_shade=True,
#                                       signature_shade_area_sqm=250,
#                                       enable_water_feature=True,
#                                       water_feature_area_sqm=150)                  # ambitious version
# =============================================================================

# Module 10 Upgrade — Schematic Sections / Elevations

Produces 2D schematic cross-sections through the concept's real zone geometry (a "long section" and a "cross section", standard landscape-architecture convention) — the remaining visual deliverable from the Scope of Work that wasn't covered by the plan-view renders or the multi-view hero shots. Explicitly schematic (illustrative silhouettes at plausible heights), matching what the Scope of Work says is expected at this design stage.

In [ ]:
# =============================================================================
# MODULE 10 UPGRADE — Schematic Sections / Elevations
# =============================================================================
# Paste this after the other Module 10 upgrade cells. Produces 2D schematic
# cross-sections (a "long section" and a "cross section", standard landscape-
# architecture convention) by cutting a line through the concept's real zone
# geometry and showing what's actually there in profile — canopy height,
# structures, seating, grade — rather than only the plan-view renders already
# built.
#
# Honesty note, consistent with everything else in this notebook: this is
# explicitly schematic, not a construction-grade section. The Scope of Work
# itself says detailed construction documentation isn't expected at this
# stage, so illustrative silhouettes (a circle for a tree canopy, a box for a
# structure) at plausible heights are the right level of fidelity here — this
# does NOT attempt real grading/topography unless a DEM is wired in below.
# =============================================================================

In [ ]:
# @title Illustrative element profile heights (schematic, not structural)
#
# height_m = approximate top height above grade; shape = how it's drawn.
# These are reasonable design-stage assumptions, not engineered dimensions —
# same "illustrative, override before construction" caveat as the cost model's
# unit rates.
SECTION_ELEMENT_PROFILES = {
    "tree_canopy_expansion":      {"height_m": 5.5, "shape": "canopy",  "trunk_m": 1.8},
    "green_space_expansion":      {"height_m": 0.3, "shape": "ground_cover"},
    "permeable_surface_conversion": {"height_m": 0.05, "shape": "ground_cover"},
    "play_area":                  {"height_m": 2.8, "shape": "structure"},
    "sports_recreation_zone":     {"height_m": 3.5, "shape": "structure"},
    "outdoor_fitness":            {"height_m": 2.2, "shape": "structure"},
    "community_gathering_plaza":  {"height_m": 4.5, "shape": "shade_sail"},
    "seating_comfort_area":       {"height_m": 0.5, "shape": "low_furniture"},
    "entrance_plaza":             {"height_m": 4.5, "shape": "shade_sail"},
    "circulation_paths":          {"height_m": 0.05, "shape": "ground_cover"},
    "jogging_track":              {"height_m": 0.05, "shape": "ground_cover"},
    "restrooms_and_services":     {"height_m": 3.6, "shape": "building"},
    "commercial_kiosk":           {"height_m": 3.2, "shape": "building"},
    "picnic_area":                {"height_m": 2.5, "shape": "shade_sail"},
    "bicycle_dropoff":            {"height_m": 1.1, "shape": "low_furniture"},
}
DEFAULT_PROFILE = {"height_m": 0.1, "shape": "ground_cover"}

In [ ]:
# @title Cut real zone geometry along a section line

def build_section_cut_lines(boundary_utm):
    '''Two standard cuts through the site: a "long section" along the longer bbox
    axis, a "cross section" along the shorter one, both through the centroid —
    the same convention any landscape architecture set uses (Section A-A / B-B).'''
    minx, miny, maxx, maxy = boundary_utm.bounds
    cx, cy = (minx + maxx) / 2, (miny + maxy) / 2
    width, height = maxx - minx, maxy - miny
    from shapely.geometry import LineString
    if width >= height:
        long_line = LineString([(minx - 5, cy), (maxx + 5, cy)])
        cross_line = LineString([(cx, miny - 5), (cx, maxy + 5)])
    else:
        long_line = LineString([(cx, miny - 5), (cx, maxy + 5)])
        cross_line = LineString([(minx - 5, cy), (maxx + 5, cy)])
    return {"Long Section A-A": long_line, "Cross Section B-B": cross_line}


def compute_section_profile(concept, cut_line):
    '''Intersects the cut line with every zone's real geometry, returning ordered
    segments of (start_distance_m, end_distance_m, purpose, name) along the line —
    the actual sequence of what the cut line passes through, in order, not a
    hand-arranged illustration.'''
    segments = []
    for z in concept.zones:
        if z.geometry is None:
            continue
        try:
            inter = cut_line.intersection(z.geometry)
        except Exception:
            continue
        if inter.is_empty:
            continue
        geoms = list(inter.geoms) if hasattr(inter, "geoms") else [inter]
        for g in geoms:
            if g.geom_type != "LineString" or g.length == 0:
                continue
            d0 = cut_line.project(Point(g.coords[0]))
            d1 = cut_line.project(Point(g.coords[-1]))
            segments.append((min(d0, d1), max(d0, d1), z.purpose, z.name))
    return sorted(segments, key=lambda s: s[0])

from shapely.geometry import Point

In [ ]:
# @title Render one schematic section

def render_section(concept, section_name, cut_line, output_dir):
    if not globals().get("_MATPLOTLIB_AVAILABLE"):
        print("matplotlib not available — skipping section rendering.")
        return None
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches

    segments = compute_section_profile(concept, cut_line)
    total_length = cut_line.length
    if not segments:
        print(f"'{section_name}' for '{concept.name}' doesn't cross any zoned area — skipping "
              f"(the cut line may fall entirely in unprogrammed lawn/buffer space).")
        return None

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.axhline(0, color="#6D4C41", linewidth=2, zorder=1)   # ground line
    ax.fill_between([0, total_length], -0.3, 0, color="#8D6E63", alpha=0.4, zorder=0)  # ground fill

    legend_seen = set()
    for start, end, purpose, name in segments:
        profile = SECTION_ELEMENT_PROFILES.get(purpose, DEFAULT_PROFILE)
        mid = (start + end) / 2
        width = max(end - start, 0.5)
        color = ZONE_PURPOSE_COLORS.get(purpose, "#607D8B")
        shape = profile["shape"]

        if shape == "canopy":
            trunk_h = profile.get("trunk_m", 1.8)
            ax.plot([mid, mid], [0, trunk_h], color="#5D4037", linewidth=3, zorder=2)
            canopy = mpatches.Ellipse((mid, trunk_h + (profile["height_m"] - trunk_h) / 2),
                                        width=max(width, 3.5), height=(profile["height_m"] - trunk_h),
                                        color=color, alpha=0.75, zorder=3)
            ax.add_patch(canopy)
        elif shape == "shade_sail":
            h = profile["height_m"]
            ax.plot([mid - width/2, mid, mid + width/2], [h*0.65, h, h*0.65], color=color, linewidth=2, zorder=3)
            ax.fill_between([mid - width/2, mid, mid + width/2], [h*0.65, h, h*0.65], 0,
                              color=color, alpha=0.25, zorder=2)
            ax.plot([mid - width/2, mid - width/2], [0, h*0.65], color="#757575", linewidth=1.5, zorder=2)
            ax.plot([mid + width/2, mid + width/2], [0, h*0.65], color="#757575", linewidth=1.5, zorder=2)
        elif shape == "building":
            rect = mpatches.Rectangle((mid - width/2, 0), width, profile["height_m"],
                                        color=color, alpha=0.8, zorder=3)
            ax.add_patch(rect)
        elif shape == "structure":
            rect = mpatches.Rectangle((mid - width/2, 0), width, profile["height_m"],
                                        color=color, alpha=0.5, zorder=3, hatch="//")
            ax.add_patch(rect)
        elif shape == "low_furniture":
            rect = mpatches.Rectangle((mid - width/2, 0), width, profile["height_m"],
                                        color=color, alpha=0.9, zorder=3)
            ax.add_patch(rect)
        else:  # ground_cover
            ax.fill_between([start, end], 0, profile["height_m"], color=color, alpha=0.5, zorder=2)

        label = ZONE_PURPOSE_LABELS.get(purpose, purpose)
        if label not in legend_seen:
            ax.plot([], [], color=color, linewidth=6, label=label)
            legend_seen.add(label)

    ax.set_xlim(-2, total_length + 2)
    ax.set_ylim(-1, 8)
    ax.set_xlabel("Distance along section (m)")
    ax.set_ylabel("Height above grade (m)")
    ax.set_title(f"{SITE.name}\n{section_name} — {concept.name} (schematic, design-stage only)", fontsize=11)
    ax.legend(loc="upper right", fontsize=7, ncol=2)
    ax.grid(axis="x", alpha=0.2)
    fig.tight_layout()

    safe_section = section_name.replace(" ", "_").replace("-", "")
    path = _os_upg.path.join(output_dir, f"{concept.id}_{safe_section}.png")
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return path


def render_all_sections(concept, output_dir):
    if SITE_UTM_CRS is None or globals().get("PARK_DESIGN_BOUNDARY") is None:
        print("No UTM CRS / design boundary available — skipping sections.")
        return {}
    boundary_gdf = gpd.GeoDataFrame({"geometry": [PARK_DESIGN_BOUNDARY]}, crs="EPSG:4326").to_crs(SITE_UTM_CRS)
    boundary_utm = boundary_gdf.geometry.iloc[0]
    cut_lines = build_section_cut_lines(boundary_utm)
    paths = {}
    for name, line in cut_lines.items():
        path = render_section(concept, name, line, output_dir)
        paths[name] = path
        print(f"  {name:20s} -> {'ok' if path else 'skipped'}")
    return paths


# =============================================================================
# USAGE:
#   for concept in DESIGN_CONCEPTS:
#       render_all_sections(concept, DESIGN_OUTPUT_DIR)
# =============================================================================

# Module 10 Upgrade — One-Minute Concept Animation

Assembles the stills already generated above (schematic plan, AI aerial impression, day/night hero shots) into a single ~60-second video using `ffmpeg` directly (no `moviepy` install needed — `ffmpeg` is available as a system binary in Colab already). Ken Burns pan/zoom per still, captions, hard-cut transitions. Every clip is optional — a missing render is skipped and the sequence is retimed to still land close to 60 seconds, rather than failing outright or padding with a blank clip.

In [ ]:
# =============================================================================
# MODULE 10 UPGRADE — One-Minute Concept Animation (ffmpeg-based, no moviepy)
# =============================================================================
# Paste this after the multi-view render pipeline cells. Assembles the stills
# already generated elsewhere (schematic plan, AI aerial impression, day/night
# hero shots) into a single ~60-second video with Ken Burns pan/zoom on each
# still, captions, and hard-cut transitions between clips — using `ffmpeg`
# directly via subprocess (available as a system binary in Colab already;
# no `moviepy` install needed, which is itself just an ffmpeg wrapper).
#
# Design intent: never fail the whole animation because one image is missing.
# Every clip is optional — if a render didn't exist (e.g. AI image-gen was
# disabled that run), its slot is skipped and the remaining clips are
# retimed so the total still lands close to the ~60s target, with the
# shortfall reported rather than silently padded with a blank clip.
# =============================================================================

In [ ]:
# @title ffmpeg clip + concat helpers

import subprocess as _subprocess_anim

def _ffmpeg_available():
    try:
        r = _subprocess_anim.run(["ffmpeg", "-version"], capture_output=True, timeout=10)
        return r.returncode == 0
    except Exception:
        return False


def _make_ken_burns_clip(image_path, output_path, duration_s, caption="",
                           resolution=(1920, 1080), fps=25, zoom_in=True):
    '''One still image -> one short video clip with a slow pan/zoom. Returns
    (success: bool, stderr_tail: str) rather than raising, so a single bad
    image degrades gracefully instead of aborting the whole animation.'''
    w, h = resolution
    n_frames = max(1, int(round(duration_s * fps)))
    zoom_expr = "min(zoom+0.0018,1.3)" if zoom_in else "if(lte(zoom,1.0),1.28,max(1.0,zoom-0.0018))"

    vf_parts = [f"scale={w*2}:{h*2}:force_original_aspect_ratio=increase,crop={w*2}:{h*2}",
                f"zoompan=z='{zoom_expr}':d={n_frames}:s={w}x{h}:fps={fps}"]
    if caption:
        safe = caption.replace("\\", "").replace(":", " -").replace("'", "")
        vf_parts.append(
            f"drawtext=text='{safe}':fontcolor=white:fontsize=40:box=1:boxcolor=black@0.55:"
            f"boxborderw=14:x=(w-text_w)/2:y=h-th-70"
        )
    vf = ",".join(vf_parts)

    cmd = ["ffmpeg", "-y", "-loop", "1", "-i", image_path, "-vf", vf,
           "-t", str(duration_s), "-r", str(fps), "-pix_fmt", "yuv420p", output_path]
    try:
        result = _subprocess_anim.run(cmd, capture_output=True, text=True, timeout=120)
        return result.returncode == 0, result.stderr[-800:] if result.returncode != 0 else ""
    except Exception as e:
        return False, str(e)


def _concat_clips(clip_paths, output_path, work_dir):
    '''Concatenates same-codec/resolution/fps clips without re-encoding (fast,
    lossless) via ffmpeg's concat demuxer.'''
    list_path = _os_upg.path.join(work_dir, "_concat_list.txt")
    with open(list_path, "w") as f:
        for p in clip_paths:
            f.write(f"file '{_os_upg.path.abspath(p)}'\n")
    cmd = ["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", list_path, "-c", "copy", output_path]
    try:
        result = _subprocess_anim.run(cmd, capture_output=True, text=True, timeout=120)
        return result.returncode == 0, result.stderr[-800:] if result.returncode != 0 else ""
    except Exception as e:
        return False, str(e)

In [ ]:
# @title Build the ~60s sequence from whatever renders actually exist this run

# (image_path_key_in_renders_dict, caption, target_duration_s, zoom_in)
ANIMATION_SEQUENCE_TEMPLATE = [
    ("_plan_png",                  "{concept_name} — Master Plan",           8,  True),
    ("_ai_aerial_png",              "Aerial Impression",                       7,  True),
    ("entrance_plaza_day",          "Entrance Plaza — Day",                    7,  True),
    ("entrance_plaza_night",        "Entrance Plaza — Evening",                6,  False),
    ("community_gathering_plaza_day",   "Community Plaza — Day",              7,  True),
    ("community_gathering_plaza_night", "Community Plaza — Evening",          6,  False),
    ("play_area_day",               "Play Area — Day",                        6,  True),
    ("play_area_night",             "Play Area — Evening",                    6,  False),
    ("jogging_track_day",           "Jogging & Walking Track — Day",          6,  True),
]
TARGET_TOTAL_DURATION_S = 60


def build_concept_animation(concept, hero_renders, plan_png_path=None, ai_aerial_path=None,
                              output_dir=None, resolution=(1920, 1080), fps=25):
    '''hero_renders: the dict returned by render_multi_view_set() — {"<purpose>_<day|night>": path}.
    Builds whichever clips have a real image behind them, retimes the sequence to land close to
    TARGET_TOTAL_DURATION_S using only the clips that exist, and reports what was skipped rather
    than padding with blank frames.'''
    if not _ffmpeg_available():
        print("ffmpeg not available on this system — cannot build the animation.")
        return None

    available = dict(hero_renders or {})
    if plan_png_path:
        available["_plan_png"] = plan_png_path
    if ai_aerial_path:
        available["_ai_aerial_png"] = ai_aerial_path

    sequence = [(key, cap.format(concept_name=concept.name), dur, zi)
                for key, cap, dur, zi in ANIMATION_SEQUENCE_TEMPLATE if available.get(key)]
    skipped = [key for key, *_ in ANIMATION_SEQUENCE_TEMPLATE if not available.get(key)]

    if not sequence:
        print(f"No renders available for '{concept.name}' — nothing to animate. Run "
              f"render_multi_view_set() and the plan/AI-aerial renders first.")
        return None

    # Retime proportionally so the sequence still lands near TARGET_TOTAL_DURATION_S even
    # when some clips are missing, rather than silently producing a shorter-than-expected video.
    template_total = sum(dur for _, _, dur, _ in sequence)
    scale = TARGET_TOTAL_DURATION_S / template_total if template_total > 0 else 1.0

    work_dir = output_dir or "."
    clip_paths = []
    failures = []
    for i, (key, caption, base_dur, zoom_in) in enumerate(sequence):
        clip_out = _os_upg.path.join(work_dir, f"_clip_{i:02d}.mp4")
        ok, err = _make_ken_burns_clip(available[key], clip_out, base_dur * scale, caption,
                                         resolution=resolution, fps=fps, zoom_in=zoom_in)
        if ok:
            clip_paths.append(clip_out)
        else:
            failures.append((key, err))

    if not clip_paths:
        print(f"All {len(sequence)} clip(s) failed to render — see failures below. Nothing produced.")
        for key, err in failures:
            print(f"  {key}: {err[:200]}")
        return None

    final_path = _os_upg.path.join(work_dir, f"{concept.id}_animation.mp4")
    ok, err = _concat_clips(clip_paths, final_path, work_dir)

    total_s = sum(base_dur * scale for _, _, base_dur, _ in sequence[:len(clip_paths)])
    print(f"Animation for '{concept.name}': {len(clip_paths)}/{len(sequence)} clip(s) succeeded, "
          f"~{total_s:.0f}s target duration.")
    if skipped:
        print(f"  Skipped (no render available): {', '.join(skipped)}")
    if failures:
        print(f"  Failed during clip rendering: {[k for k, _ in failures]}")
    if not ok:
        print(f"  Concat step failed: {err[:300]}")
        return None

    print(f"  -> {final_path}")
    return final_path


# =============================================================================
# USAGE (after Module 10's original renders + the multi-view render upgrade):
#   for concept in DESIGN_CONCEPTS:
#       renders = render_multi_view_set(concept, DESIGN_OUTPUT_DIR)
#       build_concept_animation(concept, renders,
#                                 plan_png_path=concept.png_path,
#                                 ai_aerial_path=concept.ai_render_path,
#                                 output_dir=DESIGN_OUTPUT_DIR)
# =============================================================================